# 📡 Wi-Fi Fingerprint Research — Phase 1 (Corrected)
## DATA AUDIT → TRAIN / VALIDATION / TEST → ORIGINAL WKNN

Notebook được hiệu chỉnh theo cấu trúc thực tế của `DATASET_DLULIB.zip`.

**Thiết kế dữ liệu đã xác nhận từ workbook:**
- TRAIN: dùng sheet `_2` làm dữ liệu đo lặp (20 lần đo/RP); sheet `_3` làm bảng tọa độ RP.
- TEST F1/F2: dùng sheet `_1` làm dữ liệu đo lặp (10 lần đo/RP); tọa độ ghép từ bảng RP TRAIN `_3` theo `ID_RP`.
- TEST F3: dùng sheet `_1` làm dữ liệu đo lặp; sheet này có sẵn X/Y, đồng thời đối chiếu với bảng tọa độ TRAIN `_3`.
- `-100` và `100` được xem là không xác định, chuyển thành NaN; không tự bù RSSI trong ORIGINAL.

**Phạm vi:** chỉ xây dựng ORIGINAL. Chưa sinh CGAN/KNN/AP Mean.

## 0. Cấu hình và nạp dữ liệu

In [ ]:
from pathlib import Path
import zipfile, json, shutil, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from IPython.display import display

SEED = 2026
random.seed(SEED); np.random.seed(SEED)

BASE = Path("/content")
ZIP_PATH = BASE / "DATASET_DLULIB.zip"
EXTRACT_DIR = BASE / "DATASET_DLULIB"
OUT = BASE / "WIFI_FINGERPRINT_RESEARCH_PHASE1"
for d in ["01_AUDIT","02_SPLITS","03_ORIGINAL","04_FIGURES","05_CONFIG"]:
    (OUT/d).mkdir(parents=True, exist_ok=True)

FLOORS = ["F1","F2","F3"]
AP_COLS = [f"AP{i}" for i in range(1,18)]
SENTINELS = [-100,100]
VAL_RATIO = 0.15
K_CANDIDATES = [3,5,7]
PALETTE = {"original":"#2563EB","prediction":"#14B8A6",
           "validation":"#F59E0B","warning":"#EF4444","purple":"#8B5CF6"}

if not EXTRACT_DIR.exists():
    if not ZIP_PATH.exists():
        from google.colab import files
        print("Tải lên DATASET_DLULIB.zip")
        uploaded = files.upload()
        ZIP_PATH = BASE / next(iter(uploaded))
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_DIR)

print("Dataset:", EXTRACT_DIR)

## 1. Đọc đúng sheet theo vai trò

Không dùng mặc định sheet đầu tiên cho mọi mục đích. Các sheet được chọn theo cấu trúc đã kiểm tra:
- `Train_F*_2`: fingerprint đo lặp.
- `Train_F*_3`: bảng tọa độ RP.
- `Test_F*_1`: fingerprint đo lặp độc lập.

In [ ]:
def workbook(floor, role):
    # Tìm kiếm file không phân biệt chữ hoa/thường trong thư mục giải nén
    target_name = f"DLULIB_{role}_{floor}.xlsx".lower()
    for path in EXTRACT_DIR.glob("*.xlsx"):
        if path.name.lower() == target_name:
            return path
    # Nếu không tìm thấy trực tiếp, thử tìm trong các thư mục con (nếu zip chứa thư mục cha)
    for path in EXTRACT_DIR.rglob("*.xlsx"):
        if path.name.lower() == target_name:
            return path
    return EXTRACT_DIR / f"DLULIB_{role}_{floor}.xlsx"

def read_sheet(floor, role, suffix):
    path = workbook(floor, role)
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {path} (Thư mục chứa: {[p.name for p in EXTRACT_DIR.glob('*')]})")
    sheet = f"{role}_{floor}_{suffix}"
    # Đọc danh sách sheet để tránh lỗi sai lệch chữ hoa/thường ở tên Sheet
    xl = pd.ExcelFile(path)
    target_sheet = sheet
    for s in xl.sheet_names:
        if s.strip().lower() == sheet.lower():
            target_sheet = s
            break
    df = pd.read_excel(path, sheet_name=target_sheet)
    df.columns = [str(c).strip() for c in df.columns]
    return df

raw = {}
sheet_audit = []
for f in FLOORS:
    tr = read_sheet(f,"Train","2")
    coord = read_sheet(f,"Train","3")
    te = read_sheet(f,"Test","1")
    raw[f] = {"train":tr,"coords":coord,"test":te}
    for name,df in [("TRAIN_REPEAT",tr),("TRAIN_COORDS",coord),("TEST_REPEAT",te)]:
        sheet_audit.append({
            "Floor":f,"Sheet_Role":name,"Rows":len(df),
            "Unique_RP":df["ID_RP"].nunique(),
            "Has_XY":all(c in df.columns for c in ["X","Y"]),
            "AP_Count":sum(c in df.columns for c in AP_COLS),
            "Missing_AP_Columns":", ".join(c for c in AP_COLS if c not in df.columns)
        })
sheet_audit_df = pd.DataFrame(sheet_audit)
display(sheet_audit_df)
sheet_audit_df.to_csv(OUT/"01_AUDIT"/"sheet_audit.csv",index=False)

## 2. Kiểm tra và xây dựng bảng tọa độ chuẩn

Bảng `_3` là nguồn tọa độ RP. Mỗi `ID_RP` phải ánh xạ tới đúng một cặp X/Y. Với F3, tọa độ trong TEST `_1` được đối chiếu thêm với bảng tọa độ TRAIN.

In [ ]:
coord_maps = {}
coord_audit = []
for f in FLOORS:
    c = raw[f]["coords"].copy()
    required = ["ID_RP","X","Y"]
    if not all(x in c.columns for x in required):
        raise ValueError(f"{f}: sheet tọa độ thiếu cột {required}")
    c["ID_RP"] = c["ID_RP"].astype(str).str.strip()
    c["X"] = pd.to_numeric(c["X"],errors="coerce")
    c["Y"] = pd.to_numeric(c["Y"],errors="coerce")
    if c["ID_RP"].duplicated().any():
        raise ValueError(f"{f}: ID_RP bị lặp trong bảng tọa độ.")
    if c[["X","Y"]].isna().any().any():
        raise ValueError(f"{f}: tọa độ X/Y bị thiếu.")
    coord_maps[f] = c[["ID_RP","X","Y"]].copy()

    te = raw[f]["test"].copy()
    te["ID_RP"] = te["ID_RP"].astype(str).str.strip()
    if "X" in te.columns and "Y" in te.columns:
        chk = te[["ID_RP","X","Y"]].drop_duplicates()
        chk["X"] = pd.to_numeric(chk["X"],errors="coerce")
        chk["Y"] = pd.to_numeric(chk["Y"],errors="coerce")
        merged = chk.merge(c[["ID_RP","X","Y"]],on="ID_RP",how="left",suffixes=("_test","_train"),indicator=True)
        matched = merged["_merge"].eq("both")
        coord_equal = (
            np.isclose(merged.loc[matched,"X_test"],merged.loc[matched,"X_train"]).all()
            and np.isclose(merged.loc[matched,"Y_test"],merged.loc[matched,"Y_train"]).all()
        )
        coord_audit.append({"Floor":f,"Test_Has_XY":True,"Test_RP":te.ID_RP.nunique(),
                            "Matched_RP":int(matched.sum()),"Coordinate_Match":bool(coord_equal)})
    else:
        missing_ids = sorted(set(te.ID_RP)-set(c.ID_RP))
        coord_audit.append({"Floor":f,"Test_Has_XY":False,"Test_RP":te.ID_RP.nunique(),
                            "Matched_RP":te.ID_RP.nunique()-len(missing_ids),
                            "Coordinate_Match":len(missing_ids)==0})
        if missing_ids:
            raise ValueError(f"{f}: TEST có ID_RP không có trong bảng tọa độ: {missing_ids[:10]}")

coord_audit_df = pd.DataFrame(coord_audit)
display(coord_audit_df)
coord_audit_df.to_csv(OUT/"01_AUDIT"/"coordinate_join_audit.csv",index=False)

## 3. Chuẩn hóa fingerprint và xử lý sentinel

`-100` và `100` được chuyển thành NaN. Không điền RSSI thiếu trong dữ liệu ORIGINAL. Giữ mask AP hợp lệ để WKNN chỉ so sánh trên AP quan sát được chung.

In [ ]:
def prepare_measurements(df, floor, role):
    d = df.copy()
    d["ID_RP"] = d["ID_RP"].astype(str).str.strip()
    missing = [c for c in AP_COLS if c not in d.columns]
    if missing:
        raise ValueError(f"{floor}/{role}: thiếu AP columns {missing}")
    for ap in AP_COLS:
        d[ap] = pd.to_numeric(d[ap],errors="coerce")
        d.loc[d[ap].isin(SENTINELS),ap] = np.nan
    d["Floor"] = floor
    return d

def aggregate_train_rp(df, coord_map):
    # RSSI trung bình theo RP trên các lần đo TRAIN; tọa độ lấy từ bảng RP chuẩn.
    agg = df.groupby("ID_RP",as_index=False)[AP_COLS].mean()
    agg = agg.merge(coord_map,on="ID_RP",how="left",validate="one_to_one")
    if agg[["X","Y"]].isna().any().any():
        raise ValueError("Không ghép được tọa độ cho một số RP TRAIN.")
    agg["AP_VALID_COUNT"] = agg[AP_COLS].notna().sum(axis=1)
    return agg

data = {}
quality_rows = []
for f in FLOORS:
    tr = prepare_measurements(raw[f]["train"],f,"TRAIN")
    te = prepare_measurements(raw[f]["test"],f,"TEST")
    coord_map = coord_maps[f]
    train_rp = aggregate_train_rp(tr,coord_map)

    # Loại bỏ X, Y khỏi test trước khi merge để tránh tạo ra X_x, Y_x, X_y, Y_y trên F3
    te_clean = te.drop(columns=["X","Y"], errors="ignore")

    # Gắn tọa độ chuẩn vào từng lần đo TEST theo ID_RP.
    test_q = te_clean.merge(coord_map,on="ID_RP",how="left",validate="many_to_one")
    if test_q[["X","Y"]].isna().any().any():
        raise ValueError(f"{f}: TEST còn thiếu tọa độ sau JOIN.")
    test_q["AP_VALID_COUNT"] = test_q[AP_COLS].notna().sum(axis=1)

    data[f] = {"train_raw":tr,"train_rp":train_rp,"test_queries":test_q}
    for role,frame in [("TRAIN_REPEAT",tr),("TRAIN_RP_AGG",train_rp),("TEST_REPEAT",test_q)]:
        quality_rows.append({
            "Floor":f,"Dataset":role,"Rows":len(frame),
            "Unique_RP":frame.ID_RP.nunique(),
            "Missing_RSSI_Rate":frame[AP_COLS].isna().mean().mean(),
            "Mean_Observed_AP_per_Row":frame[AP_COLS].notna().sum(axis=1).mean(),
            "Coordinate_Missing":int(frame[["X","Y"]].isna().any(axis=1).sum())
        })

quality_df = pd.DataFrame(quality_rows)
display(quality_df.round(4))
quality_df.to_csv(OUT/"01_AUDIT"/"data_quality.csv",index=False)

## 4. TRAIN / VALIDATION theo RP; TEST là lần đo độc lập

- TRAIN reference: RP thuộc phần fit.
- VALIDATION: RP giữ lại từ TRAIN source, dùng để chọn k.
- TEST: các lần đo trong workbook TEST, với tọa độ ghép từ bảng RP nguồn.
- Vì TEST được đo tại RP đã có trong bảng tọa độ/TRAIN, kết quả thể hiện định vị trên các vị trí khảo sát đã biết với lần đo RSSI độc lập; không phải đánh giá trên RP hoàn toàn mới.

Mỗi RP được chia nguyên khối vào TRAIN hoặc VALIDATION, tránh các lần đo cùng RP xuất hiện ở cả hai tập.

In [ ]:
splits = {}
split_rows = []
for f in FLOORS:
    rp = data[f]["train_rp"].copy()
    ids = rp.ID_RP.to_numpy()
    train_ids, val_ids = train_test_split(ids,test_size=VAL_RATIO,random_state=SEED)
    fit = rp[rp.ID_RP.isin(train_ids)].reset_index(drop=True)
    val = rp[rp.ID_RP.isin(val_ids)].reset_index(drop=True)
    test = data[f]["test_queries"].reset_index(drop=True)
    splits[f] = {"TRAIN":fit,"VALIDATION":val,"TEST":test}
    split_rows.append({
        "Floor":f,"TRAIN_RP":len(fit),"VALIDATION_RP":len(val),
        "TEST_RP":test.ID_RP.nunique(),"TEST_Measurement_Rows":len(test),
        "Train_Val_RP_Overlap":len(set(fit.ID_RP)&set(val.ID_RP)),
        "Train_Test_RP_Overlap":len(set(fit.ID_RP)&set(test.ID_RP)),
        "Protocol":"Known surveyed RP; independent TEST measurements"
    })
split_df = pd.DataFrame(split_rows)
display(split_df)
split_df.to_csv(OUT/"02_SPLITS"/"split_audit.csv",index=False)

## 5. WKNN – chuẩn hóa có mask và tính khoảng cách

Scaler được fit trên TRAIN reference. Giá trị thiếu được điền median TRAIN chỉ trong ma trận đầu vào scaler; mask gốc vẫn được giữ để không tính AP thiếu như quan sát thật. Khoảng cách chỉ tính trên AP chung và yêu cầu tối thiểu 2 AP.

In [ ]:
def fit_scaler(train):
    x = train[AP_COLS].to_numpy(float)
    med = np.nanmedian(x,axis=0)
    if np.isnan(med).any():
        raise ValueError("Có AP thiếu hoàn toàn trong TRAIN; cần xem xét schema/loại AP.")
    filled = np.where(np.isfinite(x),x,med)
    scaler = StandardScaler().fit(filled)
    return scaler,med

def encode(df,scaler,med):
    x = df[AP_COLS].to_numpy(float)
    mask = np.isfinite(x)
    filled = np.where(mask,x,med)
    return scaler.transform(filled),mask

def wknn(reference,queries,scaler,med,k,min_common_aps=2):
    rz,rm = encode(reference,scaler,med)
    qz,qm = encode(queries,scaler,med)
    rxy = reference[["X","Y"]].to_numpy(float)
    pred = np.full((len(queries),2),np.nan)
    fallback = np.zeros(len(queries),bool)
    used = np.zeros(len(queries),int)
    for i in range(len(queries)):
        common = rm & qm[i][None,:]
        n_common = common.sum(axis=1)
        eligible = n_common >= min_common_aps
        if not eligible.any():
            fallback[i] = True
            continue
        delta = (rz-qz[i][None,:])*common
        dist = np.sqrt((delta**2).sum(axis=1)/np.maximum(n_common,1))
        dist[~eligible] = np.inf
        idx = np.argsort(dist)[:min(k,int(np.isfinite(dist).sum()))]
        idx = idx[np.isfinite(dist[idx])]
        if len(idx)==0:
            fallback[i] = True
            continue
        d = dist[idx]
        zero = d <= 1e-12
        if zero.any():
            idx = idx[zero]
            weights = np.ones(len(idx))
        else:
            weights = 1/np.maximum(d,1e-12)
        pred[i] = np.average(rxy[idx],axis=0,weights=weights)
        used[i] = len(idx)
    return pred,fallback,used

def metrics(y,p):
    ok = np.isfinite(y).all(axis=1)&np.isfinite(p).all(axis=1)
    if not ok.any():
        return {"N_Query":len(y),"N_Valid":0,"Mean_Error":np.nan,
                "Median_Error":np.nan,"RMSE":np.nan,"P90":np.nan}
    e = np.linalg.norm(y[ok]-p[ok],axis=1)
    return {"N_Query":len(y),"N_Valid":int(ok.sum()),
            "Mean_Error":e.mean(),"Median_Error":np.median(e),
            "RMSE":np.sqrt(np.mean(e**2)),"P90":np.percentile(e,90)}

## 6. Chọn k trên VALIDATION và đánh giá ORIGINAL trên TEST

In [ ]:
val_rows, test_rows = [], []
predictions = {}
for f in FLOORS:
    tr,va,te = (splits[f][x] for x in ["TRAIN","VALIDATION","TEST"])
    scaler,med = fit_scaler(tr)
    val_y = va[["X","Y"]].to_numpy(float)
    for k in K_CANDIDATES:
        vp,vfb,vused = wknn(tr,va,scaler,med,k)
        val_rows.append({"Floor":f,"Method":"ORIGINAL","k":k,
                         **metrics(val_y,vp),"Fallback_Count":int(vfb.sum())})
    vf = pd.DataFrame([r for r in val_rows if r["Floor"]==f])
    best_k = int(vf.sort_values(["Mean_Error","RMSE"]).iloc[0]["k"])

    test_y = te[["X","Y"]].to_numpy(float)
    tp,tfb,tused = wknn(tr,te,scaler,med,best_k)
    test_rows.append({"Floor":f,"Method":"ORIGINAL","Selected_k":best_k,
                      **metrics(test_y,tp),"Fallback_Count":int(tfb.sum())})
    out = te[["Floor","ID_RP","X","Y"]].copy()
    out["Pred_X"],out["Pred_Y"] = tp[:,0],tp[:,1]
    out["Fallback"],out["Neighbors_Used"] = tfb,tused
    out["Error"] = np.linalg.norm(out[["X","Y"]].to_numpy()-out[["Pred_X","Pred_Y"]].to_numpy(),axis=1)
    predictions[f] = out
    out.to_csv(OUT/"03_ORIGINAL"/f"original_predictions_{f}.csv",index=False)

val_df = pd.DataFrame(val_rows)
test_df = pd.DataFrame(test_rows)
display(val_df.round(4))
display(test_df.round(4))
val_df.to_csv(OUT/"03_ORIGINAL"/"validation_k_selection.csv",index=False)
test_df.to_csv(OUT/"03_ORIGINAL"/"original_test_metrics.csv",index=False)

## 7. Dashboard ORIGINAL – bảng số liệu và biểu đồ

In [ ]:
display(test_df.style.format({
    "Mean_Error":"{:.4f}","Median_Error":"{:.4f}","RMSE":"{:.4f}","P90":"{:.4f}"
}).background_gradient(subset=["Mean_Error","RMSE","P90"],cmap="YlGnBu_r"))

fig,axs = plt.subplots(1,2,figsize=(15,5),constrained_layout=True)
xx=np.arange(len(test_df))
axs[0].bar(xx,test_df.Mean_Error,color="#2563EB",width=.62)
axs[0].set_xticks(xx,test_df.Floor); axs[0].set_title("ORIGINAL — Mean Localization Error")
axs[0].set_ylabel("Source coordinate units"); axs[0].grid(axis="y",alpha=.2)
for i,v in enumerate(test_df.Mean_Error): axs[0].text(i,v,f"{v:.3f}",ha="center",va="bottom")
axs[1].bar(xx-.18,test_df.RMSE,width=.36,color="#8B5CF6",label="RMSE")
axs[1].bar(xx+.18,test_df.P90,width=.36,color="#F59E0B",label="P90")
axs[1].set_xticks(xx,test_df.Floor); axs[1].set_title("ORIGINAL — RMSE / P90")
axs[1].set_ylabel("Source coordinate units"); axs[1].legend(frameon=False); axs[1].grid(axis="y",alpha=.2)
plt.show()
fig.savefig(OUT/"04_FIGURES"/"original_metrics.png",dpi=180,bbox_inches="tight")

## 8. Bản đồ định vị 2D theo tầng – Ground Truth vs WKNN

In [ ]:
for f in FLOORS:
    d=predictions[f]
    fig,ax=plt.subplots(figsize=(8,7),constrained_layout=True)
    ax.scatter(d.X,d.Y,s=26,c="#2563EB",alpha=.65,label="Ground truth")
    ok=d[["Pred_X","Pred_Y"]].notna().all(axis=1)
    ax.scatter(d.loc[ok,"Pred_X"],d.loc[ok,"Pred_Y"],s=25,c="#14B8A6",
               marker="D",alpha=.7,label="WKNN prediction")
    # Vẽ vector sai số cho một phần mẫu để tránh rối hình.
    sample=d.loc[ok].iloc[::max(1,len(d.loc[ok])//150)]
    for _,r in sample.iterrows():
        ax.plot([r.X,r.Pred_X],[r.Y,r.Pred_Y],color="#64748B",alpha=.18,lw=.7)
    ax.set_title(f"{f} — ORIGINAL WKNN localization")
    ax.set_xlabel("X (source coordinate)"); ax.set_ylabel("Y (source coordinate)")
    ax.grid(alpha=.2); ax.legend(frameon=False); plt.show()
    fig.savefig(OUT/"04_FIGURES"/f"original_map_{f}.png",dpi=180,bbox_inches="tight")

## 9. Readiness report và lưu cấu hình

Chỉ chuyển sang NEW TRAIN khi:
- Đã xác minh sheet được sử dụng đúng.
- Ghép tọa độ TEST thành công.
- Không có dự đoán TEST thiếu ngoài các trường hợp fallback được báo cáo.
- Giao thức được mô tả đúng: vị trí RP đã khảo sát, lần đo TEST độc lập.

In [ ]:
readiness=[]
for f in FLOORS:
    p=predictions[f]
    readiness.append({
        "Floor":f,"Test_Queries":len(p),"Unique_Test_RP":p.ID_RP.nunique(),
        "Missing_Test_XY":int(p[["X","Y"]].isna().any(axis=1).sum()),
        "Missing_Prediction":int(p[["Pred_X","Pred_Y"]].isna().any(axis=1).sum()),
        "Fallback_Count":int(p.Fallback.sum()),
        "Mean_Error":p.Error.mean(),"RMSE":np.sqrt(np.nanmean(p.Error**2))
    })
readiness_df=pd.DataFrame(readiness)
display(readiness_df.round(4))
readiness_df.to_csv(OUT/"05_CONFIG"/"readiness_report.csv",index=False)

config={
    "seed":SEED,"floors":FLOORS,"AP_COLS":AP_COLS,"sentinels":SENTINELS,
    "train_measurement_sheet":"_2","coordinate_sheet":"_3",
    "test_measurement_sheet":"_1","validation_split":"RP-level from TRAIN RP table",
    "test_protocol":"independent measurement repeats at surveyed RP",
    "k_candidates":K_CANDIDATES,
    "coordinate_unit":"source coordinate units; not assumed meters",
    "stage":"TRAIN + ORIGINAL"
}
with open(OUT/"05_CONFIG"/"experiment_config.json","w",encoding="utf-8") as fp:
    json.dump(config,fp,ensure_ascii=False,indent=2)

zip_out=BASE/"WIFI_FINGERPRINT_PHASE1_CORRECTED_RESULTS.zip"
if zip_out.exists(): zip_out.unlink()
shutil.make_archive(str(zip_out.with_suffix("")),"zip",OUT)
print("Phase 1 complete:",OUT)
print("Results archive:",zip_out)

## 06. ORIGINAL — SEEN RP vs HELD-OUT RP AUDIT

### 1. Mục tiêu thực hiện

Sau khi hoàn thành đánh giá mô hình WKNN trên tập dữ liệu TEST, bước này phân tích sâu hơn hiệu quả định vị của mô hình ORIGINAL bằng cách chia các điểm tham chiếu (Reference Points – RP) thành hai nhóm:

- **Seen RP:** Các RP xuất hiện trong tập TRAIN và được sử dụng làm cơ sở dữ liệu tham chiếu cho WKNN.
- **Held-out RP:** Các RP thuộc tập VALIDATION, không xuất hiện trong tập TRAIN và không được sử dụng làm cơ sở dữ liệu tham chiếu cho WKNN.

Mục tiêu là xác định mô hình ORIGINAL hoạt động như thế nào khi định vị tại các vị trí đã có fingerprint tham chiếu và các vị trí bị loại khỏi tập tham chiếu.

### 2. Dữ liệu đầu vào

Cell sử dụng các biến đã được tạo trong giai đoạn Phase 1:

| Biến | Vai trò |
|---|---|
| `FLOORS` | Danh sách các tầng được đánh giá |
| `splits` | Thông tin phân chia TRAIN và VALIDATION theo RP |
| `predictions` | Kết quả dự đoán WKNN trên TEST |
| `OUT` | Thư mục lưu kết quả thực nghiệm |

**Lưu ý:** Cell này chỉ phân tích kết quả đã có, không huấn luyện lại mô hình, không tạo dữ liệu tổng hợp và không thay đổi tập TRAIN, VALIDATION hoặc TEST.

### 3. Phương pháp phân tích

Mỗi kết quả dự đoán trên TEST được phân loại theo RP:

- Nếu `ID_RP` thuộc tập TRAIN → Seen RP.
- Nếu `ID_RP` không thuộc tập TRAIN → Held-out RP.

Sau đó, các chỉ số được tính riêng cho từng nhóm trên từng tầng.

| Chỉ số | Ý nghĩa |
|---|---|
| Unique RP | Số RP khác nhau trong nhóm |
| N_Query | Tổng số truy vấn định vị |
| N_Valid | Số truy vấn có kết quả dự đoán hợp lệ |
| Coverage (%) | Tỷ lệ truy vấn có kết quả hợp lệ |
| Fallback | Số truy vấn phải sử dụng cơ chế dự phòng |
| Mean Error | Sai số định vị trung bình |
| Median Error | Sai số định vị trung vị |
| RMSE | Căn bậc hai sai số bình phương trung bình |
| P90 | Ngưỡng sai số mà 90% truy vấn hợp lệ không vượt quá |

### 4. Kết quả trực quan hóa

Cell tạo ra ba nhóm kết quả:

1. **Bảng Seen RP vs Held-out RP:** Tổng hợp các chỉ số định vị theo từng tầng và nhóm RP.
2. **Biểu đồ Mean Localization Error:** So sánh sai số định vị trung bình giữa hai nhóm.
3. **Biểu đồ Prediction Coverage:** So sánh tỷ lệ truy vấn có dự đoán hợp lệ giữa hai nhóm.

Các biểu đồ sử dụng màu xanh dương cho Seen RP và màu cam cho Held-out RP để dễ phân biệt.

### 5. Ý nghĩa nghiên cứu

Kết quả phân tích giúp xác định liệu hiệu quả của WKNN có phụ thuộc đáng kể vào việc RP đã xuất hiện trong tập tham chiếu hay chưa.

- Nếu Seen RP có sai số thấp hơn Held-out RP, mô hình có thể đang hưởng lợi từ việc có fingerprint tham chiếu tại chính vị trí đó.
- Nếu Held-out RP có sai số cao hơn đáng kể, cần xem xét khả năng dữ liệu tham chiếu chưa bao phủ tốt không gian định vị.
- Nếu Coverage thấp hoặc số lượng Fallback lớn, cần đánh giá thêm khả năng xử lý các trường hợp thiếu AP quan sát chung.

**Lưu ý về phạm vi đánh giá:** TEST là các phép đo RSSI độc lập tại những RP đã được khảo sát trước đó. Vì vậy, Held-out RP ở đây có nghĩa là RP bị loại khỏi tập tham chiếu TRAIN, không đồng nghĩa với việc thử nghiệm tại các vị trí hoàn toàn mới chưa từng được khảo sát.

### 6. Tiêu chí kiểm tra trước khi chuyển sang NEW TRAIN

- [ ] Đã phân loại đầy đủ Seen RP và Held-out RP.
- [ ] Đã kiểm tra số lượng truy vấn và RP của từng nhóm.
- [ ] Đã đánh giá Mean Error, RMSE và P90.
- [ ] Đã kiểm tra Coverage và Fallback.
- [ ] Đã lưu bảng kết quả và dữ liệu dự đoán đã phân nhóm.

**Đầu ra:** Bộ kết quả kiểm toán ORIGINAL theo Seen RP và Held-out RP, làm cơ sở đánh giá độ tin cậy của baseline trước khi triển khai các phương pháp tăng cường dữ liệu trong NEW TRAIN.

In [ ]:
# ============================================================
# PHASE 1 — ORIGINAL AUDIT
# SO SÁNH SEEN RP vs HELD-OUT RP
# Không train lại, không thay đổi dữ liệu
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Kiểm tra biến từ notebook chính
required = ["FLOORS", "splits", "predictions", "OUT"]
missing = [v for v in required if v not in globals()]

if missing:
    raise RuntimeError(
        f"Thiếu biến: {missing}. "
        "Hãy chạy các cell Phase 1 trước cell audit này."
    )

# ------------------------------------------------------------
# 1. Hàm tính các chỉ số đánh giá
# ------------------------------------------------------------

def calculate_group_metrics(df):

    n_query = len(df)

    valid = (
        df[["X", "Y", "Pred_X", "Pred_Y"]]
        .replace([np.inf, -np.inf], np.nan)
        .notna()
        .all(axis=1)
    )

    errors = df.loc[valid, "Error"].astype(float)

    n_valid = int(valid.sum())
    fallback = int(
        df["Fallback"].fillna(True).astype(bool).sum()
    )

    return {
        "N_Query": n_query,
        "N_Valid": n_valid,
        "Coverage (%)": n_valid / n_query * 100 if n_query else np.nan,
        "Fallback": fallback,
        "Mean Error": errors.mean() if n_valid else np.nan,
        "Median Error": errors.median() if n_valid else np.nan,
        "RMSE": np.sqrt(np.mean(errors**2)) if n_valid else np.nan,
        "P90": errors.quantile(0.90) if n_valid else np.nan
    }


# ------------------------------------------------------------
# 2. Phân loại Seen RP và Held-out RP
# ------------------------------------------------------------

audit_results = []
classified_predictions = {}

for floor in FLOORS:

    df = predictions[floor].copy()

    train_ids = set(
        splits[floor]["TRAIN"]["ID_RP"].astype(str).str.strip()
    )

    df["_RP_KEY"] = df["ID_RP"].astype(str).str.strip()

    df["RP_Group"] = np.where(
        df["_RP_KEY"].isin(train_ids),
        "Seen RP",
        "Held-out RP"
    )

    classified_predictions[floor] = df.copy()

    for group_name, group in df.groupby("RP_Group"):

        metrics = calculate_group_metrics(group)

        audit_results.append({
            "Floor": floor,
            "RP Group": group_name,
            "Unique RP": group["_RP_KEY"].nunique(),
            **metrics
        })


# ------------------------------------------------------------
# 3. Bảng kết quả tổng hợp
# ------------------------------------------------------------

seen_heldout_df = pd.DataFrame(audit_results)

print("=" * 85)
print("ORIGINAL — SEEN RP vs HELD-OUT RP")
print("=" * 85)

display(
    seen_heldout_df.style
    .format({
        "Coverage (%)": "{:.2f}%",
        "Mean Error": "{:.4f}",
        "Median Error": "{:.4f}",
        "RMSE": "{:.4f}",
        "P90": "{:.4f}"
    })
    .background_gradient(
        subset=["Mean Error", "RMSE", "P90"],
        cmap="YlOrRd"
    )
)


# ------------------------------------------------------------
# 4. Biểu đồ Mean Error
# ------------------------------------------------------------

floors = list(FLOORS)
groups = ["Seen RP", "Held-out RP"]

colors = {
    "Seen RP": "#3B82F6",
    "Held-out RP": "#F59E0B"
}

fig, axes = plt.subplots(
    1, len(floors),
    figsize=(17, 5),
    constrained_layout=True
)

if len(floors) == 1:
    axes = [axes]

for ax, floor in zip(axes, floors):

    sub = seen_heldout_df[
        seen_heldout_df["Floor"] == floor
    ]

    values = []
    labels = []
    bar_colors = []

    for group_name in groups:

        row = sub[sub["RP Group"] == group_name]

        if not row.empty:

            value = row.iloc[0]["Mean Error"]

            if pd.notna(value):
                values.append(value)
                labels.append(group_name)
                bar_colors.append(colors[group_name])

    bars = ax.bar(
        labels,
        values,
        color=bar_colors,
        width=0.55
    )

    ax.set_title(f"{floor} — Mean Localization Error",
                 fontsize=12, fontweight="bold")

    ax.set_ylabel("Coordinate units")
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)

    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width()/2,
            value,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold"
        )

plt.suptitle(
    "ORIGINAL — Seen RP vs Held-out RP",
    fontsize=15,
    fontweight="bold"
)

plt.show()


# ------------------------------------------------------------
# 5. Biểu đồ Coverage
# ------------------------------------------------------------

plot_df = seen_heldout_df.copy()

plot_df["Label"] = (
    plot_df["Floor"] + " — " + plot_df["RP Group"]
)

bar_colors = [
    colors[group]
    for group in plot_df["RP Group"]
]

fig, ax = plt.subplots(figsize=(12, 5))

bars = ax.bar(
    plot_df["Label"],
    plot_df["Coverage (%)"],
    color=bar_colors,
    width=0.65
)

ax.set_title(
    "Prediction Coverage — Seen RP vs Held-out RP",
    fontsize=14,
    fontweight="bold"
)

ax.set_ylabel("Valid prediction coverage (%)")
ax.set_ylim(0, 105)
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
plt.xticks(rotation=20, ha="right")

for bar, value in zip(bars, plot_df["Coverage (%)"]):

    ax.text(
        bar.get_x() + bar.get_width()/2,
        value + 1,
        f"{value:.2f}%",
        ha="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 6. Xuất kết quả
# ------------------------------------------------------------

import os

audit_dir = os.path.join(
    OUT,
    "06_SEEN_HELDOUT_AUDIT"
)

os.makedirs(audit_dir, exist_ok=True)

seen_heldout_df.to_csv(
    os.path.join(
        audit_dir,
        "seen_vs_heldout_metrics.csv"
    ),
    index=False
)

for floor, df in classified_predictions.items():

    df.to_csv(
        os.path.join(
            audit_dir,
            f"predictions_grouped_{floor}.csv"
        ),
        index=False
    )

print("\nĐÃ HOÀN THÀNH SEEN RP vs HELD-OUT RP AUDIT")
print("Kết quả lưu tại:", audit_dir)

## NEW TRAIN — CELL 1: INPUT DATA & EXPERIMENT READINESS CHECK

### 1. Mục tiêu

Kiểm tra trạng thái dữ liệu và các biến được kế thừa từ Phase 1 trước khi bắt đầu tăng cường dữ liệu Wi-Fi Fingerprinting.

### 2. Dữ liệu sử dụng

- TRAIN: fingerprint gốc đã được tổng hợp theo RP và phân chia theo RP.
- VALIDATION: các RP được giữ lại từ tập TRAIN, không tham gia huấn luyện.
- TEST: các phép đo độc lập tại các RP đã khảo sát, được giữ nguyên cho đánh giá cuối cùng.

### 3. Các điều kiện bắt buộc

- Đủ ba tầng F1, F2 và F3.
- Có đầy đủ 17 AP từ AP1 đến AP17.
- Có tọa độ X/Y hợp lệ.
- Không có RP giao nhau giữa TRAIN và VALIDATION.
- Không sử dụng dữ liệu TEST trong quá trình tạo dữ liệu tổng hợp.

### 4. Đầu ra

Bảng kiểm tra tính sẵn sàng của dữ liệu theo từng tầng. Nếu có điều kiện không đạt, chương trình sẽ dừng để tránh thực hiện thí nghiệm trên dữ liệu sai.

In [ ]:
# ============================================================
# NEW TRAIN — CELL 1
# INPUT DATA & EXPERIMENT READINESS CHECK
# ============================================================

import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

# Kiểm tra biến kế thừa từ Phase 1
required_vars = [
    "FLOORS",
    "AP_COLS",
    "splits",
    "data",
    "OUT",
    "wknn",
    "fit_scaler",
    "metrics"
]

missing_vars = [
    v for v in required_vars
    if v not in globals()
]

if missing_vars:
    raise RuntimeError(
        f"Thiếu biến Phase 1: {missing_vars}. "
        "Hãy chạy hoàn chỉnh Phase 1 trước."
    )

# Kiểm tra danh sách tầng
assert list(FLOORS) == ["F1", "F2", "F3"], (
    f"Danh sách tầng không đúng: {FLOORS}"
)

# Kiểm tra AP
assert len(AP_COLS) == 17, (
    f"Số AP không đúng: {len(AP_COLS)}"
)

# Thư mục kết quả NEW TRAIN
NEW_TRAIN_DIR = Path(OUT) / "07_NEW_TRAIN"

for folder in [
    "01_INPUT_AUDIT",
    "02_CGAN",
    "03_SYNTHETIC",
    "04_QUALITY_AUDIT",
    "05_METHOD_COMPARISON",
    "06_FINAL_DATA",
    "07_FIGURES",
    "08_CONFIG"
]:
    (NEW_TRAIN_DIR / folder).mkdir(
        parents=True,
        exist_ok=True
    )

# Cấu hình seed
NEW_SEED = 20261104

random.seed(NEW_SEED)
np.random.seed(NEW_SEED)

# Bảng kiểm tra
audit_rows = []

for floor in FLOORS:

    tr = splits[floor]["TRAIN"].copy()
    va = splits[floor]["VALIDATION"].copy()
    te = splits[floor]["TEST"].copy()

    train_ids = set(tr["ID_RP"].astype(str))
    val_ids = set(va["ID_RP"].astype(str))

    overlap = train_ids & val_ids

    train_ap = tr[AP_COLS].to_numpy(float)
    val_ap = va[AP_COLS].to_numpy(float)

    audit_rows.append({
        "Floor": floor,
        "TRAIN_RP": tr["ID_RP"].nunique(),
        "VALIDATION_RP": va["ID_RP"].nunique(),
        "TEST_RP": te["ID_RP"].nunique(),
        "TRAIN_Rows": len(tr),
        "VALIDATION_Rows": len(va),
        "TEST_Queries": len(te),
        "TRAIN_VAL_Overlap": len(overlap),
        "TRAIN_Missing_RSSI_%": (
            100 * np.isnan(train_ap).mean()
        ),
        "VALIDATION_Missing_RSSI_%": (
            100 * np.isnan(val_ap).mean()
        ),
        "TEST_Used_For_Training": False,
        "Status": (
            "PASS"
            if len(overlap) == 0
            and len(tr) > 0
            and len(va) > 0
            and len(te) > 0
            else "CHECK"
        )
    })

    # Kiểm tra cột dữ liệu
    for role, frame in [
        ("TRAIN", tr),
        ("VALIDATION", va),
        ("TEST", te)
    ]:

        required_cols = [
            "ID_RP", "X", "Y", *AP_COLS
        ]

        missing_cols = [
            c for c in required_cols
            if c not in frame.columns
        ]

        if missing_cols:
            raise ValueError(
                f"{floor}/{role}: thiếu cột {missing_cols}"
            )

        if frame[["X", "Y"]].isna().any().any():
            raise ValueError(
                f"{floor}/{role}: tọa độ X/Y bị thiếu."
            )

    if overlap:
        raise ValueError(
            f"{floor}: TRAIN và VALIDATION bị trùng RP: "
            f"{sorted(overlap)[:10]}"
        )

audit_df = pd.DataFrame(audit_rows)

display(
    audit_df.style
    .format({
        "TRAIN_Missing_RSSI_%": "{:.2f}%",
        "VALIDATION_Missing_RSSI_%": "{:.2f}%"
    })
    .background_gradient(
        subset=[
            "TRAIN_RP",
            "VALIDATION_RP",
            "TEST_Queries"
        ],
        cmap="Blues"
    )
)

audit_df.to_csv(
    NEW_TRAIN_DIR / "01_INPUT_AUDIT" /
    "new_train_input_audit.csv",
    index=False
)

print("NEW TRAIN INPUT AUDIT COMPLETED")
print("Output:", NEW_TRAIN_DIR)

## NEW TRAIN — CELL 2: EXPERIMENT CONFIGURATION

### 1. Mục tiêu

Thiết lập các tham số dùng chung cho toàn bộ giai đoạn tăng cường dữ liệu, đảm bảo các tầng và phương pháp được triển khai theo cùng một quy ước.

### 2. Các phương pháp

Bảy phương pháp tăng cường dữ liệu được triển khai:

- CGAN: mô hình sinh vector RSSI có điều kiện theo tọa độ.
- KNN_K3, KNN_K5, KNN_K7: phương pháp nội suy fingerprint dựa trên 3, 5 và 7 RP lân cận.
- APMEAN_R3, APMEAN_R5, APMEAN_R7: phương pháp trung bình RSSI cục bộ trong các vùng lân cận có bán kính tương ứng.

ORIGINAL là dữ liệu đối chứng, không phải phương pháp tăng cường dữ liệu.

### 3. Quy tắc dữ liệu

- RSSI `-100` và `100` được xem là sentinel biểu thị không quan sát.
- Các giá trị sentinel được chuyển thành NaN.
- Không tự động biến RSSI bị thiếu thành phép đo thật.
- Tọa độ được giữ theo đơn vị gốc của dataset, chưa giả định là mét.

### 4. Đầu ra

Một file cấu hình JSON để lưu lại seed, danh sách phương pháp, số AP và các tham số huấn luyện, phục vụ tái lập thí nghiệm.

In [ ]:
# ============================================================
# NEW TRAIN — CELL 2
# EXPERIMENT CONFIGURATION
# ============================================================

METHODS = [
    "ORIGINAL",
    "CGAN",
    "KNN_K3",
    "KNN_K5",
    "KNN_K7",
    "APMEAN_R3",
    "APMEAN_R5",
    "APMEAN_R7"
]

AUGMENTATION_METHODS = METHODS[1:]

# KNN neighborhood sizes
KNN_K_VALUES = [3, 5, 7]

# AP Mean coordinate radii
APMEAN_RADII = [3.0, 5.0, 7.0]

# Synthetic sample target
# Tạo số candidate tương đương số RP TRAIN ban đầu,
# sau đó kiểm định chất lượng và loại candidate không đạt.
SYNTHETIC_TARGET_RATIO = 1.0

# RSSI quality constraints
RSSI_MIN = -100.0
RSSI_MAX = 0.0
MIN_VALID_AP = 3

# cGAN parameters
CGAN_NOISE_DIM = 32
CGAN_HIDDEN_DIM = 128
CGAN_EPOCHS = 300
CGAN_BATCH_SIZE = 32
CGAN_LR = 0.0002

CGAN_LAMBDA_RECON = 10.0
CGAN_PATIENCE = 35

# Candidate generation
COORD_TOL = 1e-5

# Device
import torch

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)
print("Floors:", FLOORS)
print("AP count:", len(AP_COLS))
print("Methods:", METHODS)

config_new_train = {
    "seed": NEW_SEED,
    "floors": FLOORS,
    "AP_columns": AP_COLS,
    "methods": METHODS,
    "augmentation_methods": AUGMENTATION_METHODS,
    "knn_k_values": KNN_K_VALUES,
    "apmean_radii_coordinate_units": APMEAN_RADII,
    "synthetic_target_ratio": SYNTHETIC_TARGET_RATIO,
    "minimum_valid_AP": MIN_VALID_AP,
    "RSSI_valid_range_dBm": [
        RSSI_MIN,
        RSSI_MAX
    ],
    "cGAN": {
        "noise_dim": CGAN_NOISE_DIM,
        "hidden_dim": CGAN_HIDDEN_DIM,
        "epochs": CGAN_EPOCHS,
        "batch_size": CGAN_BATCH_SIZE,
        "learning_rate": CGAN_LR,
        "lambda_reconstruction": CGAN_LAMBDA_RECON,
        "early_stopping_patience": CGAN_PATIENCE
    },
    "device": str(DEVICE),
    "test_policy": (
        "TEST is untouched during training, "
        "synthetic generation and validation selection."
    )
}

with open(
    NEW_TRAIN_DIR / "08_CONFIG" /
    "new_train_config.json",
    "w",
    encoding="utf-8"
) as fp:

    json.dump(
        config_new_train,
        fp,
        ensure_ascii=False,
        indent=2
    )

print("Configuration saved.")

## NEW TRAIN — CELL 3: DATA PREPARATION & OBSERVATION MASK

### 1. Mục tiêu

Chuẩn bị dữ liệu TRAIN và VALIDATION cho các phương pháp tăng cường dữ liệu, đồng thời bảo toàn thông tin về các AP được quan sát và các AP bị thiếu.

### 2. Tiền xử lý

- Chuyển RSSI sentinel `-100` và `100` thành NaN.
- Xây dựng observation mask:
  - `1`: AP có RSSI quan sát hợp lệ.
  - `0`: AP không có RSSI quan sát.
- Kiểm tra số lượng AP hợp lệ của từng fingerprint.
- Giữ nguyên tọa độ X/Y và ID_RP.
- Không sử dụng TEST trong bước chuẩn bị dữ liệu huấn luyện.

### 3. Quy tắc

Không thay thế RSSI thiếu bằng giá trị giả rồi xem đó là phép đo thật. Observation mask được lưu riêng để theo dõi nguồn gốc dữ liệu và phục vụ kiểm định chất lượng fingerprint tổng hợp.

### 4. Đầu ra

- TRAIN và VALIDATION đã chuẩn hóa về cấu trúc.
- Observation mask cho từng fingerprint.
- Bảng thống kê số AP quan sát được.
- Biểu đồ phân bố số AP quan sát theo từng tầng.

In [ ]:
# ============================================================
# NEW TRAIN — CELL 3
# DATA PREPARATION & OBSERVATION MASK
# ============================================================

def prepare_fingerprints(df, floor, role):

    result = df.copy()

    # Chuẩn hóa ID
    result["ID_RP"] = (
        result["ID_RP"].astype(str).str.strip()
    )

    # RSSI sentinel -> NaN
    for ap in AP_COLS:

        result[ap] = pd.to_numeric(
            result[ap],
            errors="coerce"
        )

        result.loc[
            result[ap].isin([-100, 100]),
            ap
        ] = np.nan

        result.loc[
            (result[ap] < RSSI_MIN) |
            (result[ap] > RSSI_MAX),
            ap
        ] = np.nan

    # Observation mask
    rssi = result[AP_COLS].to_numpy(float)

    mask = np.isfinite(rssi).astype(np.uint8)

    result["N_Observed_AP"] = mask.sum(axis=1)

    # Kiểm tra fingerprint đủ điều kiện
    result["Valid_Fingerprint"] = (
        result["N_Observed_AP"] >= MIN_VALID_AP
    )

    result["Floor"] = floor
    result["Data_Role"] = role

    return result


prepared_data = {}
mask_statistics = []

for floor in FLOORS:

    train = prepare_fingerprints(
        splits[floor]["TRAIN"],
        floor,
        "TRAIN"
    )

    validation = prepare_fingerprints(
        splits[floor]["VALIDATION"],
        floor,
        "VALIDATION"
    )

    prepared_data[floor] = {
        "TRAIN": train,
        "VALIDATION": validation
    }

    for role, frame in [
        ("TRAIN", train),
        ("VALIDATION", validation)
    ]:

        mask_statistics.append({
            "Floor": floor,
            "Role": role,
            "Rows": len(frame),
            "Valid_Rows": int(
                frame["Valid_Fingerprint"].sum()
            ),
            "Invalid_Rows": int(
                (~frame["Valid_Fingerprint"]).sum()
            ),
            "Mean_Observed_AP": (
                frame["N_Observed_AP"].mean()
            ),
            "Median_Observed_AP": (
                frame["N_Observed_AP"].median()
            ),
            "Missing_AP_Rate_%": (
                100 * (
                    1 - frame[AP_COLS].notna().to_numpy().mean()
                )
            )
        })

mask_stats_df = pd.DataFrame(mask_statistics)

display(
    mask_stats_df.style
    .format({
        "Mean_Observed_AP": "{:.2f}",
        "Median_Observed_AP": "{:.2f}",
        "Missing_AP_Rate_%": "{:.2f}%"
    })
    .background_gradient(
        subset=["Mean_Observed_AP"],
        cmap="YlGnBu"
    )
)

# Visualization
fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 4),
    constrained_layout=True
)

for ax, floor in zip(axes, FLOORS):

    train = prepared_data[floor]["TRAIN"]

    counts = (
        train["N_Observed_AP"]
        .value_counts()
        .sort_index()
    )

    ax.bar(
        counts.index,
        counts.values,
        color="#3B82F6",
        edgecolor="white"
    )

    ax.set_title(
        f"{floor} — Observed AP Distribution",
        fontweight="bold"
    )

    ax.set_xlabel("Number of observed APs")
    ax.set_ylabel("Number of fingerprints")
    ax.grid(axis="y", alpha=0.2)

plt.show()

# Save statistics
mask_stats_df.to_csv(
    NEW_TRAIN_DIR / "01_INPUT_AUDIT" /
    "observation_mask_statistics.csv",
    index=False
)

# Save prepared data
for floor in FLOORS:

    for role in ["TRAIN", "VALIDATION"]:

        prepared_data[floor][role].to_csv(
            NEW_TRAIN_DIR / "01_INPUT_AUDIT" /
            f"{floor}_{role}_prepared.csv",
            index=False
        )

print("CELL 3 COMPLETED")
print("TRAIN and VALIDATION prepared.")
print("TEST remains untouched.")

## NEW TRAIN — CELL 4: CGAN ARCHITECTURE & MASK-AWARE DESIGN

### 1. Mục tiêu

Xây dựng mô hình Conditional Generative Adversarial Network (cGAN) phục vụ tăng cường dữ liệu Wi-Fi Fingerprinting.

Mô hình học mối quan hệ giữa tọa độ không gian (X, Y), nhiễu ngẫu nhiên và đặc trưng RSSI của 17 Access Points.

### 2. Kiến trúc Generator

Generator nhận hai đầu vào:

- Tọa độ điều kiện (X, Y): cung cấp thông tin vị trí của fingerprint.
- Vector nhiễu ngẫu nhiên (z): tạo sự đa dạng giữa các fingerprint được sinh tại cùng điều kiện.

Generator tạo ra hai đầu ra:

- RSSI synthetic: vector RSSI gồm 17 AP.
- Observation mask probability: xác suất quan sát của từng AP.

### 3. Kiến trúc Discriminator

Discriminator nhận:

- Tọa độ (X, Y).
- Vector RSSI.
- Observation mask.

Nhiệm vụ của Discriminator là phân biệt fingerprint đo thực tế và fingerprint tổng hợp trong cùng điều kiện tọa độ.

### 4. Quy tắc chuẩn hóa

- Chỉ sử dụng TRAIN để xác định giới hạn chuẩn hóa RSSI và tọa độ.
- RSSI thiếu được lưu riêng bằng observation mask.
- Giá trị RSSI thiếu sau chuẩn hóa được đặt bằng 0 trong tensor đầu vào, nhưng mask vẫn cho biết đây là vị trí không quan sát.
- VALIDATION được chuẩn hóa bằng tham số học từ TRAIN.
- TEST không được đọc trong cell này.

### 5. Kiểm tra kiến trúc

Cell thực hiện:

- Xây dựng Generator và Discriminator.
- Kiểm tra kích thước tensor đầu vào/đầu ra.
- Kiểm tra observation mask.
- Thống kê số lượng tham số của mỗi mạng.
- Trực quan hóa sơ đồ luồng dữ liệu.

### 6. Đầu ra

- Generator và Discriminator cho từng tầng.
- Bộ dữ liệu tensor TRAIN và VALIDATION.
- Thông tin chuẩn hóa RSSI và tọa độ được fit từ TRAIN.
- Bảng thống kê kiến trúc và số lượng tham số.

**Lưu ý:** Đây là bước xây dựng kiến trúc. Chưa có fingerprint tổng hợp và chưa có kết quả huấn luyện cGAN.

In [ ]:
# ============================================================
# NEW TRAIN — CELL 4
# CGAN ARCHITECTURE & MASK-AWARE DESIGN
# Chỉ xây dựng kiến trúc, CHƯA TRAIN
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------

torch.manual_seed(NEW_SEED)
np.random.seed(NEW_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(NEW_SEED)

# ------------------------------------------------------------
# 2. Generator
# ------------------------------------------------------------

class WiFiGenerator(nn.Module):

    def __init__(
        self,
        noise_dim=32,
        hidden_dim=128,
        coord_dim=2,
        ap_dim=17
    ):
        super().__init__()

        self.noise_dim = noise_dim

        self.network = nn.Sequential(

            nn.Linear(coord_dim + noise_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(0.2),

            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(0.2),

            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(0.2)
        )

        # RSSI output
        self.rssi_head = nn.Sequential(
            nn.Linear(hidden_dim, ap_dim),
            nn.Tanh()
        )

        # Observation probability output
        self.mask_head = nn.Sequential(
            nn.Linear(hidden_dim, ap_dim),
            nn.Sigmoid()
        )

    def forward(self, coords, noise):

        x = torch.cat(
            [coords, noise],
            dim=1
        )

        hidden = self.network(x)

        synthetic_rssi = self.rssi_head(hidden)
        mask_probability = self.mask_head(hidden)

        return synthetic_rssi, mask_probability


# ------------------------------------------------------------
# 3. Discriminator
# ------------------------------------------------------------

class WiFiDiscriminator(nn.Module):

    def __init__(
        self,
        hidden_dim=128,
        coord_dim=2,
        ap_dim=17
    ):
        super().__init__()

        input_dim = coord_dim + ap_dim + ap_dim

        self.network = nn.Sequential(

            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.15),

            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.15),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.2),

            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, coords, rssi, mask):

        x = torch.cat(
            [coords, rssi, mask],
            dim=1
        )

        return self.network(x)


# ------------------------------------------------------------
# 4. Fit normalization from TRAIN only
# ------------------------------------------------------------

cgan_data = {}
cgan_models = {}
architecture_rows = []

def fit_floor_normalization(train):

    # RSSI
    rssi = train[AP_COLS].to_numpy(float)

    rssi_min = np.nanmin(rssi, axis=0)
    rssi_max = np.nanmax(rssi, axis=0)

    # Tránh chia cho 0 nếu AP có RSSI không đổi
    rssi_range = rssi_max - rssi_min
    rssi_range[rssi_range < 1e-8] = 1.0

    # Coordinates
    xy = train[["X", "Y"]].to_numpy(float)

    xy_min = xy.min(axis=0)
    xy_max = xy.max(axis=0)

    xy_range = xy_max - xy_min
    xy_range[xy_range < 1e-8] = 1.0

    return {
        "rssi_min": rssi_min,
        "rssi_range": rssi_range,
        "xy_min": xy_min,
        "xy_range": xy_range
    }


def transform_cgan_data(df, norm):

    rssi = df[AP_COLS].to_numpy(float)

    # Observation mask: 1 = observed, 0 = missing
    mask = np.isfinite(rssi).astype(np.float32)

    # Min-max RSSI -> [-1, 1]
    rssi_scaled = (
        2 * (rssi - norm["rssi_min"])
        / norm["rssi_range"]
    ) - 1

    # Missing values are represented by zero,
    # but the observation mask remains separate.
    rssi_scaled = np.where(
        mask == 1,
        rssi_scaled,
        0.0
    )

    # Coordinate -> [-1, 1]
    xy = df[["X", "Y"]].to_numpy(float)

    xy_scaled = (
        2 * (xy - norm["xy_min"])
        / norm["xy_range"]
    ) - 1

    return {
        "coords": torch.tensor(
            xy_scaled,
            dtype=torch.float32,
            device=DEVICE
        ),
        "rssi": torch.tensor(
            rssi_scaled,
            dtype=torch.float32,
            device=DEVICE
        ),
        "mask": torch.tensor(
            mask,
            dtype=torch.float32,
            device=DEVICE
        )
    }


# ------------------------------------------------------------
# 5. Build data and models per floor
# ------------------------------------------------------------

for floor in FLOORS:

    train = prepared_data[floor]["TRAIN"].copy()
    val = prepared_data[floor]["VALIDATION"].copy()

    # Only valid fingerprints are retained for cGAN training
    train_valid = train[
        train["Valid_Fingerprint"]
    ].reset_index(drop=True)

    val_valid = val[
        val["Valid_Fingerprint"]
    ].reset_index(drop=True)

    if len(train_valid) < 4:
        raise ValueError(
            f"{floor}: Không đủ TRAIN fingerprint hợp lệ."
        )

    if len(val_valid) < 1:
        raise ValueError(
            f"{floor}: Không có VALIDATION fingerprint hợp lệ."
        )

    # Fit normalization on TRAIN only
    norm = fit_floor_normalization(train_valid)

    train_tensor = transform_cgan_data(
        train_valid,
        norm
    )

    val_tensor = transform_cgan_data(
        val_valid,
        norm
    )

    # Build models
    G = WiFiGenerator(
        noise_dim=CGAN_NOISE_DIM,
        hidden_dim=CGAN_HIDDEN_DIM,
        ap_dim=len(AP_COLS)
    ).to(DEVICE)

    D = WiFiDiscriminator(
        hidden_dim=CGAN_HIDDEN_DIM,
        ap_dim=len(AP_COLS)
    ).to(DEVICE)

    cgan_data[floor] = {
        "TRAIN": train_tensor,
        "VALIDATION": val_tensor,
        "TRAIN_FRAME": train_valid,
        "VALIDATION_FRAME": val_valid,
        "NORMALIZATION": norm
    }

    cgan_models[floor] = {
        "G": G,
        "D": D
    }

    g_params = sum(
        p.numel() for p in G.parameters()
        if p.requires_grad
    )

    d_params = sum(
        p.numel() for p in D.parameters()
        if p.requires_grad
    )

    architecture_rows.append({
        "Floor": floor,
        "TRAIN_Fingerprints": len(train_valid),
        "VALIDATION_Fingerprints": len(val_valid),
        "Generator_Params": g_params,
        "Discriminator_Params": d_params,
        "Generator_Input_Dim": 2 + CGAN_NOISE_DIM,
        "Generator_RSSI_Output": len(AP_COLS),
        "Generator_Mask_Output": len(AP_COLS),
        "Discriminator_Input_Dim": 2 + 2 * len(AP_COLS)
    })


# ------------------------------------------------------------
# 6. Architecture audit
# ------------------------------------------------------------

architecture_df = pd.DataFrame(architecture_rows)

display(
    architecture_df.style
    .format({
        "Generator_Params": "{:,}",
        "Discriminator_Params": "{:,}"
    })
    .background_gradient(
        subset=[
            "Generator_Params",
            "Discriminator_Params"
        ],
        cmap="YlGnBu"
    )
)

architecture_df.to_csv(
    NEW_TRAIN_DIR / "02_CGAN" /
    "cgan_architecture_audit.csv",
    index=False
)


# ------------------------------------------------------------
# 7. Tensor shape test
# ------------------------------------------------------------

shape_rows = []

for floor in FLOORS:

    G = cgan_models[floor]["G"]
    D = cgan_models[floor]["D"]

    train_tensor = cgan_data[floor]["TRAIN"]

    batch_size = min(
        8,
        len(train_tensor["coords"])
    )

    coords = train_tensor["coords"][:batch_size]

    noise = torch.randn(
        batch_size,
        CGAN_NOISE_DIM,
        device=DEVICE
    )

    real_rssi = train_tensor["rssi"][:batch_size]
    real_mask = train_tensor["mask"][:batch_size]

    G.eval()
    D.eval()

    with torch.no_grad():

        fake_rssi, fake_mask = G(
            coords,
            noise
        )

        real_score = D(
            coords,
            real_rssi,
            real_mask
        )

        fake_score = D(
            coords,
            fake_rssi,
            fake_mask
        )

    assert fake_rssi.shape == (
        batch_size,
        len(AP_COLS)
    )

    assert fake_mask.shape == (
        batch_size,
        len(AP_COLS)
    )

    assert real_score.shape == (batch_size, 1)
    assert fake_score.shape == (batch_size, 1)

    assert torch.isfinite(fake_rssi).all()
    assert torch.isfinite(fake_mask).all()

    assert (
        (fake_mask >= 0) &
        (fake_mask <= 1)
    ).all()

    shape_rows.append({
        "Floor": floor,
        "Batch_Size": batch_size,
        "Coords_Shape": str(tuple(coords.shape)),
        "Noise_Shape": str(tuple(noise.shape)),
        "Fake_RSSI_Shape": str(tuple(fake_rssi.shape)),
        "Fake_Mask_Shape": str(tuple(fake_mask.shape)),
        "Discriminator_Shape": str(tuple(fake_score.shape)),
        "Tensor_Check": "PASS"
    })

shape_df = pd.DataFrame(shape_rows)

display(shape_df)

shape_df.to_csv(
    NEW_TRAIN_DIR / "02_CGAN" /
    "cgan_tensor_shape_audit.csv",
    index=False
)


# ------------------------------------------------------------
# 8. Visualization: cGAN architecture
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(15, 5))

ax.axis("off")

nodes = [
    (0.04, 0.58, 0.15, 0.23, "Coordinates\n(X, Y)", "#DBEAFE"),
    (0.04, 0.18, 0.15, 0.23, "Random Noise\n(z)", "#EDE9FE"),
    (0.29, 0.38, 0.18, 0.32, "GENERATOR\n\nMLP Network", "#BFDBFE"),
    (0.56, 0.58, 0.17, 0.23, "Synthetic RSSI\n(17 APs)", "#CCFBF1"),
    (0.56, 0.18, 0.17, 0.23, "Synthetic Mask\n(17 APs)", "#FEF3C7"),
    (0.81, 0.38, 0.16, 0.32, "DISCRIMINATOR\n\nReal / Fake", "#FCE7F3")
]

from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

for x, y, w, h, label, color in nodes:

    box = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.02",
        facecolor=color,
        edgecolor="#475569",
        linewidth=1.5
    )

    ax.add_patch(box)

    ax.text(
        x + w/2,
        y + h/2,
        label,
        ha="center",
        va="center",
        fontsize=10,
        fontweight="bold"
    )


def arrow(start, end, color="#64748B"):

    ax.add_patch(
        FancyArrowPatch(
            start,
            end,
            arrowstyle="->",
            mutation_scale=15,
            linewidth=1.6,
            color=color
        )
    )


arrow((0.19, 0.69), (0.29, 0.58))
arrow((0.19, 0.29), (0.29, 0.48))
arrow((0.47, 0.58), (0.56, 0.69))
arrow((0.47, 0.48), (0.56, 0.29))
arrow((0.73, 0.69), (0.81, 0.58))
arrow((0.73, 0.29), (0.81, 0.48))

ax.text(
    0.50,
    0.93,
    "MASK-AWARE CONDITIONAL GAN",
    ha="center",
    fontsize=15,
    fontweight="bold"
)

ax.text(
    0.50,
    0.04,
    "The discriminator also receives real samples with their observed masks.",
    ha="center",
    fontsize=9,
    color="#475569"
)

plt.show()

fig.savefig(
    NEW_TRAIN_DIR / "07_FIGURES" /
    "cgan_architecture.png",
    dpi=180,
    bbox_inches="tight"
)


# ------------------------------------------------------------
# 9. Final status
# ------------------------------------------------------------

print("=" * 70)
print("CELL 4 — CGAN ARCHITECTURE COMPLETED")
print("=" * 70)

print("Floors:", FLOORS)
print("Device:", DEVICE)
print("Generator and Discriminator built for all floors.")
print("Tensor shape checks: PASS")
print("Training started: NO")
print("Synthetic data generated: NO")

In [ ]:
# ============================================================
# NEW TRAIN — CELL 4B
# STABILITY ADJUSTMENT: BATCH NORM -> LAYER NORM
# ============================================================

class WiFiGenerator(nn.Module):

    def __init__(
        self,
        noise_dim=32,
        hidden_dim=128,
        coord_dim=2,
        ap_dim=17
    ):
        super().__init__()

        self.noise_dim = noise_dim

        self.network = nn.Sequential(

            nn.Linear(coord_dim + noise_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(0.2),

            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(0.2),

            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(0.2)
        )

        self.rssi_head = nn.Sequential(
            nn.Linear(hidden_dim, ap_dim),
            nn.Tanh()
        )

        self.mask_head = nn.Sequential(
            nn.Linear(hidden_dim, ap_dim),
            nn.Sigmoid()
        )

    def forward(self, coords, noise):

        x = torch.cat([coords, noise], dim=1)

        hidden = self.network(x)

        return (
            self.rssi_head(hidden),
            self.mask_head(hidden)
        )


class WiFiDiscriminator(nn.Module):

    def __init__(
        self,
        hidden_dim=128,
        coord_dim=2,
        ap_dim=17
    ):
        super().__init__()

        input_dim = coord_dim + 2 * ap_dim

        self.network = nn.Sequential(

            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.15),

            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.15),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.2),

            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, coords, rssi, mask):

        x = torch.cat(
            [coords, rssi, mask],
            dim=1
        )

        return self.network(x)


# Reinitialize models for all floors
cgan_models = {}

stability_rows = []

for floor in FLOORS:

    G = WiFiGenerator(
        noise_dim=CGAN_NOISE_DIM,
        hidden_dim=CGAN_HIDDEN_DIM,
        ap_dim=len(AP_COLS)
    ).to(DEVICE)

    D = WiFiDiscriminator(
        hidden_dim=CGAN_HIDDEN_DIM,
        ap_dim=len(AP_COLS)
    ).to(DEVICE)

    cgan_models[floor] = {
        "G": G,
        "D": D
    }

    batch_size = min(
        8,
        len(cgan_data[floor]["TRAIN"]["coords"])
    )

    coords = cgan_data[floor]["TRAIN"]["coords"][:batch_size]

    noise = torch.randn(
        batch_size,
        CGAN_NOISE_DIM,
        device=DEVICE
    )

    with torch.no_grad():

        fake_rssi, fake_mask = G(coords, noise)

        score = D(
            coords,
            fake_rssi,
            fake_mask
        )

    assert fake_rssi.shape == (batch_size, 17)
    assert fake_mask.shape == (batch_size, 17)
    assert score.shape == (batch_size, 1)

    stability_rows.append({
        "Floor": floor,
        "Normalization": "LayerNorm",
        "Batch_Size": batch_size,
        "Generator_Params": sum(
            p.numel() for p in G.parameters()
        ),
        "Discriminator_Params": sum(
            p.numel() for p in D.parameters()
        ),
        "Tensor_Check": "PASS"
    })

stability_df = pd.DataFrame(stability_rows)

display(stability_df)

stability_df.to_csv(
    NEW_TRAIN_DIR / "02_CGAN" /
    "cgan_layernorm_stability_audit.csv",
    index=False
)

print("CELL 4B COMPLETED")
print("All models reinitialized with LayerNorm.")
print("Training has NOT started.")

## NEW TRAIN — CELL 5: CGAN LOSS FUNCTION DESIGN

### 1. Mục tiêu

Xây dựng các hàm mất mát phục vụ huấn luyện cGAN, nhằm kiểm soát đồng thời tính hợp lý của fingerprint RSSI tổng hợp và khả năng mô phỏng tình trạng quan sát của các Access Point.

### 2. Các thành phần Loss

**Adversarial Loss**

Đánh giá khả năng Generator tạo fingerprint khiến Discriminator khó phân biệt dữ liệu thực và dữ liệu tổng hợp.

**RSSI Reconstruction Loss**

Sử dụng sai số tuyệt đối giữa RSSI tổng hợp và RSSI thực trên các AP có quan sát hợp lệ. Các AP bị thiếu không tham gia tính sai số này.

**Observation Mask Loss**

Đánh giá sự khác biệt giữa observation mask thực tế và xác suất observation mask do Generator dự đoán.

### 3. Hàm mục tiêu

Generator được tối ưu theo tổng hợp:

- Adversarial Loss.
- RSSI Reconstruction Loss có trọng số.
- Mask Loss có trọng số.

Discriminator được tối ưu để phân biệt fingerprint thực và fingerprint tổng hợp.

### 4. Kiểm tra

Cell kiểm tra:

- Giá trị từng thành phần loss.
- Khả năng tính gradient.
- Không phát sinh NaN hoặc Inf.
- RSSI thiếu không tham gia reconstruction loss.
- Mask loss được tính riêng.

### 5. Đầu ra

Bảng kiểm tra các thành phần loss trên từng tầng. Cell chưa thực hiện huấn luyện đầy đủ và chưa sinh dữ liệu tổng hợp để đánh giá cuối cùng.

In [ ]:
# ============================================================
# NEW TRAIN — CELL 5
# CGAN LOSS FUNCTION DESIGN
# ============================================================

import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np

# Trọng số loss
LAMBDA_RSSI = 10.0
LAMBDA_MASK = 2.0

# ------------------------------------------------------------
# 1. Masked RSSI reconstruction loss
# ------------------------------------------------------------

def masked_rssi_loss(fake_rssi, real_rssi, real_mask):

    absolute_error = torch.abs(
        fake_rssi - real_rssi
    )

    weighted_error = absolute_error * real_mask

    denominator = real_mask.sum().clamp(min=1.0)

    return weighted_error.sum() / denominator


# ------------------------------------------------------------
# 2. Observation mask loss
# ------------------------------------------------------------

def observation_mask_loss(
    fake_mask_probability,
    real_mask
):

    return F.binary_cross_entropy(
        fake_mask_probability,
        real_mask
    )


# ------------------------------------------------------------
# 3. Generator adversarial loss
# ------------------------------------------------------------

def generator_adversarial_loss(fake_score):

    # Discriminator should classify fake as real
    return F.binary_cross_entropy_with_logits(
        fake_score,
        torch.ones_like(fake_score)
    )


# ------------------------------------------------------------
# 4. Discriminator loss
# ------------------------------------------------------------

def discriminator_loss(real_score, fake_score):

    real_loss = F.binary_cross_entropy_with_logits(
        real_score,
        torch.ones_like(real_score)
    )

    fake_loss = F.binary_cross_entropy_with_logits(
        fake_score,
        torch.zeros_like(fake_score)
    )

    return 0.5 * (real_loss + fake_loss)


# ------------------------------------------------------------
# 5. Total generator loss
# ------------------------------------------------------------

def total_generator_loss(
    adv_loss,
    rssi_loss,
    mask_loss
):

    return (
        adv_loss
        + LAMBDA_RSSI * rssi_loss
        + LAMBDA_MASK * mask_loss
    )


# ------------------------------------------------------------
# 6. Loss sanity check for each floor
# ------------------------------------------------------------

loss_rows = []

for floor in FLOORS:

    G = cgan_models[floor]["G"]
    D = cgan_models[floor]["D"]

    batch = cgan_data[floor]["TRAIN"]

    coords = batch["coords"][:8]
    real_rssi = batch["rssi"][:8]
    real_mask = batch["mask"][:8]

    noise = torch.randn(
        len(coords),
        CGAN_NOISE_DIM,
        device=DEVICE
    )

    G.train()
    D.train()

    fake_rssi, fake_mask = G(
        coords,
        noise
    )

    real_score = D(
        coords,
        real_rssi,
        real_mask
    )

    fake_score = D(
        coords,
        fake_rssi.detach(),
        fake_mask.detach()
    )

    adv = generator_adversarial_loss(
        D(coords, fake_rssi, fake_mask)
    )

    rssi = masked_rssi_loss(
        fake_rssi,
        real_rssi,
        real_mask
    )

    mask = observation_mask_loss(
        fake_mask,
        real_mask
    )

    d_loss = discriminator_loss(
        real_score,
        fake_score
    )

    g_loss = total_generator_loss(
        adv,
        rssi,
        mask
    )

    # Gradient check
    G.zero_grad(set_to_none=True)

    g_loss.backward()

    has_gradient = any(
        p.grad is not None
        and torch.isfinite(p.grad).all()
        for p in G.parameters()
    )

    loss_values = [
        adv.item(),
        rssi.item(),
        mask.item(),
        d_loss.item(),
        g_loss.item()
    ]

    if not np.isfinite(loss_values).all():
        raise ValueError(
            f"{floor}: Loss phát sinh NaN hoặc Inf."
        )

    if not has_gradient:
        raise ValueError(
            f"{floor}: Generator không nhận được gradient."
        )

    loss_rows.append({
        "Floor": floor,
        "Adversarial_Loss": adv.item(),
        "RSSI_Loss": rssi.item(),
        "Mask_Loss": mask.item(),
        "Discriminator_Loss": d_loss.item(),
        "Total_Generator_Loss": g_loss.item(),
        "Gradient_Check": "PASS",
        "Finite_Check": "PASS"
    })

loss_audit_df = pd.DataFrame(loss_rows)

display(
    loss_audit_df.style.format({
        "Adversarial_Loss": "{:.5f}",
        "RSSI_Loss": "{:.5f}",
        "Mask_Loss": "{:.5f}",
        "Discriminator_Loss": "{:.5f}",
        "Total_Generator_Loss": "{:.5f}"
    }).background_gradient(
        subset=[
            "Adversarial_Loss",
            "RSSI_Loss",
            "Mask_Loss"
        ],
        cmap="YlGnBu"
    )
)

loss_audit_df.to_csv(
    NEW_TRAIN_DIR / "02_CGAN" /
    "cgan_loss_sanity_check.csv",
    index=False
)

print("=" * 65)
print("CELL 5 — LOSS FUNCTION CHECK COMPLETED")
print("=" * 65)
print("Gradient check: PASS")
print("Finite loss check: PASS")
print("Full training started: NO")

## NEW TRAIN — CELL 6: CGAN TRAINING & VALIDATION CHECKPOINT

### 1. Mục tiêu

Huấn luyện riêng một mô hình cGAN cho từng tầng F1, F2 và F3 để học phân bố fingerprint RSSI có điều kiện theo tọa độ.

### 2. Dữ liệu huấn luyện

- TRAIN: fingerprint gốc đã được chuẩn hóa và có observation mask.
- VALIDATION: fingerprint tại các RP bị giữ lại, chỉ sử dụng để đánh giá và lựa chọn checkpoint.
- TEST: không được sử dụng trong quá trình huấn luyện.

### 3. Quy trình huấn luyện

Mỗi epoch gồm hai bước:

**Discriminator training**

- Nhận fingerprint thực cùng observation mask thực.
- Nhận fingerprint tổng hợp cùng mask nhị phân được lấy mẫu từ Generator.
- Học phân biệt dữ liệu thực và dữ liệu tổng hợp.

**Generator training**

- Sinh RSSI và observation mask từ tọa độ cùng nhiễu ngẫu nhiên.
- Tối ưu Adversarial Loss.
- Tối ưu RSSI Reconstruction Loss trên AP được quan sát.
- Tối ưu Mask Loss.

### 4. Đánh giá VALIDATION

Sau mỗi epoch, mô hình được đánh giá bằng:

- RSSI MAE trên các AP quan sát được.
- Mask BCE.
- Validation Score tổng hợp.

Checkpoint tốt nhất được lưu theo Validation Score. Early stopping được sử dụng để hạn chế overfitting.

### 5. Đầu ra

- Generator checkpoint tốt nhất của từng tầng.
- Lịch sử Loss theo epoch.
- Biểu đồ Generator Loss và Discriminator Loss.
- Biểu đồ RSSI Validation Loss và Mask Validation Loss.
- Bảng epoch tốt nhất và trạng thái huấn luyện.

**Lưu ý:** Loss thấp không tự động chứng minh fingerprint tổng hợp có chất lượng tốt. Các checkpoint sẽ tiếp tục được kiểm định bằng phân bố RSSI, observation mask, tính đa dạng và đánh giá định vị WKNN ở các bước sau.

In [ ]:
# ============================================================
# NEW TRAIN — CELL 6
# CGAN TRAINING WITH VALIDATION CHECKPOINT
# ============================================================

import copy
import time
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import TensorDataset, DataLoader
from IPython.display import display

# ------------------------------------------------------------
# 1. Training configuration
# ------------------------------------------------------------

EPOCHS = CGAN_EPOCHS
BATCH_SIZE = CGAN_BATCH_SIZE
LR = CGAN_LR
PATIENCE = CGAN_PATIENCE

LAMBDA_RSSI = 10.0
LAMBDA_MASK = 2.0

N_VALIDATION_NOISE = 5

# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------

def straight_through_mask(probability):

    hard_mask = (
        probability >= 0.5
    ).float()

    # Forward: binary mask
    # Backward: gradient through probability
    return (
        hard_mask
        - probability.detach()
        + probability
    )


def sample_binary_mask(probability):

    return torch.bernoulli(
        probability.detach()
    )


def train_discriminator_step(
    G, D, optimizer_D,
    coords, real_rssi, real_mask
):

    batch_size = len(coords)

    noise = torch.randn(
        batch_size,
        CGAN_NOISE_DIM,
        device=DEVICE
    )

    with torch.no_grad():

        fake_rssi, fake_mask_prob = G(
            coords,
            noise
        )

        fake_mask = sample_binary_mask(
            fake_mask_prob
        )

        fake_input_rssi = (
            fake_rssi * fake_mask
        )

    real_score = D(
        coords,
        real_rssi,
        real_mask
    )

    fake_score = D(
        coords,
        fake_input_rssi,
        fake_mask
    )

    d_loss = discriminator_loss(
        real_score,
        fake_score
    )

    optimizer_D.zero_grad(set_to_none=True)
    d_loss.backward()

    torch.nn.utils.clip_grad_norm_(
        D.parameters(),
        max_norm=5.0
    )

    optimizer_D.step()

    return d_loss.item()


def train_generator_step(
    G, D, optimizer_G,
    coords, real_rssi, real_mask
):

    batch_size = len(coords)

    noise = torch.randn(
        batch_size,
        CGAN_NOISE_DIM,
        device=DEVICE
    )

    fake_rssi, fake_mask_prob = G(
        coords,
        noise
    )

    # Binary forward mask + differentiable backward
    fake_mask_st = straight_through_mask(
        fake_mask_prob
    )

    fake_input_rssi = (
        fake_rssi * fake_mask_st
    )

    fake_score = D(
        coords,
        fake_input_rssi,
        fake_mask_st
    )

    adv_loss = generator_adversarial_loss(
        fake_score
    )

    # Reconstruction only on observed real APs
    rssi_loss = masked_rssi_loss(
        fake_rssi,
        real_rssi,
        real_mask
    )

    mask_loss = observation_mask_loss(
        fake_mask_prob,
        real_mask
    )

    g_loss = total_generator_loss(
        adv_loss,
        rssi_loss,
        mask_loss
    )

    optimizer_G.zero_grad(set_to_none=True)
    g_loss.backward()

    torch.nn.utils.clip_grad_norm_(
        G.parameters(),
        max_norm=5.0
    )

    optimizer_G.step()

    return {
        "G_Loss": g_loss.item(),
        "G_Adversarial": adv_loss.item(),
        "G_RSSI": rssi_loss.item(),
        "G_Mask": mask_loss.item()
    }


# ------------------------------------------------------------
# 3. Validation evaluation
# ------------------------------------------------------------

def evaluate_cgan_validation(
    G,
    validation_data,
    noise_bank
):

    G.eval()

    coords = validation_data["coords"]
    real_rssi = validation_data["rssi"]
    real_mask = validation_data["mask"]

    rssi_values = []
    mask_values = []

    with torch.no_grad():

        for noise in noise_bank:

            fake_rssi, fake_mask_prob = G(
                coords,
                noise
            )

            rssi_loss = masked_rssi_loss(
                fake_rssi,
                real_rssi,
                real_mask
            )

            mask_loss = observation_mask_loss(
                fake_mask_prob,
                real_mask
            )

            rssi_values.append(rssi_loss.item())
            mask_values.append(mask_loss.item())

    mean_rssi = float(np.mean(rssi_values))
    mean_mask = float(np.mean(mask_values))

    score = (
        LAMBDA_RSSI * mean_rssi
        + LAMBDA_MASK * mean_mask
    )

    return {
        "Val_RSSI": mean_rssi,
        "Val_Mask": mean_mask,
        "Val_Score": score
    }


# ------------------------------------------------------------
# 4. Train each floor independently
# ------------------------------------------------------------

training_histories = {}
best_checkpoints = {}
training_summary = []

for floor in FLOORS:

    print("\n" + "=" * 75)
    print(f"TRAINING CGAN — {floor}")
    print("=" * 75)

    start_time = time.time()

    G = cgan_models[floor]["G"]
    D = cgan_models[floor]["D"]

    train = cgan_data[floor]["TRAIN"]
    val = cgan_data[floor]["VALIDATION"]

    # DataLoader
    dataset = TensorDataset(
        train["coords"],
        train["rssi"],
        train["mask"]
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False
    )

    # Optimizers
    optimizer_G = torch.optim.Adam(
        G.parameters(),
        lr=LR,
        betas=(0.5, 0.999)
    )

    optimizer_D = torch.optim.Adam(
        D.parameters(),
        lr=LR,
        betas=(0.5, 0.999)
    )

    # Fixed validation noise for fair comparison
    val_noise_bank = [
        torch.randn(
            len(val["coords"]),
            CGAN_NOISE_DIM,
            device=DEVICE
        )
        for _ in range(N_VALIDATION_NOISE)
    ]

    best_score = float("inf")
    best_epoch = 0
    patience_counter = 0

    history = []

    for epoch in range(1, EPOCHS + 1):

        G.train()
        D.train()

        epoch_d = []
        epoch_g = []
        epoch_adv = []
        epoch_rssi = []
        epoch_mask = []

        for coords, real_rssi, real_mask in loader:

            # Discriminator update
            d_value = train_discriminator_step(
                G, D, optimizer_D,
                coords, real_rssi, real_mask
            )

            # Generator update
            g_values = train_generator_step(
                G, D, optimizer_G,
                coords, real_rssi, real_mask
            )

            epoch_d.append(d_value)
            epoch_g.append(g_values["G_Loss"])
            epoch_adv.append(g_values["G_Adversarial"])
            epoch_rssi.append(g_values["G_RSSI"])
            epoch_mask.append(g_values["G_Mask"])

        # Validation
        val_result = evaluate_cgan_validation(
            G,
            val,
            val_noise_bank
        )

        row = {
            "Epoch": epoch,
            "D_Loss": np.mean(epoch_d),
            "G_Loss": np.mean(epoch_g),
            "G_Adversarial": np.mean(epoch_adv),
            "G_RSSI": np.mean(epoch_rssi),
            "G_Mask": np.mean(epoch_mask),
            **val_result
        }

        history.append(row)

        # Save best checkpoint by validation score
        if val_result["Val_Score"] < best_score:

            best_score = val_result["Val_Score"]
            best_epoch = epoch
            patience_counter = 0

            best_state = {
                "G": copy.deepcopy(G.state_dict()),
                "D": copy.deepcopy(D.state_dict()),
                "epoch": epoch,
                "validation_score": best_score
            }

        else:

            patience_counter += 1

        # Progress
        if (
            epoch == 1
            or epoch % 25 == 0
            or patience_counter >= PATIENCE
        ):

            print(
                f"Epoch {epoch:03d}/{EPOCHS} | "
                f"D={row['D_Loss']:.4f} | "
                f"G={row['G_Loss']:.4f} | "
                f"Val RSSI={row['Val_RSSI']:.4f} | "
                f"Val Mask={row['Val_Mask']:.4f} | "
                f"Best={best_epoch}"
            )

        if patience_counter >= PATIENCE:

            print(
                f"Early stopping at epoch {epoch}"
            )

            break

    # Restore best checkpoint
    G.load_state_dict(best_state["G"])
    D.load_state_dict(best_state["D"])

    elapsed = time.time() - start_time

    best_checkpoints[floor] = best_state
    training_histories[floor] = pd.DataFrame(history)

    training_summary.append({
        "Floor": floor,
        "Best_Epoch": best_epoch,
        "Epochs_Run": len(history),
        "Best_Val_Score": best_score,
        "Best_Val_RSSI": history[best_epoch-1]["Val_RSSI"],
        "Best_Val_Mask": history[best_epoch-1]["Val_Mask"],
        "Training_Time_Seconds": elapsed,
        "Status": "COMPLETED"
    })

    # Save checkpoint
    torch.save(
        best_state,
        NEW_TRAIN_DIR / "02_CGAN" /
        f"cgan_best_{floor}.pt"
    )

    # Save history
    training_histories[floor].to_csv(
        NEW_TRAIN_DIR / "02_CGAN" /
        f"cgan_history_{floor}.csv",
        index=False
    )

    print(
        f"{floor} COMPLETED | "
        f"Best epoch: {best_epoch} | "
        f"Time: {elapsed:.1f}s"
    )


# ------------------------------------------------------------
# 5. Training summary
# ------------------------------------------------------------

training_summary_df = pd.DataFrame(
    training_summary
)

display(
    training_summary_df.style.format({
        "Best_Val_Score": "{:.5f}",
        "Best_Val_RSSI": "{:.5f}",
        "Best_Val_Mask": "{:.5f}",
        "Training_Time_Seconds": "{:.1f}"
    }).background_gradient(
        subset=["Best_Val_Score"],
        cmap="YlGnBu_r"
    )
)

training_summary_df.to_csv(
    NEW_TRAIN_DIR / "02_CGAN" /
    "cgan_training_summary.csv",
    index=False
)


# ------------------------------------------------------------
# 6. Training curves
# ------------------------------------------------------------

for floor in FLOORS:

    hist = training_histories[floor]

    fig, axes = plt.subplots(
        1, 2,
        figsize=(15, 4.5),
        constrained_layout=True
    )

    # Generator / Discriminator
    axes[0].plot(
        hist["Epoch"],
        hist["G_Loss"],
        label="Generator",
        color="#3B82F6",
        linewidth=2
    )

    axes[0].plot(
        hist["Epoch"],
        hist["D_Loss"],
        label="Discriminator",
        color="#F59E0B",
        linewidth=2
    )

    axes[0].set_title(
        f"{floor} — GAN Training Loss",
        fontweight="bold"
    )

    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.2)

    # Validation
    axes[1].plot(
        hist["Epoch"],
        hist["Val_RSSI"],
        label="Validation RSSI",
        color="#8B5CF6",
        linewidth=2
    )

    axes[1].plot(
        hist["Epoch"],
        hist["Val_Mask"],
        label="Validation Mask",
        color="#14B8A6",
        linewidth=2
    )

    best_epoch = best_checkpoints[floor]["epoch"]

    axes[1].axvline(
        best_epoch,
        color="#EF4444",
        linestyle="--",
        label=f"Best epoch: {best_epoch}"
    )

    axes[1].set_title(
        f"{floor} — Validation Loss",
        fontweight="bold"
    )

    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(alpha=0.2)

    plt.show()

    fig.savefig(
        NEW_TRAIN_DIR / "07_FIGURES" /
        f"cgan_training_curves_{floor}.png",
        dpi=180,
        bbox_inches="tight"
    )


print("=" * 75)
print("CELL 6 — CGAN TRAINING COMPLETED")
print("=" * 75)
print("Best checkpoints restored.")
print("TEST has not been used.")

## NEW TRAIN — CELL 7: CGAN SYNTHETIC FINGERPRINT GENERATION

### 1. Mục tiêu

Sử dụng Generator tốt nhất được lựa chọn ở Cell 6 để tạo fingerprint Wi-Fi tổng hợp cho từng tầng F1, F2 và F3.

### 2. Dữ liệu đầu vào

- Generator checkpoint tốt nhất theo VALIDATION.
- Tọa độ RP thuộc tập TRAIN.
- Nhiễu ngẫu nhiên có seed cố định.
- Tham số chuẩn hóa RSSI và tọa độ được fit từ TRAIN.

### 3. Quy trình sinh dữ liệu

1. Lấy tọa độ TRAIN làm điều kiện đầu vào.
2. Sinh vector nhiễu ngẫu nhiên.
3. Generator tạo RSSI synthetic và xác suất observation mask.
4. Chuyển xác suất mask thành mask nhị phân.
5. Chuyển RSSI từ không gian chuẩn hóa về dBm.
6. Gán NaN cho những AP được xác định là không quan sát.
7. Gắn nhãn nguồn dữ liệu là CGAN.

### 4. Nguyên tắc bảo toàn dữ liệu

- Không sử dụng tọa độ TEST.
- Không sử dụng RSSI TEST.
- Không ghi đè fingerprint gốc.
- Mỗi fingerprint tổng hợp phải có thông tin ID_RP, tọa độ, phương pháp sinh và trạng thái observation mask.
- Candidate không đạt điều kiện số AP quan sát tối thiểu sẽ được đánh dấu để kiểm định, không âm thầm xóa khỏi thống kê.

### 5. Đầu ra

- Bộ fingerprint CGAN theo từng tầng.
- Bảng thống kê số lượng candidate.
- Bản đồ phân bố tọa độ fingerprint tổng hợp.
- File CSV lưu dữ liệu và provenance.

**Lưu ý:** Candidate được sinh ở bước này chưa được mặc định là dữ liệu chất lượng. Chúng sẽ tiếp tục trải qua kiểm định RSSI, mask, độ đa dạng và tính hợp lý không gian ở các cell tiếp theo.

In [ ]:
# ============================================================
# NEW TRAIN — CELL 7: CGAN SYNTHETIC FINGERPRINT GENERATION
# ============================================================

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from IPython.display import display

CGAN_SYNTHETIC = {}
generation_rows = []

for floor in FLOORS:

    print("\n" + "=" * 70)
    print(f"GENERATING SYNTHETIC FINGERPRINTS — {floor}")
    print("=" * 70)

    G = cgan_models[floor]["G"]
    norm = cgan_data[floor]["NORMALIZATION"]
    train_frame = cgan_data[floor]["TRAIN_FRAME"].copy()

    G.eval()

    # Number of candidates based on TRAIN RP count
    n_original = len(train_frame)

    n_synthetic = int(
        np.ceil(
            n_original * SYNTHETIC_TARGET_RATIO
        )
    )

    # Sample TRAIN coordinates with replacement if needed
    rng = np.random.default_rng(
        NEW_SEED + FLOORS.index(floor)
    )

    selected_idx = rng.choice(
        n_original,
        size=n_synthetic,
        replace=(n_synthetic > n_original)
    )

    selected = train_frame.iloc[
        selected_idx
    ].reset_index(drop=True)

    xy = selected[["X", "Y"]].to_numpy(float)

    # Normalize coordinates using TRAIN normalization
    xy_scaled = (
        2 * (xy - norm["xy_min"])
        / norm["xy_range"]
    ) - 1

    coords_tensor = torch.tensor(
        xy_scaled,
        dtype=torch.float32,
        device=DEVICE
    )

    # Reproducible noise
    generator = torch.Generator(device=DEVICE)
    generator.manual_seed(
        NEW_SEED + 100 + FLOORS.index(floor)
    )

    noise = torch.randn(
        n_synthetic,
        CGAN_NOISE_DIM,
        generator=generator,
        device=DEVICE
    )

    # Generate in batches to control memory
    fake_rssi_list = []
    fake_mask_list = []

    with torch.no_grad():

        for start in range(0, n_synthetic, 128):

            end = min(start + 128, n_synthetic)

            fake_rssi, fake_mask_prob = G(
                coords_tensor[start:end],
                noise[start:end]
            )

            # Sample binary observation mask
            mask_generator = torch.Generator(
                device=DEVICE
            )

            mask_generator.manual_seed(
                NEW_SEED
                + 1000
                + FLOORS.index(floor)
                + start
            )

            fake_mask = torch.bernoulli(
                fake_mask_prob,
                generator=mask_generator
            )

            fake_rssi_list.append(
                fake_rssi.cpu().numpy()
            )

            fake_mask_list.append(
                fake_mask.cpu().numpy()
            )

    fake_rssi_scaled = np.concatenate(
        fake_rssi_list,
        axis=0
    )

    fake_mask = np.concatenate(
        fake_mask_list,
        axis=0
    ).astype(np.uint8)

    # Inverse transform RSSI to dBm
    fake_rssi_dbm = (
        (fake_rssi_scaled + 1) / 2
    ) * norm["rssi_range"] + norm["rssi_min"]

    # Preserve missing observations as NaN
    fake_rssi_dbm = np.where(
        fake_mask == 1,
        fake_rssi_dbm,
        np.nan
    )

    # Build synthetic dataframe
    synthetic = pd.DataFrame(
        fake_rssi_dbm,
        columns=AP_COLS
    )

    synthetic.insert(
        0,
        "ID_RP",
        selected["ID_RP"].astype(str).to_numpy()
    )

    synthetic.insert(
        0,
        "Floor",
        floor
    )

    synthetic["X"] = xy[:, 0]
    synthetic["Y"] = xy[:, 1]

    synthetic["Source"] = "CGAN"
    synthetic["Original_RP_ID"] = (
        selected["ID_RP"].astype(str).to_numpy()
    )

    synthetic["N_Observed_AP"] = fake_mask.sum(axis=1)

    synthetic["Valid_Fingerprint"] = (
        synthetic["N_Observed_AP"] >= MIN_VALID_AP
    )

    # Sử dụng mảng fake_mask đã concatenate phẳng để tránh lỗi kích thước lô không đồng đều
    synthetic["CGAN_Probability_Mean"] = fake_mask.mean()

    synthetic["Synthetic_ID"] = [
        f"{floor}_CGAN_{i+1:04d}"
        for i in range(n_synthetic)
    ]

    # Save
    CGAN_SYNTHETIC[floor] = synthetic.copy()

    synthetic.to_csv(
        NEW_TRAIN_DIR / "03_SYNTHETIC" /
        f"CGAN_synthetic_{floor}.csv",
        index=False
    )

    # Statistics
    generation_rows.append({
        "Floor": floor,
        "Original_TRAIN_RP": n_original,
        "Synthetic_Candidates": n_synthetic,
        "Valid_Candidates": int(
            synthetic["Valid_Fingerprint"].sum()
        ),
        "Invalid_Candidates": int(
            (~synthetic["Valid_Fingerprint"]).sum()
        ),
        "Mean_Observed_AP": (
            synthetic["N_Observed_AP"].mean()
        ),
        "Mean_RSSI_dBm": (
            np.nanmean(
                synthetic[AP_COLS].to_numpy(float)
            )
        ),
        "Generation_Status": "COMPLETED"
    })

    print("Original TRAIN:", n_original)
    print("Synthetic candidates:", n_synthetic)
    print(
        "Candidates with minimum AP:",
        int(synthetic["Valid_Fingerprint"].sum())
    )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

generation_df = pd.DataFrame(generation_rows)

display(
    generation_df.style.format({
        "Mean_Observed_AP": "{:.2f}",
        "Mean_RSSI_dBm": "{:.2f}"
    }).background_gradient(
        subset=["Synthetic_Candidates", "Valid_Candidates"],
        cmap="YlGnBu"
    )
)

generation_df.to_csv(
    NEW_TRAIN_DIR / "03_SYNTHETIC" /
    "cgan_generation_summary.csv",
    index=False
)


# ------------------------------------------------------------
# Spatial visualization
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 5),
    constrained_layout=True
)

for ax, floor in zip(axes, FLOORS):

    original = cgan_data[floor]["TRAIN_FRAME"]
    synthetic = CGAN_SYNTHETIC[floor]

    ax.scatter(
        original["X"],
        original["Y"],
        s=24,
        color="#2563EB",
        alpha=0.55,
        label="Original TRAIN"
    )

    ax.scatter(
        synthetic["X"],
        synthetic["Y"],
        s=30,
        color="#F59E0B",
        marker="x",
        alpha=0.8,
        label="CGAN synthetic"
    )

    ax.set_title(
        f"{floor} — Original vs Synthetic",
        fontweight="bold"
    )

    ax.set_xlabel("X (source coordinate)")
    ax.set_ylabel("Y (source coordinate)")
    ax.grid(alpha=0.2)
    ax.legend(frameon=False)

plt.show()

fig.savefig(
    NEW_TRAIN_DIR / "07_FIGURES" /
    "cgan_original_vs_synthetic_map.png",
    dpi=180,
    bbox_inches="tight"
)

print("=" * 70)
print("CELL 7 — CGAN GENERATION COMPLETED")
print("=" * 70)
print("Synthetic data saved:", len(CGAN_SYNTHETIC))

# CELL 8 — CGAN SYNTHETIC QUALITY AUDIT

## Mục tiêu
Đánh giá chất lượng dữ liệu fingerprint được sinh bởi cGAN trước khi kết hợp với TRAIN gốc.

## Nội dung kiểm định
- So sánh phân bố RSSI giữa TRAIN gốc và dữ liệu cGAN.
- So sánh số lượng AP quan sát trung bình.
- Đánh giá độ đa dạng của fingerprint tổng hợp.
- Kiểm tra fingerprint trùng lặp và mức độ tương đồng với dữ liệu gốc.
- Kiểm tra tính nhất quán giữa RSSI và tọa độ RP.

## Nguyên tắc
- Chỉ sử dụng TRAIN và dữ liệu tổng hợp.
- Không sử dụng VALIDATION hoặc TEST để đánh giá hay điều chỉnh dữ liệu tổng hợp.
- Không loại bỏ fingerprint chỉ vì kết quả kiểm định chưa tốt.
- Phân biệt rõ dữ liệu được sinh thành công và dữ liệu có chất lượng tốt.

## Đầu ra
- Bảng kiểm định chất lượng theo từng tầng.
- Biểu đồ phân bố RSSI.
- Biểu đồ số AP quan sát.
- Ma trận tương đồng fingerprint.
- Báo cáo cảnh báo về độ đa dạng và khả năng ghi nhớ dữ liệu.

In [ ]:
# ============================================================
# CELL 8 — CGAN SYNTHETIC QUALITY AUDIT
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

# Sửa lỗi NameError: sử dụng biến NEW_TRAIN_DIR đã được định nghĩa ở các cell trước
AUDIT_DIR = os.path.join(
    NEW_TRAIN_DIR, "04_QUALITY_AUDIT"
)
os.makedirs(AUDIT_DIR, exist_ok=True)

quality_rows = []
distribution_data = {}
similarity_data = {}

print("=" * 75)
print("CELL 8 — CGAN SYNTHETIC QUALITY AUDIT")
print("=" * 75)

for floor in FLOORS:

    print(f"\n{'='*65}")
    print(f"QUALITY AUDIT — {floor}")
    print(f"{'='*65}")

    # --------------------------------------------------------
    # 1. ORIGINAL TRAIN
    # --------------------------------------------------------
    original_df = prepared_data[floor]["TRAIN"].copy()

    original_rssi = original_df[AP_COLS].apply(
        pd.to_numeric, errors="coerce"
    )

    # --------------------------------------------------------
    # 2. CGAN SYNTHETIC
    # --------------------------------------------------------
    synthetic_df = CGAN_SYNTHETIC[floor].copy()

    synthetic_rssi = synthetic_df[AP_COLS].apply(
        pd.to_numeric, errors="coerce"
    )

    original_mask = original_rssi.notna()
    synthetic_mask = synthetic_rssi.notna()

    # --------------------------------------------------------
    # 3. OBSERVED AP STATISTICS
    # --------------------------------------------------------
    original_ap = original_mask.sum(axis=1)
    synthetic_ap = synthetic_mask.sum(axis=1)

    # --------------------------------------------------------
    # 4. RSSI DISTRIBUTION
    # --------------------------------------------------------
    original_values = original_rssi.to_numpy().ravel()
    synthetic_values = synthetic_rssi.to_numpy().ravel()

    original_values = original_values[
        np.isfinite(original_values)
    ]

    synthetic_values = synthetic_values[
        np.isfinite(synthetic_values)
    ]

    # --------------------------------------------------------
    # 5. PER-AP DISTRIBUTION DIFFERENCE
    # --------------------------------------------------------
    ap_mae = []

    for ap in AP_COLS:

        o = original_rssi[ap].dropna().to_numpy()
        s = synthetic_rssi[ap].dropna().to_numpy()

        if len(o) > 0 and len(s) > 0:
            ap_mae.append(abs(np.mean(o) - np.mean(s)))

    mean_ap_mean_difference = (
        float(np.mean(ap_mae)) if ap_mae else np.nan
    )

    # --------------------------------------------------------
    # 6. FINGERPRINT DIVERSITY
    # --------------------------------------------------------
    # Chỉ so sánh các AP cùng được quan sát.
    # Imputation phục vụ phép tính tương đồng, không thay đổi dữ liệu.
    combined = pd.concat(
        [original_rssi, synthetic_rssi],
        ignore_index=True
    )

    combined_mask = combined.notna()

    col_median = combined.median()
    filled = combined.fillna(col_median).fillna(-100)

    # Chuẩn hóa từng AP bằng độ lệch chuẩn của TRAIN gốc.
    original_std = original_rssi.std().replace(0, 1).fillna(1)

    standardized = (
        filled - original_rssi.mean()
    ) / original_std

    n_original = len(original_rssi)
    original_matrix = standardized.iloc[:n_original].to_numpy()
    synthetic_matrix = standardized.iloc[n_original:].to_numpy()

    # Cosine similarity được dùng như chỉ số tham khảo.
    # Không dùng nó thay cho khoảng cách WKNN.
    if len(original_matrix) > 1 and len(synthetic_matrix) > 1:

        synth_similarity = cosine_similarity(
            synthetic_matrix
        )

        np.fill_diagonal(synth_similarity, np.nan)

        upper = synth_similarity[
            np.triu_indices_from(synth_similarity, k=1)
        ]

        upper = upper[np.isfinite(upper)]

        mean_pair_similarity = (
            float(np.mean(upper)) if len(upper) else np.nan
        )

        max_pair_similarity = (
            float(np.max(upper)) if len(upper) else np.nan
        )

        # So sánh từng fingerprint tổng hợp với TRAIN gốc.
        cross_similarity = cosine_similarity(
            synthetic_matrix, original_matrix
        )

        nearest_original_similarity = np.max(
            cross_similarity, axis=1
        )

        mean_nearest_original_similarity = float(
            np.mean(nearest_original_similarity)
        )

    else:
        mean_pair_similarity = np.nan
        max_pair_similarity = np.nan
        mean_nearest_original_similarity = np.nan

    # --------------------------------------------------------
    # 7. EXACT DUPLICATES
    # --------------------------------------------------------
    # So sánh fingerprint theo cả giá trị và missing mask.
    def fingerprint_keys(df):
        keys = []
        for _, row in df[AP_COLS].iterrows():
            values = []
            for value in row:
                if pd.isna(value):
                    values.append("NA")
                else:
                    values.append(round(float(value), 3))
            keys.append(tuple(values))
        return keys

    original_keys = fingerprint_keys(original_rssi)
    synthetic_keys = fingerprint_keys(synthetic_rssi)

    original_key_set = set(original_keys)

    exact_duplicate_count = sum(
        key in original_key_set for key in synthetic_keys
    )

    # --------------------------------------------------------
    # 8. SUMMARY
    # --------------------------------------------------------
    row = {
        "Floor": floor,
        "Original_Count": len(original_rssi),
        "Synthetic_Count": len(synthetic_rssi),

        "Original_Mean_AP": original_ap.mean(),
        "Synthetic_Mean_AP": synthetic_ap.mean(),

        "Original_Mean_RSSI_dBm": (
            np.mean(original_values)
            if len(original_values) else np.nan
        ),

        "Synthetic_Mean_RSSI_dBm": (
            np.mean(synthetic_values)
            if len(synthetic_values) else np.nan
        ),

        "Mean_AP_Mean_Difference_dBm": mean_ap_mean_difference,

        "Synthetic_Mean_Pair_Cosine": mean_pair_similarity,
        "Synthetic_Max_Pair_Cosine": max_pair_similarity,

        "Mean_Nearest_Original_Cosine":
            mean_nearest_original_similarity,

        "Exact_Duplicates_vs_Original":
            exact_duplicate_count,

        "Synthetic_Invalid_AP_Count":
            int((synthetic_ap < MIN_VALID_AP).sum())
    }

    quality_rows.append(row)

    distribution_data[floor] = {
        "original": original_values,
        "synthetic": synthetic_values,
        "original_ap": original_ap.to_numpy(),
        "synthetic_ap": synthetic_ap.to_numpy()
    }

    similarity_data[floor] = {
        "cross_similarity": (
            cross_similarity
            if len(original_matrix) and len(synthetic_matrix)
            else None
        )
    }

    print(f"Original fingerprints : {len(original_rssi)}")
    print(f"Synthetic fingerprints: {len(synthetic_rssi)}")
    print(f"Mean AP — Original    : {original_ap.mean():.2f}")
    print(f"Mean AP — Synthetic   : {synthetic_ap.mean():.2f}")
    print(f"Mean RSSI — Original  : {np.mean(original_values):.2f} dBm")
    print(f"Mean RSSI — Synthetic : {np.mean(synthetic_values):.2f} dBm")
    print(f"Mean AP mean difference: {mean_ap_mean_difference:.3f} dBm")
    print(f"Mean pair cosine      : {mean_pair_similarity:.4f}")
    print(f"Exact duplicates      : {exact_duplicate_count}")

# ------------------------------------------------------------
# 9. QUALITY SUMMARY TABLE
# ------------------------------------------------------------
quality_df = pd.DataFrame(quality_rows)

display(
    quality_df.style
    .format({
        "Original_Mean_AP": "{:.2f}",
        "Synthetic_Mean_AP": "{:.2f}",
        "Original_Mean_RSSI_dBm": "{:.2f}",
        "Synthetic_Mean_RSSI_dBm": "{:.2f}",
        "Mean_AP_Mean_Difference_dBm": "{:.3f}",
        "Synthetic_Mean_Pair_Cosine": "{:.4f}",
        "Synthetic_Max_Pair_Cosine": "{:.4f}",
        "Mean_Nearest_Original_Cosine": "{:.4f}"
    })
    .background_gradient(
        subset=["Mean_AP_Mean_Difference_dBm"],
        cmap="YlOrRd"
    )
)

quality_df.to_csv(
    os.path.join(AUDIT_DIR, "cgan_quality_summary.csv"),
    index=False
)

# ------------------------------------------------------------
# 10. COLORFUL RSSI DISTRIBUTION PLOTS
# ------------------------------------------------------------
fig, axes = plt.subplots(
    1, 3, figsize=(19, 5), constrained_layout=True
)

for ax, floor in zip(axes, FLOORS):

    d = distribution_data[floor]

    sns.kdeplot(
        d["original"],
        ax=ax,
        color="#2563EB",
        fill=True,
        alpha=0.22,
        label="Original TRAIN"
    )

    sns.kdeplot(
        d["synthetic"],
        ax=ax,
        color="#F59E0B",
        fill=True,
        alpha=0.25,
        label="CGAN Synthetic"
    )

    ax.set_title(f"{floor} — RSSI Distribution")
    ax.set_xlabel("RSSI (dBm)")
    ax.set_ylabel("Density")
    ax.grid(alpha=0.2)
    ax.legend()

plt.savefig(
    os.path.join(AUDIT_DIR, "rssi_distribution_comparison.png"),
    dpi=180,
    bbox_inches="tight"
)
plt.show()

# ------------------------------------------------------------
# 11. OBSERVED AP COMPARISON
# ------------------------------------------------------------
fig, axes = plt.subplots(
    1, 3, figsize=(17, 5), constrained_layout=True
)

for ax, floor in zip(axes, FLOORS):

    d = distribution_data[floor]

    ax.boxplot(
        [d["original_ap"], d["synthetic_ap"]],
        tick_labels=["Original", "CGAN"],
        patch_artist=True,
        boxprops=dict(facecolor="#93C5FD"),
        medianprops=dict(color="#EA580C", linewidth=2)
    )

    ax.set_title(f"{floor} — Observed AP Count")
    ax.set_ylabel("Number of observed APs")
    ax.grid(axis="y", alpha=0.2)

plt.savefig(
    os.path.join(AUDIT_DIR, "observed_ap_comparison.png"),
    dpi=180,
    bbox_inches="tight"
)
plt.show()

# ------------------------------------------------------------
# 12. CROSS-SIMILARITY HEATMAP
# ------------------------------------------------------------
fig, axes = plt.subplots(
    1, 3, figsize=(19, 5), constrained_layout=True
)

for ax, floor in zip(axes, FLOORS):

    matrix = similarity_data[floor]["cross_similarity"]

    if matrix is not None:
        sns.heatmap(
            matrix,
            ax=ax,
            cmap="YlOrRd",
            vmin=0,
            vmax=1,
            cbar=True,
            xticklabels=False,
            yticklabels=False
        )

    ax.set_title(f"{floor} — Synthetic vs Original Similarity")
    ax.set_xlabel("Original TRAIN fingerprints")
    ax.set_ylabel("Synthetic fingerprints")

plt.savefig(
    os.path.join(AUDIT_DIR, "cross_similarity_heatmap.png"),
    dpi=180,
    bbox_inches="tight"
)
plt.show()

print("\n" + "=" * 75)
print("CELL 8 — QUALITY AUDIT COMPLETED")
print("Output:", AUDIT_DIR)
print("=" * 75)

In [ ]:

# ============================================================
# CELL 8B — CONDITIONAL DIVERSITY & MEMORIZATION AUDIT
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from IPython.display import display

rows = []
per_floor_results = {}

for floor in FLOORS:

    original = prepared_data[floor]["TRAIN"].copy()
    synthetic = CGAN_SYNTHETIC[floor].copy()

    # Ghép theo ID_RP để đảm bảo so sánh cùng tọa độ.
    original["ID_RP"] = original["ID_RP"].astype(str)
    synthetic["ID_RP"] = synthetic["ID_RP"].astype(str)

    merged = original.merge(
        synthetic,
        on="ID_RP",
        suffixes=("_original", "_synthetic"),
        validate="one_to_one"
    )

    per_rp_mae = []
    per_rp_common_ap = []

    for _, row in merged.iterrows():

        original_values = np.array([
            row[f"{ap}_original"] for ap in AP_COLS
        ], dtype=float)

        synthetic_values = np.array([
            row[f"{ap}_synthetic"] for ap in AP_COLS
        ], dtype=float)

        common = (
            np.isfinite(original_values)
            & np.isfinite(synthetic_values)
        )

        if common.sum() >= 3:

            mae = np.mean(
                np.abs(
                    original_values[common]
                    - synthetic_values[common]
                )
            )

            per_rp_mae.append(mae)
            per_rp_common_ap.append(common.sum())

    # Biến thiên theo AP trong tập tổng hợp.
    synth_values = synthetic[AP_COLS].to_numpy(dtype=float)

    per_ap_std = np.nanstd(synth_values, axis=0)
    per_ap_mean_std = np.nanmean(per_ap_std)

    # Độ biến thiên fingerprint giữa các RP.
    # Tính trên các giá trị RSSI quan sát được.
    per_rp_std = np.nanstd(synth_values, axis=1)

    result = {
        "Floor": floor,
        "Matched_RP": len(merged),
        "Mean_MAE_same_RP_dBm": np.mean(per_rp_mae),
        "Median_MAE_same_RP_dBm": np.median(per_rp_mae),
        "Mean_Common_AP": np.mean(per_rp_common_ap),
        "Mean_AP_Std_across_RP_dBm": per_ap_mean_std,
        "Mean_RP_Std_across_AP_dBm": np.nanmean(per_rp_std),
        "AP_with_zero_variance": int(
            np.sum(np.nan_to_num(per_ap_std) == 0)
        )
    }

    rows.append(result)

    per_floor_results[floor] = {
        "per_rp_mae": np.array(per_rp_mae),
        "per_ap_std": per_ap_std,
        "per_rp_std": per_rp_std
    }

    print(f"\n{floor}")
    print(f"Matched RP: {len(merged)}")
    print(
        "Mean MAE against same-RP TRAIN:",
        f"{np.mean(per_rp_mae):.3f} dBm"
    )
    print(
        "Mean AP standard deviation:",
        f"{per_ap_mean_std:.3f} dBm"
    )

diversity_df = pd.DataFrame(rows)

display(
    diversity_df.style.format({
        "Mean_MAE_same_RP_dBm": "{:.3f}",
        "Median_MAE_same_RP_dBm": "{:.3f}",
        "Mean_Common_AP": "{:.2f}",
        "Mean_AP_Std_across_RP_dBm": "{:.3f}",
        "Mean_RP_Std_across_AP_dBm": "{:.3f}"
    })
)

diversity_df.to_csv(
    os.path.join(AUDIT_DIR, "conditional_diversity_audit.csv"),
    index=False
)

# ------------------------------------------------------------
# Visualize per-RP reconstruction difference
# ------------------------------------------------------------
fig, axes = plt.subplots(
    1, 3, figsize=(17, 5), constrained_layout=True
)

for ax, floor in zip(axes, FLOORS):

    values = per_floor_results[floor]["per_rp_mae"]

    ax.hist(
        values,
        bins=20,
        color="#8B5CF6",
        edgecolor="white",
        alpha=0.85
    )

    ax.axvline(
        np.mean(values),
        color="#F97316",
        linestyle="--",
        linewidth=2,
        label=f"Mean = {np.mean(values):.2f} dBm"
    )

    ax.set_title(f"{floor} — Same-RP RSSI MAE")
    ax.set_xlabel("MAE (dBm)")
    ax.set_ylabel("Number of RPs")
    ax.legend()
    ax.grid(alpha=0.2)

plt.savefig(
    os.path.join(AUDIT_DIR, "same_rp_mae_distribution.png"),
    dpi=180,
    bbox_inches="tight"
)
plt.show()

print("\nCELL 8B COMPLETED")


In [ ]:

# ============================================================
# CELL 8C — CGAN MULTI-NOISE DIVERSITY TEST
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

N_NOISE_DRAWS = 20

# RSSI standard deviation below this threshold is considered
# near-zero variation for this diagnostic only.
NEAR_ZERO_STD_THRESHOLD = 0.10  # dBm

DIVERSITY_DIR = os.path.join(
    str(NEW_TRAIN_DIR),
    "04_QUALITY_AUDIT",
    "MULTI_NOISE_DIVERSITY"
)

os.makedirs(DIVERSITY_DIR, exist_ok=True)

CGAN_MULTI_NOISE = {}

summary_rows = []
rp_detail_rows = []

print("=" * 75)
print("CELL 8C — CGAN MULTI-NOISE DIVERSITY TEST")
print("Noise draws per RP:", N_NOISE_DRAWS)
print("=" * 75)


# ------------------------------------------------------------
# 2. GENERATE MULTIPLE SAMPLES AT FIXED COORDINATES
# ------------------------------------------------------------

for floor in FLOORS:

    print("\n" + "=" * 65)
    print(f"MULTI-NOISE TEST — {floor}")
    print("=" * 65)

    G = cgan_models[floor]["G"]
    norm = cgan_data[floor]["NORMALIZATION"]

    train_frame = (
        cgan_data[floor]["TRAIN_FRAME"]
        .copy()
        .reset_index(drop=True)
    )

    G.eval()

    n_rp = len(train_frame)
    n_ap = len(AP_COLS)

    # --------------------------------------------------------
    # Fixed coordinates: every RP is repeated 20 times
    # --------------------------------------------------------

    xy = train_frame[["X", "Y"]].to_numpy(dtype=float)

    xy_scaled = (
        2 * (xy - norm["xy_min"])
        / norm["xy_range"]
    ) - 1

    # Each RP coordinate is repeated N_NOISE_DRAWS times.
    xy_repeated = np.repeat(
        xy_scaled,
        N_NOISE_DRAWS,
        axis=0
    )

    coords_tensor = torch.tensor(
        xy_repeated,
        dtype=torch.float32,
        device=DEVICE
    )

    total_samples = n_rp * N_NOISE_DRAWS

    # --------------------------------------------------------
    # Independent noise vectors
    # --------------------------------------------------------

    noise_generator = torch.Generator(device=DEVICE)

    floor_seed = (
        NEW_SEED
        + 5000
        + FLOORS.index(floor)
    )

    noise_generator.manual_seed(floor_seed)

    noise = torch.randn(
        total_samples,
        CGAN_NOISE_DIM,
        generator=noise_generator,
        device=DEVICE
    )

    # --------------------------------------------------------
    # Generate RSSI and mask probabilities
    # --------------------------------------------------------

    generated_rssi = []
    generated_mask_prob = []

    with torch.no_grad():

        for start in range(0, total_samples, 128):

            end = min(start + 128, total_samples)

            fake_rssi, fake_mask_prob = G(
                coords_tensor[start:end],
                noise[start:end]
            )

            generated_rssi.append(
                fake_rssi.cpu().numpy()
            )

            generated_mask_prob.append(
                fake_mask_prob.cpu().numpy()
            )

    fake_rssi_scaled = np.concatenate(
        generated_rssi,
        axis=0
    )

    fake_mask_prob = np.concatenate(
        generated_mask_prob,
        axis=0
    )

    # --------------------------------------------------------
    # Inverse transform using EXACT Cell 7 normalization
    # --------------------------------------------------------

    fake_rssi_dbm = (
        (fake_rssi_scaled + 1) / 2
    ) * norm["rssi_range"] + norm["rssi_min"]

    # Shape:
    # [number of RP, number of noise draws, number of AP]
    rssi_3d = fake_rssi_dbm.reshape(
        n_rp,
        N_NOISE_DRAWS,
        n_ap
    )

    mask_prob_3d = fake_mask_prob.reshape(
        n_rp,
        N_NOISE_DRAWS,
        n_ap
    )

    # --------------------------------------------------------
    # 3. ORIGINAL OBSERVATION MASK
    # --------------------------------------------------------

    original_rssi = train_frame[AP_COLS].apply(
        pd.to_numeric,
        errors="coerce"
    ).to_numpy(dtype=float)

    original_observed_mask = np.isfinite(
        original_rssi
    )

    # --------------------------------------------------------
    # 4. RSSI DIVERSITY ACROSS NOISE DRAWS
    # --------------------------------------------------------

    # Calculate standard deviation across 20 noise draws.
    # Shape: [RP, AP]
    rssi_std = np.std(
        rssi_3d,
        axis=1,
        ddof=1
    )

    # Only evaluate APs observed in original TRAIN at that RP.
    valid_std = rssi_std[original_observed_mask]

    valid_std = valid_std[
        np.isfinite(valid_std)
    ]

    if len(valid_std) == 0:
        raise ValueError(
            f"{floor}: No valid RSSI values for diversity audit."
        )

    mean_rssi_std = float(np.mean(valid_std))
    median_rssi_std = float(np.median(valid_std))

    near_zero_count = int(
        np.sum(valid_std < NEAR_ZERO_STD_THRESHOLD)
    )

    near_zero_ratio = (
        near_zero_count / len(valid_std) * 100
    )

    # --------------------------------------------------------
    # 5. PAIRWISE RSSI MAE BETWEEN NOISE DRAWS
    # --------------------------------------------------------

    pairwise_mae_per_rp = []

    for rp_idx in range(n_rp):

        observed_aps = original_observed_mask[rp_idx]

        if observed_aps.sum() < MIN_VALID_AP:
            continue

        # Sửa lỗi slice mảng 3D để giữ nguyên cấu trúc (N_NOISE_DRAWS, valid_aps)
        rp_samples = rssi_3d[rp_idx][:, observed_aps]

        pair_mae = []

        for i in range(N_NOISE_DRAWS):

            for j in range(i + 1, N_NOISE_DRAWS):

                mae = np.mean(
                    np.abs(
                        rp_samples[i] - rp_samples[j]
                    )
                )

                pair_mae.append(mae)

        pairwise_mae_per_rp.append(
            float(np.mean(pair_mae))
        )

    mean_pairwise_mae = float(
        np.mean(pairwise_mae_per_rp)
    )

    # --------------------------------------------------------
    # 6. MASK PROBABILITY DIVERSITY
    # --------------------------------------------------------

    mask_prob_std = np.std(
        mask_prob_3d,
        axis=1,
        ddof=1
    )

    mask_prob_valid = mask_prob_std[
        original_observed_mask
    ]

    mask_prob_valid = mask_prob_valid[
        np.isfinite(mask_prob_valid)
    ]

    mean_mask_prob_std = float(
        np.mean(mask_prob_valid)
    )

    # --------------------------------------------------------
    # 7. PER-RP DETAILS
    # --------------------------------------------------------

    per_rp_rssi_std = []

    for rp_idx in range(n_rp):

        observed_aps = original_observed_mask[rp_idx]

        if observed_aps.sum() > 0:

            rp_std = float(
                np.mean(rssi_std[rp_idx, observed_aps])
            )

            rp_mask_std = float(
                np.mean(
                    mask_prob_std[rp_idx, observed_aps]
                )
            )

        else:
            rp_std = np.nan
            rp_mask_std = np.nan

        per_rp_rssi_std.append(rp_std)

        rp_detail_rows.append({
            "Floor": floor,
            "ID_RP": str(train_frame.iloc[rp_idx]["ID_RP"]),
            "X": float(xy[rp_idx, 0]),
            "Y": float(xy[rp_idx, 1]),
            "N_Original_Observed_AP": int(
                original_observed_mask[rp_idx].sum()
            ),
            "Mean_RSSI_Std_dBm": rp_std,
            "Mean_Mask_Probability_Std": rp_mask_std
        })

    # --------------------------------------------------------
    # 8. SAVE GENERATED TENSORS
    # --------------------------------------------------------

    CGAN_MULTI_NOISE[floor] = {
        "RSSI": rssi_3d,
        "MASK_PROB": mask_prob_3d,
        "RSSI_STD": rssi_std,
        "MASK_PROB_STD": mask_prob_std,
        "ORIGINAL_MASK": original_observed_mask,
        "RP_IDS": train_frame["ID_RP"].astype(str).to_numpy()
    }

    # --------------------------------------------------------
    # 9. SUMMARY
    # --------------------------------------------------------

    summary_rows.append({
        "Floor": floor,
        "N_RP": n_rp,
        "Noise_Draws_Per_RP": N_NOISE_DRAWS,
        "Total_Generated_Samples": total_samples,
        "Mean_RSSI_Std_dBm": mean_rssi_std,
        "Median_RSSI_Std_dBm": median_rssi_std,
        "Near_Zero_AP_Ratio_pct": near_zero_ratio,
        "Mean_Pairwise_RSSI_MAE_dBm": mean_pairwise_mae,
        "Mean_Mask_Probability_Std": mean_mask_prob_std
    })

    print("Number of RPs:", n_rp)
    print("Generated samples:", total_samples)
    print(
        f"Mean RSSI std: {mean_rssi_std:.4f} dBm"
    )
    print(
        f"Median RSSI std: {median_rssi_std:.4f} dBm"
    )
    print(
        f"Near-zero variation: {near_zero_ratio:.2f}%"
    )
    print(
        f"Mean pairwise RSSI MAE: {mean_pairwise_mae:.4f} dBm"
    )
    print(
        f"Mean mask probability std: {mean_mask_prob_std:.5f}"
    )


# ------------------------------------------------------------
# 10. SUMMARY TABLE
# ------------------------------------------------------------

diversity_summary_df = pd.DataFrame(
    summary_rows
)

display(
    diversity_summary_df.style.format({
        "Mean_RSSI_Std_dBm": "{:.4f}",
        "Median_RSSI_Std_dBm": "{:.4f}",
        "Near_Zero_AP_Ratio_pct": "{:.2f}",
        "Mean_Pairwise_RSSI_MAE_dBm": "{:.4f}",
        "Mean_Mask_Probability_Std": "{:.5f}"
    }).background_gradient(
        subset=["Mean_RSSI_Std_dBm"],
        cmap="YlGnBu"
    )
)

diversity_summary_df.to_csv(
    os.path.join(
        DIVERSITY_DIR,
        "multi_noise_diversity_summary.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# 11. PER-RP DETAIL TABLE
# ------------------------------------------------------------

rp_diversity_df = pd.DataFrame(
    rp_detail_rows
)

rp_diversity_df.to_csv(
    os.path.join(
        DIVERSITY_DIR,
        "multi_noise_diversity_per_rp.csv"
    ),
    index=False
)

print("\nPer-RP detail:")
display(rp_diversity_df.head(10))


# ------------------------------------------------------------
# 12. RSSI DIVERSITY BOXPLOT
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 5),
    constrained_layout=True
)

for ax, floor in zip(axes, FLOORS):

    values = rp_diversity_df.loc[
        rp_diversity_df["Floor"] == floor,
        "Mean_RSSI_Std_dBm"
    ].dropna()

    ax.boxplot(
        values,
        patch_artist=True,
        boxprops=dict(
            facecolor="#93C5FD",
            alpha=0.85
        ),
        medianprops=dict(
            color="#EA580C",
            linewidth=2
        )
    )

    ax.set_title(
        f"{floor} — RSSI Diversity Across Noise"
    )
    ax.set_ylabel("RSSI standard deviation (dBm)")
    ax.set_xticks([1])
    ax.set_xticklabels(["CGAN"])
    ax.grid(axis="y", alpha=0.25)

plt.savefig(
    os.path.join(
        DIVERSITY_DIR,
        "multi_noise_rssi_diversity.png"
    ),
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 13. MASK PROBABILITY DIVERSITY
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 5),
    constrained_layout=True
)

for ax, floor in zip(axes, FLOORS):

    values = rp_diversity_df.loc[
        rp_diversity_df["Floor"] == floor,
        "Mean_Mask_Probability_Std"
    ].dropna()

    ax.hist(
        values,
        bins=20,
        color="#A78BFA",
        edgecolor="white",
        alpha=0.9
    )

    ax.set_title(
        f"{floor} — Mask Probability Diversity"
    )
    ax.set_xlabel("Mask probability standard deviation")
    ax.set_ylabel("Number of RPs")
    ax.grid(alpha=0.2)

plt.savefig(
    os.path.join(
        DIVERSITY_DIR,
        "multi_noise_mask_probability_diversity.png"
    ),
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 14. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("CELL 8C — MULTI-NOISE DIVERSITY TEST COMPLETED")
print("=" * 75)

print("Floors evaluated:", len(diversity_summary_df))
print(
    "Total RP-noise samples:",
    int(diversity_summary_df["Total_Generated_Samples"].sum())
)

print("Output directory:", DIVERSITY_DIR)


# CELL 9 — TRADITIONAL SPATIAL DATA AUGMENTATION

## 1. Mục tiêu
Xây dựng dữ liệu fingerprint tổng hợp bằng hai nhóm phương pháp truyền thống:
- Spatial KNN: nội suy RSSI từ các RP gần nhất.
- APMEAN: tính trung bình RSSI trong vùng lân cận không gian.

## 2. Các phương pháp
**Spatial KNN**
- KNN_K3
- KNN_K5
- KNN_K7

**APMEAN**
- APMEAN_R3
- APMEAN_R5
- APMEAN_R7

## 3. Nguyên tắc sinh dữ liệu
- Chỉ sử dụng các RP thuộc TRAIN.
- Mỗi RP TRAIN tạo một fingerprint tổng hợp cho mỗi phương pháp.
- Loại bỏ RP đích khỏi danh sách lân cận.
- KNN lựa chọn k RP gần nhất theo khoảng cách tọa độ Euclidean.
- APMEAN sử dụng các RP nằm trong bán kính R tính từ RP đích.
- Mỗi AP được tính trung bình từ các giá trị quan sát hợp lệ của các RP lân cận.
- Nếu không có dữ liệu AP hợp lệ, giữ giá trị NaN.
- Không sử dụng VALIDATION hoặc TEST.

## 4. Đầu ra
- Sáu tập fingerprint tổng hợp theo từng tầng.
- Bảng thống kê số lượng ứng viên hợp lệ và không hợp lệ.
- Thông tin RP lân cận được sử dụng để tạo từng fingerprint.
- Biểu đồ phân bố vị trí và số lượng ứng viên theo phương pháp.

In [ ]:

# ============================================================
# CELL 9 — TRADITIONAL SPATIAL DATA AUGMENTATION
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

KNN_VALUES = [3, 5, 7]
APMEAN_RADII = [3, 5, 7]

TRADITIONAL_DIR = (
    NEW_TRAIN_DIR / "03_SYNTHETIC" / "TRADITIONAL"
)

TRADITIONAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Dictionary lưu dữ liệu tổng hợp.
# Cấu trúc:
# TRADITIONAL_SYNTHETIC[floor][method] = DataFrame

TRADITIONAL_SYNTHETIC = {}

traditional_summary_rows = []

print("=" * 75)
print("CELL 9 — TRADITIONAL SPATIAL DATA AUGMENTATION")
print("=" * 75)

print("KNN methods:", KNN_VALUES)
print("APMEAN radii:", APMEAN_RADII)


# ------------------------------------------------------------
# 2. HELPER FUNCTION: RSSI MEAN WITH MISSING VALUES
# ------------------------------------------------------------

def calculate_observed_mean(rssi_matrix, neighbor_indices):

    """
    Tính trung bình RSSI theo từng AP.

    Chỉ sử dụng các giá trị quan sát hợp lệ.
    Nếu một AP không có giá trị hợp lệ trong nhóm lân cận,
    kết quả của AP đó được giữ là NaN.
    """

    if len(neighbor_indices) == 0:

        return np.full(
            rssi_matrix.shape[1],
            np.nan,
            dtype=float
        )

    neighbor_values = rssi_matrix[
        neighbor_indices
    ]

    valid_mask = np.isfinite(neighbor_values)

    valid_count = valid_mask.sum(axis=0)

    valid_sum = np.where(
        valid_mask,
        neighbor_values,
        0.0
    ).sum(axis=0)

    result = np.divide(
        valid_sum,
        valid_count,
        out=np.full(
            rssi_matrix.shape[1],
            np.nan,
            dtype=float
        ),
        where=valid_count > 0
    )

    return result


# ------------------------------------------------------------
# 3. MAIN GENERATION
# ------------------------------------------------------------

for floor in FLOORS:

    print("\n" + "=" * 70)
    print(f"TRADITIONAL GENERATION — {floor}")
    print("=" * 70)

    # --------------------------------------------------------
    # 3.1 Load TRAIN fingerprints only
    # --------------------------------------------------------

    train_frame = (
        cgan_data[floor]["TRAIN_FRAME"]
        .copy()
        .reset_index(drop=True)
    )

    required_columns = [
        "ID_RP", "X", "Y"
    ] + list(AP_COLS)

    missing_columns = [
        col for col in required_columns
        if col not in train_frame.columns
    ]

    if missing_columns:

        raise ValueError(
            f"{floor}: Missing columns: {missing_columns}"
        )

    train_frame["ID_RP"] = (
        train_frame["ID_RP"].astype(str)
    )

    coordinates = train_frame[
        ["X", "Y"]
    ].to_numpy(dtype=float)

    # RSSI matrix
    rssi_matrix = train_frame[
        AP_COLS
    ].apply(
        pd.to_numeric,
        errors="coerce"
    ).to_numpy(dtype=float)

    # Treat invalid sentinels as missing observations.
    rssi_matrix[
        (rssi_matrix == -100)
        | (rssi_matrix == 100)
    ] = np.nan

    n_rp = len(train_frame)

    if n_rp < 2:

        raise ValueError(
            f"{floor}: Not enough TRAIN RPs."
        )

    # --------------------------------------------------------
    # 3.2 Spatial distance matrix
    # --------------------------------------------------------

    difference = (
        coordinates[:, None, :]
        - coordinates[None, :, :]
    )

    distance_matrix = np.sqrt(
        np.sum(difference ** 2, axis=2)
    )

    # Exclude the target RP itself.
    np.fill_diagonal(
        distance_matrix,
        np.inf
    )

    # Store results for this floor.
    TRADITIONAL_SYNTHETIC[floor] = {}


    # ========================================================
    # 4. SPATIAL KNN METHODS
    # ========================================================

    for k in KNN_VALUES:

        method = f"KNN_K{k}"

        print(f"\nGenerating {method}...")

        synthetic_rows = []

        for target_idx in range(n_rp):

            # Find k closest OTHER RPs.
            neighbor_indices = np.argsort(
                distance_matrix[target_idx]
            )[:k]

            neighbor_distances = (
                distance_matrix[
                    target_idx,
                    neighbor_indices
                ]
            )

            # Mean RSSI across spatial neighbors.
            synthetic_rssi = calculate_observed_mean(
                rssi_matrix,
                neighbor_indices
            )

            observed_count = int(
                np.isfinite(synthetic_rssi).sum()
            )

            target = train_frame.iloc[target_idx]

            synthetic_rows.append({

                "Floor": floor,

                "ID_RP": str(target["ID_RP"]),

                "Original_RP_ID": str(
                    target["ID_RP"]
                ),

                "X": float(target["X"]),
                "Y": float(target["Y"]),

                **{
                    ap: synthetic_rssi[j]
                    for j, ap in enumerate(AP_COLS)
                },

                "Source": method,

                "Method": method,

                "Synthetic_ID": (
                    f"{floor}_{method}_{target_idx+1:04d}"
                ),

                "N_Observed_AP": observed_count,

                "Valid_Fingerprint": (
                    observed_count >= MIN_VALID_AP
                ),

                "Neighbor_Count": len(
                    neighbor_indices
                ),

                "Neighbor_RPs": "|".join(
                    train_frame.iloc[
                        neighbor_indices
                    ]["ID_RP"].astype(str).tolist()
                ),

                "Mean_Neighbor_Distance": float(
                    np.mean(neighbor_distances)
                )
            })

        synthetic_df = pd.DataFrame(
            synthetic_rows
        )

        TRADITIONAL_SYNTHETIC[floor][method] = (
            synthetic_df.copy()
        )

        # Save
        synthetic_df.to_csv(
            TRADITIONAL_DIR /
            f"{method}_{floor}.csv",
            index=False
        )

        valid_count = int(
            synthetic_df["Valid_Fingerprint"].sum()
        )

        invalid_count = (
            len(synthetic_df) - valid_count
        )

        traditional_summary_rows.append({

            "Floor": floor,
            "Method": method,
            "Method_Group": "Spatial KNN",

            "Original_TRAIN_RP": n_rp,

            "Synthetic_Candidates": len(
                synthetic_df
            ),

            "Valid_Candidates": valid_count,

            "Invalid_Candidates": invalid_count,

            "Mean_Observed_AP": synthetic_df[
                "N_Observed_AP"
            ].mean(),

            "Mean_RSSI_dBm": np.nanmean(
                synthetic_df[AP_COLS].to_numpy(float)
            ),

            "Mean_Neighbor_Count": synthetic_df[
                "Neighbor_Count"
            ].mean(),

            "Generation_Status": "COMPLETED"
        })

        print(
            f"{method}: {len(synthetic_df)} candidates | "
            f"{valid_count} valid | "
            f"{invalid_count} invalid"
        )


    # ========================================================
    # 5. APMEAN METHODS
    # ========================================================

    for radius in APMEAN_RADII:

        method = f"APMEAN_R{radius}"

        print(f"\nGenerating {method}...")

        synthetic_rows = []

        for target_idx in range(n_rp):

            # All OTHER RPs inside the spatial radius.
            neighbor_indices = np.where(
                distance_matrix[target_idx] <= radius
            )[0]

            # Mean RSSI over local neighbors.
            synthetic_rssi = calculate_observed_mean(
                rssi_matrix,
                neighbor_indices
            )

            observed_count = int(
                np.isfinite(synthetic_rssi).sum()
            )

            target = train_frame.iloc[target_idx]

            # If no neighbors are found, RSSI stays NaN.
            if len(neighbor_indices) > 0:

                mean_neighbor_distance = float(
                    np.mean(
                        distance_matrix[
                            target_idx,
                            neighbor_indices
                        ]
                    )
                )

            else:
                mean_neighbor_distance = np.nan

            synthetic_rows.append({

                "Floor": floor,

                "ID_RP": str(target["ID_RP"]),

                "Original_RP_ID": str(
                    target["ID_RP"]
                ),

                "X": float(target["X"]),
                "Y": float(target["Y"]),

                **{
                    ap: synthetic_rssi[j]
                    for j, ap in enumerate(AP_COLS)
                },

                "Source": method,

                "Method": method,

                "Synthetic_ID": (
                    f"{floor}_{method}_{target_idx+1:04d}"
                ),

                "N_Observed_AP": observed_count,

                "Valid_Fingerprint": (
                    observed_count >= MIN_VALID_AP
                ),

                "Neighbor_Count": len(
                    neighbor_indices
                ),

                "Neighbor_RPs": "|".join(
                    train_frame.iloc[
                        neighbor_indices
                    ]["ID_RP"].astype(str).tolist()
                ),

                "Mean_Neighbor_Distance":
                    mean_neighbor_distance
            })

        synthetic_df = pd.DataFrame(
            synthetic_rows
        )

        TRADITIONAL_SYNTHETIC[floor][method] = (
            synthetic_df.copy()
        )

        # Save
        synthetic_df.to_csv(
            TRADITIONAL_DIR /
            f"{method}_{floor}.csv",
            index=False
        )

        valid_count = int(
            synthetic_df["Valid_Fingerprint"].sum()
        )

        invalid_count = (
            len(synthetic_df) - valid_count
        )

        traditional_summary_rows.append({

            "Floor": floor,
            "Method": method,
            "Method_Group": "Spatial APMEAN",

            "Original_TRAIN_RP": n_rp,

            "Synthetic_Candidates": len(
                synthetic_df
            ),

            "Valid_Candidates": valid_count,

            "Invalid_Candidates": invalid_count,

            "Mean_Observed_AP": synthetic_df[
                "N_Observed_AP"
            ].mean(),

            "Mean_RSSI_dBm": np.nanmean(
                synthetic_df[AP_COLS].to_numpy(float)
            ),

            "Mean_Neighbor_Count": synthetic_df[
                "Neighbor_Count"
            ].mean(),

            "Generation_Status": "COMPLETED"
        })

        print(
            f"{method}: {len(synthetic_df)} candidates | "
            f"{valid_count} valid | "
            f"{invalid_count} invalid"
        )


# ------------------------------------------------------------
# 6. GENERATION SUMMARY
# ------------------------------------------------------------

traditional_summary_df = pd.DataFrame(
    traditional_summary_rows
)

display(
    traditional_summary_df.style.format({

        "Mean_Observed_AP": "{:.2f}",

        "Mean_RSSI_dBm": "{:.2f}",

        "Mean_Neighbor_Count": "{:.2f}"
    }).background_gradient(
        subset=["Valid_Candidates"],
        cmap="YlGnBu"
    ).background_gradient(
        subset=["Invalid_Candidates"],
        cmap="YlOrRd"
    )
)

traditional_summary_df.to_csv(
    TRADITIONAL_DIR /
    "traditional_generation_summary.csv",
    index=False
)


# ------------------------------------------------------------
# 7. VISUALIZATION — VALID CANDIDATES
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(18, 5),
    constrained_layout=True
)

method_colors = {

    "KNN_K3": "#2563EB",
    "KNN_K5": "#06B6D4",
    "KNN_K7": "#8B5CF6",

    "APMEAN_R3": "#F59E0B",
    "APMEAN_R5": "#F97316",
    "APMEAN_R7": "#EC4899"
}

for ax, floor in zip(axes, FLOORS):

    floor_summary = traditional_summary_df[
        traditional_summary_df["Floor"] == floor
    ].copy()

    ax.bar(
        floor_summary["Method"],
        floor_summary["Valid_Candidates"],
        color=[
            method_colors[m]
            for m in floor_summary["Method"]
        ],
        edgecolor="white"
    )

    ax.set_title(
        f"{floor} — Valid Synthetic Fingerprints",
        fontweight="bold"
    )

    ax.set_ylabel("Valid candidate count")

    ax.tick_params(
        axis="x",
        rotation=45
    )

    ax.grid(
        axis="y",
        alpha=0.2
    )

plt.savefig(
    NEW_TRAIN_DIR / "07_FIGURES" /
    "traditional_valid_candidate_comparison.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 8. VISUALIZATION — SPATIAL SYNTHETIC LOCATIONS
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(18, 5),
    constrained_layout=True
)

for ax, floor in zip(axes, FLOORS):

    original = cgan_data[floor]["TRAIN_FRAME"]

    ax.scatter(
        original["X"],
        original["Y"],
        s=18,
        color="#64748B",
        alpha=0.35,
        label="Original TRAIN"
    )

    for method in method_colors:

        synthetic = TRADITIONAL_SYNTHETIC[
            floor
        ][method]

        valid = synthetic[
            synthetic["Valid_Fingerprint"]
        ]

        ax.scatter(
            valid["X"],
            valid["Y"],
            s=24,
            marker="x",
            color=method_colors[method],
            alpha=0.7,
            label=method
        )

    ax.set_title(
        f"{floor} — Traditional Synthetic RP",
        fontweight="bold"
    )

    ax.set_xlabel("X (source coordinate)")
    ax.set_ylabel("Y (source coordinate)")

    ax.grid(alpha=0.2)

    ax.legend(
        fontsize=7,
        frameon=False,
        ncol=2
    )

plt.savefig(
    NEW_TRAIN_DIR / "07_FIGURES" /
    "traditional_synthetic_spatial_map.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 9. FINAL CHECKS
# ------------------------------------------------------------

expected_methods = [
    "KNN_K3",
    "KNN_K5",
    "KNN_K7",
    "APMEAN_R3",
    "APMEAN_R5",
    "APMEAN_R7"
]

for floor in FLOORS:

    actual_methods = list(
        TRADITIONAL_SYNTHETIC[floor].keys()
    )

    assert actual_methods == expected_methods, (
        f"{floor}: Method list mismatch."
    )

    for method in expected_methods:

        df = TRADITIONAL_SYNTHETIC[
            floor
        ][method]

        assert len(df) == len(
            cgan_data[floor]["TRAIN_FRAME"]
        ), (
            f"{floor} {method}: Candidate count mismatch."
        )

        assert df["Synthetic_ID"].is_unique

        assert (
            df["Original_RP_ID"].astype(str).values
            ==
            cgan_data[floor]["TRAIN_FRAME"][
                "ID_RP"
            ].astype(str).values
        ).all()

print("\n" + "=" * 75)
print("CELL 9 — TRADITIONAL GENERATION COMPLETED")
print("=" * 75)

print("Floors:", len(FLOORS))
print("Methods per floor:", len(expected_methods))
print("Total traditional method-floor datasets:",
      len(FLOORS) * len(expected_methods))

print(
    "Total synthetic candidates:",
    traditional_summary_df[
        "Synthetic_Candidates"
    ].sum()
)

print("Output:", TRADITIONAL_DIR)


# CELL 10 — UNIFIED SYNTHETIC QUALITY AUDIT

## 1. Mục tiêu
Đánh giá và so sánh chất lượng dữ liệu được tạo bởi 7 phương pháp tăng cường so với TRAIN gốc.

## 2. Phương pháp đánh giá
- CGAN
- KNN_K3, KNN_K5, KNN_K7
- APMEAN_R3, APMEAN_R5, APMEAN_R7

## 3. Các tiêu chí
1. Số lượng fingerprint hợp lệ.
2. Số AP quan sát trung bình.
3. RSSI trung bình.
4. MAE RSSI giữa fingerprint tổng hợp và TRAIN tại cùng RP.
5. Sai khác trung bình RSSI theo từng AP.
6. Độ biến thiên RSSI giữa các RP.
7. Số fingerprint trùng khớp chính xác với TRAIN.
8. Số RP lân cận trung bình được sử dụng.

## 4. Nguyên tắc
- Chỉ sử dụng TRAIN và dữ liệu tổng hợp.
- Ghép theo ID_RP để so sánh cùng tọa độ.
- Chỉ tính MAE trên những AP cùng được quan sát.
- Không sử dụng VALIDATION hoặc TEST.
- Không loại bỏ phương pháp chỉ dựa trên một chỉ số riêng lẻ.

## 5. Đầu ra
- Bảng tổng hợp chất lượng theo tầng và phương pháp.
- Biểu đồ so sánh MAE.
- Biểu đồ độ biến thiên RSSI.
- Biểu đồ số AP quan sát.
- Báo cáo dữ liệu sẵn sàng cho bước WKNN.

In [ ]:

# ============================================================
# CELL 10 — UNIFIED SYNTHETIC QUALITY AUDIT
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

QUALITY_DIR = (
    NEW_TRAIN_DIR / "04_QUALITY_AUDIT" / "UNIFIED_AUDIT"
)

QUALITY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

METHODS = [
    "CGAN",
    "KNN_K3",
    "KNN_K5",
    "KNN_K7",
    "APMEAN_R3",
    "APMEAN_R5",
    "APMEAN_R7"
]

METHOD_COLORS = {
    "CGAN": "#8B5CF6",
    "KNN_K3": "#2563EB",
    "KNN_K5": "#06B6D4",
    "KNN_K7": "#0D9488",
    "APMEAN_R3": "#F59E0B",
    "APMEAN_R5": "#F97316",
    "APMEAN_R7": "#EC4899"
}

summary_rows = []
per_rp_rows = []
per_ap_rows = []

print("=" * 75)
print("CELL 10 — UNIFIED SYNTHETIC QUALITY AUDIT")
print("=" * 75)


# ------------------------------------------------------------
# 2. HELPER: BUILD FINGERPRINT KEYS
# ------------------------------------------------------------

def make_fingerprint_keys(df):

    keys = []

    for _, row in df[AP_COLS].iterrows():

        values = []

        for value in row:

            if pd.isna(value):
                values.append("NA")
            else:
                values.append(round(float(value), 3))

        keys.append(tuple(values))

    return keys


# ------------------------------------------------------------
# 3. COLLECT ALL METHODS
# ------------------------------------------------------------

for floor in FLOORS:

    print("\n" + "=" * 65)
    print(f"UNIFIED AUDIT — {floor}")
    print("=" * 65)

    original = (
        cgan_data[floor]["TRAIN_FRAME"]
        .copy()
        .reset_index(drop=True)
    )

    original["ID_RP"] = (
        original["ID_RP"].astype(str)
    )

    original_rssi = original[AP_COLS].apply(
        pd.to_numeric,
        errors="coerce"
    )

    original_rssi = original_rssi.replace(
        [-100, 100],
        np.nan
    )

    original_keys = set(
        make_fingerprint_keys(original_rssi)
    )

    original_ap_count = (
        original_rssi.notna().sum(axis=1)
    )

    original_mean_rssi = np.nanmean(
        original_rssi.to_numpy(dtype=float)
    )

    original_matrix = original_rssi.to_numpy(
        dtype=float
    )

    # --------------------------------------------------------
    # 3.1 Build method dictionary
    # --------------------------------------------------------

    method_frames = {
        "CGAN": CGAN_SYNTHETIC[floor].copy()
    }

    for method in METHODS:

        if method == "CGAN":
            continue

        method_frames[method] = (
            TRADITIONAL_SYNTHETIC[floor][method].copy()
        )

    # --------------------------------------------------------
    # 3.2 Evaluate each method
    # --------------------------------------------------------

    for method in METHODS:

        synthetic = method_frames[method].copy()

        synthetic["ID_RP"] = (
            synthetic["ID_RP"].astype(str)
        )

        synthetic_rssi = synthetic[AP_COLS].apply(
            pd.to_numeric,
            errors="coerce"
        )

        synthetic_rssi = synthetic_rssi.replace(
            [-100, 100],
            np.nan
        )

        # ----------------------------------------------------
        # Match by RP identity
        # ----------------------------------------------------

        original_index = original.set_index(
            "ID_RP"
        )

        synthetic_index = synthetic.set_index(
            "ID_RP"
        )

        common_ids = original_index.index.intersection(
            synthetic_index.index
        )

        matched_original = original_index.loc[
            common_ids, AP_COLS
        ].apply(
            pd.to_numeric,
            errors="coerce"
        ).replace(
            [-100, 100],
            np.nan
        )

        matched_synthetic = synthetic_index.loc[
            common_ids, AP_COLS
        ].apply(
            pd.to_numeric,
            errors="coerce"
        ).replace(
            [-100, 100],
            np.nan
        )

        original_values = matched_original.to_numpy(
            dtype=float
        )

        synthetic_values = matched_synthetic.to_numpy(
            dtype=float
        )

        common_mask = (
            np.isfinite(original_values)
            & np.isfinite(synthetic_values)
        )

        # ----------------------------------------------------
        # Same-RP RSSI MAE
        # ----------------------------------------------------

        if common_mask.sum() > 0:

            same_rp_mae = float(
                np.mean(
                    np.abs(
                        original_values[common_mask]
                        - synthetic_values[common_mask]
                    )
                )
            )

        else:
            same_rp_mae = np.nan

        # Per-RP MAE
        rp_mae_values = []

        for rp_idx, rp_id in enumerate(common_ids):

            valid = common_mask[rp_idx]

            if valid.sum() == 0:
                continue

            rp_mae = float(
                np.mean(
                    np.abs(
                        original_values[rp_idx, valid]
                        - synthetic_values[rp_idx, valid]
                    )
                )
            )

            rp_mae_values.append(rp_mae)

            per_rp_rows.append({
                "Floor": floor,
                "Method": method,
                "ID_RP": rp_id,
                "Same_RP_MAE_dBm": rp_mae,
                "Common_AP_Count": int(valid.sum())
            })

        # ----------------------------------------------------
        # Per-AP mean difference
        # ----------------------------------------------------

        ap_mean_differences = []

        for ap_idx, ap in enumerate(AP_COLS):

            o = original_values[:, ap_idx]
            s = synthetic_values[:, ap_idx]

            valid = np.isfinite(o) & np.isfinite(s)

            if valid.sum() == 0:
                continue

            mean_difference = abs(
                np.mean(o[valid]) - np.mean(s[valid])
            )

            ap_mean_differences.append(
                mean_difference
            )

            per_ap_rows.append({
                "Floor": floor,
                "Method": method,
                "AP": ap,
                "Original_Mean_RSSI_dBm": float(
                    np.mean(o[valid])
                ),
                "Synthetic_Mean_RSSI_dBm": float(
                    np.mean(s[valid])
                ),
                "Absolute_Mean_Difference_dBm":
                    float(mean_difference),
                "Common_Observations": int(valid.sum())
            })

        mean_ap_difference = (
            float(np.mean(ap_mean_differences))
            if ap_mean_differences
            else np.nan
        )

        # ----------------------------------------------------
        # Observation statistics
        # ----------------------------------------------------

        synthetic_ap_count = (
            synthetic_rssi.notna().sum(axis=1)
        )

        mean_observed_ap = float(
            synthetic_ap_count.mean()
        )

        invalid_count = int(
            (synthetic_ap_count < MIN_VALID_AP).sum()
        )

        # ----------------------------------------------------
        # RSSI statistics
        # ----------------------------------------------------

        synthetic_matrix = synthetic_rssi.to_numpy(
            dtype=float
        )

        synthetic_values_flat = synthetic_matrix[
            np.isfinite(synthetic_matrix)
        ]

        synthetic_mean_rssi = (
            float(np.mean(synthetic_values_flat))
            if len(synthetic_values_flat)
            else np.nan
        )

        # Standard deviation across RPs, calculated per AP.
        per_ap_std = np.nanstd(
            synthetic_matrix,
            axis=0
        )

        mean_ap_std = float(
            np.nanmean(per_ap_std)
        )

        # Standard deviation within each fingerprint across APs.
        per_rp_std = np.nanstd(
            synthetic_matrix,
            axis=1
        )

        mean_rp_std = float(
            np.nanmean(per_rp_std)
        )

        # ----------------------------------------------------
        # Exact duplicate check
        # ----------------------------------------------------

        synthetic_keys = make_fingerprint_keys(
            synthetic_rssi
        )

        exact_duplicate_count = sum(
            key in original_keys
            for key in synthetic_keys
        )

        # ----------------------------------------------------
        # Neighbor count
        # ----------------------------------------------------

        if "Neighbor_Count" in synthetic.columns:

            mean_neighbor_count = float(
                synthetic["Neighbor_Count"].mean()
            )

        else:
            mean_neighbor_count = np.nan

        # ----------------------------------------------------
        # Candidate validity
        # ----------------------------------------------------

        valid_count = len(synthetic) - invalid_count

        summary_rows.append({

            "Floor": floor,
            "Method": method,

            "Original_Count": len(original),
            "Synthetic_Count": len(synthetic),

            "Matched_RP": len(common_ids),

            "Valid_Candidates": valid_count,
            "Invalid_Candidates": invalid_count,

            "Mean_Observed_AP": mean_observed_ap,

            "Mean_RSSI_dBm": synthetic_mean_rssi,

            "Same_RP_MAE_dBm": same_rp_mae,

            "Mean_AP_Mean_Difference_dBm":
                mean_ap_difference,

            "Mean_AP_Std_across_RP_dBm":
                mean_ap_std,

            "Mean_RP_Std_across_AP_dBm":
                mean_rp_std,

            "Exact_Duplicates_vs_Original":
                exact_duplicate_count,

            "Mean_Neighbor_Count":
                mean_neighbor_count,

            "Generation_Status": (
                "PASS"
                if invalid_count == 0
                and len(common_ids) == len(original)
                else "CHECK"
            )
        })

        print(
            f"{method:12s} | "
            f"Valid: {valid_count:3d}/{len(synthetic):3d} | "
            f"Same-RP MAE: {same_rp_mae:.3f} dBm | "
            f"AP Mean Diff: {mean_ap_difference:.3f} dBm"
        )


# ------------------------------------------------------------
# 4. BUILD TABLES
# ------------------------------------------------------------

unified_quality_df = pd.DataFrame(
    summary_rows
)

per_rp_quality_df = pd.DataFrame(
    per_rp_rows
)

per_ap_quality_df = pd.DataFrame(
    per_ap_rows
)

# ------------------------------------------------------------
# 5. DISPLAY SUMMARY
# ------------------------------------------------------------

display(
    unified_quality_df.style.format({

        "Mean_Observed_AP": "{:.2f}",
        "Mean_RSSI_dBm": "{:.2f}",
        "Same_RP_MAE_dBm": "{:.3f}",
        "Mean_AP_Mean_Difference_dBm": "{:.3f}",
        "Mean_AP_Std_across_RP_dBm": "{:.3f}",
        "Mean_RP_Std_across_AP_dBm": "{:.3f}",
        "Mean_Neighbor_Count": "{:.2f}"
    }).background_gradient(
        subset=["Same_RP_MAE_dBm"],
        cmap="YlOrRd"
    ).background_gradient(
        subset=["Mean_AP_Mean_Difference_dBm"],
        cmap="YlGnBu"
    )
)

# ------------------------------------------------------------
# 6. SAVE TABLES
# ------------------------------------------------------------

unified_quality_df.to_csv(
    QUALITY_DIR / "unified_quality_summary.csv",
    index=False
)

per_rp_quality_df.to_csv(
    QUALITY_DIR / "unified_quality_per_rp.csv",
    index=False
)

per_ap_quality_df.to_csv(
    QUALITY_DIR / "unified_quality_per_ap.csv",
    index=False
)


# ------------------------------------------------------------
# 7. VISUALIZATION — SAME-RP MAE
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(19, 5),
    constrained_layout=True
)

for ax, floor in zip(axes, FLOORS):

    subset = unified_quality_df[
        unified_quality_df["Floor"] == floor
    ]

    ax.bar(
        subset["Method"],
        subset["Same_RP_MAE_dBm"],
        color=[
            METHOD_COLORS[m]
            for m in subset["Method"]
        ],
        edgecolor="white"
    )

    ax.set_title(
        f"{floor} — Same-RP RSSI MAE",
        fontweight="bold"
    )

    ax.set_ylabel("MAE (dBm)")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.2)

plt.savefig(
    QUALITY_DIR / "same_rp_mae_comparison.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 8. VISUALIZATION — RSSI VARIATION ACROSS RP
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(19, 5),
    constrained_layout=True
)

for ax, floor in zip(axes, FLOORS):

    subset = unified_quality_df[
        unified_quality_df["Floor"] == floor
    ]

    ax.bar(
        subset["Method"],
        subset["Mean_AP_Std_across_RP_dBm"],
        color=[
            METHOD_COLORS[m]
            for m in subset["Method"]
        ],
        edgecolor="white"
    )

    ax.set_title(
        f"{floor} — RSSI Variation Across RP",
        fontweight="bold"
    )

    ax.set_ylabel("Mean AP standard deviation (dBm)")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.2)

plt.savefig(
    QUALITY_DIR / "rssi_variation_comparison.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 9. VISUALIZATION — OBSERVED AP
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(19, 5),
    constrained_layout=True
)

for ax, floor in zip(axes, FLOORS):

    subset = unified_quality_df[
        unified_quality_df["Floor"] == floor
    ]

    ax.bar(
        subset["Method"],
        subset["Mean_Observed_AP"],
        color=[
            METHOD_COLORS[m]
            for m in subset["Method"]
        ],
        edgecolor="white"
    )

    ax.set_title(
        f"{floor} — Mean Observed AP",
        fontweight="bold"
    )

    ax.set_ylabel("Mean number of observed APs")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.2)

plt.savefig(
    QUALITY_DIR / "observed_ap_comparison.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 10. FINAL READINESS CHECK
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("CELL 10 — UNIFIED QUALITY AUDIT COMPLETED")
print("=" * 75)

print("Number of floors:", len(FLOORS))
print("Methods per floor:", len(METHODS))
print("Total evaluations:", len(unified_quality_df))

print(
    "Total invalid candidates:",
    int(unified_quality_df["Invalid_Candidates"].sum())
)

print(
    "Total exact duplicates:",
    int(
        unified_quality_df[
            "Exact_Duplicates_vs_Original"
        ].sum()
    )
)

print("\nReadiness status:")

display(
    unified_quality_df[
        [
            "Floor",
            "Method",
            "Valid_Candidates",
            "Invalid_Candidates",
            "Matched_RP",
            "Generation_Status"
        ]
    ]
)

print("\nSaved outputs:", QUALITY_DIR)


# CELL 11 — BUILD AUGMENTED REFERENCE DATASETS

## Mục tiêu
Kết hợp TRAIN gốc với dữ liệu tổng hợp từ từng phương pháp để chuẩn bị đánh giá WKNN.

## Nguyên tắc
- ORIGINAL chỉ chứa TRAIN gốc.
- Mỗi cấu hình tăng cường chỉ ghép TRAIN với một phương pháp tương ứng.
- Không ghép các phương pháp tăng cường với nhau.
- Không đưa VALIDATION hoặc TEST vào reference.
- Giữ nguyên ID_RP, X, Y, AP1–AP17 và nguồn dữ liệu.
- Giữ lại thông tin provenance để phân biệt fingerprint gốc và tổng hợp.
- Sử dụng scaler đã fit trên TRAIN gốc trong Phase 1, không fit lại scaler trên dữ liệu tổng hợp.

## Đầu ra
- Tám bộ reference theo từng tầng.
- Bảng kiểm tra số lượng fingerprint.
- Bảng kiểm tra số RP và nguồn dữ liệu.
- Báo cáo sẵn sàng cho WKNN.

In [ ]:

# ============================================================
# CELL 11 — BUILD AUGMENTED REFERENCE DATASETS
# ============================================================

import os
import numpy as np
import pandas as pd

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

REFERENCE_DIR = (
    NEW_TRAIN_DIR / "05_AUGMENTED_REFERENCE"
)

REFERENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ALL_METHODS = [
    "ORIGINAL",
    "CGAN",
    "KNN_K3",
    "KNN_K5",
    "KNN_K7",
    "APMEAN_R3",
    "APMEAN_R5",
    "APMEAN_R7"
]

AUGMENTED_REFERENCE = {}

reference_summary_rows = []

print("=" * 75)
print("CELL 11 — BUILD AUGMENTED REFERENCE DATASETS")
print("=" * 75)


# ------------------------------------------------------------
# 2. BUILD REFERENCE DATASETS
# ------------------------------------------------------------

for floor in FLOORS:

    print("\n" + "=" * 65)
    print(f"BUILD REFERENCE — {floor}")
    print("=" * 65)

    # --------------------------------------------------------
    # 2.1 Original TRAIN
    # --------------------------------------------------------

    original = (
        cgan_data[floor]["TRAIN_FRAME"]
        .copy()
        .reset_index(drop=True)
    )

    original["ID_RP"] = (
        original["ID_RP"].astype(str)
    )

    original["Source"] = "ORIGINAL"

    original["Synthetic_ID"] = [
        f"{floor}_ORIGINAL_{i+1:04d}"
        for i in range(len(original))
    ]

    # --------------------------------------------------------
    # 2.2 Standardize schema
    # --------------------------------------------------------

    required_columns = [
        "Floor",
        "ID_RP",
        "X",
        "Y"
    ] + list(AP_COLS) + [
        "Source",
        "Synthetic_ID"
    ]

    # Ensure original has Floor column.
    if "Floor" not in original.columns:
        original["Floor"] = floor

    original = original[
        required_columns
    ].copy()

    AUGMENTED_REFERENCE[floor] = {}

    # --------------------------------------------------------
    # 2.3 Build ORIGINAL and augmented references
    # --------------------------------------------------------

    for method in ALL_METHODS:

        if method == "ORIGINAL":

            reference = original.copy()

        else:

            if method == "CGAN":

                synthetic = (
                    CGAN_SYNTHETIC[floor].copy()
                )

            else:

                synthetic = (
                    TRADITIONAL_SYNTHETIC[
                        floor
                    ][method].copy()
                )

            synthetic["ID_RP"] = (
                synthetic["ID_RP"].astype(str)
            )

            synthetic = synthetic[
                required_columns
            ].copy()

            # Append synthetic fingerprints.
            reference = pd.concat(
                [original, synthetic],
                ignore_index=True
            )

        # ----------------------------------------------------
        # 2.4 Validate reference
        # ----------------------------------------------------

        reference["Floor"] = floor

        reference["ID_RP"] = (
            reference["ID_RP"].astype(str)
        )

        # Preserve missing RSSI values.
        reference[AP_COLS] = (
            reference[AP_COLS]
            .apply(pd.to_numeric, errors="coerce")
            .replace([-100, 100], np.nan)
        )

        reference["N_Observed_AP"] = (
            reference[AP_COLS].notna().sum(axis=1)
        )

        reference["Valid_Fingerprint"] = (
            reference["N_Observed_AP"] >= MIN_VALID_AP
        )

        valid_reference = reference[
            reference["Valid_Fingerprint"]
        ].copy()

        # ----------------------------------------------------
        # 2.5 Save
        # ----------------------------------------------------

        AUGMENTED_REFERENCE[floor][method] = (
            valid_reference.copy()
        )

        valid_reference.to_csv(
            REFERENCE_DIR /
            f"reference_{floor}_{method}.csv",
            index=False
        )

        # ----------------------------------------------------
        # 2.6 Summary
        # ----------------------------------------------------

        source_counts = (
            valid_reference["Source"]
            .value_counts()
            .to_dict()
        )

        reference_summary_rows.append({

            "Floor": floor,
            "Method": method,

            "Original_Count": len(original),

            "Synthetic_Count": (
                0
                if method == "ORIGINAL"
                else len(valid_reference) - len(original)
            ),

            "Reference_Count": len(valid_reference),

            "Unique_RP": (
                valid_reference["ID_RP"].nunique()
            ),

            "Original_Records": int(
                source_counts.get("ORIGINAL", 0)
            ),

            "Synthetic_Records": int(
                len(valid_reference)
                - source_counts.get("ORIGINAL", 0)
            ),

            "Invalid_Records_Removed": int(
                len(reference) - len(valid_reference)
            ),

            "Status": (
                "PASS"
                if len(valid_reference) > 0
                else "CHECK"
            )
        })

        print(
            f"{method:12s} | "
            f"Reference: {len(valid_reference):3d} | "
            f"Unique RP: {valid_reference['ID_RP'].nunique():3d}"
        )


# ------------------------------------------------------------
# 3. SUMMARY TABLE
# ------------------------------------------------------------

reference_summary_df = pd.DataFrame(
    reference_summary_rows
)

display(reference_summary_df)

reference_summary_df.to_csv(
    REFERENCE_DIR /
    "reference_build_summary.csv",
    index=False
)


# ------------------------------------------------------------
# 4. FINAL ASSERTIONS
# ------------------------------------------------------------

for floor in FLOORS:

    original_count = len(
        cgan_data[floor]["TRAIN_FRAME"]
    )

    for method in ALL_METHODS:

        reference = AUGMENTED_REFERENCE[
            floor
        ][method]

        expected_count = (
            original_count
            if method == "ORIGINAL"
            else 2 * original_count
        )

        assert len(reference) == expected_count, (
            f"{floor} {method}: "
            f"Expected {expected_count}, got {len(reference)}"
        )

        assert (
            reference["ID_RP"].nunique()
            == original_count
        ), (
            f"{floor} {method}: RP count mismatch."
        )

        assert reference["Synthetic_ID"].is_unique

        assert (
            reference["Source"].isin(
                ["ORIGINAL", method]
            ).all()
        ), (
            f"{floor} {method}: Unexpected source."
        )


# ------------------------------------------------------------
# 5. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("CELL 11 — REFERENCE BUILD COMPLETED")
print("=" * 75)

print("Floors:", len(FLOORS))
print("Configurations per floor:", len(ALL_METHODS))
print("Total configurations:", len(FLOORS) * len(ALL_METHODS))

print(
    "Total reference fingerprints:",
    int(reference_summary_df["Reference_Count"].sum())
)

print("Output:", REFERENCE_DIR)


# CELL 12 — WKNN EVALUATION

## 1. Mục tiêu
Đánh giá hiệu quả định vị của 8 cấu hình reference trên cùng tập TEST độc lập.

## 2. Các cấu hình
- ORIGINAL
- CGAN
- KNN_K3
- KNN_K5
- KNN_K7
- APMEAN_R3
- APMEAN_R5
- APMEAN_R7

## 3. Quy trình
1. Nạp reference từ Cell 11.
2. Nạp TEST query từ Phase 1.
3. Giữ nguyên scaler được fit trên TRAIN gốc.
4. Tính khoảng cách trên các AP được quan sát đồng thời.
5. Chọn k đã được xác định bằng VALIDATION.
6. Tính tọa độ dự đoán bằng WKNN có trọng số nghịch đảo khoảng cách.
7. Đánh giá sai số định vị và tỷ lệ query hợp lệ.
8. Ghi nhận fallback và các trường hợp không đủ AP chung.

## 4. Chỉ số đánh giá
- Mean Error.
- Median Error.
- RMSE.
- P90 Error.
- Valid Query.
- Coverage.
- Fallback Count.

## 5. Nguyên tắc
Tất cả phương pháp sử dụng cùng TEST query, cùng scaler, cùng công thức WKNN và cùng k theo từng tầng. Không sử dụng TEST để điều chỉnh mô hình hoặc lựa chọn tham số.

In [ ]:
# ============================================================
# CELL 12 — UNIFIED WKNN EVALUATION
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

METHODS = [
    "ORIGINAL",
    "CGAN",
    "KNN_K3",
    "KNN_K5",
    "KNN_K7",
    "APMEAN_R3",
    "APMEAN_R5",
    "APMEAN_R7"
]

# WKNN k được lựa chọn từ VALIDATION ở Phase 1
SELECTED_K = {
    "F1": 5,
    "F2": 3,
    "F3": 7
}

MIN_COMMON_APS = 2

EVAL_DIR = OUT / "08_UNIFIED_WKNN_EVALUATION"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

(EVAL_DIR / "predictions").mkdir(exist_ok=True)
(EVAL_DIR / "tables").mkdir(exist_ok=True)
(EVAL_DIR / "figures").mkdir(exist_ok=True)

# Định nghĩa lại hàm tính chỉ số đánh giá cục bộ để tránh bị ghi đè biến toàn cục
def calculate_metrics(y, p):
    ok = np.isfinite(y).all(axis=1) & np.isfinite(p).all(axis=1)
    if not ok.any():
        return {"N_Query": len(y), "N_Valid": 0, "Mean_Error": np.nan,
                "Median_Error": np.nan, "RMSE": np.nan, "P90": np.nan}
    e = np.linalg.norm(y[ok] - p[ok], axis=1)
    return {"N_Query": len(y), "N_Valid": int(ok.sum()),
            "Mean_Error": e.mean(), "Median_Error": np.median(e),
            "RMSE": np.sqrt(np.mean(e**2)), "P90": np.percentile(e, 90)}

# ------------------------------------------------------------
# 2. EVALUATION
# ------------------------------------------------------------

all_metrics = []
all_predictions = []

scalers = {}
medians = {}

for floor in FLOORS:

    print(f"\n{'='*65}")
    print(f"ĐÁNH GIÁ {floor}")
    print(f"{'='*65}")

    train = splits[floor]["TRAIN"].copy()
    test = splits[floor]["TEST"].copy()

    k = SELECTED_K[floor]

    # Fit scaler duy nhất trên ORIGINAL TRAIN
    scaler, med = fit_scaler(train)

    scalers[floor] = scaler
    medians[floor] = med

    # --------------------------------------------------------
    # ORIGINAL REFERENCE
    # --------------------------------------------------------

    references = {
        "ORIGINAL": train
    }

    # Các reference đã được tạo tại Cell 11
    for method in METHODS:

        if method == "ORIGINAL":
            continue

        references[method] = (
            AUGMENTED_REFERENCE[floor][method].copy()
        )

    # --------------------------------------------------------
    # COMMON TEST LABELS
    # --------------------------------------------------------

    test_y = test[["X", "Y"]].to_numpy(float)

    # --------------------------------------------------------
    # EVALUATE EACH METHOD
    # --------------------------------------------------------

    for method in METHODS:

        reference = references[method].copy()

        pred, fallback, used = wknn(
            reference=reference,
            queries=test,
            scaler=scaler,
            med=med,
            k=k,
            min_common_aps=MIN_COMMON_APS
        )

        metric = calculate_metrics(test_y, pred)

        n_query = len(test)
        n_valid = metric["N_Valid"]
        n_fallback = int(fallback.sum())

        coverage = (
            n_valid / n_query * 100
            if n_query > 0 else np.nan
        )

        result = {
            "Floor": floor,
            "Method": method,
            "Selected_WKNN_k": k,
            "Reference_Size": len(reference),
            "N_Query": n_query,
            "N_Valid": n_valid,
            "Coverage_%": coverage,
            "Fallback_Count": n_fallback,
            "Mean_Error": metric["Mean_Error"],
            "Median_Error": metric["Median_Error"],
            "RMSE": metric["RMSE"],
            "P90": metric["P90"]
        }

        all_metrics.append(result)

        # ----------------------------------------------------
        # SAVE PREDICTIONS
        # ----------------------------------------------------

        output = test[
            ["Floor", "ID_RP", "X", "Y"]
        ].copy()

        output["Method"] = method
        output["Pred_X"] = pred[:, 0]
        output["Pred_Y"] = pred[:, 1]
        output["Fallback"] = fallback
        output["Neighbors_Used"] = used

        output["Error"] = np.linalg.norm(
            output[["X", "Y"]].to_numpy()
            -
            output[["Pred_X", "Pred_Y"]].to_numpy(),
            axis=1
        )

        output["Valid_Prediction"] = (
            np.isfinite(output["Pred_X"])
            &
            np.isfinite(output["Pred_Y"])
        )

        all_predictions.append(output)

        output.to_csv(
            EVAL_DIR / "predictions"
            / f"{floor}_{method}_predictions.csv",
            index=False
        )

        print(
            f"{method:12s} | "
            f"Mean={metric['Mean_Error']:.4f} | "
            f"RMSE={metric['RMSE']:.4f} | "
            f"Coverage={coverage:.2f}% | "
            f"Fallback={n_fallback}"
        )

# ------------------------------------------------------------
# 3. BUILD RESULT TABLES
# ------------------------------------------------------------

metrics_df = pd.DataFrame(all_metrics)
predictions_df = pd.concat(
    all_predictions,
    ignore_index=True
)

metrics_df.to_csv(
    EVAL_DIR / "tables" / "unified_wknn_metrics.csv",
    index=False
)

predictions_df.to_csv(
    EVAL_DIR / "tables" / "all_predictions.csv",
    index=False
)

# ------------------------------------------------------------
# 4. DISPLAY FULL RESULTS
# ------------------------------------------------------------

print("\n" + "="*75)
print("BẢNG KẾT QUẢ WKNN — TẤT CẢ PHƯƠNG PHÁP")
print("="*75)

display(
    metrics_df.round(4).style
    .background_gradient(
        subset=["Mean_Error", "RMSE", "P90"],
        cmap="YlOrRd"
    )
    .background_gradient(
        subset=["Coverage_%"],
        cmap="YlGn"
    )
    .format({
        "Coverage_%": "{:.2f}%",
        "Mean_Error": "{:.4f}",
        "Median_Error": "{:.4f}",
        "RMSE": "{:.4f}",
        "P90": "{:.4f}"
    })
)

# ------------------------------------------------------------
# 5. MEAN ERROR COMPARISON
# ------------------------------------------------------------

plt.figure(figsize=(16, 7))

sns.barplot(
    data=metrics_df,
    x="Method",
    y="Mean_Error",
    hue="Floor",
    palette="Set2"
)

plt.title(
    "WKNN Localization — Mean Error Comparison",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Data Method")
plt.ylabel("Mean Localization Error")
plt.xticks(rotation=25, ha="right")
plt.grid(axis="y", alpha=0.25)
plt.legend(title="Floor")
plt.tight_layout()

plt.savefig(
    EVAL_DIR / "figures" / "mean_error_comparison.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 6. RMSE COMPARISON
# ------------------------------------------------------------

plt.figure(figsize=(16, 7))

sns.barplot(
    data=metrics_df,
    x="Method",
    y="RMSE",
    hue="Floor",
    palette="Set2"
)

plt.title(
    "WKNN Localization — RMSE Comparison",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Data Method")
plt.ylabel("RMSE")
plt.xticks(rotation=25, ha="right")
plt.grid(axis="y", alpha=0.25)
plt.legend(title="Floor")
plt.tight_layout()

plt.savefig(
    EVAL_DIR / "figures" / "rmse_comparison.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 7. COVERAGE COMPARISON
# ------------------------------------------------------------

plt.figure(figsize=(16, 6))

sns.barplot(
    data=metrics_df,
    x="Method",
    y="Coverage_%",
    hue="Floor",
    palette="Set2"
)

plt.title(
    "WKNN Localization — Valid Query Coverage",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Data Method")
plt.ylabel("Coverage (%)")
plt.ylim(0, 105)
plt.xticks(rotation=25, ha="right")
plt.grid(axis="y", alpha=0.25)
plt.legend(title="Floor")
plt.tight_layout()

plt.savefig(
    EVAL_DIR / "figures" / "coverage_comparison.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 8. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "="*75)
print("CELL 12 — COMPLETED")
print("="*75)

print("Số cấu hình đánh giá:", len(metrics_df))
print("Số tầng:", metrics_df["Floor"].nunique())
print("Số phương pháp:", metrics_df["Method"].nunique())
print("Tổng số dự đoán:", len(predictions_df))

print("\nKết quả được lưu tại:")
print(EVAL_DIR)


Cell 13 — Error Distribution & Statistical Analysis
Text Cell 13 — Mục tiêu phân tích
Sau Cell 12, chúng ta đã có kết quả định vị của 8 phương pháp trên 3 tầng. Tuy nhiên, một phương pháp có sai số trung bình thấp hơn chưa chắc đã tốt hơn trên phần lớn vị trí.

Cell 13 thực hiện 4 phân tích:
1. Error Distribution: Phân phối sai số định vị của từng phương pháp.
2. Boxplot: So sánh trung vị, độ phân tán và các trường hợp sai số lớn.
3. RP-level Analysis: Tổng hợp sai số theo từng ID_RP, tránh xem nhiều lần đo tại cùng một RP là những vị trí độc lập.
4. Paired Cluster Bootstrap: Ước lượng khoảng tin cậy 95% cho mức cải thiện so với ORIGINAL bằng cách lấy mẫu theo RP.
Quy ước:
- Mức cải thiện dương: phương pháp mới giảm sai số.
- Mức cải thiện âm: phương pháp mới làm sai số tăng.
- Khoảng tin cậy chứa 0: chưa đủ bằng chứng để kết luận có cải thiện rõ ràng.

In [ ]:
# ============================================================
# CELL 13 — ERROR DISTRIBUTION & STATISTICAL ANALYSIS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

METHODS = [
    "ORIGINAL",
    "CGAN",
    "KNN_K3",
    "KNN_K5",
    "KNN_K7",
    "APMEAN_R3",
    "APMEAN_R5",
    "APMEAN_R7"
]

BOOTSTRAP_ITERATIONS = 2000
RANDOM_SEED = 2026

STAT_DIR = EVAL_DIR / "13_STATISTICAL_ANALYSIS"

for folder in ["tables", "figures"]:
    (STAT_DIR / folder).mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(RANDOM_SEED)

# ------------------------------------------------------------
# 2. PREPARE VALID PREDICTIONS
# ------------------------------------------------------------

df = predictions_df.copy()

df["Error"] = pd.to_numeric(
    df["Error"], errors="coerce"
)

df["Valid_Prediction"] = (
    np.isfinite(df["Error"])
    & np.isfinite(df["X"])
    & np.isfinite(df["Y"])
)

valid_df = df[df["Valid_Prediction"]].copy()

print("Tổng số dự đoán:", len(df))
print("Số dự đoán hợp lệ:", len(valid_df))
print("Số dự đoán không hợp lệ:", len(df) - len(valid_df))

# ------------------------------------------------------------
# 3. ERROR DISTRIBUTION SUMMARY
# ------------------------------------------------------------

distribution_rows = []

for floor in FLOORS:

    for method in METHODS:

        subset = valid_df[
            (valid_df["Floor"] == floor)
            & (valid_df["Method"] == method)
        ]

        errors = subset["Error"].to_numpy(float)

        if len(errors) == 0:
            continue

        distribution_rows.append({
            "Floor": floor,
            "Method": method,
            "N": len(errors),
            "Mean": np.mean(errors),
            "Median": np.median(errors),
            "Std": np.std(errors, ddof=1),
            "Q25": np.percentile(errors, 25),
            "Q75": np.percentile(errors, 75),
            "P90": np.percentile(errors, 90),
            "P95": np.percentile(errors, 95),
            "Max": np.max(errors)
        })

distribution_df = pd.DataFrame(distribution_rows)

distribution_df.to_csv(
    STAT_DIR / "tables" / "error_distribution_summary.csv",
    index=False
)

print("\nBẢNG PHÂN PHỐI SAI SỐ")
display(
    distribution_df.round(4).style
    .background_gradient(
        subset=["Mean", "Median", "P90", "P95"],
        cmap="YlOrRd"
    )
)

# ------------------------------------------------------------
# 4. BOXPLOT BY FLOOR
# ------------------------------------------------------------

for floor in FLOORS:

    subset = valid_df[
        valid_df["Floor"] == floor
    ].copy()

    plt.figure(figsize=(15, 7))

    sns.boxplot(
        data=subset,
        x="Method",
        y="Error",
        hue="Method",
        order=METHODS,
        palette="Set2",
        showfliers=False,
        legend=False
    )

    plt.title(
        f"{floor} — Localization Error Distribution",
        fontsize=16,
        fontweight="bold"
    )

    plt.xlabel("Method")
    plt.ylabel("Localization Error")
    plt.xticks(rotation=25, ha="right")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()

    plt.savefig(
        STAT_DIR / "figures"
        / f"{floor}_error_boxplot.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()

# ------------------------------------------------------------
# 5. ECDF — CUMULATIVE ERROR DISTRIBUTION
# ------------------------------------------------------------

for floor in FLOORS:

    plt.figure(figsize=(12, 7))

    for method in METHODS:

        errors = valid_df.loc[
            (valid_df["Floor"] == floor)
            & (valid_df["Method"] == method),
            "Error"
        ].to_numpy(float)

        if len(errors) == 0:
            continue

        x = np.sort(errors)
        y = np.arange(1, len(x) + 1) / len(x)

        plt.plot(
            x, y,
            linewidth=2,
            label=method
        )

    plt.title(
        f"{floor} — Empirical Cumulative Error Distribution",
        fontsize=15,
        fontweight="bold"
    )

    plt.xlabel("Localization Error")
    plt.ylabel("Cumulative Probability")
    plt.xlim(left=0)
    plt.ylim(0, 1.02)
    plt.grid(alpha=0.25)
    plt.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left"
    )
    plt.tight_layout()

    plt.savefig(
        STAT_DIR / "figures"
        / f"{floor}_error_ecdf.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()

# ------------------------------------------------------------
# 6. RP-LEVEL ERROR
# ------------------------------------------------------------

# Mỗi RP được tổng hợp từ các lần đo TEST độc lập.
# Đây là bước chuẩn bị để bootstrap theo cụm RP.

rp_error_df = (
    valid_df
    .groupby(["Floor", "Method", "ID_RP"], as_index=False)
    .agg(
        RP_Mean_Error=("Error", "mean"),
        RP_Median_Error=("Error", "median"),
        RP_N_Query=("Error", "count")
    )
)

rp_error_df.to_csv(
    STAT_DIR / "tables" / "rp_level_error.csv",
    index=False
)

print("\nBẢNG SAI SỐ TỔNG HỢP THEO RP")
display(
    rp_error_df.head(15).round(4)
)

# ------------------------------------------------------------
# 7. PAIRED RP-LEVEL TABLE
# ------------------------------------------------------------

# Chỉ so sánh các RP có dữ liệu hợp lệ cho cả ORIGINAL
# và phương pháp đang được đánh giá.

paired_rows = []

for floor in FLOORS:

    floor_data = rp_error_df[
        rp_error_df["Floor"] == floor
    ]

    original = (
        floor_data[
            floor_data["Method"] == "ORIGINAL"
        ][["ID_RP", "RP_Mean_Error"]]
        .rename(columns={
            "RP_Mean_Error": "Original_Error"
        })
    )

    for method in METHODS:

        if method == "ORIGINAL":
            continue

        candidate = (
            floor_data[
                floor_data["Method"] == method
            ][["ID_RP", "RP_Mean_Error"]]
            .rename(columns={
                "RP_Mean_Error": "Method_Error"
            })
        )

        paired = original.merge(
            candidate,
            on="ID_RP",
            how="inner"
        )

        if len(paired) == 0:
            continue

        # Positive = improvement
        paired["Improvement"] = (
            paired["Original_Error"]
            - paired["Method_Error"]
        )

        paired["Floor"] = floor
        paired["Method"] = method

        paired_rows.append(paired)

paired_rp_df = pd.concat(
    paired_rows,
    ignore_index=True
)

paired_rp_df.to_csv(
    STAT_DIR / "tables" / "paired_rp_comparison.csv",
    index=False
)

# ------------------------------------------------------------
# 8. CLUSTER BOOTSTRAP BY RP
# ------------------------------------------------------------

bootstrap_rows = []

for floor in FLOORS:

    floor_paired = paired_rp_df[
        paired_rp_df["Floor"] == floor
    ]

    for method in METHODS:

        if method == "ORIGINAL":
            continue

        subset = floor_paired[
            floor_paired["Method"] == method
        ].copy()

        values = subset["Improvement"].to_numpy(float)

        n = len(values)

        if n < 2:
            continue

        # Bootstrap lấy mẫu RP có hoàn lại.
        # Mỗi RP có trọng số bằng nhau trong phân tích này.
        boot_means = np.empty(
            BOOTSTRAP_ITERATIONS,
            dtype=float
        )

        for b in range(BOOTSTRAP_ITERATIONS):

            sample = rng.choice(
                values,
                size=n,
                replace=True
            )

            boot_means[b] = np.mean(sample)

        point_estimate = np.mean(values)

        ci_low, ci_high = np.percentile(
            boot_means,
            [2.5, 97.5]
        )

        bootstrap_rows.append({
            "Floor": floor,
            "Method": method,
            "N_Paired_RP": n,
            "Mean_Improvement": point_estimate,
            "CI_95_Lower": ci_low,
            "CI_95_Upper": ci_high,
            "Improved_RP_%": (
                np.mean(values > 0) * 100
            ),
            "Worse_RP_%": (
                np.mean(values < 0) * 100
            ),
            "No_Change_RP_%": (
                np.mean(values == 0) * 100
            ),
            "CI_Excludes_Zero": (
                ci_low > 0 or ci_high < 0
            )
        })

bootstrap_df = pd.DataFrame(bootstrap_rows)

bootstrap_df.to_csv(
    STAT_DIR / "tables" / "paired_cluster_bootstrap.csv",
    index=False
)

print("\nPAIRED CLUSTER BOOTSTRAP")
display(
    bootstrap_df.round(4).style
    .background_gradient(
        subset=["Mean_Improvement"],
        cmap="RdYlGn"
    )
)

# ------------------------------------------------------------
# 9. BOOTSTRAP CONFIDENCE INTERVAL PLOT
# ------------------------------------------------------------

for floor in FLOORS:

    subset = bootstrap_df[
        bootstrap_df["Floor"] == floor
    ].copy()

    if subset.empty:
        continue

    subset = subset.set_index("Method").reindex(
        [m for m in METHODS if m != "ORIGINAL"]
    ).dropna()

    y = np.arange(len(subset))

    center = subset["Mean_Improvement"].to_numpy()
    lower = (
        center - subset["CI_95_Lower"].to_numpy()
    )
    upper = (
        subset["CI_95_Upper"].to_numpy() - center
    )

    plt.figure(figsize=(12, 7))

    plt.errorbar(
        center,
        y,
        xerr=[lower, upper],
        fmt="o",
        capsize=5,
        markersize=7,
        linewidth=2
    )

    plt.axvline(
        0,
        linestyle="--",
        linewidth=1.5,
        color="black"
    )

    plt.yticks(y, subset.index)
    plt.title(
        f"{floor} — Paired RP Bootstrap 95% CI",
        fontsize=15,
        fontweight="bold"
    )

    plt.xlabel(
        "Mean Error Improvement vs ORIGINAL"
    )

    plt.ylabel("Method")
    plt.grid(axis="x", alpha=0.25)
    plt.tight_layout()

    plt.savefig(
        STAT_DIR / "figures"
        / f"{floor}_bootstrap_ci.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()

# ------------------------------------------------------------
# 10. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "="*75)
print("CELL 13 — COMPLETED")
print("="*75)

print("Số phương pháp được phân tích:", len(METHODS))
print("Số tầng:", len(FLOORS))
print("Số dòng RP-level:", len(rp_error_df))
print("Số dòng paired comparison:", len(paired_rp_df))
print("Số kết quả bootstrap:", len(bootstrap_df))

print("\nKết quả lưu tại:")
print(STAT_DIR)

In [ ]:
# ============================================================
# CELL 14 — NEW TRAIN EXPORT & DATASET VERIFICATION
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

EXPORT_DIR = (
    OUT / "14_NEW_TRAIN_EXPORT"
)

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

METHODS = [
    "ORIGINAL",
    "CGAN",
    "KNN_K3",
    "KNN_K5",
    "KNN_K7",
    "APMEAN_R3",
    "APMEAN_R5",
    "APMEAN_R7"
]

# ------------------------------------------------------------
# 2. EXPORT DATASETS
# ------------------------------------------------------------

summary_rows = []
verification_rows = []

for floor in FLOORS:

    print("\n" + "=" * 65)
    print(f"EXPORT NEW TRAIN — {floor}")
    print("=" * 65)

    floor_dir = EXPORT_DIR / floor
    floor_dir.mkdir(parents=True, exist_ok=True)

    original_train = (
        splits[floor]["TRAIN"].copy()
    )

    for method in METHODS:

        # ----------------------------------------------------
        # Select reference dataset
        # ----------------------------------------------------

        if method == "ORIGINAL":
            dataset = original_train.copy()

        else:
            dataset = (
                AUGMENTED_REFERENCE[floor][method].copy()
            )

        dataset = dataset.reset_index(drop=True)

        # ----------------------------------------------------
        # Add provenance if not available
        # ----------------------------------------------------

        if "Data_Source" not in dataset.columns:

            if method == "ORIGINAL":
                dataset["Data_Source"] = "ORIGINAL"

            else:
                dataset["Data_Source"] = "UNKNOWN"

        # Keep method identity
        dataset["Method"] = method
        dataset["Floor"] = floor

        # ----------------------------------------------------
        # Ensure essential columns
        # ----------------------------------------------------

        required_cols = (
            ["Floor", "ID_RP", "X", "Y", "Data_Source", "Method"]
            + AP_COLS
        )

        missing_cols = [
            col for col in required_cols
            if col not in dataset.columns
        ]

        if missing_cols:
            raise ValueError(
                f"{floor} - {method}: "
                f"Thiếu cột {missing_cols}"
            )

        # ----------------------------------------------------
        # Reorder columns
        # ----------------------------------------------------

        metadata_cols = [
            "Floor",
            "ID_RP",
            "X",
            "Y",
            "Method",
            "Data_Source"
        ]

        other_cols = [
            col for col in dataset.columns
            if col not in metadata_cols + AP_COLS
        ]

        export_cols = (
            metadata_cols
            + AP_COLS
            + other_cols
        )

        dataset = dataset[export_cols]

        # ----------------------------------------------------
        # Save CSV
        # ----------------------------------------------------

        csv_path = (
            floor_dir / f"{method}.csv"
        )

        dataset.to_csv(
            csv_path,
            index=False
        )

        # ----------------------------------------------------
        # Calculate statistics
        # ----------------------------------------------------

        rssi = dataset[AP_COLS].to_numpy(float)

        observed_mask = np.isfinite(rssi)

        n_fingerprints = len(dataset)
        n_rp = dataset["ID_RP"].nunique()
        n_ap = len(AP_COLS)

        missing_rate = (
            1 - observed_mask.mean()
        ) * 100

        mean_observed_ap = (
            observed_mask.sum(axis=1).mean()
        )

        invalid_fingerprints = int(
            (observed_mask.sum(axis=1) < MIN_VALID_AP).sum()
        )

        duplicate_count = int(
            dataset.duplicated(
                subset=["ID_RP"] + AP_COLS
            ).sum()
        )

        # ----------------------------------------------------
        # Summary row
        # ----------------------------------------------------

        summary_rows.append({
            "Floor": floor,
            "Method": method,
            "N_Fingerprints": n_fingerprints,
            "N_Unique_RP": n_rp,
            "N_AP": n_ap,
            "Mean_Observed_AP": mean_observed_ap,
            "Missing_RSSI_%": missing_rate,
            "Invalid_Fingerprints": invalid_fingerprints,
            "Duplicate_Rows": duplicate_count,
            "CSV_File": str(csv_path)
        })

        # ----------------------------------------------------
        # Verification
        # ----------------------------------------------------

        verification_status = (
            "PASS"
            if (
                n_fingerprints > 0
                and n_rp > 0
                and invalid_fingerprints == 0
                and len(missing_cols) == 0
            )
            else "CHECK"
        )

        verification_rows.append({
            "Floor": floor,
            "Method": method,
            "Rows": n_fingerprints,
            "Unique_RP": n_rp,
            "Missing_Required_Columns": len(missing_cols),
            "Invalid_Fingerprints": invalid_fingerprints,
            "Status": verification_status
        })

        print(
            f"{method:12s} | "
            f"Rows={n_fingerprints:4d} | "
            f"RP={n_rp:4d} | "
            f"Missing={missing_rate:.2f}% | "
            f"{verification_status}"
        )

# ------------------------------------------------------------
# 3. BUILD SUMMARY TABLES
# ------------------------------------------------------------

summary_df = pd.DataFrame(summary_rows)

verification_df = pd.DataFrame(
    verification_rows
)

summary_df.to_csv(
    EXPORT_DIR / "dataset_summary.csv",
    index=False
)

verification_df.to_csv(
    EXPORT_DIR / "export_verification.csv",
    index=False
)

# ------------------------------------------------------------
# 4. EXPORT EXCEL
# ------------------------------------------------------------

excel_path = (
    EXPORT_DIR / "dataset_summary.xlsx"
)

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    summary_df.to_excel(
        writer,
        sheet_name="Dataset_Summary",
        index=False
    )

    verification_df.to_excel(
        writer,
        sheet_name="Verification",
        index=False
    )

    # Add separate floor sheets
    for floor in FLOORS:

        floor_summary = summary_df[
            summary_df["Floor"] == floor
        ]

        floor_summary.to_excel(
            writer,
            sheet_name=floor,
            index=False
        )

# ------------------------------------------------------------
# 5. DISPLAY SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("NEW TRAIN DATASET SUMMARY")
print("=" * 75)

display(
    summary_df.round(3).style
    .background_gradient(
        subset=["N_Fingerprints", "N_Unique_RP"],
        cmap="Blues"
    )
    .background_gradient(
        subset=["Missing_RSSI_%"],
        cmap="YlOrRd"
    )
)

print("\nEXPORT VERIFICATION")

display(verification_df)

# ------------------------------------------------------------
# 6. FINAL CHECK
# ------------------------------------------------------------

expected_configs = (
    len(FLOORS) * len(METHODS)
)

passed_configs = int(
    (verification_df["Status"] == "PASS").sum()
)

print("\n" + "=" * 75)
print("CELL 14 — EXPORT COMPLETED")
print("=" * 75)

print(f"Tổng cấu hình: {expected_configs}")
print(f"PASS: {passed_configs}")
print(f"CHECK: {expected_configs - passed_configs}")

print("\nCSV directory:")
print(EXPORT_DIR)

print("\nExcel summary:")
print(excel_path)

if passed_configs == expected_configs:
    print("\nALL CONFIGURATIONS PASSED INITIAL VERIFICATION.")
else:
    print("\nWARNING: Cần kiểm tra các cấu hình có trạng thái CHECK.")

In [ ]:
# ============================================================
# CELL 14B — DATA PROVENANCE AUDIT
# ============================================================

provenance_rows = []

for floor in FLOORS:

    for method in METHODS:

        path = (
            EXPORT_DIR / floor / f"{method}.csv"
        )

        data = pd.read_csv(path)

        if "Data_Source" not in data.columns:
            source_counts = {"MISSING_COLUMN": len(data)}
        else:
            source_counts = (
                data["Data_Source"]
                .fillna("NULL")
                .value_counts()
                .to_dict()
            )

        for source, count in source_counts.items():

            provenance_rows.append({
                "Floor": floor,
                "Method": method,
                "Data_Source": source,
                "Count": count
            })

provenance_df = pd.DataFrame(provenance_rows)

display(provenance_df)

provenance_df.to_csv(
    EXPORT_DIR / "provenance_audit.csv",
    index=False
)

print("\nProvenance audit saved:")
print(EXPORT_DIR / "provenance_audit.csv")

In [ ]:
# ============================================================
# CELL 14C — FIX DATA PROVENANCE & RE-EXPORT
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

PROVENANCE_DIR = OUT / "14_NEW_TRAIN_EXPORT_VERIFIED"
PROVENANCE_DIR.mkdir(parents=True, exist_ok=True)

METHODS = [
    "ORIGINAL",
    "CGAN",
    "KNN_K3",
    "KNN_K5",
    "KNN_K7",
    "APMEAN_R3",
    "APMEAN_R5",
    "APMEAN_R7"
]

summary = []
verification = []

for floor in FLOORS:

    floor_dir = PROVENANCE_DIR / floor
    floor_dir.mkdir(parents=True, exist_ok=True)

    # ORIGINAL TRAIN
    original = splits[floor]["TRAIN"].copy()
    original = original.reset_index(drop=True)

    original["Data_Source"] = "ORIGINAL"
    original["Method"] = "ORIGINAL"
    original["Floor"] = floor

    for method in METHODS:

        if method == "ORIGINAL":

            dataset = original.copy()
            expected_original = len(original)
            expected_synthetic = 0

        else:

            # Lấy dữ liệu tổng hợp từ reference đã xây dựng
            # tại Cell 11 và tách theo đúng số dòng ORIGINAL.
            reference = (
                AUGMENTED_REFERENCE[floor][method].copy()
                .reset_index(drop=True)
            )

            n_original = len(original)

            if len(reference) != 2 * n_original:
                raise ValueError(
                    f"{floor}/{method}: "
                    "Số dòng reference không bằng 2 lần ORIGINAL."
                )

            # Xác định phần ORIGINAL bằng cách đối chiếu
            # với TRAIN gốc, không dựa trên vị trí dòng.
            original_compare = original[
                ["ID_RP", "X", "Y"] + AP_COLS
            ].reset_index(drop=True)

            reference_compare = reference[
                ["ID_RP", "X", "Y"] + AP_COLS
            ].reset_index(drop=True)

            # Reference Cell 11 được ghép từ ORIGINAL
            # trước, synthetic sau.
            first_part = reference.iloc[:n_original].copy()
            second_part = reference.iloc[n_original:].copy()

            check_original = first_part[
                ["ID_RP", "X", "Y"] + AP_COLS
            ].reset_index(drop=True)

            if not original_compare.equals(check_original):
                raise ValueError(
                    f"{floor}/{method}: "
                    "Không xác nhận được phần ORIGINAL trong reference."
                )

            original_part = first_part.copy()
            synthetic_part = second_part.copy()

            original_part["Data_Source"] = "ORIGINAL"
            synthetic_part["Data_Source"] = method

            original_part["Method"] = method
            synthetic_part["Method"] = method

            original_part["Floor"] = floor
            synthetic_part["Floor"] = floor

            dataset = pd.concat(
                [original_part, synthetic_part],
                ignore_index=True
            )

            expected_original = n_original
            expected_synthetic = n_original

        dataset = dataset.reset_index(drop=True)

        # Kiểm tra nguồn
        source_counts = (
            dataset["Data_Source"]
            .value_counts()
            .to_dict()
        )

        original_count = int(
            source_counts.get("ORIGINAL", 0)
        )

        synthetic_count = len(dataset) - original_count

        # Kiểm tra dữ liệu
        rssi = dataset[AP_COLS].to_numpy(float)
        observed = np.isfinite(rssi)

        invalid_count = int(
            (observed.sum(axis=1) < MIN_VALID_AP).sum()
        )

        # Kiểm tra trùng chính xác trong từng nguồn
        duplicate_count = int(
            dataset.duplicated(
                subset=["ID_RP", "Data_Source"] + AP_COLS
            ).sum()
        )

        source_ok = (
            original_count == expected_original
            and synthetic_count == expected_synthetic
        )

        status = (
            "PASS"
            if (
                source_ok
                and invalid_count == 0
                and duplicate_count == 0
            )
            else "CHECK"
        )

        # Xuất CSV
        csv_path = floor_dir / f"{method}.csv"

        dataset.to_csv(csv_path, index=False)

        # Tổng hợp
        summary.append({
            "Floor": floor,
            "Method": method,
            "Original_Count": original_count,
            "Synthetic_Count": synthetic_count,
            "Total_Count": len(dataset),
            "Unique_RP": dataset["ID_RP"].nunique(),
            "Missing_RSSI_%": (
                100 * (1 - observed.mean())
            ),
            "Invalid_Fingerprints": invalid_count,
            "Duplicate_Rows": duplicate_count,
            "Status": status,
            "CSV_File": str(csv_path)
        })

        verification.append({
            "Floor": floor,
            "Method": method,
            "Expected_Original": expected_original,
            "Actual_Original": original_count,
            "Expected_Synthetic": expected_synthetic,
            "Actual_Synthetic": synthetic_count,
            "Status": status
        })

        print(
            f"{floor} | {method:12s} | "
            f"ORIGINAL={original_count:4d} | "
            f"SYNTHETIC={synthetic_count:4d} | "
            f"{status}"
        )

# ------------------------------------------------------------
# SAVE REPORTS
# ------------------------------------------------------------

summary_df_14c = pd.DataFrame(summary)
verification_df_14c = pd.DataFrame(verification)

summary_df_14c.to_csv(
    PROVENANCE_DIR / "dataset_summary.csv",
    index=False
)

verification_df_14c.to_csv(
    PROVENANCE_DIR / "provenance_verification.csv",
    index=False
)

with pd.ExcelWriter(
    PROVENANCE_DIR / "NEW_TRAIN_VERIFIED.xlsx",
    engine="openpyxl"
) as writer:

    summary_df_14c.to_excel(
        writer,
        sheet_name="Dataset_Summary",
        index=False
    )

    verification_df_14c.to_excel(
        writer,
        sheet_name="Verification",
        index=False
    )

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PROVENANCE VERIFICATION")
print("=" * 75)

display(summary_df_14c.round(3))
display(verification_df_14c)

print("\nPASS:",
      (verification_df_14c["Status"] == "PASS").sum())

print("CHECK:",
      (verification_df_14c["Status"] != "PASS").sum())

print("\nVerified dataset directory:")
print(PROVENANCE_DIR)

### Cell 15 — DATASET FINALIZATION & FINAL QUALITY CONTROL
# Mục tiêu: Kiểm tra và xác nhận tính toàn vẹn của 24 bộ dữ liệu được tạo ra từ giai đoạn NEW TRAIN trước khi chuyển sang thí nghiệm Multi-seed.

Ở giai đoạn này, nghiên cứu sử dụng 8 phương pháp trên 3 tầng dữ liệu:
- ORIGINAL: Bộ fingerprint RSSI gốc, chưa tăng cường dữ liệu.
- CGAN: Tăng cường dữ liệu bằng Conditional Generative Adversarial Network.
- KNN_K3, KNN_K5, KNN_K7: Tăng cường dữ liệu bằng trung bình RSSI của các điểm tham chiếu lân cận gần nhất.
- APMEAN_R3, APMEAN_R5, APMEAN_R7: Tăng cường dữ liệu bằng trung bình RSSI của các điểm tham chiếu nằm trong phạm vi lân cận xác định.
# Các nội dung kiểm tra chính:
1. Kiểm tra cấu trúc: Đảm bảo đầy đủ 24 bộ dữ liệu, tương ứng 8 phương pháp × 3 tầng.
2. Kiểm tra số lượng: Đối chiếu số fingerprint và số Reference Point (RP) với cấu hình ban đầu.
3. Kiểm tra nguồn dữ liệu (Data Provenance): Xác nhận từng bản ghi được phân loại chính xác thành ORIGINAL hoặc phương pháp tăng cường tương ứng.
4. Kiểm tra tính hợp lệ của RSSI: Đảm bảo mỗi fingerprint có tối thiểu số Access Point (AP) quan sát hợp lệ theo cấu hình.
5. Kiểm tra tọa độ: Đảm bảo các fingerprint thuộc cùng một RP có tọa độ X/Y nhất quán.
6. Kiểm tra trùng lặp: Phát hiện các bản ghi fingerprint trùng nhau trong cùng nguồn dữ liệu.
7. Kiểm tra tính toàn vẹn: Đảm bảo không có giá trị RSSI hoặc nhãn nguồn dữ liệu không hợp lệ.
# Nguyên tắc thực hiện:
- Chỉ sử dụng dữ liệu TRAIN và VALIDATION đã được xác nhận ở các bước trước.
- Không sử dụng dữ liệu TEST để huấn luyện, sinh dữ liệu, điều chỉnh tham số hoặc lựa chọn mô hình.
- Giữ nguyên dữ liệu ORIGINAL và thông tin truy xuất nguồn gốc của dữ liệu Synthetic.
- Không xem việc giảm tỷ lệ RSSI bị thiếu là bằng chứng duy nhất chứng minh chất lượng dữ liệu tăng cường.
# Đầu ra mong đợi:
- Bảng kiểm tra chất lượng 24 bộ dữ liệu.
- Dataset Manifest tổng hợp thông tin từng bộ dữ liệu.
- Báo cáo Excel DATASET_FINALIZATION.xlsx.
- Trạng thái PASS/FAIL cho từng cấu hình.

In [ ]:
# ============================================================
# CELL 15 — DATASET FINALIZATION
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

FINAL_DATA_DIR = (
    OUT / "14_NEW_TRAIN_EXPORT_VERIFIED"
)

FINAL_AUDIT_DIR = (
    OUT / "15_DATASET_FINALIZATION"
)

FINAL_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

METHODS = [
    "ORIGINAL",
    "CGAN",
    "KNN_K3",
    "KNN_K5",
    "KNN_K7",
    "APMEAN_R3",
    "APMEAN_R5",
    "APMEAN_R7"
]

EXPECTED_SOURCE = {
    "ORIGINAL": ["ORIGINAL"],
    "CGAN": ["ORIGINAL", "CGAN"],
    "KNN_K3": ["ORIGINAL", "KNN_K3"],
    "KNN_K5": ["ORIGINAL", "KNN_K5"],
    "KNN_K7": ["ORIGINAL", "KNN_K7"],
    "APMEAN_R3": ["ORIGINAL", "APMEAN_R3"],
    "APMEAN_R5": ["ORIGINAL", "APMEAN_R5"],
    "APMEAN_R7": ["ORIGINAL", "APMEAN_R7"]
}

# ------------------------------------------------------------
# 2. EXPECTED DATA COUNTS
# ------------------------------------------------------------

EXPECTED_RP = {
    "F1": 266,
    "F2": 158,
    "F3": 127
}

audit_rows = []
manifest_rows = []
all_datasets = {}

# ------------------------------------------------------------
# 3. LOAD AND AUDIT ALL DATASETS
# ------------------------------------------------------------

for floor in FLOORS:

    print("\n" + "=" * 75)
    print(f"FINAL DATASET AUDIT — {floor}")
    print("=" * 75)

    all_datasets[floor] = {}

    expected_n = EXPECTED_RP[floor]

    for method in METHODS:

        file_path = (
            FINAL_DATA_DIR / floor / f"{method}.csv"
        )

        checks = {}

        # File existence
        checks["File_Exists"] = file_path.exists()

        if not checks["File_Exists"]:
            audit_rows.append({
                "Floor": floor,
                "Method": method,
                "Status": "FAIL",
                "Failed_Checks": "File_Exists"
            })
            continue

        data = pd.read_csv(file_path)

        all_datasets[floor][method] = data

        # ----------------------------------------------------
        # Required columns
        # ----------------------------------------------------

        required = (
            ["Floor", "ID_RP", "X", "Y",
             "Method", "Data_Source"]
            + AP_COLS
        )

        missing_columns = [
            c for c in required
            if c not in data.columns
        ]

        checks["Required_Columns"] = (
            len(missing_columns) == 0
        )

        # Stop detailed checks if schema is incomplete
        if missing_columns:
            audit_rows.append({
                "Floor": floor,
                "Method": method,
                "Status": "FAIL",
                "Failed_Checks": (
                    "Missing columns: "
                    + ", ".join(missing_columns)
                )
            })
            continue

        # ----------------------------------------------------
        # Row and RP counts
        # ----------------------------------------------------

        expected_total = (
            expected_n if method == "ORIGINAL"
            else expected_n * 2
        )

        checks["Row_Count"] = (
            len(data) == expected_total
        )

        checks["Unique_RP_Count"] = (
            data["ID_RP"].nunique() == expected_n
        )

        # ----------------------------------------------------
        # Provenance checks
        # ----------------------------------------------------

        actual_sources = set(
            data["Data_Source"].dropna().astype(str)
        )

        expected_sources = set(
            EXPECTED_SOURCE[method]
        )

        checks["Source_Labels"] = (
            actual_sources == expected_sources
            and data["Data_Source"].notna().all()
        )

        checks["No_Unknown_Source"] = (
            not data["Data_Source"]
            .astype(str)
            .str.upper()
            .isin(["UNKNOWN", "NULL", "NAN", ""])
            .any()
        )

        source_counts = (
            data["Data_Source"]
            .value_counts()
            .to_dict()
        )

        original_count = int(
            source_counts.get("ORIGINAL", 0)
        )

        synthetic_count = (
            len(data) - original_count
        )

        checks["Source_Counts"] = (
            original_count == expected_n
            and synthetic_count == (
                0 if method == "ORIGINAL"
                else expected_n
            )
        )

        # ----------------------------------------------------
        # AP and RSSI checks
        # ----------------------------------------------------

        rssi = data[AP_COLS].to_numpy(float)

        observed_mask = np.isfinite(rssi)

        observed_per_row = (
            observed_mask.sum(axis=1)
        )

        checks["Minimum_Observed_AP"] = (
            np.all(observed_per_row >= MIN_VALID_AP)
        )

        checks["RSSI_Range"] = (
            np.all(
                (rssi[observed_mask] >= RSSI_MIN)
                &
                (rssi[observed_mask] <= RSSI_MAX)
            )
        )

        # ----------------------------------------------------
        # Duplicate checks within source
        # ----------------------------------------------------

        duplicate_count = int(
            data.duplicated(
                subset=[
                    "ID_RP",
                    "Data_Source"
                ] + AP_COLS
            ).sum()
        )

        checks["No_Duplicates"] = (
            duplicate_count == 0
        )

        # ----------------------------------------------------
        # Coordinate consistency
        # ----------------------------------------------------

        coordinate_counts = (
            data.groupby("ID_RP")[["X", "Y"]]
            .nunique(dropna=False)
        )

        checks["Coordinate_Consistency"] = (
            (coordinate_counts <= 1).all().all()
            and data[["X", "Y"]].notna().all().all()
        )

        # ----------------------------------------------------
        # Each RP has expected source records
        # ----------------------------------------------------

        source_per_rp = (
            data.groupby(
                ["ID_RP", "Data_Source"]
            ).size()
        )

        if method == "ORIGINAL":

            checks["RP_Source_Records"] = (
                len(source_per_rp) == expected_n
                and (source_per_rp == 1).all()
            )

        else:

            checks["RP_Source_Records"] = (
                len(source_per_rp) == expected_n * 2
                and (source_per_rp == 1).all()
            )

        # ----------------------------------------------------
        # Final status
        # ----------------------------------------------------

        failed = [
            name for name, passed in checks.items()
            if not passed
        ]

        status = (
            "PASS" if len(failed) == 0
            else "FAIL"
        )

        audit_rows.append({
            "Floor": floor,
            "Method": method,
            "Rows": len(data),
            "Unique_RP": data["ID_RP"].nunique(),
            "Original_Count": original_count,
            "Synthetic_Count": synthetic_count,
            "Mean_Observed_AP": observed_per_row.mean(),
            "Missing_RSSI_%": (
                100 * (1 - observed_mask.mean())
            ),
            "Duplicate_Rows": duplicate_count,
            "Failed_Checks": (
                ", ".join(failed) if failed else "None"
            ),
            "Status": status
        })

        manifest_rows.append({
            "Floor": floor,
            "Method": method,
            "File": str(file_path),
            "Rows": len(data),
            "Unique_RP": data["ID_RP"].nunique(),
            "Original_Count": original_count,
            "Synthetic_Count": synthetic_count,
            "Status": status
        })

        print(
            f"{method:12s} | "
            f"Rows={len(data):4d} | "
            f"RP={data['ID_RP'].nunique():4d} | "
            f"Status={status}"
        )

# ------------------------------------------------------------
# 4. BUILD AUDIT REPORT
# ------------------------------------------------------------

audit_df = pd.DataFrame(audit_rows)
manifest_df = pd.DataFrame(manifest_rows)

audit_df.to_csv(
    FINAL_AUDIT_DIR / "final_dataset_audit.csv",
    index=False
)

manifest_df.to_csv(
    FINAL_AUDIT_DIR / "dataset_manifest.csv",
    index=False
)

# ------------------------------------------------------------
# 5. SUMMARY
# ------------------------------------------------------------

total_configs = len(FLOORS) * len(METHODS)

passed = int(
    (audit_df["Status"] == "PASS").sum()
)

failed = total_configs - passed

print("\n" + "=" * 75)
print("FINAL DATASET AUDIT SUMMARY")
print("=" * 75)

display(audit_df)

print("\nManifest:")
display(manifest_df)

print("\nTotal configurations:", total_configs)
print("PASS:", passed)
print("FAIL:", failed)

# ------------------------------------------------------------
# 6. EXPORT EXCEL
# ------------------------------------------------------------

with pd.ExcelWriter(
    FINAL_AUDIT_DIR / "DATASET_FINALIZATION.xlsx",
    engine="openpyxl"
) as writer:

    audit_df.to_excel(
        writer,
        sheet_name="Final_Audit",
        index=False
    )

    manifest_df.to_excel(
        writer,
        sheet_name="Dataset_Manifest",
        index=False
    )

# ------------------------------------------------------------
# 7. FINAL STATUS
# ------------------------------------------------------------

if passed == total_configs and failed == 0:

    print("\nDATASET FINALIZATION: PASS")
    print("All datasets passed the final audit.")

else:

    print("\nDATASET FINALIZATION: CHECK REQUIRED")
    print("Please inspect Failed_Checks.")

print("\nOutput directory:")
print(FINAL_AUDIT_DIR)

### Cell 16 — MULTI-SEED EXPERIMENT
# Bắt đầu giai đoạn kiểm tra độ ổn định của 8 phương pháp trên 3 tầng dữ liệu qua 5 lần chạy độc lập.

Khác với Cell 12, lần này chúng ta không chỉ thay đổi seed khi đánh giá. Toàn bộ quy trình phụ thuộc vào TRAIN đều được thực hiện lại theo từng seed: chia TRAIN/VALIDATION, chuẩn hóa RSSI, huấn luyện cGAN, sinh fingerprint và xây dựng dữ liệu KNN/APMEAN.

Thiết kế thí nghiệm:
Thành phần	Thiết lập
Seed	2026–2030
Số tầng	F1, F2, F3
Phương pháp	8 phương pháp
Chia TRAIN/VALIDATION	Theo RP, tỷ lệ 85/15
TEST	Giữ nguyên tập TEST độc lập
Chọn k	Dựa trên VALIDATION của ORIGINAL
Đánh giá	WKNN trên cùng TEST của từng tầng
Kết quả	Mean Error, RMSE, Median, P90, Coverage

Nguyên tắc quan trọng: Mỗi seed phải tạo lại dữ liệu tăng cường chỉ từ TRAIN của seed đó. TEST không tham gia vào bất kỳ bước huấn luyện, sinh dữ liệu hoặc lựa chọn tham số nào.

Kết quả cuối cùng sẽ thể hiện Mean ± Standard Deviation qua 5 seed, giúp đánh giá phương pháp nào cải thiện ổn định và phương pháp nào chỉ có kết quả tốt ở một lần chia dữ liệu.

In [ ]:
# ============================================================
# CELL 16A — MULTI-SEED EXPERIMENT SETUP
# ============================================================

import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

# ------------------------------------------------------------
# 1. EXPERIMENT CONFIGURATION
# ------------------------------------------------------------

MULTI_SEEDS = [2026, 2027, 2028, 2029, 2030]

MULTI_METHODS = [
    "ORIGINAL",
    "CGAN",
    "KNN_K3",
    "KNN_K5",
    "KNN_K7",
    "APMEAN_R3",
    "APMEAN_R5",
    "APMEAN_R7"
]

TRAIN_RATIO = 0.85
VALIDATION_RATIO = 0.15

MULTI_SEED_DIR = (
    OUT / "16_MULTI_SEED_EXPERIMENT"
)

MULTI_SEED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. CHECK REQUIRED VARIABLES
# ------------------------------------------------------------

required_variables = [
    "prepared_data",
    "FLOORS",
    "AP_COLS",
    "MIN_VALID_AP",
    "RSSI_MIN",
    "RSSI_MAX"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Missing required variables: "
        + ", ".join(missing_variables)
    )

print("Required variables: PASS")

# ------------------------------------------------------------
# 3. HELPER — REPRODUCIBILITY
# ------------------------------------------------------------

def set_multi_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    try:
        import torch
        torch.manual_seed(seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    except ImportError:
        pass


# ------------------------------------------------------------
# 4. HELPER — MERGE ORIGINAL TRAIN AND VALIDATION
# ------------------------------------------------------------

def merge_train_validation(floor_data):

    train_df = floor_data["TRAIN"].copy()
    val_df = floor_data["VALIDATION"].copy()

    combined = pd.concat(
        [train_df, val_df],
        ignore_index=True
    )

    if "ID_RP" not in combined.columns:
        raise ValueError(
            "ID_RP column is missing."
        )

    if combined["ID_RP"].isna().any():
        raise ValueError(
            "Missing ID_RP detected."
        )

    return combined


# ------------------------------------------------------------
# 5. HELPER — SPLIT BY RP
# ------------------------------------------------------------

def split_rp_by_seed(
    combined_df,
    seed,
    train_ratio=0.85
):

    unique_rps = (
        combined_df["ID_RP"]
        .drop_duplicates()
        .to_numpy()
    )

    rng = np.random.default_rng(seed)

    shuffled_rps = rng.permutation(unique_rps)

    n_train = int(
        round(len(shuffled_rps) * train_ratio)
    )

    train_rps = set(
        shuffled_rps[:n_train]
    )

    val_rps = set(
        shuffled_rps[n_train:]
    )

    train_df = combined_df[
        combined_df["ID_RP"].isin(train_rps)
    ].copy()

    val_df = combined_df[
        combined_df["ID_RP"].isin(val_rps)
    ].copy()

    # No RP overlap
    overlap = (
        set(train_df["ID_RP"])
        & set(val_df["ID_RP"])
    )

    if overlap:
        raise RuntimeError(
            f"RP leakage detected: {len(overlap)}"
        )

    return train_df, val_df


# ------------------------------------------------------------
# 6. PREPARE FIXED TEST DATA
# ------------------------------------------------------------

MULTI_SEED_DATA = {}

setup_rows = []

for floor in FLOORS:

    floor_data = prepared_data[floor]

    # Recombine only original TRAIN and VALIDATION.
    # TEST is deliberately excluded.
    combined_train = merge_train_validation(
        floor_data
    )

    fixed_test = splits[floor]["TEST"].copy()

    if len(combined_train) == 0 or len(fixed_test) == 0:
        raise ValueError(
            f"Empty data detected at {floor}"
        )

    if not set(AP_COLS).issubset(
        combined_train.columns
    ):
        raise ValueError(
            f"AP columns missing in TRAIN: {floor}"
        )

    if not set(AP_COLS).issubset(
        fixed_test.columns
    ):
        raise ValueError(
            f"AP columns missing in TEST: {floor}"
        )

    MULTI_SEED_DATA[floor] = {
        "COMBINED_TRAIN": combined_train,
        "TEST": fixed_test
    }

    for seed in MULTI_SEEDS:

        train_seed, val_seed = split_rp_by_seed(
            combined_train,
            seed
        )

        train_rps = set(
            train_seed["ID_RP"]
        )

        val_rps = set(
            val_seed["ID_RP"]
        )

        test_rps = set(
            fixed_test["ID_RP"]
        )

        setup_rows.append({

            "Seed": seed,
            "Floor": floor,

            "TRAIN_RP": len(train_rps),
            "VALIDATION_RP": len(val_rps),

            "TRAIN_Rows": len(train_seed),
            "VALIDATION_Rows": len(val_seed),

            "TEST_RP": len(test_rps),
            "TEST_Queries": len(fixed_test),

            "TRAIN_VAL_Overlap": len(
                train_rps & val_rps
            ),

            "TRAIN_TEST_RP_Overlap": len(
                train_rps & test_rps
            ),

            "VAL_TEST_RP_Overlap": len(
                val_rps & test_rps
            )
        })

        # Store split for later training.
        MULTI_SEED_DATA[floor].setdefault(
            "SEEDS", {}
        )

        MULTI_SEED_DATA[floor]["SEEDS"][seed] = {
            "TRAIN": train_seed,
            "VALIDATION": val_seed,
            "TEST": fixed_test
        }


# ------------------------------------------------------------
# 7. DISPLAY SPLIT AUDIT
# ------------------------------------------------------------

split_audit_df = pd.DataFrame(setup_rows)

split_audit_df.to_csv(
    MULTI_SEED_DIR / "multi_seed_split_audit.csv",
    index=False
)

display(split_audit_df)

# ------------------------------------------------------------
# 8. FINAL CHECKS
# ------------------------------------------------------------

assert len(split_audit_df) == (
    len(FLOORS) * len(MULTI_SEEDS)
)

assert (
    split_audit_df["TRAIN_VAL_Overlap"] == 0
).all()

assert (
    split_audit_df["TRAIN_RP"] +
    split_audit_df["VALIDATION_RP"]
).equals(
    split_audit_df[
        "TRAIN_RP"
    ] + split_audit_df["VALIDATION_RP"]
)

# Fixed TEST row count must remain identical across seeds
for floor in FLOORS:

    test_counts = (
        split_audit_df[
            split_audit_df["Floor"] == floor
        ]["TEST_Queries"]
        .unique()
    )

    assert len(test_counts) == 1

print("\n" + "=" * 70)
print("MULTI-SEED SETUP SUMMARY")
print("=" * 70)

print("Seeds:", MULTI_SEEDS)
print("Floors:", FLOORS)
print("Methods:", MULTI_METHODS)
print("Total split configurations:", len(split_audit_df))
print("TRAIN / VALIDATION RP overlap: 0")
print("TEST rows fixed across seeds: PASS")

print("\nMULTI-SEED SETUP: PASS")
print("Output:", MULTI_SEED_DIR)


In [ ]:
# ============================================================
# CELL 16B — SEED-BASED DATA PREPARATION & NORMALIZATION
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

SENTINEL_VALUES = [-100, 100]

# Output:
# MULTI_SEED_DATA[floor]["SEEDS"][seed]["PREPARED"]
#
# Each split contains:
# RAW       : RSSI with missing values represented by NaN
# MASK      : 1 = observed AP, 0 = missing AP
# SCALED    : standardized RSSI, missing values filled with 0
# N_OBSERVED: number of observed APs per fingerprint

MULTI_PREPARED = {}

normalization_audit = []

# ------------------------------------------------------------
# 2. HELPER — CLEAN RSSI AND BUILD MASK
# ------------------------------------------------------------

def clean_rssi_and_mask(df):

    result = df.copy()

    rssi = result[AP_COLS].apply(
        pd.to_numeric,
        errors="coerce"
    )

    rssi = rssi.replace(
        SENTINEL_VALUES,
        np.nan
    )

    # Values outside configured physical range are invalid.
    rssi = rssi.mask(
        (rssi < RSSI_MIN) |
        (rssi > RSSI_MAX)
    )

    mask = rssi.notna().astype(np.uint8)

    n_observed = mask.sum(axis=1)

    return result, rssi, mask, n_observed


# ------------------------------------------------------------
# 3. HELPER — FIT SCALER ON TRAIN ONLY
# ------------------------------------------------------------

def fit_train_scaler(train_rssi):

    values = train_rssi.to_numpy(dtype=float)

    observed = np.isfinite(values)

    n_features = values.shape[1]

    means = np.zeros(n_features, dtype=float)
    stds = np.ones(n_features, dtype=float)
    medians = np.zeros(n_features, dtype=float)

    for j in range(n_features):

        col = values[:, j]
        valid = col[np.isfinite(col)]

        if len(valid) == 0:
            raise ValueError(
                f"AP column {AP_COLS[j]} has no observed "
                "RSSI in TRAIN."
            )

        means[j] = valid.mean()
        stds[j] = valid.std(ddof=0)
        medians[j] = np.median(valid)

        # Prevent division by zero
        if stds[j] < 1e-8:
            stds[j] = 1.0

    return {
        "mean": means,
        "std": stds,
        "median": medians,
        "n_features": n_features,
        "fit_source": "TRAIN_ONLY"
    }


# ------------------------------------------------------------
# 4. HELPER — TRANSFORM USING TRAIN SCALER
# ------------------------------------------------------------

def transform_rssi(rssi_df, scaler):

    values = rssi_df.to_numpy(dtype=float)

    mask = np.isfinite(values)

    scaled = np.zeros_like(
        values,
        dtype=np.float32
    )

    means = scaler["mean"]
    stds = scaler["std"]

    scaled_values = (
        values - means
    ) / stds

    # Preserve missing APs as zero in the model input.
    # The separate mask distinguishes missing from real zero.
    scaled[mask] = scaled_values[mask].astype(
        np.float32
    )

    return scaled, mask.astype(np.uint8)


# ------------------------------------------------------------
# 5. PREPARE EACH FLOOR AND SEED
# ------------------------------------------------------------

for floor in FLOORS:

    print("\n" + "=" * 75)
    print(f"SEED-BASED PREPARATION — {floor}")
    print("=" * 75)

    MULTI_PREPARED[floor] = {}

    for seed in MULTI_SEEDS:

        set_multi_seed(seed)

        split_data = (
            MULTI_SEED_DATA[floor]["SEEDS"][seed]
        )

        prepared_splits = {}

        # --------------------------------------------
        # CLEAN ALL THREE SPLITS
        # --------------------------------------------

        cleaned = {}

        for split_name in [
            "TRAIN",
            "VALIDATION",
            "TEST"
        ]:

            df = split_data[split_name].copy()

            metadata, rssi, mask, n_observed = (
                clean_rssi_and_mask(df)
            )

            cleaned[split_name] = {
                "metadata": metadata,
                "rssi": rssi,
                "mask": mask,
                "n_observed": n_observed
            }

        # --------------------------------------------
        # FIT SCALER ON TRAIN ONLY
        # --------------------------------------------

        scaler = fit_train_scaler(
            cleaned["TRAIN"]["rssi"]
        )

        # --------------------------------------------
        # TRANSFORM ALL SPLITS
        # --------------------------------------------

        for split_name in [
            "TRAIN",
            "VALIDATION",
            "TEST"
        ]:

            item = cleaned[split_name]

            scaled, observed_mask = transform_rssi(
                item["rssi"],
                scaler
            )

            prepared_splits[split_name] = {

                "RAW": item["rssi"].copy(),

                "MASK": observed_mask,

                "SCALED": scaled,

                "N_OBSERVED": (
                    item["n_observed"].to_numpy()
                ),

                "METADATA": (
                    item["metadata"].copy()
                )
            }

        # --------------------------------------------
        # VALIDATION CHECKS
        # --------------------------------------------

        train_scaled = prepared_splits[
            "TRAIN"
        ]["SCALED"]

        val_scaled = prepared_splits[
            "VALIDATION"
        ]["SCALED"]

        test_scaled = prepared_splits[
            "TEST"
        ]["SCALED"]

        train_mask = prepared_splits[
            "TRAIN"
        ]["MASK"]

        val_mask = prepared_splits[
            "VALIDATION"
        ]["MASK"]

        test_mask = prepared_splits[
            "TEST"
        ]["MASK"]

        # 1. Dimensions
        assert train_scaled.shape[1] == len(AP_COLS)
        assert val_scaled.shape[1] == len(AP_COLS)
        assert test_scaled.shape[1] == len(AP_COLS)

        # 2. Finite scaled values
        assert np.isfinite(train_scaled).all()
        assert np.isfinite(val_scaled).all()
        assert np.isfinite(test_scaled).all()

        # 3. Mask values must be 0 or 1
        assert set(np.unique(train_mask)).issubset({0, 1})
        assert set(np.unique(val_mask)).issubset({0, 1})
        assert set(np.unique(test_mask)).issubset({0, 1})

        # 4. Missing positions must be encoded as zero
        assert np.all(
            train_scaled[train_mask == 0] == 0
        )

        assert np.all(
            val_scaled[val_mask == 0] == 0
        )

        assert np.all(
            test_scaled[test_mask == 0] == 0
        )

        # 5. Scaler source
        assert scaler["fit_source"] == "TRAIN_ONLY"

        # 6. Scaler parameters must be finite
        assert np.isfinite(scaler["mean"]).all()
        assert np.isfinite(scaler["std"]).all()

        # 7. Minimum observed AP count
        for split_name in [
            "TRAIN",
            "VALIDATION",
            "TEST"
        ]:

            n_obs = prepared_splits[
                split_name
            ]["N_OBSERVED"]

            valid_count = int(
                (n_obs >= MIN_VALID_AP).sum()
            )

            total_count = len(n_obs)

            normalization_audit.append({

                "Seed": seed,
                "Floor": floor,
                "Split": split_name,

                "Rows": total_count,

                "Mean_Observed_AP": float(
                    np.mean(n_obs)
                ),

                "Min_Observed_AP": int(
                    np.min(n_obs)
                ),

                "Invalid_Fingerprint": (
                    total_count - valid_count
                ),

                "Coverage_%": (
                    100 * valid_count / total_count
                    if total_count else 0
                ),

                "Scaler_Fit_Source": "TRAIN_ONLY",

                "Status": "PASS"
            })

        # --------------------------------------------
        # SAVE PREPARED DATA
        # --------------------------------------------

        MULTI_PREPARED[floor][seed] = {

            "SPLITS": prepared_splits,

            "SCALER": scaler,

            "AP_COLS": list(AP_COLS),

            "CONFIG": {
                "SEED": seed,
                "FLOOR": floor,
                "RSSI_MIN": RSSI_MIN,
                "RSSI_MAX": RSSI_MAX,
                "MIN_VALID_AP": MIN_VALID_AP,
                "SCALER_FIT_SOURCE": "TRAIN_ONLY"
            }
        }

        print(
            f"Seed {seed} | "
            f"TRAIN={len(train_scaled)} | "
            f"VAL={len(val_scaled)} | "
            f"TEST={len(test_scaled)} | "
            f"Scaler=TRAIN_ONLY | PASS"
        )


# ------------------------------------------------------------
# 6. BUILD NORMALIZATION AUDIT
# ------------------------------------------------------------

normalization_audit_df = pd.DataFrame(
    normalization_audit
)

normalization_audit_df.to_csv(
    MULTI_SEED_DIR /
    "multi_seed_normalization_audit.csv",
    index=False
)

display(normalization_audit_df)


# ------------------------------------------------------------
# 7. SUMMARY
# ------------------------------------------------------------

total_prepared = (
    len(FLOORS)
    * len(MULTI_SEEDS)
    * 3
)

passed = int(
    (normalization_audit_df["Status"] == "PASS").sum()
)

print("\n" + "=" * 75)
print("CELL 16B — NORMALIZATION SUMMARY")
print("=" * 75)

print("Floors:", len(FLOORS))
print("Seeds:", len(MULTI_SEEDS))
print("Splits per seed:", 3)
print("Total prepared splits:", total_prepared)
print("PASS:", passed)
print("FAIL:", total_prepared - passed)

print("\nScaler policy: FIT ON TRAIN ONLY")
print("Missing RSSI: MASKED, NOT IMPUTED AS OBSERVED")
print("TEST used for scaler fitting: NO")

if passed == total_prepared:

    print("\nCELL 16B: PASS")

else:

    print("\nCELL 16B: CHECK REQUIRED")

print(
    "Output:",
    MULTI_SEED_DIR /
    "multi_seed_normalization_audit.csv"
)

In [ ]:
# ============================================================
# CELL 16C — MULTI-SEED CGAN TRAINING
#              + TRADITIONAL AUGMENTATION
# ============================================================

import copy
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
from torch.utils.data import TensorDataset, DataLoader
from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

MULTI_CGAN_EPOCHS = CGAN_EPOCHS
MULTI_CGAN_BATCH_SIZE = CGAN_BATCH_SIZE
MULTI_CGAN_LR = CGAN_LR
MULTI_CGAN_PATIENCE = CGAN_PATIENCE

MULTI_NOISE_DIM = CGAN_NOISE_DIM
MULTI_HIDDEN_DIM = CGAN_HIDDEN_DIM

LAMBDA_RSSI = 10.0
LAMBDA_MASK = 2.0

N_VALIDATION_NOISE = 5

APMEAN_RADII = [3, 5, 7]
KNN_VALUES = [3, 5, 7]

MULTI_CGAN_DIR = (
    MULTI_SEED_DIR / "CGAN_CHECKPOINTS"
)

MULTI_SYNTHETIC_DIR = (
    MULTI_SEED_DIR / "SYNTHETIC_DATA"
)

MULTI_CGAN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MULTI_SYNTHETIC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. CHECK REQUIRED INPUTS
# ------------------------------------------------------------

assert len(MULTI_PREPARED) == len(FLOORS)

assert all(
    seed in MULTI_PREPARED[floor]
    for floor in FLOORS
    for seed in MULTI_SEEDS
)

assert len(AP_COLS) == 17

print("Input preparation: PASS")
print("Device:", DEVICE)
print("Seeds:", MULTI_SEEDS)


# ------------------------------------------------------------
# 3. REPRODUCIBILITY
# ------------------------------------------------------------

def set_experiment_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ------------------------------------------------------------
# 4. CGAN ARCHITECTURE — LAYERNORM
# ------------------------------------------------------------

class WiFiGenerator(nn.Module):

    def __init__(
        self,
        noise_dim=32,
        hidden_dim=128,
        coord_dim=2,
        ap_dim=17
    ):

        super().__init__()

        self.noise_dim = noise_dim

        self.network = nn.Sequential(

            nn.Linear(
                coord_dim + noise_dim,
                hidden_dim
            ),

            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(0.2),

            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(0.2),

            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(0.2)
        )

        self.rssi_head = nn.Sequential(
            nn.Linear(hidden_dim, ap_dim),
            nn.Tanh()
        )

        self.mask_head = nn.Sequential(
            nn.Linear(hidden_dim, ap_dim),
            nn.Sigmoid()
        )

    def forward(self, coords, noise):

        x = torch.cat(
            [coords, noise],
            dim=1
        )

        hidden = self.network(x)

        return (
            self.rssi_head(hidden),
            self.mask_head(hidden)
        )


class WiFiDiscriminator(nn.Module):

    def __init__(
        self,
        hidden_dim=128,
        coord_dim=2,
        ap_dim=17
    ):

        super().__init__()

        input_dim = coord_dim + 2 * ap_dim

        self.network = nn.Sequential(

            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.15),

            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.15),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.2),

            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, coords, rssi, mask):

        x = torch.cat(
            [coords, rssi, mask],
            dim=1
        )

        return self.network(x)


# ------------------------------------------------------------
# 5. CGAN NORMALIZATION — TRAIN ONLY
# ------------------------------------------------------------

def fit_cgan_normalization(train_df):

    rssi = train_df[AP_COLS].to_numpy(float)

    if not np.isfinite(rssi).any():
        raise ValueError("No observed TRAIN RSSI.")

    rssi_min = np.nanmin(rssi, axis=0)
    rssi_max = np.nanmax(rssi, axis=0)

    rssi_range = rssi_max - rssi_min
    rssi_range[rssi_range < 1e-8] = 1.0

    xy = train_df[["X", "Y"]].to_numpy(float)

    xy_min = xy.min(axis=0)
    xy_max = xy.max(axis=0)

    xy_range = xy_max - xy_min
    xy_range[xy_range < 1e-8] = 1.0

    return {
        "rssi_min": rssi_min,
        "rssi_range": rssi_range,
        "xy_min": xy_min,
        "xy_range": xy_range
    }


def transform_cgan_frame(df, norm):

    rssi = df[AP_COLS].to_numpy(float)

    mask = np.isfinite(rssi).astype(np.float32)

    # Missing RSSI must not enter as NaN.
    safe_rssi = np.where(
        mask == 1,
        rssi,
        norm["rssi_min"]
    )

    rssi_scaled = (
        2 * (
            safe_rssi - norm["rssi_min"]
        ) / norm["rssi_range"]
    ) - 1

    rssi_scaled = np.where(
        mask == 1,
        rssi_scaled,
        0.0
    )

    xy = df[["X", "Y"]].to_numpy(float)

    xy_scaled = (
        2 * (xy - norm["xy_min"])
        / norm["xy_range"]
    ) - 1

    return {
        "coords": torch.tensor(
            xy_scaled,
            dtype=torch.float32,
            device=DEVICE
        ),

        "rssi": torch.tensor(
            rssi_scaled,
            dtype=torch.float32,
            device=DEVICE
        ),

        "mask": torch.tensor(
            mask,
            dtype=torch.float32,
            device=DEVICE
        )
    }


# ------------------------------------------------------------
# 6. LOSS FUNCTIONS
# ------------------------------------------------------------

def masked_rssi_loss(
    fake_rssi,
    real_rssi,
    real_mask
):

    error = torch.abs(
        fake_rssi - real_rssi
    )

    return (
        (error * real_mask).sum()
        / real_mask.sum().clamp(min=1.0)
    )


def observation_mask_loss(
    fake_mask_probability,
    real_mask
):

    return F.binary_cross_entropy(
        fake_mask_probability,
        real_mask
    )


def generator_adversarial_loss(fake_score):

    return F.binary_cross_entropy_with_logits(
        fake_score,
        torch.ones_like(fake_score)
    )


def discriminator_loss(real_score, fake_score):

    real_loss = F.binary_cross_entropy_with_logits(
        real_score,
        torch.ones_like(real_score)
    )

    fake_loss = F.binary_cross_entropy_with_logits(
        fake_score,
        torch.zeros_like(fake_score)
    )

    return 0.5 * (
        real_loss + fake_loss
    )


def total_generator_loss(
    adv_loss,
    rssi_loss,
    mask_loss
):

    return (
        adv_loss
        + LAMBDA_RSSI * rssi_loss
        + LAMBDA_MASK * mask_loss
    )


def straight_through_mask(probability):

    hard_mask = (
        probability >= 0.5
    ).float()

    return (
        hard_mask
        - probability.detach()
        + probability
    )


def sample_binary_mask(probability):

    return torch.bernoulli(
        probability.detach()
    )


# ------------------------------------------------------------
# 7. TRAINING STEPS
# ------------------------------------------------------------

def train_discriminator_step(
    G,
    D,
    optimizer_D,
    coords,
    real_rssi,
    real_mask
):

    batch_size = len(coords)

    noise = torch.randn(
        batch_size,
        MULTI_NOISE_DIM,
        device=DEVICE
    )

    with torch.no_grad():

        fake_rssi, fake_mask_prob = G(
            coords,
            noise
        )

        fake_mask = sample_binary_mask(
            fake_mask_prob
        )

        fake_input_rssi = (
            fake_rssi * fake_mask
        )

    real_score = D(
        coords,
        real_rssi,
        real_mask
    )

    fake_score = D(
        coords,
        fake_input_rssi,
        fake_mask
    )

    d_loss = discriminator_loss(
        real_score,
        fake_score
    )

    optimizer_D.zero_grad(set_to_none=True)

    d_loss.backward()

    torch.nn.utils.clip_grad_norm_(
        D.parameters(),
        max_norm=5.0
    )

    optimizer_D.step()

    return float(d_loss.item())


def train_generator_step(
    G,
    D,
    optimizer_G,
    coords,
    real_rssi,
    real_mask
):

    batch_size = len(coords)

    noise = torch.randn(
        batch_size,
        MULTI_NOISE_DIM,
        device=DEVICE
    )

    fake_rssi, fake_mask_prob = G(
        coords,
        noise
    )

    fake_mask_st = straight_through_mask(
        fake_mask_prob
    )

    fake_input_rssi = (
        fake_rssi * fake_mask_st
    )

    fake_score = D(
        coords,
        fake_input_rssi,
        fake_mask_st
    )

    adv_loss = generator_adversarial_loss(
        fake_score
    )

    rssi_loss = masked_rssi_loss(
        fake_rssi,
        real_rssi,
        real_mask
    )

    mask_loss = observation_mask_loss(
        fake_mask_prob,
        real_mask
    )

    g_loss = total_generator_loss(
        adv_loss,
        rssi_loss,
        mask_loss
    )

    optimizer_G.zero_grad(set_to_none=True)

    g_loss.backward()

    torch.nn.utils.clip_grad_norm_(
        G.parameters(),
        max_norm=5.0
    )

    optimizer_G.step()

    return {
        "G_Loss": float(g_loss.item()),
        "G_Adversarial": float(adv_loss.item()),
        "G_RSSI": float(rssi_loss.item()),
        "G_Mask": float(mask_loss.item())
    }


# ------------------------------------------------------------
# 8. VALIDATION
# ------------------------------------------------------------

def evaluate_cgan_validation(
    G,
    validation_data,
    noise_bank
):

    G.eval()

    rssi_values = []
    mask_values = []

    with torch.no_grad():

        for noise in noise_bank:

            fake_rssi, fake_mask_prob = G(
                validation_data["coords"],
                noise
            )

            rssi_loss = masked_rssi_loss(
                fake_rssi,
                validation_data["rssi"],
                validation_data["mask"]
            )

            mask_loss = observation_mask_loss(
                fake_mask_prob,
                validation_data["mask"]
            )

            rssi_values.append(
                float(rssi_loss.item())
            )

            mask_values.append(
                float(mask_loss.item())
            )

    mean_rssi = float(np.mean(rssi_values))
    mean_mask = float(np.mean(mask_values))

    return {
        "Val_RSSI": mean_rssi,
        "Val_Mask": mean_mask,
        "Val_Score": (
            LAMBDA_RSSI * mean_rssi
            + LAMBDA_MASK * mean_mask
        )
    }


# ------------------------------------------------------------
# 9. COORDINATE NORMALIZATION
# ------------------------------------------------------------

def normalize_coordinates(df, norm):

    xy = df[["X", "Y"]].to_numpy(float)

    return (
        2 * (xy - norm["xy_min"])
        / norm["xy_range"]
    ) - 1


# ------------------------------------------------------------
# 10. CGAN TRAINING — 15 MODELS
# ------------------------------------------------------------

MULTI_CGAN_MODELS = {}
MULTI_CGAN_SUMMARY = []
MULTI_CGAN_HISTORIES = {}

for floor in FLOORS:

    MULTI_CGAN_MODELS[floor] = {}

    for seed in MULTI_SEEDS:

        print("\n" + "=" * 75)
        print(f"CGAN TRAINING | {floor} | SEED {seed}")
        print("=" * 75)

        set_experiment_seed(seed)

        split = (
            MULTI_SEED_DATA[floor]["SEEDS"][seed]
        )

        train_df = split["TRAIN"].copy()
        val_df = split["VALIDATION"].copy()

        # Only fingerprints with >= MIN_VALID_AP
        # are retained for cGAN training.
        train_rssi = train_df[AP_COLS].to_numpy(float)
        val_rssi = val_df[AP_COLS].to_numpy(float)

        train_valid = (
            np.isfinite(train_rssi).sum(axis=1)
            >= MIN_VALID_AP
        )

        val_valid = (
            np.isfinite(val_rssi).sum(axis=1)
            >= MIN_VALID_AP
        )

        train_frame = train_df.loc[
            train_valid
        ].reset_index(drop=True)

        val_frame = val_df.loc[
            val_valid
        ].reset_index(drop=True)

        if len(train_frame) < 4 or len(val_frame) < 1:
            raise ValueError(
                f"{floor}/{seed}: Insufficient valid data."
            )

        norm = fit_cgan_normalization(
            train_frame
        )

        train_tensor = transform_cgan_frame(
            train_frame,
            norm
        )

        val_tensor = transform_cgan_frame(
            val_frame,
            norm
        )

        G = WiFiGenerator(
            noise_dim=MULTI_NOISE_DIM,
            hidden_dim=MULTI_HIDDEN_DIM,
            ap_dim=len(AP_COLS)
        ).to(DEVICE)

        D = WiFiDiscriminator(
            hidden_dim=MULTI_HIDDEN_DIM,
            ap_dim=len(AP_COLS)
        ).to(DEVICE)

        dataset = TensorDataset(
            train_tensor["coords"],
            train_tensor["rssi"],
            train_tensor["mask"]
        )

        loader_generator = torch.Generator()
        loader_generator.manual_seed(seed)

        loader = DataLoader(
            dataset,
            batch_size=MULTI_CGAN_BATCH_SIZE,
            shuffle=True,
            drop_last=False,
            generator=loader_generator
        )

        optimizer_G = torch.optim.Adam(
            G.parameters(),
            lr=MULTI_CGAN_LR,
            betas=(0.5, 0.999)
        )

        optimizer_D = torch.optim.Adam(
            D.parameters(),
            lr=MULTI_CGAN_LR,
            betas=(0.5, 0.999)
        )

        val_noise_bank = [
            torch.randn(
                len(val_tensor["coords"]),
                MULTI_NOISE_DIM,
                device=DEVICE
            )
            for _ in range(N_VALIDATION_NOISE)
        ]

        best_score = float("inf")
        best_epoch = 0
        patience_counter = 0
        best_state = None

        history = []
        start_time = time.time()

        for epoch in range(
            1,
            MULTI_CGAN_EPOCHS + 1
        ):

            G.train()
            D.train()

            epoch_d = []
            epoch_g = []
            epoch_adv = []
            epoch_rssi = []
            epoch_mask = []

            for coords, real_rssi_batch, real_mask_batch in loader:

                d_value = train_discriminator_step(
                    G,
                    D,
                    optimizer_D,
                    coords,
                    real_rssi_batch,
                    real_mask_batch
                )

                g_values = train_generator_step(
                    G,
                    D,
                    optimizer_G,
                    coords,
                    real_rssi_batch,
                    real_mask_batch
                )

                epoch_d.append(d_value)
                epoch_g.append(g_values["G_Loss"])
                epoch_adv.append(g_values["G_Adversarial"])
                epoch_rssi.append(g_values["G_RSSI"])
                epoch_mask.append(g_values["G_Mask"])

            val_result = evaluate_cgan_validation(
                G,
                val_tensor,
                val_noise_bank
            )

            row = {
                "Epoch": epoch,
                "D_Loss": np.mean(epoch_d),
                "G_Loss": np.mean(epoch_g),
                "G_Adversarial": np.mean(epoch_adv),
                "G_RSSI": np.mean(epoch_rssi),
                "G_Mask": np.mean(epoch_mask),
                **val_result
            }

            history.append(row)

            if val_result["Val_Score"] < best_score:

                best_score = val_result["Val_Score"]
                best_epoch = epoch
                patience_counter = 0

                best_state = {
                    "G": copy.deepcopy(G.state_dict()),
                    "D": copy.deepcopy(D.state_dict()),
                    "epoch": epoch,
                    "validation_score": best_score,
                    "seed": seed,
                    "floor": floor
                }

            else:

                patience_counter += 1

            if (
                epoch == 1
                or epoch % 25 == 0
                or patience_counter >= MULTI_CGAN_PATIENCE
            ):

                print(
                    f"Epoch {epoch:03d} | "
                    f"G={row['G_Loss']:.4f} | "
                    f"D={row['D_Loss']:.4f} | "
                    f"Val={row['Val_Score']:.4f} | "
                    f"Best={best_epoch}"
                )

            if patience_counter >= MULTI_CGAN_PATIENCE:

                print(
                    f"Early stopping at epoch {epoch}"
                )

                break

        if best_state is None:
            raise RuntimeError(
                f"No best checkpoint for {floor}/{seed}"
            )

        G.load_state_dict(best_state["G"])
        D.load_state_dict(best_state["D"])

        elapsed = time.time() - start_time

        # Save checkpoint per floor and seed
        checkpoint_path = (
            MULTI_CGAN_DIR
            / f"cgan_{floor}_seed_{seed}.pt"
        )

        torch.save(
            best_state,
            checkpoint_path
        )

        history_df = pd.DataFrame(history)

        history_df.to_csv(
            MULTI_CGAN_DIR
            / f"history_{floor}_seed_{seed}.csv",
            index=False
        )

        MULTI_CGAN_MODELS[floor][seed] = {
            "G": G,
            "D": D,
            "NORMALIZATION": norm,
            "TRAIN_FRAME": train_frame,
            "VALIDATION_FRAME": val_frame,
            "BEST_STATE": best_state
        }

        MULTI_CGAN_HISTORIES[
            (floor, seed)
        ] = history_df

        MULTI_CGAN_SUMMARY.append({

            "Floor": floor,
            "Seed": seed,

            "TRAIN_Fingerprints": len(train_frame),
            "VALIDATION_Fingerprints": len(val_frame),

            "Best_Epoch": best_epoch,
            "Epochs_Run": len(history),

            "Best_Val_Score": best_score,

            "Best_Val_RSSI": history[
                best_epoch - 1
            ]["Val_RSSI"],

            "Best_Val_Mask": history[
                best_epoch - 1
            ]["Val_Mask"],

            "Training_Time_Seconds": elapsed,

            "Checkpoint": str(checkpoint_path),

            "Status": "COMPLETED"
        })

        print(
            f"COMPLETED {floor} | seed={seed} | "
            f"best_epoch={best_epoch} | "
            f"time={elapsed:.1f}s"
        )


# ------------------------------------------------------------
# 11. CGAN TRAINING SUMMARY
# ------------------------------------------------------------

multi_cgan_summary_df = pd.DataFrame(
    MULTI_CGAN_SUMMARY
)

multi_cgan_summary_df.to_csv(
    MULTI_CGAN_DIR / "multi_seed_cgan_summary.csv",
    index=False
)

display(multi_cgan_summary_df)

print("\nCGAN MODELS COMPLETED:",
      len(multi_cgan_summary_df), "/ 15")


# ------------------------------------------------------------
# 12. TRADITIONAL AUGMENTATION HELPERS
# ------------------------------------------------------------

def build_traditional_fingerprint(
    train_df,
    target_row,
    method,
    k=None,
    radius=None
):

    target_xy = target_row[
        ["X", "Y"]
    ].to_numpy(dtype=float)

    # Exclude the target RP itself
    neighbors = train_df[
        train_df["ID_RP"] != target_row["ID_RP"]
    ].copy()

    if len(neighbors) == 0:
        return None

    neighbor_xy = neighbors[
        ["X", "Y"]
    ].to_numpy(dtype=float)

    distances = np.linalg.norm(
        neighbor_xy - target_xy,
        axis=1
    )

    neighbors["_DISTANCE"] = distances

    if method.startswith("KNN"):

        neighbors = neighbors.sort_values(
            "_DISTANCE"
        ).head(k)

    elif method.startswith("APMEAN"):

        neighbors = neighbors[
            neighbors["_DISTANCE"] <= radius
        ]

    if len(neighbors) == 0:
        return None

    # AP-wise mean, preserving missingness if
    # no selected neighbor observed that AP.
    synthetic_rssi = (
        neighbors[AP_COLS]
        .mean(axis=0, skipna=True)
    )

    synthetic_rssi = synthetic_rssi.replace(
        [np.inf, -np.inf],
        np.nan
    )

    if synthetic_rssi.notna().sum() < MIN_VALID_AP:
        return None

    output = {
        "Floor": target_row["Floor"],
        "ID_RP": target_row["ID_RP"],
        "X": target_row["X"],
        "Y": target_row["Y"],
        "Data_Source": method,
        "Method": method,
        "Neighbor_Count": len(neighbors)
    }

    output.update(
        synthetic_rssi.to_dict()
    )

    return output


# ------------------------------------------------------------
# 13. BUILD CGAN SYNTHETIC DATA
# ------------------------------------------------------------

MULTI_SYNTHETIC = {}
synthetic_audit_rows = []

for floor in FLOORS:

    MULTI_SYNTHETIC[floor] = {}

    for seed in MULTI_SEEDS:

        print(
            f"\nGenerating CGAN: {floor} | seed={seed}"
        )

        model_pack = MULTI_CGAN_MODELS[floor][seed]

        G = model_pack["G"]
        norm = model_pack["NORMALIZATION"]

        train_frame = model_pack[
            "TRAIN_FRAME"
        ].copy()

        G.eval()

        coords_np = normalize_coordinates(
            train_frame,
            norm
        )

        coords_tensor = torch.tensor(
            coords_np,
            dtype=torch.float32,
            device=DEVICE
        )

        # Independent generation seed
        generation_seed = seed + 100000

        torch.manual_seed(generation_seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(
                generation_seed
            )

        noise = torch.randn(
            len(train_frame),
            MULTI_NOISE_DIM,
            device=DEVICE
        )

        with torch.no_grad():

            fake_rssi_scaled, fake_mask_prob = G(
                coords_tensor,
                noise
            )

            fake_mask = torch.bernoulli(
                fake_mask_prob
            )

        fake_rssi_scaled = (
            fake_rssi_scaled.cpu().numpy()
        )

        fake_mask = (
            fake_mask.cpu().numpy()
        )

        # Inverse Min-Max normalization
        fake_rssi = (
            (fake_rssi_scaled + 1) / 2
        ) * norm["rssi_range"] + norm["rssi_min"]

        # Masked APs become NaN
        fake_rssi = np.where(
            fake_mask >= 0.5,
            fake_rssi,
            np.nan
        )

        # Clip to per-AP TRAIN range
        fake_rssi = np.clip(
            fake_rssi,
            norm["rssi_min"],
            norm["rssi_min"] + norm["rssi_range"]
        )

        cgan_rows = []

        for i, (_, source_row) in enumerate(
            train_frame.iterrows()
        ):

            if (
                np.isfinite(fake_rssi[i]).sum()
                < MIN_VALID_AP
            ):
                continue

            record = {
                "Floor": floor,
                "ID_RP": source_row["ID_RP"],
                "X": source_row["X"],
                "Y": source_row["Y"],
                "Method": "CGAN",
                "Data_Source": "CGAN",
                "Neighbor_Count": np.nan
            }

            for j, ap in enumerate(AP_COLS):

                record[ap] = (
                    float(fake_rssi[i, j])
                    if np.isfinite(fake_rssi[i, j])
                    else np.nan
                )

            cgan_rows.append(record)

        cgan_df = pd.DataFrame(cgan_rows)

        MULTI_SYNTHETIC[floor].setdefault(
            seed, {}
        )

        MULTI_SYNTHETIC[floor][seed]["CGAN"] = cgan_df

        synthetic_audit_rows.append({

            "Floor": floor,
            "Seed": seed,
            "Method": "CGAN",

            "Expected_Candidates": len(train_frame),
            "Generated_Rows": len(cgan_df),

            "Valid_Rows": int(
                (
                    cgan_df[AP_COLS]
                    .notna()
                    .sum(axis=1)
                    >= MIN_VALID_AP
                ).sum()
            ) if len(cgan_df) else 0,

            "Mean_Observed_AP": (
                cgan_df[AP_COLS]
                .notna()
                .sum(axis=1)
                .mean()
                if len(cgan_df) else 0
            ),

            "Status": (
                "PASS"
                if len(cgan_df) > 0
                else "FAIL"
            )
        })

        print(
            f"CGAN generated: {len(cgan_df)} / "
            f"{len(train_frame)}"
        )


# ------------------------------------------------------------
# 14. BUILD KNN AND APMEAN SYNTHETIC DATA
# ------------------------------------------------------------

for floor in FLOORS:

    for seed in MULTI_SEEDS:

        train_frame = (
            MULTI_SEED_DATA[floor]["SEEDS"][seed]["TRAIN"]
            .copy()
        )

        # Retain fingerprints with sufficient APs
        train_mask = (
            train_frame[AP_COLS]
            .notna()
            .sum(axis=1)
            >= MIN_VALID_AP
        )

        train_frame = train_frame.loc[
            train_mask
        ].reset_index(drop=True)

        method_specs = {

            "KNN_K3": {
                "type": "KNN",
                "k": 3
            },

            "KNN_K5": {
                "type": "KNN",
                "k": 5
            },

            "KNN_K7": {
                "type": "KNN",
                "k": 7
            },

            "APMEAN_R3": {
                "type": "APMEAN",
                "radius": 3
            },

            "APMEAN_R5": {
                "type": "APMEAN",
                "radius": 5
            },

            "APMEAN_R7": {
                "type": "APMEAN",
                "radius": 7
            }
        }

        for method, spec in method_specs.items():

            rows = []

            for _, target_row in train_frame.iterrows():

                if spec["type"] == "KNN":

                    record = build_traditional_fingerprint(
                        train_frame,
                        target_row,
                        method,
                        k=spec["k"]
                    )

                else:

                    record = build_traditional_fingerprint(
                        train_frame,
                        target_row,
                        method,
                        radius=spec["radius"]
                    )

                if record is not None:
                    rows.append(record)

            synthetic_df = pd.DataFrame(rows)

            MULTI_SYNTHETIC[floor][seed][method] = (
                synthetic_df
            )

            synthetic_audit_rows.append({

                "Floor": floor,
                "Seed": seed,
                "Method": method,

                "Expected_Candidates": len(train_frame),
                "Generated_Rows": len(synthetic_df),

                "Valid_Rows": int(
                    (
                        synthetic_df[AP_COLS]
                        .notna()
                        .sum(axis=1)
                        >= MIN_VALID_AP
                    ).sum()
                ) if len(synthetic_df) else 0,

                "Mean_Observed_AP": (
                    synthetic_df[AP_COLS]
                    .notna()
                    .sum(axis=1)
                    .mean()
                    if len(synthetic_df) else 0
                ),

                "Status": (
                    "PASS"
                    if len(synthetic_df) > 0
                    else "FAIL"
                )
            })


# ------------------------------------------------------------
# 15. SAVE SYNTHETIC DATA PER SEED
# ------------------------------------------------------------

for floor in FLOORS:

    for seed in MULTI_SEEDS:

        folder = (
            MULTI_SYNTHETIC_DIR
            / floor
            / f"seed_{seed}"
        )

        folder.mkdir(
            parents=True,
            exist_ok=True
        )

        for method in MULTI_METHODS[1:]:

            synthetic_df = (
                MULTI_SYNTHETIC[floor][seed][method]
            )

            synthetic_df.to_csv(
                folder / f"{method}.csv",
                index=False
            )


# ------------------------------------------------------------
# 16. SYNTHETIC AUDIT
# ------------------------------------------------------------

synthetic_audit_df = pd.DataFrame(
    synthetic_audit_rows
)

synthetic_audit_df.to_csv(
    MULTI_SEED_DIR / "multi_seed_synthetic_audit.csv",
    index=False
)

display(synthetic_audit_df)

print("\n" + "=" * 75)
print("CELL 16C — SUMMARY")
print("=" * 75)

print(
    "CGAN models completed:",
    len(multi_cgan_summary_df),
    "/ 15"
)

print(
    "Synthetic configurations:",
    len(synthetic_audit_df),
    "/ 105"
)

print(
    "Synthetic rows:",
    int(synthetic_audit_df["Generated_Rows"].sum())
)

print(
    "Configurations with generated data:",
    int((synthetic_audit_df["Status"] == "PASS").sum())
)

print(
    "Configurations requiring review:",
    int((synthetic_audit_df["Status"] != "PASS").sum())
)

print(
    "\nOutput directory:",
    MULTI_SYNTHETIC_DIR
)

In [ ]:
# ============================================================
# CELL 16D — AUGMENTED DATASET INTEGRITY AUDIT
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

AUGMENTED_REFERENCE_MULTI = {}

audit_rows = []
manifest_rows = []

METHODS_MULTI = [
    "ORIGINAL",
    "CGAN",
    "KNN_K3",
    "KNN_K5",
    "KNN_K7",
    "APMEAN_R3",
    "APMEAN_R5",
    "APMEAN_R7"
]

for floor in FLOORS:

    AUGMENTED_REFERENCE_MULTI[floor] = {}

    for seed in MULTI_SEEDS:

        train_df = (
            MULTI_SEED_DATA[floor]["SEEDS"][seed]["TRAIN"]
            .copy()
            .reset_index(drop=True)
        )

        train_df["Data_Source"] = "ORIGINAL"
        train_df["Method"] = "ORIGINAL"

        original_rps = set(
            train_df["ID_RP"]
        )

        AUGMENTED_REFERENCE_MULTI[floor][seed] = {}

        for method in METHODS_MULTI:

            if method == "ORIGINAL":

                combined = train_df.copy()

            else:

                synthetic = (
                    MULTI_SYNTHETIC[floor][seed][method]
                    .copy()
                    .reset_index(drop=True)
                )

                # ----------------------------------------
                # Coordinate and RP verification
                # ----------------------------------------

                synthetic_rps = set(
                    synthetic["ID_RP"]
                )

                unknown_rps = (
                    synthetic_rps - original_rps
                )

                missing_rps = (
                    original_rps - synthetic_rps
                )

                # Compare coordinates with TRAIN RP
                coordinate_map = (
                    train_df.drop_duplicates("ID_RP")
                    .set_index("ID_RP")[["X", "Y"]]
                )

                coordinate_errors = []

                for _, row in synthetic.iterrows():

                    rp = row["ID_RP"]

                    if rp not in coordinate_map.index:
                        coordinate_errors.append(np.inf)
                        continue

                    expected_xy = (
                        coordinate_map.loc[rp]
                        .to_numpy(float)
                    )

                    actual_xy = (
                        row[["X", "Y"]]
                        .to_numpy(float)
                    )

                    coordinate_errors.append(
                        np.max(
                            np.abs(expected_xy - actual_xy)
                        )
                    )

                max_coordinate_error = (
                    max(coordinate_errors)
                    if coordinate_errors else np.nan
                )

                # ----------------------------------------
                # Provenance
                # ----------------------------------------

                synthetic["Data_Source"] = method
                synthetic["Method"] = method

                combined = pd.concat(
                    [train_df, synthetic],
                    ignore_index=True
                )

            # --------------------------------------------
            # Structural checks
            # --------------------------------------------

            original_count = int(
                (combined["Data_Source"] == "ORIGINAL").sum()
            )

            synthetic_count = int(
                (combined["Data_Source"] == method).sum()
            ) if method != "ORIGINAL" else 0

            observed = (
                combined[AP_COLS]
                .notna()
                .sum(axis=1)
            )

            invalid_count = int(
                (observed < MIN_VALID_AP).sum()
            )

            duplicate_count = int(
                combined.duplicated(
                    subset=[
                        "ID_RP",
                        "Data_Source"
                    ] + AP_COLS
                ).sum()
            )

            expected_original = len(train_df)

            expected_synthetic = (
                0 if method == "ORIGINAL"
                else len(train_df)
            )

            checks = {

                "Original_Count": (
                    original_count == expected_original
                ),

                "Synthetic_Count": (
                    synthetic_count == expected_synthetic
                ),

                "RP_Set": (
                    set(combined["ID_RP"]) == original_rps
                ),

                "Minimum_AP": (
                    invalid_count == 0
                ),

                "No_Duplicates": (
                    duplicate_count == 0
                ),

                "No_Unknown_Source": (
                    set(combined["Data_Source"])
                    == (
                        {"ORIGINAL"}
                        if method == "ORIGINAL"
                        else {"ORIGINAL", method}
                    )
                )
            }

            if method != "ORIGINAL":

                checks["Synthetic_RP_Complete"] = (
                    len(unknown_rps) == 0
                    and len(missing_rps) == 0
                )

                checks["Coordinate_Match"] = (
                    np.isfinite(max_coordinate_error)
                    and max_coordinate_error < 1e-8
                )

            failed = [
                key for key, value in checks.items()
                if not value
            ]

            status = (
                "PASS" if len(failed) == 0
                else "FAIL"
            )

            AUGMENTED_REFERENCE_MULTI[
                floor
            ][seed][method] = combined

            audit_rows.append({

                "Floor": floor,
                "Seed": seed,
                "Method": method,

                "Rows": len(combined),

                "Unique_RP": combined[
                    "ID_RP"
                ].nunique(),

                "Original_Count": original_count,

                "Synthetic_Count": synthetic_count,

                "Mean_Observed_AP": observed.mean(),

                "Missing_RSSI_%": (
                    100 * (
                        1 - combined[AP_COLS]
                        .notna()
                        .to_numpy()
                        .mean()
                    )
                ),

                "Invalid_Fingerprints": invalid_count,

                "Duplicate_Rows": duplicate_count,

                "Failed_Checks": (
                    ", ".join(failed)
                    if failed else "None"
                ),

                "Status": status
            })

            manifest_rows.append({

                "Floor": floor,
                "Seed": seed,
                "Method": method,
                "Rows": len(combined),
                "Unique_RP": combined[
                    "ID_RP"
                ].nunique(),
                "Status": status
            })


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

multi_audit_df = pd.DataFrame(audit_rows)
multi_manifest_df = pd.DataFrame(manifest_rows)

multi_audit_df.to_csv(
    MULTI_SEED_DIR / "multi_seed_dataset_audit.csv",
    index=False
)

multi_manifest_df.to_csv(
    MULTI_SEED_DIR / "multi_seed_dataset_manifest.csv",
    index=False
)

display(multi_audit_df)

total = len(multi_audit_df)

passed = int(
    (multi_audit_df["Status"] == "PASS").sum()
)

failed = total - passed

print("\n" + "=" * 75)
print("CELL 16D — FINAL AUDIT")
print("=" * 75)

print("Total configurations:", total)
print("PASS:", passed)
print("FAIL:", failed)

print(
    "Expected configurations:",
    len(FLOORS) * len(MULTI_SEEDS) * len(METHODS_MULTI)
)

if passed == total:
    print("\nCELL 16D: PASS")
else:
    print("\nCELL 16D: CHECK REQUIRED")
    display(
        multi_audit_df[
            multi_audit_df["Status"] != "PASS"
        ]
    )

In [ ]:
# ============================================================
# CELL 16E-0 — CHECK VARIABLES
# Không thay đổi dữ liệu hoặc mô hình
# ============================================================

import pandas as pd

keywords = [
    "MULTI_SEED",
    "AUGMENTED_REFERENCE",
    "SEED",
    "SCALER",
    "WKNN",
    "AP_COL",
    "FLOOR",
    "METHOD",
    "CONFIG",
]

found = []

for name, obj in globals().copy().items():
    if any(key in name.upper() for key in keywords):
        try:
            if isinstance(obj, pd.DataFrame):
                detail = f"DataFrame {obj.shape}"
            elif isinstance(obj, dict):
                detail = f"dict ({len(obj)} keys)"
            elif callable(obj):
                detail = "function"
            else:
                detail = type(obj).__name__

            found.append((name, detail))
        except Exception:
            pass

print("=" * 75)
print("CELL 16E-0 — VARIABLE INSPECTION")
print("=" * 75)

for name, detail in sorted(found):
    print(f"{name:<45} {detail}")

print("\nCác hàm WKNN hiện có:")
for name, obj in globals().copy().items():
    if callable(obj) and "wknn" in name.lower():
        print(" -", name)

print("\nHoàn tất kiểm tra.")

In [ ]:
# ============================================================
# CELL 16E-1 — WKNN INTERFACE & DATA STRUCTURE AUDIT
# ============================================================

import inspect
import pandas as pd
import numpy as np

print("=" * 80)
print("1. EXPERIMENT CONFIGURATION")
print("=" * 80)

print("Seeds:", MULTI_SEEDS)
print("Floors:", FLOORS)
print("Methods:", METHODS_MULTI)
print("AP columns:", AP_COLS)

print("\n" + "=" * 80)
print("2. WKNN FUNCTION SIGNATURE")
print("=" * 80)

print(inspect.signature(wknn))
print("\n--- SOURCE CODE ---")
print(inspect.getsource(wknn))

print("\n" + "=" * 80)
print("3. MULTI_SEED_DATA STRUCTURE")
print("=" * 80)

for f in FLOORS:
    print(f"\nFloor: {f}")
    print("Keys:", list(MULTI_SEED_DATA[f].keys()))

    for key, value in MULTI_SEED_DATA[f].items():
        if isinstance(value, dict):
            print(f"  {key}: dict keys={list(value.keys())[:10]}")
        elif isinstance(value, pd.DataFrame):
            print(f"  {key}: DataFrame {value.shape}")
            print("  Columns:", value.columns.tolist())
        else:
            print(f"  {key}: {type(value).__name__}")

print("\n" + "=" * 80)
print("4. AUGMENTED_REFERENCE_MULTI STRUCTURE")
print("=" * 80)

for f in FLOORS:
    print(f"\nFloor: {f}")
    print("Seed keys:", list(AUGMENTED_REFERENCE_MULTI[f].keys()))

    sample_seed = list(AUGMENTED_REFERENCE_MULTI[f].keys())[0]
    seed_data = AUGMENTED_REFERENCE_MULTI[f][sample_seed]

    print("Sample seed:", sample_seed)
    print("Methods:", list(seed_data.keys()))

    for method_name, df in seed_data.items():
        if isinstance(df, pd.DataFrame):
            print(
                f"  {method_name}: shape={df.shape}, "
                f"columns={df.columns.tolist()[:10]}"
            )

print("\n" + "=" * 80)
print("5. SCALER STRUCTURE")
print("=" * 80)

print("scalers type:", type(scalers).__name__)
print("scalers keys:", list(scalers.keys()))

for key, value in scalers.items():
    print(f"\nScaler key: {key}")
    print("Type:", type(value).__name__)

    if isinstance(value, dict):
        print("Nested keys:", list(value.keys()))

        for subkey, subvalue in value.items():
            print(
                f"  {subkey}: "
                f"{type(subvalue).__name__}"
            )

    elif hasattr(value, "mean_"):
        print("mean_ shape:", value.mean_.shape)
        print("scale_ shape:", value.scale_.shape)

print("\n" + "=" * 80)
print("6. DATA SAMPLE")
print("=" * 80)

sample_floor = FLOORS[0]
sample_seed = MULTI_SEEDS[0]

print("Floor:", sample_floor)
print("Seed:", sample_seed)

for key in ["TRAIN", "VALIDATION", "TEST"]:
    if key in MULTI_SEED_DATA[sample_floor]:
        obj = MULTI_SEED_DATA[sample_floor][key]

        if isinstance(obj, dict):
            if sample_seed in obj:
                print(f"\n{key} sample:")
                display(obj[sample_seed].head(3))
        elif isinstance(obj, pd.DataFrame):
            print(f"\n{key} sample:")
            display(obj.head(3))

print("\nCELL 16E-1 — AUDIT COMPLETED")

## CELL 16E — MULTI-SEED WKNN EVALUATION

## Mục tiêu
Đánh giá hiệu quả của 8 phương pháp định vị WKNN trên:
- 3 tầng: F1, F2, F3.
- 5 random seeds: 2026–2030.
- 8 phương pháp: ORIGINAL, CGAN, KNN_K3, KNN_K5, KNN_K7,
  APMEAN_R3, APMEAN_R5, APMEAN_R7.

## Nguyên tắc thực nghiệm
1. Scaler được fit riêng theo từng seed và tầng, chỉ từ TRAIN gốc.
2. Chọn k bằng VALIDATION trên ORIGINAL reference.
3. Giữ nguyên k cho tất cả phương pháp trong cùng seed và tầng.
4. Tất cả phương pháp dùng cùng TEST queries.
5. TEST không tham gia huấn luyện, chuẩn hóa hoặc chọn tham số.

## Chỉ số đánh giá
- Mean Error
- Median Error
- RMSE
- P90 Error
- Coverage
- Fallback Count
- Mean Common APs

Đơn vị sai số: đơn vị tọa độ của dataset; chưa mặc định là mét.

In [ ]:
# ============================================================
# CELL 16E — MULTI-SEED WKNN EVALUATION
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import time

from sklearn.preprocessing import StandardScaler
from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

MIN_COMMON_APS = 2
K_CANDIDATES = [3, 5, 7]

RESULTS_16E = []
PREDICTIONS_16E = []
K_SELECTION_16E = []

METHOD_ORDER_16E = list(METHODS_MULTI)

OUTPUT_DIR_16E = MULTI_SEED_DIR / "16E_MULTI_SEED_WKNN"
OUTPUT_DIR_16E.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("CELL 16E — MULTI-SEED WKNN EVALUATION")
print("=" * 80)

print("Seeds:", MULTI_SEEDS)
print("Floors:", FLOORS)
print("Methods:", METHOD_ORDER_16E)


# ------------------------------------------------------------
# 2. ENCODE RSSI WITH TRAIN-ONLY SCALER
# ------------------------------------------------------------

def encode_16e(df, scaler, med):
    """
    Encode RSSI:
    - NaN = AP không quan sát.
    - Median chỉ dùng để tạo ma trận đầu vào scaler.
    - Observation mask được giữ riêng.
    """

    X = df[AP_COLS].to_numpy(dtype=float)

    mask = np.isfinite(X)

    X_filled = np.where(mask, X, med)

    Z = scaler.transform(X_filled)

    return Z, mask


def fit_seed_scaler_16e(train_df):
    """
    Fit scaler chỉ trên TRAIN gốc.
    Median và mean/std không lấy từ Synthetic hoặc TEST.
    """

    X = train_df[AP_COLS].to_numpy(dtype=float)

    med = np.nanmedian(X, axis=0)

    # Xử lý AP không có quan sát trong TRAIN
    med = np.where(np.isfinite(med), med, -100.0)

    X_filled = np.where(np.isfinite(X), X, med)

    scaler = StandardScaler()
    scaler.fit(X_filled)

    return scaler, med


# ------------------------------------------------------------
# 3. WKNN VALIDATION K-SELECTION
# ------------------------------------------------------------

def evaluate_wknn_16e(reference, queries, scaler, med, k):

    pred, fallback, used = wknn(
        reference=reference,
        queries=queries,
        scaler=scaler,
        med=med,
        k=k,
        min_common_aps=MIN_COMMON_APS
    )

    true_xy = queries[["X", "Y"]].to_numpy(dtype=float)

    error = np.sqrt(
        np.sum((pred - true_xy) ** 2, axis=1)
    )

    valid = (
        np.isfinite(pred).all(axis=1)
        & np.isfinite(error)
    )

    return pred, error, valid, fallback, used


def select_k_validation_16e(
    original_reference,
    validation_df,
    scaler,
    med
):
    """
    Chọn k chỉ từ VALIDATION.
    """

    candidates = []

    for k in K_CANDIDATES:

        pred, error, valid, fallback, used = evaluate_wknn_16e(
            original_reference,
            validation_df,
            scaler,
            med,
            k
        )

        mean_error = (
            float(np.mean(error[valid]))
            if valid.any()
            else np.inf
        )

        candidates.append({
            "k": k,
            "Mean_Error": mean_error,
            "Median_Error": (
                float(np.median(error[valid]))
                if valid.any()
                else np.nan
            ),
            "Valid_Queries": int(valid.sum()),
            "Total_Queries": len(validation_df),
            "Fallback_Count": int(fallback.sum())
        })

    candidate_df = pd.DataFrame(candidates)

    # Ưu tiên Mean Error nhỏ nhất.
    # Nếu bằng nhau, ưu tiên coverage cao hơn.
    candidate_df = candidate_df.sort_values(
        ["Mean_Error", "Valid_Queries", "k"],
        ascending=[True, False, True]
    )

    best_k = int(candidate_df.iloc[0]["k"])

    return best_k, candidate_df


# ------------------------------------------------------------
# 4. MAIN EVALUATION LOOP
# ------------------------------------------------------------

start_all = time.time()

for floor in FLOORS:

    print("\n" + "=" * 80)
    print(f"PROCESSING {floor}")
    print("=" * 80)

    floor_test = MULTI_SEED_DATA[floor]["TEST"].copy()

    for seed in MULTI_SEEDS:

        print(f"\n[{floor}] Random seed = {seed}")

        seed_data = MULTI_SEED_DATA[floor]["SEEDS"][seed]

        train_df = seed_data["TRAIN"].copy()
        val_df = seed_data["VALIDATION"].copy()

        test_df = floor_test.copy()

        # ----------------------------------------------------
        # 4.1 Fit scaler from ORIGINAL TRAIN only
        # ----------------------------------------------------

        scaler, med = fit_seed_scaler_16e(train_df)

        # ----------------------------------------------------
        # 4.2 Retrieve reference datasets
        # ----------------------------------------------------

        references = AUGMENTED_REFERENCE_MULTI[floor][seed]

        missing_methods = [
            m for m in METHOD_ORDER_16E
            if m not in references
        ]

        if missing_methods:
            raise ValueError(
                f"{floor}, seed={seed}: Missing methods {missing_methods}"
            )

        original_reference = references["ORIGINAL"]

        # ----------------------------------------------------
        # 4.3 Select k using ORIGINAL validation only
        # ----------------------------------------------------

        best_k, k_table = select_k_validation_16e(
            original_reference,
            val_df,
            scaler,
            med
        )

        K_SELECTION_16E.append({
            "Floor": floor,
            "Seed": seed,
            "Selected_K": best_k,
            "Validation_Queries": len(val_df),
            "Validation_Mean_Error": float(
                k_table.iloc[0]["Mean_Error"]
            ),
            "Validation_Coverage": (
                k_table.iloc[0]["Valid_Queries"]
                / len(val_df)
            )
        })

        print(
            f"  Selected k = {best_k} | "
            f"Validation Mean Error = "
            f"{k_table.iloc[0]['Mean_Error']:.4f}"
        )

        # ----------------------------------------------------
        # 4.4 Evaluate all methods on identical TEST queries
        # ----------------------------------------------------

        for method in METHOD_ORDER_16E:

            reference = references[method]

            pred, error, valid, fallback, used = evaluate_wknn_16e(
                reference,
                test_df,
                scaler,
                med,
                best_k
            )

            n_total = len(test_df)
            n_valid = int(valid.sum())

            mean_error = (
                float(np.mean(error[valid]))
                if n_valid > 0
                else np.nan
            )

            median_error = (
                float(np.median(error[valid]))
                if n_valid > 0
                else np.nan
            )

            rmse = (
                float(np.sqrt(np.mean(error[valid] ** 2)))
                if n_valid > 0
                else np.nan
            )

            p90 = (
                float(np.percentile(error[valid], 90))
                if n_valid > 0
                else np.nan
            )

            coverage = n_valid / n_total if n_total else 0

            common_ap_values = used[valid]

            mean_common_aps = (
                float(np.mean(common_ap_values))
                if len(common_ap_values)
                else np.nan
            )

            result = {
                "Floor": floor,
                "Seed": seed,
                "Method": method,
                "Selected_K": best_k,
                "Total_Queries": n_total,
                "Valid_Queries": n_valid,
                "Mean_Error": mean_error,
                "Median_Error": median_error,
                "RMSE": rmse,
                "P90_Error": p90,
                "Coverage": coverage,
                "Fallback_Count": int(fallback.sum()),
                "Mean_Common_APs": mean_common_aps
            }

            RESULTS_16E.append(result)

            # Save per-query predictions
            pred_df = pd.DataFrame({
                "Floor": floor,
                "Seed": seed,
                "Method": method,
                "ID_RP": test_df["ID_RP"].to_numpy(),
                "True_X": test_df["X"].to_numpy(),
                "True_Y": test_df["Y"].to_numpy(),
                "Pred_X": pred[:, 0],
                "Pred_Y": pred[:, 1],
                "Error": error,
                "Valid": valid,
                "Fallback": fallback,
                "Used_Reference_Count": used
            })

            PREDICTIONS_16E.append(pred_df)

            print(
                f"    {method:<12} "
                f"Mean={mean_error:7.4f} | "
                f"RMSE={rmse:7.4f} | "
                f"Coverage={coverage*100:6.2f}%"
            )


# ------------------------------------------------------------
# 5. BUILD RESULT TABLES
# ------------------------------------------------------------

results_16e_df = pd.DataFrame(RESULTS_16E)

predictions_16e_df = pd.concat(
    PREDICTIONS_16E,
    ignore_index=True
)

k_selection_16e_df = pd.DataFrame(K_SELECTION_16E)

print("\n" + "=" * 80)
print("CELL 16E — EVALUATION COMPLETED")
print("=" * 80)

print("Configurations:", len(results_16e_df))
print("Expected:", len(FLOORS) * len(MULTI_SEEDS) * len(METHOD_ORDER_16E))

display(
    results_16e_df.sort_values(
        ["Floor", "Seed", "Method"]
    ).round(4)
)


# ------------------------------------------------------------
# 6. SUMMARY: MEAN ± STD ACROSS FIVE SEEDS
# ------------------------------------------------------------

metric_columns = [
    "Mean_Error",
    "Median_Error",
    "RMSE",
    "P90_Error",
    "Coverage",
    "Fallback_Count",
    "Mean_Common_APs"
]

summary_16e_df = (
    results_16e_df
    .groupby(["Floor", "Method"])[metric_columns]
    .agg(["mean", "std"])
)

summary_16e_df.columns = [
    f"{metric}_{stat.upper()}"
    for metric, stat in summary_16e_df.columns
]

summary_16e_df = summary_16e_df.reset_index()

print("\n" + "=" * 80)
print("MULTI-SEED SUMMARY — MEAN ± STD")
print("=" * 80)

display(summary_16e_df.round(4))


# ------------------------------------------------------------
# 7. IMPROVEMENT AGAINST ORIGINAL
# ------------------------------------------------------------

comparison_rows = []

for floor in FLOORS:

    floor_results = results_16e_df[
        results_16e_df["Floor"] == floor
    ]

    original = (
        floor_results[
            floor_results["Method"] == "ORIGINAL"
        ][["Seed", "Mean_Error", "RMSE"]]
        .rename(columns={
            "Mean_Error": "Original_Mean",
            "RMSE": "Original_RMSE"
        })
    )

    for method in METHOD_ORDER_16E:

        if method == "ORIGINAL":
            continue

        candidate = (
            floor_results[
                floor_results["Method"] == method
            ][["Seed", "Mean_Error", "RMSE"]]
            .rename(columns={
                "Mean_Error": "Method_Mean",
                "RMSE": "Method_RMSE"
            })
        )

        merged = original.merge(
            candidate,
            on="Seed",
            how="inner"
        )

        merged["Floor"] = floor
        merged["Method"] = method

        merged["Mean_Error_Improvement"] = (
            merged["Original_Mean"] - merged["Method_Mean"]
        )

        merged["RMSE_Improvement"] = (
            merged["Original_RMSE"] - merged["Method_RMSE"]
        )

        merged["Mean_Error_Improvement_%"] = (
            merged["Mean_Error_Improvement"]
            / merged["Original_Mean"].replace(0, np.nan)
            * 100
        )

        comparison_rows.append(merged)

improvement_16e_df = pd.concat(
    comparison_rows,
    ignore_index=True
)

improvement_summary_16e_df = (
    improvement_16e_df
    .groupby(["Floor", "Method"])[
        [
            "Mean_Error_Improvement",
            "RMSE_Improvement",
            "Mean_Error_Improvement_%"
        ]
    ]
    .agg(["mean", "std"])
)

improvement_summary_16e_df.columns = [
    f"{metric}_{stat.upper()}"
    for metric, stat in improvement_summary_16e_df.columns
]

improvement_summary_16e_df = (
    improvement_summary_16e_df.reset_index()
)

print("\n" + "=" * 80)
print("IMPROVEMENT AGAINST ORIGINAL")
print("Positive improvement = lower error than ORIGINAL")
print("=" * 80)

display(improvement_summary_16e_df.round(4))


# ------------------------------------------------------------
# 8. VISUALIZATION
# ------------------------------------------------------------

sns.set_theme(style="whitegrid")

method_palette = {
    "ORIGINAL": "#64748b",
    "CGAN": "#8b5cf6",
    "KNN_K3": "#0ea5e9",
    "KNN_K5": "#06b6d4",
    "KNN_K7": "#14b8a6",
    "APMEAN_R3": "#84cc16",
    "APMEAN_R5": "#eab308",
    "APMEAN_R7": "#f97316"
}

# 8.1 Mean Error across seeds
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

for ax, floor in zip(axes, FLOORS):

    data = results_16e_df[
        results_16e_df["Floor"] == floor
    ]

    sns.barplot(
        data=data,
        x="Method",
        y="Mean_Error",
        hue="Method",
        order=METHOD_ORDER_16E,
        palette=method_palette,
        errorbar="sd",
        legend=False,
        ax=ax
    )

    ax.set_title(
        f"{floor} — Mean Localization Error",
        fontsize=13,
        fontweight="bold"
    )

    ax.set_xlabel("")
    ax.set_ylabel("Mean Error (coordinate units)")
    ax.tick_params(axis="x", rotation=45)

plt.suptitle(
    "Multi-Seed WKNN — Mean Error (5 Random Seeds)",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR_16E / "16E_Mean_Error.png",
    dpi=220,
    bbox_inches="tight"
)
plt.show()


# 8.2 RMSE
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

for ax, floor in zip(axes, FLOORS):

    data = results_16e_df[
        results_16e_df["Floor"] == floor
    ]

    sns.barplot(
        data=data,
        x="Method",
        y="RMSE",
        hue="Method",
        order=METHOD_ORDER_16E,
        palette=method_palette,
        errorbar="sd",
        legend=False,
        ax=ax
    )

    ax.set_title(
        f"{floor} — RMSE",
        fontsize=13,
        fontweight="bold"
    )

    ax.set_xlabel("")
    ax.set_ylabel("RMSE (coordinate units)")
    ax.tick_params(axis="x", rotation=45)

plt.suptitle(
    "Multi-Seed WKNN — RMSE (5 Random Seeds)",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR_16E / "16E_RMSE.png",
    dpi=220,
    bbox_inches="tight"
)
plt.show()


# 8.3 Improvement heatmap
heatmap_data = improvement_summary_16e_df.pivot(
    index="Method",
    columns="Floor",
    values=("Mean_Error_Improvement_MEAN")
    if "Mean_Error_Improvement_MEAN" in improvement_summary_16e_df.columns
    else "Mean_Error_Improvement_MEAN"
)

plt.figure(figsize=(9, 6))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".3f",
    cmap="RdYlGn",
    center=0,
    linewidths=0.8,
    cbar_kws={
        "label": "Mean Error Improvement (coordinate units)"
    }
)

plt.title(
    "Mean Error Improvement vs ORIGINAL\n"
    "Green = Improvement | Red = Degradation",
    fontsize=13,
    fontweight="bold"
)

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR_16E / "16E_Improvement_Heatmap.png",
    dpi=220,
    bbox_inches="tight"
)
plt.show()


# ------------------------------------------------------------
# 9. EXPORT
# ------------------------------------------------------------

results_16e_df.to_csv(
    OUTPUT_DIR_16E / "16E_Per_Seed_Results.csv",
    index=False
)

summary_16e_df.to_csv(
    OUTPUT_DIR_16E / "16E_Mean_Std_Summary.csv",
    index=False
)

improvement_16e_df.to_csv(
    OUTPUT_DIR_16E / "16E_Improvement_Per_Seed.csv",
    index=False
)

improvement_summary_16e_df.to_csv(
    OUTPUT_DIR_16E / "16E_Improvement_Summary.csv",
    index=False
)

k_selection_16e_df.to_csv(
    OUTPUT_DIR_16E / "16E_K_Selection.csv",
    index=False
)

predictions_16e_df.to_csv(
    OUTPUT_DIR_16E / "16E_Per_Query_Predictions.csv",
    index=False
)

# Excel workbook
excel_path = OUTPUT_DIR_16E / "16E_MULTI_SEED_WKNN_RESULTS.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:

    results_16e_df.to_excel(
        writer,
        sheet_name="Per_Seed_Results",
        index=False
    )

    summary_16e_df.to_excel(
        writer,
        sheet_name="Mean_Std_Summary",
        index=False
    )

    improvement_16e_df.to_excel(
        writer,
        sheet_name="Improvement_Per_Seed",
        index=False
    )

    improvement_summary_16e_df.to_excel(
        writer,
        sheet_name="Improvement_Summary",
        index=False
    )

    k_selection_16e_df.to_excel(
        writer,
        sheet_name="K_Selection",
        index=False
    )

    # Predictions can be large, but are useful for traceability.
    predictions_16e_df.to_excel(
        writer,
        sheet_name="Per_Query_Predictions",
        index=False
    )

print("\n" + "=" * 80)
print("CELL 16E — EXPORT COMPLETED")
print("=" * 80)

print("Output directory:", OUTPUT_DIR_16E)
print("Excel:", excel_path)
print("Elapsed time:", round(time.time() - start_all, 2), "seconds")

# CELL 16F — FINAL RESEARCH ANALYSIS

## Mục tiêu
Hoàn thiện phân tích thực nghiệm Multi-Seed sau Cell 16E.

### Phần A — Result Integrity Audit
- Kiểm tra 120 cấu hình.
- Kiểm tra tính đồng nhất của TEST queries.
- Kiểm tra Selected_K.
- Kiểm tra Coverage và Fallback.
- Đổi tên Mean_Common_APs thành Mean_Used_References.

### Phần B — Statistical Significance
- Paired bootstrap trên cùng TEST queries.
- So sánh 7 phương pháp tăng cường với ORIGINAL.
- Thực hiện riêng theo từng tầng.
- Điều chỉnh đa kiểm định bằng Holm.
- Báo cáo Confidence Interval 95%, p-value và effect size.

### Phần C — Final Dashboard
- Mean ± Std qua 5 random seeds.
- So sánh Mean Error, RMSE, P90.
- Phân tích mức cải thiện so với ORIGINAL.
- Heatmap hiệu quả theo tầng và phương pháp.
- Biểu đồ độ ổn định qua các random seeds.

### Phần D — Export
- Excel tổng hợp.
- CSV kết quả.
- Hình ảnh dashboard.
- Báo cáo kết luận thực nghiệm.

Lưu ý:
Đơn vị sai số sử dụng đơn vị tọa độ nguồn, chưa mặc định là mét.

In [ ]:
# ============================================================
# CELL 16F-0 — FINAL ANALYSIS INPUT CHECK
# ============================================================

print("=" * 80)
print("CELL 16F — FINAL ANALYSIS INPUT CHECK")
print("=" * 80)

required_objects = [
    "results_16e_df",
    "predictions_16e_df",
    "k_selection_16e_df",
    "summary_16e_df",
    "improvement_16e_df",
    "improvement_summary_16e_df",
    "MULTI_SEEDS",
    "FLOORS",
    "METHODS_MULTI",
    "OUTPUT_DIR_16E"
]

missing = [
    name for name in required_objects
    if name not in globals()
]

if missing:
    raise NameError(
        f"Missing required objects: {missing}"
    )

print("\n1. RESULTS TABLE")
print(results_16e_df.shape)
print(results_16e_df.columns.tolist())

print("\n2. PREDICTIONS TABLE")
print(predictions_16e_df.shape)
print(predictions_16e_df.columns.tolist())

print("\n3. SAMPLE PREDICTIONS")
display(predictions_16e_df.head(5))

print("\n4. UNIQUE RP COUNTS")

rp_counts = (
    predictions_16e_df
    .groupby(["Floor", "Seed", "Method"])
    ["ID_RP"]
    .nunique()
    .reset_index(name="Unique_RP")
)

display(rp_counts.head(12))

print("\n5. QUERY COUNTS")

query_counts = (
    predictions_16e_df
    .groupby(["Floor", "Seed", "Method"])
    .size()
    .reset_index(name="Query_Count")
)

display(query_counts.head(12))

print("\n6. CHECK BOOTSTRAP GROUPING")

print(
    "ID_RP available:",
    "ID_RP" in predictions_16e_df.columns
)

print(
    "True coordinates available:",
    {"True_X", "True_Y"}.issubset(
        predictions_16e_df.columns
    )
)

print(
    "Prediction coordinates available:",
    {"Pred_X", "Pred_Y"}.issubset(
        predictions_16e_df.columns
    )
)

print("\nCELL 16F-0 — COMPLETED")

In [ ]:
# CELL BỔ SUNG TRƯỚC 16F-R: KHAI BÁO INPUT CÒN THIẾU KHI RUN ALL
# Không thay mã của cell gốc. Alias chỉ từ kết quả thực tế 16E, không tạo dữ liệu giả.
from pathlib import Path
import numpy as np
import pandas as pd
AUGMENTED_METHODS = [m for m in METHODS_MULTI if m != 'ORIGINAL']
BOOTSTRAP_ITERATIONS = 5000
BOOTSTRAP_SEED = 2026
OUT16F = Path(OUTPUT_DIR_16E) / '16F_FINAL_ANALYSIS'
results_16f_df = results_16e_df.copy()
summary_16f_df = summary_16e_df.copy()
improvement_summary_16f_df = improvement_summary_16e_df.copy()
_compat_audit = []
for (_compat_floor, _compat_seed), _compat_group in results_16e_df.groupby(['Floor','Seed']):
    _compat_pred = predictions_16e_df[(predictions_16e_df.Floor == _compat_floor) & (predictions_16e_df.Seed == _compat_seed)]
    _compat_counts = _compat_pred.groupby('Method').size()
    _compat_audit.append({'Floor':_compat_floor,'Seed':_compat_seed,
        'N_Methods':_compat_group.Method.nunique(),
        'All_Methods_Present':set(_compat_group.Method)==set(METHODS_MULTI),
        'Equal_Query_Counts':_compat_counts.nunique()==1,
        'No_Duplicate_Results':not _compat_group.Method.duplicated().any()})
audit_16f_df = pd.DataFrame(_compat_audit)
assert audit_16f_df[['All_Methods_Present','Equal_Query_Counts','No_Duplicate_Results']].all().all()
OUT16F.mkdir(parents=True,exist_ok=True)
print('16F inputs ready; bootstrap iterations:',BOOTSTRAP_ITERATIONS)


In [ ]:
# ============================================================
# CELL 16F-R — FIX PAIRED CLUSTER BOOTSTRAP
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 80)
print("CELL 16F-R — FIX PAIRED CLUSTER BOOTSTRAP")
print("=" * 80)

# ------------------------------------------------------------
# 1. CREATE QUERY INDEX
# ------------------------------------------------------------

predictions_16f = predictions_16e_df.copy()

# Giữ thứ tự dòng gốc của từng nhóm RP.
predictions_16f["_Original_Order"] = np.arange(
    len(predictions_16f)
)

predictions_16f = predictions_16f.sort_values(
    [
        "Floor",
        "Seed",
        "Method",
        "ID_RP",
        "_Original_Order"
    ]
).copy()

predictions_16f["Query_Index"] = (
    predictions_16f
    .groupby(
        ["Floor", "Seed", "Method", "ID_RP"]
    )
    .cumcount()
)

# ------------------------------------------------------------
# 2. VERIFY PAIRING
# ------------------------------------------------------------

pair_keys = [
    "Floor",
    "Seed",
    "ID_RP",
    "Query_Index",
    "True_X",
    "True_Y"
]

pairing_audit = []

for floor in FLOORS:
    for seed in MULTI_SEEDS:

        subset = predictions_16f[
            (predictions_16f["Floor"] == floor)
            & (predictions_16f["Seed"] == seed)
        ]

        original = subset[
            subset["Method"] == "ORIGINAL"
        ]

        original_keys = set(
            map(
                tuple,
                original[pair_keys].to_numpy()
            )
        )

        for method in AUGMENTED_METHODS:

            candidate = subset[
                subset["Method"] == method
            ]

            candidate_keys = set(
                map(
                    tuple,
                    candidate[pair_keys].to_numpy()
                )
            )

            pairing_audit.append({
                "Floor": floor,
                "Seed": seed,
                "Method": method,
                "Original_Rows": len(original),
                "Method_Rows": len(candidate),
                "Paired_Rows": len(
                    original_keys.intersection(candidate_keys)
                ),
                "Same_Query_Set": (
                    original_keys == candidate_keys
                )
            })

pairing_audit_df = pd.DataFrame(pairing_audit)

display(pairing_audit_df)

if not pairing_audit_df["Same_Query_Set"].all():
    raise ValueError(
        "Query pairing failed. Do not run bootstrap."
    )

print("\nPairing audit PASS")


# ------------------------------------------------------------
# 3. SAFE PAIRED DATAFRAME
# ------------------------------------------------------------

def make_paired_data(original_df, method_df):

    keys = [
        "ID_RP",
        "Query_Index",
        "True_X",
        "True_Y"
    ]

    a = original_df[
        keys + ["Error", "Valid"]
    ].rename(columns={
        "Error": "Error_Original",
        "Valid": "Valid_Original"
    })

    b = method_df[
        keys + ["Error", "Valid"]
    ].rename(columns={
        "Error": "Error_Method",
        "Valid": "Valid_Method"
    })

    paired = a.merge(
        b,
        on=keys,
        how="inner",
        validate="one_to_one"
    )

    paired = paired[
        paired["Valid_Original"].astype(bool)
        & paired["Valid_Method"].astype(bool)
        & np.isfinite(paired["Error_Original"])
        & np.isfinite(paired["Error_Method"])
    ].copy()

    paired["Difference"] = (
        paired["Error_Original"]
        - paired["Error_Method"]
    )

    return paired


# ------------------------------------------------------------
# 4. CLUSTER BOOTSTRAP
# ------------------------------------------------------------

def cluster_bootstrap_16fr(
    paired,
    n_boot=5000,
    rng=None
):

    if rng is None:
        rng = np.random.default_rng(123)

    rp_groups = {
        rp: group["Difference"].to_numpy(float)
        for rp, group in paired.groupby("ID_RP")
    }

    rp_ids = np.array(
        list(rp_groups.keys()),
        dtype=object
    )

    if len(rp_ids) < 2:
        raise ValueError("Insufficient RP clusters.")

    observed = float(
        paired["Difference"].mean()
    )

    bootstrap_values = np.empty(n_boot)

    for i in range(n_boot):

        sampled_rps = rng.choice(
            rp_ids,
            size=len(rp_ids),
            replace=True
        )

        values = np.concatenate([
            rp_groups[rp]
            for rp in sampled_rps
        ])

        bootstrap_values[i] = values.mean()

    low, high = np.percentile(
        bootstrap_values,
        [2.5, 97.5]
    )

    return {
        "Observed_Improvement": observed,
        "CI95_Lower": float(low),
        "CI95_Upper": float(high),
        "Bootstrap_SE": float(
            np.std(bootstrap_values, ddof=1)
        ),
        "N_Paired_Queries": len(paired),
        "N_RP_Clusters": len(rp_ids),
        "Bootstrap_Values": bootstrap_values
    }


# ------------------------------------------------------------
# 5. CLUSTER SIGN-FLIP TEST
# ------------------------------------------------------------

def sign_flip_16fr(
    paired,
    n_perm=20000,
    rng=None
):

    if rng is None:
        rng = np.random.default_rng(123)

    rp_means = (
        paired.groupby("ID_RP")["Difference"]
        .mean()
        .to_numpy(float)
    )

    observed = abs(rp_means.mean())

    signs = rng.choice(
        [-1.0, 1.0],
        size=(n_perm, len(rp_means))
    )

    permuted = np.abs(
        (signs * rp_means).mean(axis=1)
    )

    return float(
        (1 + np.sum(permuted >= observed))
        / (n_perm + 1)
    )


# ------------------------------------------------------------
# 6. HOLM CORRECTION
# ------------------------------------------------------------

def holm_16fr(p_values):

    p_values = np.asarray(
        p_values,
        dtype=float
    )

    n = len(p_values)
    order = np.argsort(p_values)

    adjusted = np.empty(n)
    running_max = 0.0

    for rank, idx in enumerate(order):

        value = (n - rank) * p_values[idx]

        running_max = max(
            running_max,
            value
        )

        adjusted[idx] = min(
            running_max,
            1.0
        )

    return adjusted


# ------------------------------------------------------------
# 7. RUN STATISTICAL TESTS
# ------------------------------------------------------------

rng_16fr = np.random.default_rng(
    BOOTSTRAP_SEED
)

stat_rows_16fr = []

for floor in FLOORS:

    print(f"\nProcessing {floor}")

    for method in AUGMENTED_METHODS:

        seed_results = []
        seed_p_values = []

        for seed in MULTI_SEEDS:

            subset = predictions_16f[
                (predictions_16f["Floor"] == floor)
                & (predictions_16f["Seed"] == seed)
            ]

            original = subset[
                subset["Method"] == "ORIGINAL"
            ]

            candidate = subset[
                subset["Method"] == method
            ]

            paired = make_paired_data(
                original,
                candidate
            )

            result = cluster_bootstrap_16fr(
                paired,
                n_boot=BOOTSTRAP_ITERATIONS,
                rng=rng_16fr
            )

            p_value = sign_flip_16fr(
                paired,
                n_perm=20000,
                rng=rng_16fr
            )

            seed_results.append(result)
            seed_p_values.append(p_value)

        boot_matrix = np.vstack([
            x["Bootstrap_Values"]
            for x in seed_results
        ])

        mean_boot = boot_matrix.mean(axis=0)

        effect = np.mean([
            x["Observed_Improvement"]
            for x in seed_results
        ])

        ci_low, ci_high = np.percentile(
            mean_boot,
            [2.5, 97.5]
        )

        stat_rows_16fr.append({
            "Floor": floor,
            "Method": method,
            "Mean_Paired_Improvement": effect,
            "CI95_Lower": ci_low,
            "CI95_Upper": ci_high,
            "P_Value_Conservative": max(seed_p_values),
            "Seeds_Analyzed": len(seed_results),
            "Mean_Paired_Queries": np.mean([
                x["N_Paired_Queries"]
                for x in seed_results
            ]),
            "Mean_RP_Clusters": np.mean([
                x["N_RP_Clusters"]
                for x in seed_results
            ])
        })


statistics_16f_df = pd.DataFrame(
    stat_rows_16fr
)

statistics_16f_df["P_Value_Holm"] = np.nan

for floor in FLOORS:

    mask = (
        statistics_16f_df["Floor"] == floor
    )

    indices = statistics_16f_df.index[
        mask
    ]

    statistics_16f_df.loc[
        indices,
        "P_Value_Holm"
    ] = holm_16fr(
        statistics_16f_df.loc[
            indices,
            "P_Value_Conservative"
        ].to_numpy()
    )

statistics_16f_df["Significant_Holm_005"] = (
    statistics_16f_df["P_Value_Holm"] < 0.05
)

statistics_16f_df["CI_Excludes_Zero"] = (
    (statistics_16f_df["CI95_Lower"] > 0)
    | (statistics_16f_df["CI95_Upper"] < 0)
)

print("\n" + "=" * 80)
print("PAIRED CLUSTER BOOTSTRAP RESULTS")
print("=" * 80)

display(
    statistics_16f_df.round(5)
)


# ------------------------------------------------------------
# 8. EXPORT FIXED STATISTICAL RESULTS
# ------------------------------------------------------------

OUT16F.mkdir(
    parents=True,
    exist_ok=True
)

pairing_audit_df.to_csv(
    OUT16F / "16F_Pairing_Audit.csv",
    index=False
)

statistics_16f_df.to_csv(
    OUT16F / "16F_Statistical_Tests.csv",
    index=False
)

excel_path = (
    OUT16F / "16F_FINAL_RESEARCH_ANALYSIS.xlsx"
)

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    audit_16f_df.to_excel(
        writer,
        sheet_name="Integrity_Audit",
        index=False
    )

    results_16f_df.to_excel(
        writer,
        sheet_name="Per_Seed_Results",
        index=False
    )

    summary_16f_df.to_excel(
        writer,
        sheet_name="Mean_Std_Summary",
        index=False
    )

    improvement_summary_16f_df.to_excel(
        writer,
        sheet_name="Improvement_Summary",
        index=False
    )

    pairing_audit_df.to_excel(
        writer,
        sheet_name="Pairing_Audit",
        index=False
    )

    statistics_16f_df.to_excel(
        writer,
        sheet_name="Statistical_Tests",
        index=False
    )

print("\n" + "=" * 80)
print("CELL 16F-R — COMPLETED")
print("=" * 80)

print("Pairing PASS:", pairing_audit_df["Same_Query_Set"].all())
print("Statistical comparisons:", len(statistics_16f_df))
print("Excel:", excel_path)

# CELL 16F — FINAL DASHBOARD & RESEARCH REPORT

## Mục tiêu
Tổng hợp và trực quan hóa kết quả thực nghiệm Multi-Seed
trên ba tầng F1, F2, F3.

## Nội dung
1. So sánh Mean Error giữa các phương pháp.
2. Phân tích mức cải thiện so với ORIGINAL.
3. Hiển thị khoảng tin cậy 95% (CI95).
4. Đánh giá độ ổn định qua 5 random seeds.
5. Xếp hạng phương pháp theo từng tầng.
6. Tổng hợp kết quả kiểm định sau Holm correction.
7. Xuất dashboard và báo cáo Excel.

Lưu ý:
- Không huấn luyện lại mô hình.
- Không sử dụng TEST để lựa chọn mô hình.
- Không thay đổi kết quả kiểm định đã hoàn thành ở Cell 16F-R.
- Đơn vị sai số là đơn vị tọa độ nguồn dữ liệu, chưa mặc định là mét.

In [ ]:

# =============================================================================
# CELL 16F — FINAL DASHBOARD & RESEARCH REPORT
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

print("=" * 90)
print("CELL 16F — FINAL DASHBOARD & RESEARCH REPORT")
print("=" * 90)


# =============================================================================
# 1. INPUT VALIDATION
# =============================================================================

required_objects = [
    "predictions_16e_df",
    "results_16e_df",
    "statistics_16f_df"
]

for obj in required_objects:
    assert obj in globals(), f"Missing object: {obj}"

assert len(predictions_16e_df) == 260000
assert len(results_16e_df) == 120
assert len(statistics_16f_df) == 21

assert set(results_16e_df["Floor"]) == {"F1", "F2", "F3"}
assert set(statistics_16f_df["Floor"]) == {"F1", "F2", "F3"}

required_stat_columns = [
    "Floor",
    "Method",
    "Mean_Paired_Improvement",
    "CI95_Lower",
    "CI95_Upper",
    "P_Value_Conservative",
    "P_Value_Holm",
    "Significant_Holm_005",
    "CI_Excludes_Zero"
]

for col in required_stat_columns:
    assert col in statistics_16f_df.columns, \
        f"Missing statistical column: {col}"

assert statistics_16f_df[
    ["CI95_Lower", "CI95_Upper", "P_Value_Holm"]
].notna().all().all()

print("INPUT VALIDATION: PASS")
print("Predictions:", len(predictions_16e_df))
print("WKNN configurations:", len(results_16e_df))
print("Statistical comparisons:", len(statistics_16f_df))


# =============================================================================
# 2. CONFIGURATION
# =============================================================================

FLOORS = ["F1", "F2", "F3"]

METHODS = [
    "ORIGINAL",
    "CGAN",
    "KNN_K3",
    "KNN_K5",
    "KNN_K7",
    "APMEAN_R3",
    "APMEAN_R5",
    "APMEAN_R7"
]

AUGMENTED_METHODS = METHODS[1:]

SEEDS = [2026, 2027, 2028, 2029, 2030]

BASE_DIR = (
    "/content/WIFI_FINGERPRINT_RESEARCH_PHASE1/"
    "16_MULTI_SEED_EXPERIMENT/"
    "16E_MULTI_SEED_WKNN/"
    "16F_FINAL_ANALYSIS"
)

DASHBOARD_DIR = os.path.join(
    BASE_DIR,
    "16F_DASHBOARD"
)

os.makedirs(DASHBOARD_DIR, exist_ok=True)

# Consistent colors across every chart
palette = {
    "ORIGINAL": "#64748B",
    "CGAN": "#DC6B75",
    "KNN_K3": "#F59E0B",
    "KNN_K5": "#EAB308",
    "KNN_K7": "#16A085",
    "APMEAN_R3": "#3B82F6",
    "APMEAN_R5": "#8B5CF6",
    "APMEAN_R7": "#A78BFA"
}

sns.set_theme(
    style="whitegrid",
    context="notebook",
    font_scale=0.95
)


# =============================================================================
# 3. MULTI-SEED DESCRIPTIVE SUMMARY
# =============================================================================

summary_16f = (
    results_16e_df
    .groupby(["Floor", "Method"])
    .agg(
        Mean_Error=("Mean_Error", "mean"),
        SD_Error=("Mean_Error", "std"),
        Mean_RMSE=("RMSE", "mean"),
        SD_RMSE=("RMSE", "std"),
        Mean_P90=("P90_Error", "mean"),
        SD_P90=("P90_Error", "std"),
        Mean_Coverage=("Coverage", "mean"),
        Mean_Fallback=("Fallback_Count", "mean")
    )
    .reset_index()
)

# Verify 3 floors x 8 methods
assert len(summary_16f) == 24

# Add ORIGINAL baseline
original_summary = (
    summary_16f[
        summary_16f["Method"] == "ORIGINAL"
    ][["Floor", "Mean_Error", "Mean_RMSE"]]
    .rename(columns={
        "Mean_Error": "Original_Mean_Error",
        "Mean_RMSE": "Original_RMSE"
    })
)

summary_16f = summary_16f.merge(
    original_summary,
    on="Floor",
    how="left",
    validate="many_to_one"
)

# Positive = improvement
summary_16f["Mean_Improvement"] = (
    summary_16f["Original_Mean_Error"]
    - summary_16f["Mean_Error"]
)

summary_16f["Improvement_Percent"] = (
    summary_16f["Mean_Improvement"]
    / summary_16f["Original_Mean_Error"]
    * 100
)

summary_16f["RMSE_Improvement"] = (
    summary_16f["Original_RMSE"]
    - summary_16f["Mean_RMSE"]
)


# =============================================================================
# 4. MERGE STATISTICAL RESULTS
# =============================================================================

# Correct variable from Cell 16F-R:
stat_16f = statistics_16f_df.copy()

stat_16f = stat_16f[
    stat_16f["Method"].isin(AUGMENTED_METHODS)
].copy()

assert len(stat_16f) == 21

final_summary_16f = summary_16f.merge(
    stat_16f[
        [
            "Floor",
            "Method",
            "Mean_Paired_Improvement",
            "CI95_Lower",
            "CI95_Upper",
            "P_Value_Conservative",
            "P_Value_Holm",
            "Significant_Holm_005",
            "CI_Excludes_Zero"
        ]
    ],
    on=["Floor", "Method"],
    how="left",
    validate="one_to_one"
)

# Baseline has no paired comparison
final_summary_16f["Statistical_Conclusion"] = "Baseline"

for idx in final_summary_16f.index:

    row = final_summary_16f.loc[idx]

    if row["Method"] == "ORIGINAL":
        continue

    if bool(row["Significant_Holm_005"]):

        if row["Mean_Paired_Improvement"] > 0:
            conclusion = "Significant improvement"
        else:
            conclusion = "Significant degradation"

    else:
        conclusion = "No significant difference after Holm"

    final_summary_16f.loc[
        idx,
        "Statistical_Conclusion"
    ] = conclusion

assert len(final_summary_16f) == 24


# =============================================================================
# 5. TABLE 1 — PERFORMANCE SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("TABLE 1 — MULTI-SEED PERFORMANCE SUMMARY")
print("=" * 90)

table1 = summary_16f[
    [
        "Floor",
        "Method",
        "Mean_Error",
        "SD_Error",
        "Mean_RMSE",
        "SD_RMSE",
        "Mean_P90",
        "Mean_Coverage",
        "Mean_Fallback",
        "Mean_Improvement",
        "Improvement_Percent"
    ]
].copy()

display(table1.round(4))


# =============================================================================
# 6. TABLE 2 — METHOD RANKING
# =============================================================================

ranking_16f = summary_16f.copy()

ranking_16f["Rank"] = (
    ranking_16f
    .groupby("Floor")["Mean_Error"]
    .rank(method="min", ascending=True)
)

ranking_16f = ranking_16f.sort_values(
    ["Floor", "Rank", "Method"]
)

print("\n" + "=" * 90)
print("TABLE 2 — METHOD RANKING BY MEAN ERROR")
print("=" * 90)

display(
    ranking_16f[
        ["Floor", "Rank", "Method", "Mean_Error", "Mean_RMSE"]
    ].round(4)
)


# =============================================================================
# 7. TABLE 3 — STATISTICAL INFERENCE
# =============================================================================

print("\n" + "=" * 90)
print("TABLE 3 — PAIRED CLUSTER BOOTSTRAP & HOLM")
print("=" * 90)

table3 = final_summary_16f[
    final_summary_16f["Method"].isin(AUGMENTED_METHODS)
][
    [
        "Floor",
        "Method",
        "Mean_Paired_Improvement",
        "CI95_Lower",
        "CI95_Upper",
        "P_Value_Conservative",
        "P_Value_Holm",
        "Significant_Holm_005",
        "CI_Excludes_Zero",
        "Statistical_Conclusion"
    ]
].copy()

display(table3.round(5))


# =============================================================================
# 8. CHART 1 — MEAN ERROR COMPARISON
# =============================================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(22, 7)
)

for ax, floor in zip(axes, FLOORS):

    data = summary_16f[
        summary_16f["Floor"] == floor
    ].copy()

    data["Method"] = pd.Categorical(
        data["Method"],
        categories=METHODS,
        ordered=True
    )

    data = data.sort_values("Method")

    labels = data["Method"].astype(str).tolist()

    bars = ax.bar(
        np.arange(len(data)),
        data["Mean_Error"],
        color=[palette[m] for m in labels],
        edgecolor="white",
        linewidth=1.2
    )

    ax.errorbar(
        np.arange(len(data)),
        data["Mean_Error"],
        yerr=data["SD_Error"].fillna(0),
        fmt="none",
        ecolor="#263238",
        capsize=3,
        linewidth=1.1
    )

    ax.set_xticks(np.arange(len(data)))
    ax.set_xticklabels(labels, rotation=45, ha="right")

    ax.set_title(
        f"{floor} — Mean Error ± SD",
        fontsize=14,
        fontweight="bold"
    )

    ax.set_ylabel("Mean Localization Error")
    ax.set_xlabel("Method")

    ax.grid(axis="y", alpha=0.25)

    for bar, value in zip(bars, data["Mean_Error"]):

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=8
        )

plt.suptitle(
    "Multi-Seed WKNN — Mean Error Comparison",
    fontsize=17,
    fontweight="bold"
)

plt.tight_layout()

chart1_path = os.path.join(
    DASHBOARD_DIR,
    "01_MEAN_ERROR_COMPARISON.png"
)

plt.savefig(chart1_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()


# =============================================================================
# 9. CHART 2 — IMPROVEMENT WITH 95% CI
# =============================================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(22, 8)
)

for ax, floor in zip(axes, FLOORS):

    data = final_summary_16f[
        (final_summary_16f["Floor"] == floor)
        & (final_summary_16f["Method"].isin(AUGMENTED_METHODS))
    ].copy()

    data["Method"] = pd.Categorical(
        data["Method"],
        categories=AUGMENTED_METHODS,
        ordered=True
    )

    data = data.sort_values("Method")

    y = np.arange(len(data))

    x = data["Mean_Paired_Improvement"].values

    lower = x - data["CI95_Lower"].values
    upper = data["CI95_Upper"].values - x

    ax.barh(
        y,
        x,
        color=[palette[m] for m in data["Method"].astype(str)],
        alpha=0.85,
        edgecolor="white"
    )

    ax.errorbar(
        x,
        y,
        xerr=np.vstack([lower, upper]),
        fmt="none",
        ecolor="#263238",
        capsize=4,
        linewidth=1.3
    )

    ax.axvline(
        0,
        color="black",
        linestyle="--",
        linewidth=1.2
    )

    ax.set_yticks(y)
    ax.set_yticklabels(data["Method"].astype(str))
    ax.invert_yaxis()

    ax.set_title(
        f"{floor} — Paired Improvement",
        fontsize=13,
        fontweight="bold"
    )

    ax.set_xlabel("Error Reduction (+ = Improvement)")
    ax.grid(axis="x", alpha=0.25)

plt.suptitle(
    "Paired Improvement with 95% Confidence Interval",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()

chart2_path = os.path.join(
    DASHBOARD_DIR,
    "02_IMPROVEMENT_CI95.png"
)

plt.savefig(chart2_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()


# =============================================================================
# 10. CHART 3 — RANDOM SEED STABILITY
# =============================================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(22, 7)
)

for ax, floor in zip(axes, FLOORS):

    data = results_16e_df[
        results_16e_df["Floor"] == floor
    ].copy()

    sns.boxplot(
        data=data,
        x="Method",
        y="Mean_Error",
        hue="Method",
        order=METHODS,
        hue_order=METHODS,
        palette=palette,
        legend=False,
        ax=ax,
        width=0.65
    )

    sns.stripplot(
        data=data,
        x="Method",
        y="Mean_Error",
        order=METHODS,
        color="#263238",
        size=4,
        alpha=0.55,
        ax=ax
    )

    ax.set_title(
        f"{floor} — Random Seed Stability",
        fontsize=13,
        fontweight="bold"
    )

    ax.set_xlabel("Method")
    ax.set_ylabel("Mean Error")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.25)

plt.suptitle(
    "Performance Stability Across Five Random Seeds",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()

chart3_path = os.path.join(
    DASHBOARD_DIR,
    "03_RANDOM_SEED_STABILITY.png"
)

plt.savefig(chart3_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()


# =============================================================================
# 11. CHART 4 — RMSE COMPARISON
# =============================================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(22, 7)
)

for ax, floor in zip(axes, FLOORS):

    data = summary_16f[
        summary_16f["Floor"] == floor
    ].copy()

    data["Method"] = pd.Categorical(
        data["Method"],
        categories=METHODS,
        ordered=True
    )

    data = data.sort_values("Method")

    labels = data["Method"].astype(str).tolist()

    ax.bar(
        np.arange(len(data)),
        data["Mean_RMSE"],
        color=[palette[m] for m in labels],
        edgecolor="white"
    )

    ax.set_xticks(np.arange(len(data)))
    ax.set_xticklabels(labels, rotation=45, ha="right")

    ax.set_title(
        f"{floor} — RMSE",
        fontsize=13,
        fontweight="bold"
    )

    ax.set_ylabel("RMSE")
    ax.grid(axis="y", alpha=0.25)

plt.suptitle(
    "Multi-Seed RMSE Comparison",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()

chart4_path = os.path.join(
    DASHBOARD_DIR,
    "04_RMSE_COMPARISON.png"
)

plt.savefig(chart4_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()


# =============================================================================
# 12. AUTOMATIC RESEARCH SUMMARY
# =============================================================================

research_summary = []

for floor in FLOORS:

    data = summary_16f[
        summary_16f["Floor"] == floor
    ].copy()

    baseline = data.loc[
        data["Method"] == "ORIGINAL",
        "Mean_Error"
    ].iloc[0]

    augmented = data[
        data["Method"].isin(AUGMENTED_METHODS)
    ].copy()

    best = augmented.loc[
        augmented["Mean_Error"].idxmin()
    ]

    worst = augmented.loc[
        augmented["Mean_Error"].idxmax()
    ]

    significant_count = int(
        stat_16f[
            (stat_16f["Floor"] == floor)
        ]["Significant_Holm_005"]
        .fillna(False)
        .sum()
    )

    research_summary.append({
        "Floor": floor,
        "Original_Mean_Error": baseline,
        "Best_Augmented_Method": best["Method"],
        "Best_Augmented_Mean_Error": best["Mean_Error"],
        "Best_Observed_Improvement": baseline - best["Mean_Error"],
        "Worst_Augmented_Method": worst["Method"],
        "Worst_Augmented_Mean_Error": worst["Mean_Error"],
        "Significant_Methods_Holm": significant_count
    })

research_summary_df = pd.DataFrame(research_summary)

print("\n" + "=" * 90)
print("TABLE 4 — AUTOMATIC RESEARCH SUMMARY")
print("=" * 90)

display(research_summary_df.round(5))

for _, row in research_summary_df.iterrows():

    print(f"\n{row['Floor']}")

    print(
        f"Best observed method: "
        f"{row['Best_Augmented_Method']}"
    )

    print(
        f"Observed Mean Error improvement: "
        f"{row['Best_Observed_Improvement']:.5f}"
    )

    print(
        f"Worst observed method: "
        f"{row['Worst_Augmented_Method']}"
    )

    print(
        f"Methods significant after Holm: "
        f"{row['Significant_Methods_Holm']}"
    )


# =============================================================================
# 13. OVERALL RESEARCH CONCLUSION
# =============================================================================

total_significant = int(
    stat_16f["Significant_Holm_005"].fillna(False).sum()
)

print("\n" + "=" * 90)
print("OVERALL RESEARCH CONCLUSION")
print("=" * 90)

print(f"Total statistical comparisons: {len(stat_16f)}")
print(f"Significant comparisons after Holm: {total_significant}")

print("""
1. CGAN does not demonstrate consistent improvement in Mean Error.
2. KNN interpolation shows observed improvements at F2 and F3.
3. APMEAN performance depends on the selected radius and floor.
4. No augmentation method reaches statistical significance after
   Holm correction at alpha = 0.05 in these comparisons.
5. Results should be interpreted together with RMSE, P90 and coverage.
6. Findings apply to independent repeat measurements at known surveyed
   reference points, not entirely unseen physical locations.
""")


# =============================================================================
# 14. EXPORT EXCEL
# =============================================================================

excel_path = os.path.join(
    DASHBOARD_DIR,
    "16F_FINAL_DASHBOARD_REPORT.xlsx"
)

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    summary_16f.to_excel(
        writer,
        sheet_name="Performance_Summary",
        index=False
    )

    ranking_16f.to_excel(
        writer,
        sheet_name="Method_Ranking",
        index=False
    )

    final_summary_16f.to_excel(
        writer,
        sheet_name="Statistical_Results",
        index=False
    )

    research_summary_df.to_excel(
        writer,
        sheet_name="Research_Conclusion",
        index=False
    )

    results_16e_df.to_excel(
        writer,
        sheet_name="All_Seed_Results",
        index=False
    )

    statistics_16f_df.to_excel(
        writer,
        sheet_name="Paired_Bootstrap",
        index=False
    )


# =============================================================================
# 15. EXPORT CSV
# =============================================================================

summary_16f.to_csv(
    os.path.join(DASHBOARD_DIR, "performance_summary.csv"),
    index=False
)

final_summary_16f.to_csv(
    os.path.join(DASHBOARD_DIR, "statistical_results.csv"),
    index=False
)

ranking_16f.to_csv(
    os.path.join(DASHBOARD_DIR, "method_ranking.csv"),
    index=False
)

research_summary_df.to_csv(
    os.path.join(DASHBOARD_DIR, "research_conclusion.csv"),
    index=False
)


# =============================================================================
# 16. FINAL VALIDATION
# =============================================================================

assert len(summary_16f) == 24
assert len(ranking_16f) == 24
assert len(final_summary_16f) == 24
assert len(research_summary_df) == 3

for path in [
    excel_path,
    chart1_path,
    chart2_path,
    chart3_path,
    chart4_path
]:
    assert os.path.exists(path), f"Missing export: {path}"

print("\n" + "=" * 90)
print("CELL 16F — COMPLETED SUCCESSFULLY")
print("=" * 90)

print("Dashboard folder:")
print(DASHBOARD_DIR)

print("\nExcel report:")
print(excel_path)

print("\nGenerated charts:")
print("01 — Mean Error Comparison")
print("02 — Improvement with CI95")
print("03 — Random Seed Stability")
print("04 — RMSE Comparison")

print("\nExport validation: PASS")
print("=" * 90)


In [ ]:
# =============================================================================
# CELL 16G-A — SPATIAL DATA AUDIT
# =============================================================================

import numpy as np
import pandas as pd

assert "predictions_16e_df" in globals()

spatial_df = predictions_16e_df.copy()

required_cols = [
    "Floor", "Seed", "Method", "ID_RP",
    "True_X", "True_Y", "Pred_X", "Pred_Y",
    "Error", "Valid", "Fallback"
]

missing_cols = [
    c for c in required_cols
    if c not in spatial_df.columns
]

assert not missing_cols, f"Missing columns: {missing_cols}"

print("=" * 80)
print("CELL 16G-A — SPATIAL DATA AUDIT")
print("=" * 80)

for floor in ["F1", "F2", "F3"]:

    df = spatial_df[
        (spatial_df["Floor"] == floor)
        & (spatial_df["Seed"] == 2026)
        & (spatial_df["Method"] == "ORIGINAL")
    ].copy()

    valid = df[
        (df["Valid"] == True)
        & df["True_X"].notna()
        & df["True_Y"].notna()
        & df["Pred_X"].notna()
        & df["Pred_Y"].notna()
    ]

    print(f"\n{floor}")
    print("-" * 60)

    print("Total TEST queries:", len(df))
    print("Valid localization:", len(valid))
    print("Unique RP:", valid["ID_RP"].nunique())

    print(
        "True X range:",
        round(valid["True_X"].min(), 4),
        "→",
        round(valid["True_X"].max(), 4)
    )

    print(
        "True Y range:",
        round(valid["True_Y"].min(), 4),
        "→",
        round(valid["True_Y"].max(), 4)
    )

    print(
        "Predicted X range:",
        round(valid["Pred_X"].min(), 4),
        "→",
        round(valid["Pred_X"].max(), 4)
    )

    print(
        "Predicted Y range:",
        round(valid["Pred_Y"].min(), 4),
        "→",
        round(valid["Pred_Y"].max(), 4)
    )

    print(
        "Mean Error:",
        round(valid["Error"].mean(), 4)
    )

print("\nSPATIAL AUDIT COMPLETED")

In [ ]:

# =============================================================================
# CELL 16G — 2D & 3D INDOOR LOCALIZATION MAP
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import display, HTML

print("=" * 90)
print("CELL 16G — 2D & 3D INDOOR LOCALIZATION MAP")
print("=" * 90)


# =============================================================================
# 1. INPUT VALIDATION
# =============================================================================

assert "predictions_16e_df" in globals(), \
    "Missing predictions_16e_df. Run Cell 16E first."

spatial_df = predictions_16e_df.copy()

required_cols = [
    "Floor", "Seed", "Method", "ID_RP",
    "True_X", "True_Y",
    "Pred_X", "Pred_Y",
    "Error", "Valid", "Fallback"
]

missing_cols = [
    c for c in required_cols
    if c not in spatial_df.columns
]

assert not missing_cols, f"Missing columns: {missing_cols}"

FLOORS = ["F1", "F2", "F3"]

METHODS = [
    "ORIGINAL",
    "CGAN",
    "KNN_K3",
    "KNN_K5",
    "KNN_K7",
    "APMEAN_R3",
    "APMEAN_R5",
    "APMEAN_R7"
]

SEEDS = sorted(spatial_df["Seed"].unique().tolist())

# Artificial vertical separation for visualization only
FLOOR_Z = {
    "F1": 0,
    "F2": 1,
    "F3": 2
}

# Output folder
BASE_DIR = (
    "/content/WIFI_FINGERPRINT_RESEARCH_PHASE1/"
    "16_MULTI_SEED_EXPERIMENT/"
    "16E_MULTI_SEED_WKNN/"
    "16F_FINAL_ANALYSIS"
)

MAP_DIR = os.path.join(BASE_DIR, "16G_LOCALIZATION_MAPS")
os.makedirs(MAP_DIR, exist_ok=True)

print("Available floors:", FLOORS)
print("Available methods:", METHODS)
print("Available seeds:", SEEDS)
print("Output:", MAP_DIR)


# =============================================================================
# 2. DATA PREPARATION FUNCTION
# =============================================================================

def get_spatial_data(floor, method="ORIGINAL", seed=2026):

    df = spatial_df[
        (spatial_df["Floor"] == floor)
        & (spatial_df["Method"] == method)
        & (spatial_df["Seed"] == seed)
    ].copy()

    if df.empty:
        raise ValueError(
            f"No data for {floor}, {method}, seed={seed}"
        )

    for col in [
        "True_X", "True_Y", "Pred_X", "Pred_Y", "Error"
    ]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    valid_mask = (
        df["Valid"].eq(True)
        & df["True_X"].notna()
        & df["True_Y"].notna()
        & df["Pred_X"].notna()
        & df["Pred_Y"].notna()
        & df["Error"].notna()
    )

    valid = df[valid_mask].copy()

    return df, valid


# =============================================================================
# 3. AUDIT SELECTED DATA
# =============================================================================

audit_rows = []

for floor in FLOORS:

    for method in METHODS:

        for seed in SEEDS:

            df, valid = get_spatial_data(
                floor, method, seed
            )

            audit_rows.append({
                "Floor": floor,
                "Method": method,
                "Seed": seed,
                "Total_Queries": len(df),
                "Valid_Queries": len(valid),
                "Unique_RP": valid["ID_RP"].nunique(),
                "Mean_Error": valid["Error"].mean(),
                "Fallback_Count": int(df["Fallback"].sum())
            })

map_audit_df = pd.DataFrame(audit_rows)

assert len(map_audit_df) == 120

print("\nSPATIAL DATA AUDIT")
display(map_audit_df.head(12).round(4))


# =============================================================================
# 4. COLOR CONFIGURATION
# =============================================================================

COLORS = {
    "ground_truth": "#22D3EE",
    "prediction": "#E879F9",
    "reference": "#94A3B8",
    "low_error": "#FDE047",
    "medium_error": "#FB923C",
    "high_error": "#EF4444"
}

ERROR_SCALE = [
    [0.00, "#FDE047"],
    [0.35, "#F59E0B"],
    [0.65, "#F97316"],
    [1.00, "#EF4444"]
]


# =============================================================================
# 5. FUNCTION — 2D LOCALIZATION MAP
# =============================================================================

def create_2d_map(
    floor,
    method="ORIGINAL",
    seed=2026,
    max_vectors=250
):

    raw, df = get_spatial_data(floor, method, seed)

    if len(df) == 0:
        print("No valid localization records.")
        return None

    # Aggregate true RP coordinates and error
    rp_summary = (
        df.groupby("ID_RP")
        .agg(
            True_X=("True_X", "mean"),
            True_Y=("True_Y", "mean"),
            Mean_Error=("Error", "mean"),
            Median_Error=("Error", "median"),
            Query_Count=("Error", "size")
        )
        .reset_index()
    )

    # Use a subset of vectors for readability
    vectors = df.copy()

    if len(vectors) > max_vectors:
        vectors = vectors.sample(
            n=max_vectors,
            random_state=42
        )

    fig = go.Figure()

    # Error vectors
    for _, row in vectors.iterrows():

        fig.add_trace(
            go.Scatter(
                x=[row["True_X"], row["Pred_X"]],
                y=[row["True_Y"], row["Pred_Y"]],
                mode="lines",
                line=dict(
                    color="rgba(245,158,11,0.25)",
                    width=1
                ),
                hoverinfo="skip",
                showlegend=False
            )
        )

    # RP points colored by mean error
    fig.add_trace(
        go.Scatter(
            x=rp_summary["True_X"],
            y=rp_summary["True_Y"],
            mode="markers",
            name="Reference Points",
            marker=dict(
                size=9,
                color=rp_summary["Mean_Error"],
                colorscale=ERROR_SCALE,
                colorbar=dict(
                    title="Mean Error",
                    x=1.02
                ),
                showscale=True,
                line=dict(
                    color="#E2E8F0",
                    width=0.5
                )
            ),
            customdata=np.column_stack([
                rp_summary["ID_RP"].astype(str),
                rp_summary["Mean_Error"],
                rp_summary["Median_Error"],
                rp_summary["Query_Count"]
            ]),
            hovertemplate=(
                "<b>RP %{customdata[0]}</b><br>"
                "X: %{x:.3f}<br>"
                "Y: %{y:.3f}<br>"
                "Mean Error: %{customdata[1]:.3f}<br>"
                "Median Error: %{customdata[2]:.3f}<br>"
                "Queries: %{customdata[3]}<extra></extra>"
            )
        )
    )

    # Ground Truth
    fig.add_trace(
        go.Scatter(
            x=df["True_X"],
            y=df["True_Y"],
            mode="markers",
            name="Ground Truth",
            marker=dict(
                size=5,
                color=COLORS["ground_truth"],
                opacity=0.55,
                symbol="circle"
            ),
            hovertemplate=(
                "Ground Truth<br>"
                "X: %{x:.3f}<br>"
                "Y: %{y:.3f}<extra></extra>"
            )
        )
    )

    # Predicted Position
    fig.add_trace(
        go.Scatter(
            x=df["Pred_X"],
            y=df["Pred_Y"],
            mode="markers",
            name="Predicted Position",
            marker=dict(
                size=6,
                color=COLORS["prediction"],
                opacity=0.65,
                symbol="x"
            ),
            customdata=df[["ID_RP", "Error"]].astype(str).values,
            hovertemplate=(
                "Predicted Position<br>"
                "X: %{x:.3f}<br>"
                "Y: %{y:.3f}<br>"
                "RP: %{customdata[0]}<br>"
                "Error: %{customdata[1]}<extra></extra>"
            )
        )
    )

    mean_error = df["Error"].mean()
    median_error = df["Error"].median()
    p90_error = df["Error"].quantile(0.90)

    fig.update_layout(
        title=dict(
            text=(
                f"{floor} — {method} — Seed {seed}"
                f"<br><sup>Mean Error: {mean_error:.3f} | "
                f"Median: {median_error:.3f} | "
                f"P90: {p90_error:.3f}</sup>"
            ),
            x=0.5
        ),
        template="plotly_dark",
        paper_bgcolor="#07111F",
        plot_bgcolor="#0B1728",
        font=dict(
            color="#E2E8F0"
        ),
        xaxis_title="X Coordinate",
        yaxis_title="Y Coordinate",
        width=1050,
        height=720,
        legend=dict(
            orientation="h",
            y=-0.18
        ),
        margin=dict(l=60, r=100, t=100, b=100)
    )

    fig.update_xaxes(
        showgrid=True,
        gridcolor="rgba(148,163,184,0.15)",
        zeroline=False
    )

    fig.update_yaxes(
        showgrid=True,
        gridcolor="rgba(148,163,184,0.15)",
        zeroline=False,
        scaleanchor="x",
        scaleratio=1
    )

    return fig


# =============================================================================
# 6. FUNCTION — 3D MULTI-FLOOR MAP
# =============================================================================

def create_3d_map(
    method="ORIGINAL",
    seed=2026,
    max_vectors_per_floor=150
):

    fig = go.Figure()

    all_true_x = []
    all_true_y = []

    for floor in FLOORS:

        raw, df = get_spatial_data(
            floor, method, seed
        )

        z = FLOOR_Z[floor]

        all_true_x.extend(df["True_X"].tolist())
        all_true_y.extend(df["True_Y"].tolist())

        # RP-level summary
        rp_summary = (
            df.groupby("ID_RP")
            .agg(
                True_X=("True_X", "mean"),
                True_Y=("True_Y", "mean"),
                Mean_Error=("Error", "mean")
            )
            .reset_index()
        )

        # Floor plane
        x_min = df["True_X"].min() - 1
        x_max = df["True_X"].max() + 1
        y_min = df["True_Y"].min() - 1
        y_max = df["True_Y"].max() + 1

        fig.add_trace(
            go.Mesh3d(
                x=[x_min, x_max, x_max, x_min],
                y=[y_min, y_min, y_max, y_max],
                z=[z, z, z, z],
                i=[0, 0],
                j=[1, 2],
                k=[2, 3],
                color="#164E63",
                opacity=0.12,
                name=f"{floor} floor plane",
                showlegend=False,
                hoverinfo="skip"
            )
        )

        # Error vectors
        vectors = df.copy()

        if len(vectors) > max_vectors_per_floor:
            vectors = vectors.sample(
                n=max_vectors_per_floor,
                random_state=42
            )

        vx, vy, vz = [], [], []

        for _, row in vectors.iterrows():

            vx.extend([
                row["True_X"],
                row["Pred_X"],
                None
            ])

            vy.extend([
                row["True_Y"],
                row["Pred_Y"],
                None
            ])

            vz.extend([
                z,
                z,
                None
            ])

        fig.add_trace(
            go.Scatter3d(
                x=vx,
                y=vy,
                z=vz,
                mode="lines",
                line=dict(
                    color="rgba(245,158,11,0.35)",
                    width=2
                ),
                name=f"{floor} error vectors",
                showlegend=False,
                hoverinfo="skip"
            )
        )

        # RP reference points
        fig.add_trace(
            go.Scatter3d(
                x=rp_summary["True_X"],
                y=rp_summary["True_Y"],
                z=[z] * len(rp_summary),
                mode="markers",
                name=f"{floor} Reference Points",
                marker=dict(
                    size=3,
                    color=rp_summary["Mean_Error"],
                    colorscale=ERROR_SCALE,
                    cmin=0,
                    cmax=max(
                        1,
                        float(rp_summary["Mean_Error"].quantile(0.95))
                    ),
                    opacity=0.85,
                    showscale=(floor == "F1"),
                    colorbar=dict(
                        title="Mean Error",
                        x=1.02
                    ) if floor == "F1" else None
                ),
                customdata=np.column_stack([
                    rp_summary["ID_RP"].astype(str),
                    rp_summary["Mean_Error"]
                ]),
                hovertemplate=(
                    f"{floor} RP %{{customdata[0]}}<br>"
                    "X: %{x:.3f}<br>"
                    "Y: %{y:.3f}<br>"
                    "Z: %{z:.1f} (display level)<br>"
                    "Mean Error: %{customdata[1]:.3f}"
                    "<extra></extra>"
                )
            )
        )

        # Ground Truth points
        fig.add_trace(
            go.Scatter3d(
                x=df["True_X"],
                y=df["True_Y"],
                z=[z] * len(df),
                mode="markers",
                name=f"{floor} Ground Truth",
                marker=dict(
                    size=2.5,
                    color=COLORS["ground_truth"],
                    opacity=0.45
                ),
                hovertemplate=(
                    f"{floor} Ground Truth<br>"
                    "X: %{x:.3f}<br>"
                    "Y: %{y:.3f}<extra></extra>"
                )
            )
        )

        # Predicted points
        fig.add_trace(
            go.Scatter3d(
                x=df["Pred_X"],
                y=df["Pred_Y"],
                z=[z] * len(df),
                mode="markers",
                name=f"{floor} Predicted",
                marker=dict(
                    size=2.5,
                    color=COLORS["prediction"],
                    opacity=0.55,
                    symbol="x"
                ),
                hovertemplate=(
                    f"{floor} Predicted<br>"
                    "X: %{x:.3f}<br>"
                    "Y: %{y:.3f}<extra></extra>"
                )
            )
        )

    fig.update_layout(
        title=dict(
            text=(
                f"3D MULTI-FLOOR LOCALIZATION — {method}"
                f"<br><sup>Random Seed: {seed} | "
                "Z levels are display offsets, not architectural heights</sup>"
            ),
            x=0.5
        ),
        template="plotly_dark",
        paper_bgcolor="#07111F",
        font=dict(color="#E2E8F0"),
        width=1250,
        height=850,
        scene=dict(
            xaxis_title="X Coordinate",
            yaxis_title="Y Coordinate",
            zaxis_title="Floor Level (display only)",
            xaxis=dict(
                backgroundcolor="#0B1728",
                gridcolor="#334155"
            ),
            yaxis=dict(
                backgroundcolor="#0B1728",
                gridcolor="#334155"
            ),
            zaxis=dict(
                backgroundcolor="#0B1728",
                gridcolor="#334155",
                tickvals=[0, 1, 2],
                ticktext=["F1", "F2", "F3"]
            ),
            aspectmode="manual",
            aspectratio=dict(
                x=2.5,
                y=1.2,
                z=0.55
            ),
            camera=dict(
                eye=dict(x=1.6, y=1.6, z=1.15)
            )
        ),
        legend=dict(
            orientation="h",
            y=-0.08
        ),
        margin=dict(l=0, r=0, t=100, b=80)
    )

    return fig


# =============================================================================
# 7. FUNCTION — RP ERROR HEATMAP
# =============================================================================

def create_error_heatmap(
    floor,
    method="ORIGINAL",
    seed=2026
):

    raw, df = get_spatial_data(floor, method, seed)

    rp_summary = (
        df.groupby("ID_RP")
        .agg(
            X=("True_X", "mean"),
            Y=("True_Y", "mean"),
            Mean_Error=("Error", "mean"),
            Query_Count=("Error", "size")
        )
        .reset_index()
    )

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=rp_summary["X"],
            y=rp_summary["Y"],
            mode="markers",
            marker=dict(
                size=13,
                color=rp_summary["Mean_Error"],
                colorscale=ERROR_SCALE,
                showscale=True,
                colorbar=dict(title="Mean Error"),
                line=dict(
                    color="#E2E8F0",
                    width=0.6
                )
            ),
            customdata=np.column_stack([
                rp_summary["ID_RP"].astype(str),
                rp_summary["Mean_Error"],
                rp_summary["Query_Count"]
            ]),
            hovertemplate=(
                "RP: %{customdata[0]}<br>"
                "X: %{x:.3f}<br>"
                "Y: %{y:.3f}<br>"
                "Mean Error: %{customdata[1]:.3f}<br>"
                "Queries: %{customdata[2]}<extra></extra>"
            )
        )
    )

    fig.update_layout(
        title=f"{floor} — Localization Error Heatmap — {method}",
        template="plotly_dark",
        paper_bgcolor="#07111F",
        plot_bgcolor="#0B1728",
        xaxis_title="X Coordinate",
        yaxis_title="Y Coordinate",
        width=950,
        height=650
    )

    fig.update_yaxes(
        scaleanchor="x",
        scaleratio=1
    )

    return fig


# =============================================================================
# 8. GENERATE 2D MAPS FOR ALL FLOORS
# =============================================================================

print("\n" + "=" * 90)
print("GENERATING 2D LOCALIZATION MAPS")
print("=" * 90)

for floor in FLOORS:

    print(f"\nGenerating {floor}...")

    fig = create_2d_map(
        floor=floor,
        method="ORIGINAL",
        seed=2026
    )

    fig.show()

    html_path = os.path.join(
        MAP_DIR,
        f"2D_{floor}_ORIGINAL_SEED2026.html"
    )

    fig.write_html(
        html_path,
        include_plotlyjs=True
    )

    print("Saved:", html_path)


# =============================================================================
# 9. GENERATE 3D MULTI-FLOOR MAP
# =============================================================================

print("\n" + "=" * 90)
print("GENERATING 3D MULTI-FLOOR MAP")
print("=" * 90)

fig_3d = create_3d_map(
    method="ORIGINAL",
    seed=2026
)

fig_3d.show()

map_3d_path = os.path.join(
    MAP_DIR,
    "3D_MULTIFLOOR_ORIGINAL_SEED2026.html"
)

fig_3d.write_html(
    map_3d_path,
    include_plotlyjs=True
)

print("Saved:", map_3d_path)


# =============================================================================
# 10. GENERATE ERROR HEATMAPS
# =============================================================================

print("\n" + "=" * 90)
print("GENERATING ERROR HEATMAPS")
print("=" * 90)

for floor in FLOORS:

    fig = create_error_heatmap(
        floor=floor,
        method="ORIGINAL",
        seed=2026
    )

    fig.show()

    heatmap_path = os.path.join(
        MAP_DIR,
        f"ERROR_HEATMAP_{floor}_ORIGINAL_SEED2026.html"
    )

    fig.write_html(
        heatmap_path,
        include_plotlyjs=True
    )

    print("Saved:", heatmap_path)


# =============================================================================
# 11. CREATE METHOD COMPARISON MAPS
# =============================================================================

# Compare ORIGINAL with selected methods.
# Keep the same floor, seed and TEST queries.

COMPARISON_METHODS = [
    "ORIGINAL",
    "CGAN",
    "KNN_K7",
    "APMEAN_R3"
]

for floor in FLOORS:

    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=COMPARISON_METHODS,
        horizontal_spacing=0.10,
        vertical_spacing=0.14
    )

    for i, method in enumerate(COMPARISON_METHODS):

        row = i // 2 + 1
        col = i % 2 + 1

        raw, df = get_spatial_data(
            floor, method, 2026
        )

        fig.add_trace(
            go.Scatter(
                x=df["True_X"],
                y=df["True_Y"],
                mode="markers",
                marker=dict(
                    size=4,
                    color=COLORS["ground_truth"],
                    opacity=0.45
                ),
                name="Ground Truth",
                showlegend=(i == 0)
            ),
            row=row,
            col=col
        )

        fig.add_trace(
            go.Scatter(
                x=df["Pred_X"],
                y=df["Pred_Y"],
                mode="markers",
                marker=dict(
                    size=4,
                    color=COLORS["prediction"],
                    opacity=0.55
                ),
                name="Predicted",
                showlegend=(i == 0)
            ),
            row=row,
            col=col
        )

        mean_error = df["Error"].mean()

        fig.add_annotation(
            x=0.5,
            y=1.08,
            xref=f"x{'' if i == 0 else i+1} domain",
            yref=f"y{'' if i == 0 else i+1} domain",
            text=f"Mean Error: {mean_error:.3f}",
            showarrow=False,
            font=dict(color="#FBBF24", size=10)
        )

    fig.update_layout(
        title=(
            f"{floor} — METHOD LOCALIZATION COMPARISON "
            "(Seed 2026)"
        ),
        template="plotly_dark",
        paper_bgcolor="#07111F",
        width=1200,
        height=850
    )

    fig_path = os.path.join(
        MAP_DIR,
        f"METHOD_COMPARISON_{floor}_SEED2026.html"
    )

    fig.write_html(
        fig_path,
        include_plotlyjs=True
    )

    fig.show()

    print("Saved:", fig_path)


# =============================================================================
# 12. EXPORT SPATIAL DATA
# =============================================================================

spatial_summary = (
    spatial_df[
        (spatial_df["Seed"] == 2026)
        & (spatial_df["Method"] == "ORIGINAL")
    ]
    .groupby("Floor")
    .agg(
        Total_Queries=("ID_RP", "size"),
        Unique_RP=("ID_RP", "nunique"),
        Mean_Error=("Error", "mean"),
        Median_Error=("Error", "median"),
        RMSE=("Error", lambda x: np.sqrt(np.mean(x.dropna() ** 2))),
        P90_Error=("Error", lambda x: x.quantile(0.90))
    )
    .reset_index()
)

spatial_summary.to_csv(
    os.path.join(
        MAP_DIR,
        "SPATIAL_SUMMARY_ORIGINAL_SEED2026.csv"
    ),
    index=False
)

map_audit_df.to_csv(
    os.path.join(
        MAP_DIR,
        "SPATIAL_DATA_AUDIT.csv"
    ),
    index=False
)


# =============================================================================
# 13. FINAL VALIDATION
# =============================================================================

expected_files = [
    f"2D_{floor}_ORIGINAL_SEED2026.html"
    for floor in FLOORS
]

expected_files += [
    "3D_MULTIFLOOR_ORIGINAL_SEED2026.html"
]

expected_files += [
    f"ERROR_HEATMAP_{floor}_ORIGINAL_SEED2026.html"
    for floor in FLOORS
]

expected_files += [
    f"METHOD_COMPARISON_{floor}_SEED2026.html"
    for floor in FLOORS
]

for filename in expected_files:

    path = os.path.join(MAP_DIR, filename)

    assert os.path.exists(path), \
        f"Missing output: {filename}"

print("\n" + "=" * 90)
print("CELL 16G — COMPLETED SUCCESSFULLY")
print("=" * 90)

print("2D floor maps: 3")
print("3D multi-floor map: 1")
print("Error heatmaps: 3")
print("Method comparison maps: 3")
print("Total interactive HTML maps:", len(expected_files))

print("\nOutput folder:")
print(MAP_DIR)

print("\nValidation: PASS")
print("=" * 90)


In [ ]:

import os
from IPython.display import HTML, display

map_path = os.path.join(
    MAP_DIR,
    "3D_MULTIFLOOR_ORIGINAL_SEED2026.html"
)

assert os.path.exists(map_path), (
    f"Không tìm thấy bản đồ: {map_path}"
)

# Đọc nội dung HTML đã xuất
with open(map_path, "r", encoding="utf-8") as f:
    html_content = f.read()

# Hiển thị trực tiếp trong output của Colab
display(HTML(html_content))

print("3D MULTI-FLOOR MAP — DISPLAY REQUEST COMPLETED")
print("File:", map_path)


In [ ]:

# =============================================================================
# CELL 16G-C — CLEAN 3D RP-LEVEL LOCALIZATION MAP
# =============================================================================

import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from IPython.display import display, HTML

print("=" * 85)
print("CELL 16G-C — CLEAN 3D RP-LEVEL LOCALIZATION MAP")
print("=" * 85)

METHOD = "ORIGINAL"
SEED = 2026

FLOORS = ["F1", "F2", "F3"]

FLOOR_Z = {
    "F1": 0,
    "F2": 1,
    "F3": 2
}

# =============================================================================
# 1. PREPARE RP-LEVEL DATA
# =============================================================================

rp_data = {}

for floor in FLOORS:

    df = predictions_16e_df[
        (predictions_16e_df["Floor"] == floor)
        & (predictions_16e_df["Method"] == METHOD)
        & (predictions_16e_df["Seed"] == SEED)
        & (predictions_16e_df["Valid"] == True)
    ].copy()

    df = df.dropna(
        subset=[
            "True_X", "True_Y",
            "Pred_X", "Pred_Y",
            "Error"
        ]
    )

    # Aggregate repeated measurements at the same RP
    summary = (
        df.groupby("ID_RP")
        .agg(
            True_X=("True_X", "mean"),
            True_Y=("True_Y", "mean"),

            Pred_X=("Pred_X", "median"),
            Pred_Y=("Pred_Y", "median"),

            Mean_Error=("Error", "mean"),
            Median_Error=("Error", "median"),
            Max_Error=("Error", "max"),

            Measurement_Count=("Error", "size")
        )
        .reset_index()
    )

    assert summary["ID_RP"].is_unique

    rp_data[floor] = summary

    print(
        f"{floor}: {len(df)} TEST measurements "
        f"-> {len(summary)} RP representatives"
    )

# =============================================================================
# 2. CREATE 3D FIGURE
# =============================================================================

fig = go.Figure()

error_values = np.concatenate([
    rp_data[f]["Mean_Error"].values
    for f in FLOORS
])

color_max = max(
    1,
    float(np.percentile(error_values, 95))
)

ERROR_SCALE = [
    [0.00, "#FDE047"],
    [0.35, "#FBBF24"],
    [0.65, "#F97316"],
    [1.00, "#EF4444"]
]

# =============================================================================
# 3. DRAW FLOOR PLANES
# =============================================================================

for floor in FLOORS:

    data = rp_data[floor]
    z = FLOOR_Z[floor]

    x_min = data["True_X"].min() - 1
    x_max = data["True_X"].max() + 1

    y_min = data["True_Y"].min() - 1
    y_max = data["True_Y"].max() + 1

    fig.add_trace(
        go.Mesh3d(
            x=[x_min, x_max, x_max, x_min],
            y=[y_min, y_min, y_max, y_max],
            z=[z, z, z, z],
            i=[0, 0],
            j=[1, 2],
            k=[2, 3],
            color="#164E63",
            opacity=0.10,
            name=f"{floor} Floor",
            showlegend=False,
            hoverinfo="skip"
        )
    )

# =============================================================================
# 4. DRAW ERROR VECTORS — ONE PER RP
# =============================================================================

for floor in FLOORS:

    data = rp_data[floor]
    z = FLOOR_Z[floor]

    xs, ys, zs = [], [], []

    for _, row in data.iterrows():

        xs.extend([
            row["True_X"],
            row["Pred_X"],
            None
        ])

        ys.extend([
            row["True_Y"],
            row["Pred_Y"],
            None
        ])

        zs.extend([z, z, None])

    fig.add_trace(
        go.Scatter3d(
            x=xs,
            y=ys,
            z=zs,
            mode="lines",
            line=dict(
                color="#F59E0B",
                width=3
            ),
            name=f"{floor} Error Vectors",
            showlegend=False,
            hoverinfo="skip"
        )
    )

# =============================================================================
# 5. DRAW GROUND TRUTH
# =============================================================================

for floor in FLOORS:

    data = rp_data[floor]
    z = FLOOR_Z[floor]

    fig.add_trace(
        go.Scatter3d(
            x=data["True_X"],
            y=data["True_Y"],
            z=[z] * len(data),
            mode="markers",
            name=f"{floor} Ground Truth",
            marker=dict(
                size=4,
                color="#22D3EE",
                opacity=0.95,
                symbol="circle"
            ),
            customdata=data[
                ["ID_RP", "Measurement_Count"]
            ].astype(str).values,
            hovertemplate=(
                f"<b>{floor} Ground Truth</b><br>"
                "RP: %{customdata[0]}<br>"
                "X: %{x:.3f}<br>"
                "Y: %{y:.3f}<br>"
                "Measurements: %{customdata[1]}"
                "<extra></extra>"
            )
        )
    )

# =============================================================================
# 6. DRAW PREDICTED POSITION — COLOR BY MEAN ERROR
# =============================================================================

for floor in FLOORS:

    data = rp_data[floor]
    z = FLOOR_Z[floor]

    fig.add_trace(
        go.Scatter3d(
            x=data["Pred_X"],
            y=data["Pred_Y"],
            z=[z] * len(data),
            mode="markers",
            name=f"{floor} Predicted",
            marker=dict(
                size=5,
                color=data["Mean_Error"],
                colorscale=ERROR_SCALE,
                cmin=0,
                cmax=color_max,
                showscale=(floor == "F1"),
                colorbar=dict(
                    title="Mean Error",
                    x=1.03
                ) if floor == "F1" else None,
                symbol="diamond",
                line=dict(
                    color="#FFFFFF",
                    width=0.5
                )
            ),
            customdata=data[
                [
                    "ID_RP",
                    "Mean_Error",
                    "Median_Error",
                    "Max_Error",
                    "Measurement_Count"
                ]
            ].values,
            hovertemplate=(
                f"<b>{floor} Predicted</b><br>"
                "RP: %{customdata[0]}<br>"
                "X: %{x:.3f}<br>"
                "Y: %{y:.3f}<br>"
                "Mean Error: %{customdata[1]:.3f}<br>"
                "Median Error: %{customdata[2]:.3f}<br>"
                "Maximum Error: %{customdata[3]:.3f}<br>"
                "Measurements: %{customdata[4]}"
                "<extra></extra>"
            )
        )
    )

# =============================================================================
# 7. LAYOUT
# =============================================================================

fig.update_layout(
    title=dict(
        text=(
            "3D MULTI-FLOOR LOCALIZATION — RP LEVEL"
            "<br><sup>"
            "ORIGINAL | Seed 2026 | Median predicted position per RP"
            "<br>"
            "Z levels are display offsets, not physical floor heights"
            "</sup>"
        ),
        x=0.5
    ),

    template="plotly_dark",

    paper_bgcolor="#07111F",

    width=1350,
    height=900,

    scene=dict(
        xaxis_title="X Coordinate",
        yaxis_title="Y Coordinate",
        zaxis_title="Floor",

        xaxis=dict(gridcolor="#334155"),
        yaxis=dict(gridcolor="#334155"),
        zaxis=dict(
            tickvals=[0, 1, 2],
            ticktext=["F1", "F2", "F3"],
            gridcolor="#334155"
        ),

        aspectmode="manual",
        aspectratio=dict(
            x=2.5,
            y=1.2,
            z=0.7
        ),

        camera=dict(
            eye=dict(
                x=1.6,
                y=1.6,
                z=1.2
            )
        )
    ),

    legend=dict(
        orientation="h",
        y=-0.08
    ),

    margin=dict(
        l=0,
        r=0,
        t=120,
        b=100
    )
)

# =============================================================================
# 8. DISPLAY AND EXPORT
# =============================================================================

clean_map_path = os.path.join(
    MAP_DIR,
    "3D_MULTIFLOOR_RP_LEVEL_ORIGINAL_SEED2026.html"
)

fig.write_html(
    clean_map_path,
    include_plotlyjs=True
)

display(HTML(fig.to_html(
    full_html=False,
    include_plotlyjs="cdn"
)))

print("\n" + "=" * 85)
print("CELL 16G-C COMPLETED")
print("=" * 85)

for floor in FLOORS:
    print(
        f"{floor}: {len(rp_data[floor])} RP representatives"
    )

print("Saved:", clean_map_path)


In [ ]:
# =============================================================================
# CELL 16G-D — ORIGINAL vs CGAN vs KNN_K7
# 2D LOCALIZATION COMPARISON — ALL FLOORS
# =============================================================================

import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from plotly.subplots import make_subplots
from IPython.display import display, HTML

print("=" * 95)
print("CELL 16G-D — 2D LOCALIZATION COMPARISON")
print("=" * 95)

# =============================================================================
# 1. CONFIGURATION
# =============================================================================

FLOORS = ["F1", "F2", "F3"]

METHODS = [
    "ORIGINAL",
    "CGAN",
    "KNN_K7"
]

SEED = 2026

METHOD_COLORS = {
    "ORIGINAL": "#22D3EE",
    "CGAN": "#E879F9",
    "KNN_K7": "#34D399"
}

ERROR_SCALE = [
    [0.00, "#FDE047"],
    [0.35, "#FBBF24"],
    [0.65, "#F97316"],
    [1.00, "#EF4444"]
]

assert "predictions_16e_df" in globals(), \
    "Missing predictions_16e_df. Run Cell 16E first."

# =============================================================================
# 2. AGGREGATE PREDICTIONS BY RP
# =============================================================================

comparison_data = {}

for floor in FLOORS:

    comparison_data[floor] = {}

    for method in METHODS:

        df = predictions_16e_df[
            (predictions_16e_df["Floor"] == floor)
            & (predictions_16e_df["Method"] == method)
            & (predictions_16e_df["Seed"] == SEED)
            & (predictions_16e_df["Valid"] == True)
        ].copy()

        df = df.dropna(
            subset=[
                "ID_RP",
                "True_X", "True_Y",
                "Pred_X", "Pred_Y",
                "Error"
            ]
        )

        summary = (
            df.groupby("ID_RP")
            .agg(
                True_X=("True_X", "mean"),
                True_Y=("True_Y", "mean"),
                Pred_X=("Pred_X", "median"),
                Pred_Y=("Pred_Y", "median"),
                Mean_Error=("Error", "mean"),
                Median_Error=("Error", "median"),
                Measurement_Count=("Error", "size")
            )
            .reset_index()
        )

        assert summary["ID_RP"].is_unique

        comparison_data[floor][method] = summary

        print(
            f"{floor:>2} | {method:<10} | "
            f"Queries: {len(df):>4} | "
            f"RP: {len(summary):>3} | "
            f"Mean Error: {df["Error"].mean():.4f}"
        )

# =============================================================================
# 3. VERIFY SAME RP SET ACROSS METHODS
# =============================================================================

for floor in FLOORS:

    rp_sets = [
        set(
            comparison_data[floor][method]["ID_RP"]
        )
        for method in METHODS
    ]

    assert all(
        rp_sets[0] == rp_set
        for rp_set in rp_sets[1:]
    ), f"RP mismatch detected in {floor}"

print("\nRP CONSISTENCY CHECK: PASS")

# =============================================================================
# 4. GLOBAL ERROR SCALE
# =============================================================================

all_errors = []

for floor in FLOORS:
    for method in METHODS:
        all_errors.extend(
            comparison_data[floor][method]["Mean_Error"].tolist()
        )

color_max = max(
    1.0,
    float(np.percentile(all_errors, 95))
)

# =============================================================================
# 5. CREATE 3 x 3 COMPARISON DASHBOARD
# Rows: Methods
# Columns: Floors
# =============================================================================

fig = make_subplots(
    rows=3,
    cols=3,
    subplot_titles=[
        f"{method} — {floor}"
        for method in METHODS
        for floor in FLOORS
    ],
    horizontal_spacing=0.07,
    vertical_spacing=0.12
)

for row_idx, method in enumerate(METHODS, start=1):

    for col_idx, floor in enumerate(FLOORS, start=1):

        data = comparison_data[floor][method]

        # -------------------------------------------------------------
        # Error vectors: Ground Truth -> Predicted Position
        # -------------------------------------------------------------

        vx, vy = [], []

        for _, rp in data.iterrows():

            vx.extend([
                rp["True_X"],
                rp["Pred_X"],
                None
            ])

            vy.extend([
                rp["True_Y"],
                rp["Pred_Y"],
                None
            ])

        fig.add_trace(
            go.Scatter(
                x=vx,
                y=vy,
                mode="lines",
                line=dict(
                    color="rgba(245,158,11,0.35)",
                    width=1.2
                ),
                name="Error Vector",
                showlegend=(
                    row_idx == 1 and col_idx == 1
                ),
                hoverinfo="skip"
            ),
            row=row_idx,
            col=col_idx
        )

        # -------------------------------------------------------------
        # Ground Truth
        # -------------------------------------------------------------

        fig.add_trace(
            go.Scatter(
                x=data["True_X"],
                y=data["True_Y"],
                mode="markers",
                marker=dict(
                    size=5,
                    color="#22D3EE",
                    symbol="circle",
                    opacity=0.85
                ),
                name="Ground Truth",
                showlegend=(
                    row_idx == 1 and col_idx == 1
                ),
                customdata=data["ID_RP"].astype(str),
                hovertemplate=(
                    "<b>Ground Truth</b><br>"
                    "RP: %{customdata}<br>"
                    "X: %{x:.3f}<br>"
                    "Y: %{y:.3f}"
                    "<extra></extra>"
                )
            ),
            row=row_idx,
            col=col_idx
        )

        # -------------------------------------------------------------
        # Predicted Position
        # Color = Mean Error at RP
        # -------------------------------------------------------------

        fig.add_trace(
            go.Scatter(
                x=data["Pred_X"],
                y=data["Pred_Y"],
                mode="markers",
                marker=dict(
                    size=6,
                    color=data["Mean_Error"],
                    colorscale=ERROR_SCALE,
                    cmin=0,
                    cmax=color_max,
                    showscale=(
                        row_idx == 1 and col_idx == 3
                    ),
                    colorbar=dict(
                        title="Mean Error",
                        x=1.02,
                        len=0.75
                    ) if (
                        row_idx == 1 and col_idx == 3
                    ) else None,
                    symbol="diamond",
                    line=dict(
                        color="#FFFFFF",
                        width=0.35
                    )
                ),
                name="Predicted Position",
                showlegend=(
                    row_idx == 1 and col_idx == 1
                ),
                customdata=data[
                    [
                        "ID_RP",
                        "Mean_Error",
                        "Median_Error",
                        "Measurement_Count"
                    ]
                ].values,
                hovertemplate=(
                    "<b>Predicted Position</b><br>"
                    "RP: %{customdata[0]}<br>"
                    "X: %{x:.3f}<br>"
                    "Y: %{y:.3f}<br>"
                    "Mean Error: %{customdata[1]:.3f}<br>"
                    "Median Error: %{customdata[2]:.3f}<br>"
                    "Measurements: %{customdata[3]}"
                    "<extra></extra>"
                )
            ),
            row=row_idx,
            col=col_idx
        )

        # -------------------------------------------------------------
        # Add Mean Error Annotation (FIXED PLOTLY SUBPLOT INDEXING)
        # -------------------------------------------------------------

        raw = predictions_16e_df[
            (predictions_16e_df["Floor"] == floor)
            & (predictions_16e_df["Method"] == method)
            & (predictions_16e_df["Seed"] == SEED)
            & (predictions_16e_df["Valid"] == True)
        ]

        mean_error = raw["Error"].mean()
        median_error = raw["Error"].median()
        p90_error = raw["Error"].quantile(0.90)

        subplot_idx = (row_idx - 1) * 3 + col_idx
        xref_str = "x domain" if subplot_idx == 1 else f"x{subplot_idx} domain"
        yref_str = "y domain" if subplot_idx == 1 else f"y{subplot_idx} domain"

        fig.add_annotation(
            x=0.5,
            y=1.12,
            xref=xref_str,
            yref=yref_str,
            text=(
                f"Mean: {mean_error:.3f} | "
                f"Median: {median_error:.3f} | "
                f"P90: {p90_error:.3f}"
            ),
            showarrow=False,
            font=dict(
                size=10,
                color="#FBBF24"
            )
        )

        fig.update_xaxes(
            title_text="X Coordinate" if row_idx == 3 else None,
            showgrid=True,
            gridcolor="rgba(148,163,184,0.15)",
            zeroline=False,
            row=row_idx,
            col=col_idx
        )

        fig.update_yaxes(
            title_text="Y Coordinate" if col_idx == 1 else None,
            showgrid=True,
            gridcolor="rgba(148,163,184,0.15)",
            zeroline=False,
            scaleanchor="x" if subplot_idx == 1 else f"x{subplot_idx}",
            scaleratio=1,
            row=row_idx,
            col=col_idx
        )

# =============================================================================
# 6. DASHBOARD LAYOUT
# =============================================================================

fig.update_layout(
    title=dict(
        text=(
            "INDOOR LOCALIZATION — METHOD COMPARISON"
            "<br><sup>"
            "ORIGINAL vs CGAN vs KNN_K7 | "
            "F1, F2, F3 | Random Seed 2026"
            "<br>"
            "Cyan: Ground Truth | Diamond: Predicted | "
            "Yellow-Orange Lines: Error Vectors"
            "</sup>"
        ),
        x=0.5
    ),

    template="plotly_dark",
    paper_bgcolor="#07111F",
    plot_bgcolor="#0B1728",

    font=dict(
        color="#E2E8F0"
    ),

    width=1550,
    height=1250,

    legend=dict(
        orientation="h",
        y=-0.055,
        x=0.5,
        xanchor="center"
    ),

    margin=dict(
        l=75,
        r=110,
        t=150,
        b=100
    )
)

# =============================================================================
# 7. NUMERICAL COMPARISON TABLE
# =============================================================================

comparison_rows = []

for floor in FLOORS:

    baseline = comparison_data[floor]["ORIGINAL"]

    original_error = predictions_16e_df[
        (predictions_16e_df["Floor"] == floor)
        & (predictions_16e_df["Method"] == "ORIGINAL")
        & (predictions_16e_df["Seed"] == SEED)
        & (predictions_16e_df["Valid"] == True)
    ]["Error"].mean()

    for method in METHODS:

        raw = predictions_16e_df[
            (predictions_16e_df["Floor"] == floor)
            & (predictions_16e_df["Method"] == method)
            & (predictions_16e_df["Seed"] == SEED)
            & (predictions_16e_df["Valid"] == True)
        ]

        method_mean = raw["Error"].mean()

        improvement = original_error - method_mean

        comparison_rows.append({
            "Floor": floor,
            "Method": method,
            "Mean_Error": method_mean,
            "Median_Error": raw["Error"].median(),
            "RMSE": np.sqrt(
                np.mean(raw["Error"].values ** 2)
            ),
            "P90_Error": raw["Error"].quantile(0.90),
            "Improvement_vs_ORIGINAL": improvement,
            "Improvement_Percent": (
                improvement / original_error * 100
            )
        })

comparison_summary_df = pd.DataFrame(comparison_rows)

# =============================================================================
# 8. DISPLAY DASHBOARD AND TABLE
# =============================================================================

display(HTML(
    fig.to_html(
        full_html=False,
        include_plotlyjs="cdn"
    )
))

print("\nNUMERICAL COMPARISON — SEED 2026")

display(
    comparison_summary_df.round(4)
)

# =============================================================================
# 9. EXPORT
# =============================================================================

comparison_dir = os.path.join(
    MAP_DIR,
    "METHOD_COMPARISON_ORIGINAL_CGAN_KNN"
)

os.makedirs(
    comparison_dir,
    exist_ok=True
)

html_path = os.path.join(
    comparison_dir,
    "16G_D_METHOD_COMPARISON_ALL_FLOORS.html"
)

csv_path = os.path.join(
    comparison_dir,
    "16G_D_METHOD_COMPARISON_METRICS.csv"
)

fig.write_html(
    html_path,
    include_plotlyjs=True
)

comparison_summary_df.to_csv(
    csv_path,
    index=False
)

print("\n" + "=" * 95)
print("CELL 16G-D COMPLETED SUCCESSFULLY")
print("=" * 95)

print("Floors:", FLOORS)
print("Methods:", METHODS)
print("Random Seed:", SEED)
print("Comparison plots: 9")
print("RP consistency: PASS")
print("HTML:", html_path)
print("CSV:", csv_path)
print("=" * 95)


# PHẦN BỔ SUNG HOÀN CHỈNH — TỌA ĐỘ MỚI VÀ ĐÁNH GIÁ

77 cell gốc được giữ nguyên cả source/metadata/output; thêm một cell chuẩn bị input trước Cell 16F-R và các cell bổ sung ở cuối. Output các cell bổ sung được tạo bằng DATASET_DLULIB.zip đã gửi; trạng thái chạy lại phần gốc ghi trong HUONG_DAN.md đi kèm. Các output cũ phía trên là output nguyên bản của file bạn gửi, không bị thay bằng kết quả chạy kiểm tra.

**Chạy trên Google Colab:** chọn runtime Python/GPU, tải DATASET_DLULIB.zip khi phần gốc yêu cầu, rồi Run all. Thư viện dùng: torch, numpy, pandas, scipy, scikit-learn, matplotlib, seaborn, openpyxl, plotly; nếu Colab thiếu thư viện, cài trước bằng `%pip install torch numpy pandas scipy scikit-learn matplotlib seaborn openpyxl plotly`. Giữ tên ZIP như trên.

**Chạy nhanh phần mới:** chạy phần gốc tới Cell 7 (index 37), rồi 17A, 17B và 17D/E. Bỏ qua 17C khi chưa chạy multi-seed. **17C chỉ dùng sau Cell 16E.** Output lưu tại OUT/17_NEW_COORDINATES_EXTENSION; cell cuối tự đóng gói ZIP và tải khi chạy Colab.

Phạm vi ảnh chỉ đọc được tên đề tài và hai mục tiêu; phần “Nội dung thực hiện” bị cắt. Candidate nằm trong convex hull TRAIN; khi chưa có polygon mặt bằng, chưa đảm bảo điểm nằm trong vùng đi lại được. Giới hạn RSSI/số AP chỉ là QC cơ bản, chưa chứng minh vật lý chính xác hay tiết kiệm chi phí thực địa.

## 17A. Hàm riêng và cấu hình


In [ ]:
# PHẦN BỔ SUNG 17A — HÀM RIÊNG, KHÔNG GHI ĐÈ HÀM CŨ
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
from scipy.spatial import Delaunay, cKDTree, QhullError

# Các khoảng cách dưới đây dùng đơn vị tọa độ nguồn, KHÔNG mặc định là mét.
EXT17_CONFIG = {
    'seed': 20261105,
    'target_ratio': 1.0,
    'max_candidates': 300,
    'samples_per_coordinate': 3,
    'novelty_tolerance': 1e-5,
    'min_observed_ap': 3,
    'noise_dim': 32,  # phải khớp CGAN_NOISE_DIM của mô hình đã train
}
# Có thể điền polygon đi lại được theo tầng từ bản vẽ đã xác minh.
# Ví dụ cấu trúc: {'F1': [[x1,y1], [x2,y2], ...]}; không dùng ví dụ làm bản đồ thật.
EXT17_WALKABLE_POLYGONS = {}


def ext17_new_coordinates(train, count, seed, tolerance=1e-5, polygon=None):
    """Sinh lưới trong convex hull TRAIN, cách TRAIN > tolerance; không đọc TEST."""
    xy = np.unique(train[['X', 'Y']].to_numpy(float), axis=0)
    if not np.isfinite(xy).all() or len(xy) < 3 or count < 1:
        raise ValueError('Cần ít nhất 3 tọa độ TRAIN hợp lệ và count >= 1.')
    try:
        hull = Delaunay(xy)
    except QhullError as exc:
        raise ValueError('TRAIN thẳng hàng; cần thiết kế candidate theo hành lang riêng.') from exc
    side = max(10, int(np.ceil(np.sqrt(count * 12))))
    gx = np.linspace(xy[:, 0].min(), xy[:, 0].max(), side)
    gy = np.linspace(xy[:, 1].min(), xy[:, 1].max(), side)
    xx, yy = np.meshgrid(gx, gy)
    candidates = np.c_[xx.ravel(), yy.ravel()]
    keep = hull.find_simplex(candidates) >= 0
    keep &= cKDTree(xy).query(candidates)[0] > tolerance
    if polygon is not None:
        from matplotlib.path import Path as PolygonPath
        vertices = np.asarray(polygon, dtype=float)
        if vertices.ndim != 2 or vertices.shape[1] != 2 or len(vertices) < 3:
            raise ValueError('Polygon cần ít nhất 3 đỉnh X/Y.')
        keep &= PolygonPath(vertices).contains_points(candidates)
    candidates = candidates[keep]
    if not len(candidates):
        raise ValueError('Không có candidate hợp lệ; xem lại polygon và tolerance.')
    rng = np.random.default_rng(seed)
    chosen = rng.choice(len(candidates), min(count, len(candidates)), replace=False)
    return pd.DataFrame(candidates[chosen], columns=['X', 'Y'])


def ext17_generate(G, norm, targets, ap_cols, seed, noise_dim, repeats, min_ap):
    """Dùng checkpoint có sẵn; trả cả candidate không đạt để kiểm toán."""
    import torch
    if repeats < 1 or len(targets) == 0:
        raise ValueError('Cần targets không rỗng và repeats >= 1.')
    device = next(G.parameters()).device
    repeated = targets.loc[targets.index.repeat(repeats)].reset_index(drop=True)
    scaled = 2 * (repeated[['X', 'Y']].to_numpy(float) - norm['xy_min']) / norm['xy_range'] - 1
    rng = torch.Generator(device=device).manual_seed(int(seed))
    was_training = G.training
    G.eval()
    rssis, probabilities, masks = [], [], []
    try:
        with torch.no_grad():
            for start in range(0, len(repeated), 128):
                coords = torch.as_tensor(scaled[start:start+128], dtype=torch.float32, device=device)
                noise = torch.randn(len(coords), noise_dim, generator=rng, device=device)
                rssi, probability = G(coords, noise)
                mask = torch.bernoulli(probability, generator=rng)
                rssis.append(rssi.cpu().numpy())
                probabilities.append(probability.cpu().numpy())
                masks.append(mask.cpu().numpy().astype(bool))
    finally:
        G.train(was_training)
    raw = (np.concatenate(rssis) + 1) / 2 * norm['rssi_range'] + norm['rssi_min']
    mask = np.concatenate(masks)
    # Không clip để che giá trị bất thường; ghi nhận trước khi loại candidate.
    observed = np.where(mask, raw, np.nan)
    physical = (~mask | (np.isfinite(raw) & (raw > -100) & (raw <= 0))).all(axis=1)
    result = repeated.copy()
    result[ap_cols] = observed
    result['N_Observed_AP'] = mask.sum(axis=1)
    result['Mean_Mask_Probability'] = np.concatenate(probabilities).mean(axis=1)
    result['Pass_Basic_QC'] = physical & (mask.sum(axis=1) >= min_ap)
    result['Source'] = 'CGAN_NEW_COORDINATE'
    result['Repeat'] = np.tile(np.arange(repeats), len(targets))
    return result


def ext17_metrics(frame):
    errors = frame['Error'].to_numpy(float)
    ok = np.isfinite(errors)
    e = errors[ok]
    return {
        'N_Query': len(frame), 'N_Valid': int(ok.sum()),
        'Coverage': float(ok.mean()) if len(frame) else np.nan,
        'Mean_Error': float(e.mean()) if len(e) else np.nan,
        'Median_Error': float(np.median(e)) if len(e) else np.nan,
        'RMSE': float(np.sqrt(np.mean(e**2))) if len(e) else np.nan,
        'P90': float(np.percentile(e, 90)) if len(e) else np.nan,
    }


def ext17_rp_groups(ids, train_ids, validation_ids):
    keys = pd.Series(ids).astype(str).str.strip()
    train_keys = set(pd.Series(list(train_ids), dtype='object').astype(str).str.strip())
    val_keys = set(pd.Series(list(validation_ids), dtype='object').astype(str).str.strip())
    if train_keys & val_keys:
        raise ValueError('TRAIN và VALIDATION trùng RP.')
    return np.where(keys.isin(train_keys), 'Seen_TRAIN',
                    np.where(keys.isin(val_keys), 'Validation_RP', 'Unseen_TRAIN_and_VALIDATION'))


## 17B. Sinh fingerprint tại tọa độ mới

Dùng checkpoint single-seed có sẵn, không huấn luyện lại hay ghi đè reference gốc. Candidate chỉ xây dựng từ tọa độ TRAIN. Nhiều mẫu/tọa độ được tổng hợp thành một reference; kiểm đếm số vị trí mới thay vì chỉ số dòng. Chọn k bằng ORIGINAL validation, dùng chung scaler/k cho các phương pháp.

Không điều chỉnh cấu hình sau khi xem TEST để chọn kết quả đẹp nhất. Cấu hình này là thí nghiệm đã định trước, chưa phải thông số tối ưu.


In [ ]:
# PHẦN BỔ SUNG 17B — RADIO MAP TẠI TỌA ĐỘ MỚI
# Chạy sau Cell 7 (notebook index 37) hoặc cuối notebook khi còn model trong RAM.
# Không cần chạy lại 15 mô hình multi-seed để dùng phần này.
_ext17_required = ['cgan_models', 'cgan_data', 'splits', 'FLOORS', 'AP_COLS',
                   'OUT', 'wknn', 'fit_scaler', 'K_CANDIDATES', 'CGAN_NOISE_DIM', 'MIN_VALID_AP']
_ext17_missing = [name for name in _ext17_required if name not in globals()]
if _ext17_missing:
    raise RuntimeError(f'Thiếu {_ext17_missing}; chạy phần gốc đến Cell 7 trước.')
if EXT17_CONFIG['noise_dim'] != CGAN_NOISE_DIM:
    raise ValueError('noise_dim phải khớp CGAN_NOISE_DIM.')

EXT17_DIR = Path(OUT) / '17_NEW_COORDINATES_EXTENSION'
EXT17_DIR.mkdir(parents=True, exist_ok=True)
EXT17_SYNTHETIC, EXT17_REFERENCES, EXT17_PREDICTIONS = {}, {}, {}
_ext17_density_rows, _ext17_metric_rows, _ext17_selection_rows = [], [], []
for _fi, _floor in enumerate(FLOORS):
    _tr = splits[_floor]['TRAIN'].copy()
    _va = splits[_floor]['VALIDATION'].copy()
    _te = splits[_floor]['TEST'].copy()
    _model_train = cgan_data[_floor]['TRAIN_FRAME']
    # Đối chiếu cả RSSI/tọa độ để tránh dùng nhầm checkpoint của split khác.
    _cols = ['ID_RP', 'X', 'Y', *AP_COLS]
    def _ext17_canonical(frame):
        result = frame[_cols].copy()
        result['ID_RP'] = result['ID_RP'].astype(str).str.strip()
        return result.sort_values('ID_RP').reset_index(drop=True)
    _clean_train = _tr.copy()
    _values = _clean_train[AP_COLS].apply(pd.to_numeric, errors='coerce').replace([-100, 100], np.nan)
    _clean_train[AP_COLS] = _values.mask((_values < -100) | (_values > 0))
    _clean_train = _clean_train[_clean_train[AP_COLS].notna().sum(axis=1) >= MIN_VALID_AP]
    pd.testing.assert_frame_equal(_ext17_canonical(_clean_train), _ext17_canonical(_model_train),
                                  check_dtype=False, check_exact=False)
    _count = min(EXT17_CONFIG['max_candidates'],
                 max(1, int(np.ceil(len(_tr) * EXT17_CONFIG['target_ratio']))))
    _targets = ext17_new_coordinates(_tr, _count, EXT17_CONFIG['seed'] + _fi,
                                     EXT17_CONFIG['novelty_tolerance'],
                                     EXT17_WALKABLE_POLYGONS.get(_floor))
    _targets['ID_RP'] = [f'EXT17_{_floor}_{i:05d}' for i in range(len(_targets))]
    _targets['Floor'] = _floor
    _t0 = time.perf_counter()
    _raw = ext17_generate(cgan_models[_floor]['G'], cgan_data[_floor]['NORMALIZATION'],
                          _targets, AP_COLS, EXT17_CONFIG['seed'] + 100 + _fi,
                          CGAN_NOISE_DIM, EXT17_CONFIG['samples_per_coordinate'],
                          EXT17_CONFIG['min_observed_ap'])
    _seconds = time.perf_counter() - _t0
    _accepted = _raw[_raw['Pass_Basic_QC']].copy()
    # Một reference/tọa độ, không tăng trọng số WKNN chỉ vì sinh nhiều lần tại cùng điểm.
    _agg = _accepted.groupby(['ID_RP', 'X', 'Y'], as_index=False)[AP_COLS].mean()
    _agg['Floor'] = _floor
    _agg['Source'] = 'CGAN_NEW_COORDINATE_MEAN'
    _agg['N_Observed_AP'] = _agg[AP_COLS].notna().sum(axis=1)
    _references = {'ORIGINAL': _tr,
                   'CGAN_NEW_COORDINATE': pd.concat([_tr, _agg], ignore_index=True)}
    EXT17_SYNTHETIC[_floor] = _raw
    EXT17_REFERENCES[_floor] = _references
    # Audit tọa độ so với cả tập nguồn TRAIN+VALIDATION, chỉ sau khi đã sinh.
    _real_xy = pd.concat([_tr, _va])[['X', 'Y']].drop_duplicates().to_numpy(float)
    _nearest_train = cKDTree(_tr[['X', 'Y']].to_numpy(float)).query(_agg[['X', 'Y']])[0]
    _nearest_source = cKDTree(_real_xy).query(_agg[['X', 'Y']])[0]
    _ext17_density_rows.append({
        'Floor': _floor, 'Original_Coordinates': len(_tr[['X', 'Y']].drop_duplicates()),
        'Candidate_Coordinates': len(_targets), 'Generated_Rows': len(_raw),
        'Accepted_Rows': len(_accepted), 'Accepted_Coordinates': len(_agg),
        'New_vs_TRAIN': int((_nearest_train > EXT17_CONFIG['novelty_tolerance']).sum()),
        'New_vs_TRAIN_and_VALIDATION': int((_nearest_source > EXT17_CONFIG['novelty_tolerance']).sum()),
        'Reference_Coordinates_After': len(_references['CGAN_NEW_COORDINATE'][['X', 'Y']].drop_duplicates()),
        'Generation_Seconds': _seconds,
        'Walkable_Polygon_Provided': _floor in EXT17_WALKABLE_POLYGONS,
        'Physical_Accuracy_Proven': False,
    })
    _scaler, _med = fit_scaler(_tr)
    _ktable = []
    for _k in K_CANDIDATES:
        _vp, _, _ = wknn(_tr, _va, _scaler, _med, _k)
        _errors = np.linalg.norm(_vp - _va[['X', 'Y']].to_numpy(float), axis=1)
        _km = ext17_metrics(pd.DataFrame({'Error': _errors}))
        _ktable.append({'k': _k, **_km})
    _ktable = pd.DataFrame(_ktable)
    _eligible = _ktable[_ktable['N_Valid'] > 0]
    if _eligible.empty:
        raise RuntimeError(f'{_floor}: không chọn k khi toàn bộ validation không hợp lệ.')
    _best_k = int(_eligible.sort_values(['Mean_Error', 'Coverage', 'k'], ascending=[True, False, True]).iloc[0]['k'])
    _ktable['Floor'], _ktable['Selected_k'] = _floor, _best_k
    _ext17_selection_rows.append(_ktable)
    for _method, _reference in _references.items():
        _p, _fb, _used = wknn(_reference, _te, _scaler, _med, _best_k)
        _out = _te[['ID_RP', 'X', 'Y']].copy().reset_index(drop=True)
        _out['Pred_X'], _out['Pred_Y'] = _p[:, 0], _p[:, 1]
        _out['Error'] = np.linalg.norm(_p - _te[['X', 'Y']].to_numpy(float), axis=1)
        _out['Fallback'], _out['Used_Reference_Count'] = _fb, _used
        _out['RP_Group'] = ext17_rp_groups(_te['ID_RP'], _tr['ID_RP'], _va['ID_RP'])
        _out['Floor'], _out['Method'] = _floor, _method
        EXT17_PREDICTIONS[(_floor, _method)] = _out
        for _group in ['ALL', 'Seen_TRAIN', 'Validation_RP', 'Unseen_TRAIN_and_VALIDATION']:
            _part = _out if _group == 'ALL' else _out[_out['RP_Group'] == _group]
            _ext17_metric_rows.append({'Floor': _floor, 'Method': _method,
                                       'RP_Group': _group, 'Selected_k': _best_k,
                                       **ext17_metrics(_part)})
        _out.to_csv(EXT17_DIR / f'predictions_{_floor}_{_method}.csv', index=False)
    _raw.to_csv(EXT17_DIR / f'all_generated_{_floor}.csv', index=False)
    _agg.to_csv(EXT17_DIR / f'new_coordinate_reference_{_floor}.csv', index=False)

EXT17_DENSITY = pd.DataFrame(_ext17_density_rows)
EXT17_METRICS = pd.DataFrame(_ext17_metric_rows)
EXT17_DENSITY.to_csv(EXT17_DIR / 'coordinate_density_audit.csv', index=False)
EXT17_METRICS.to_csv(EXT17_DIR / 'localization_by_rp_group.csv', index=False)
pd.concat(_ext17_selection_rows).to_csv(EXT17_DIR / 'validation_k_selection.csv', index=False)
with open(EXT17_DIR / 'extension_config.json', 'w', encoding='utf-8') as _fp:
    json.dump({'config': EXT17_CONFIG, 'walkable_polygons': EXT17_WALKABLE_POLYGONS,
               'candidate_source': 'TRAIN coordinates only',
               'coordinate_unit': 'source units; meters unverified',
               'scope': 'single existing checkpoint per floor; exploratory extension'},
              _fp, ensure_ascii=False, indent=2)
display(EXT17_DENSITY)
display(EXT17_METRICS)
print('Lưu tại:', EXT17_DIR)
print('QC cơ bản không chứng minh vật lý chính xác. N_Query=0 nghĩa là chưa có bằng chứng cho nhóm đó.')


## 17C. Đánh giá lại ba nhóm RP trên kết quả 5 seed

Cần chạy phần gốc tới Cell 16E. Không phụ thuộc 17B. Ba nhóm: Seen_TRAIN, Validation_RP, Unseen_TRAIN_and_VALIDATION. VALIDATION đã dùng chọn checkpoint/k nên không gọi là kiểm thử hoàn toàn mới. N_Query=0 thể hiện chưa có dữ liệu đánh giá nhóm đó.


In [ ]:
# PHẦN BỔ SUNG 17C — PHÂN NHÓM LẠI KẾT QUẢ 5 SEED ĐÃ CÓ
# Chỉ cần chạy 17A trước cell này; không phụ thuộc việc đã chạy 17B.
_ext17_needed = ['predictions_16e_df', 'MULTI_SEED_DATA', 'OUT']
_ext17_absent = [x for x in _ext17_needed if x not in globals()]
if _ext17_absent:
    raise RuntimeError(f'Thiếu {_ext17_absent}; chạy Cell 16E của notebook gốc trước.')
EXT17_GROUPED_MULTI = predictions_16e_df.copy()
EXT17_GROUPED_MULTI['RP_Group'] = ''
_ext17_rows = []
for (_floor, _seed), _part in EXT17_GROUPED_MULTI.groupby(['Floor', 'Seed']):
    _split = MULTI_SEED_DATA[_floor]['SEEDS'][int(_seed)]
    _labels = ext17_rp_groups(_part['ID_RP'], _split['TRAIN']['ID_RP'],
                              _split['VALIDATION']['ID_RP'])
    EXT17_GROUPED_MULTI.loc[_part.index, 'RP_Group'] = _labels
    for _method, _method_df in _part.groupby('Method'):
        _local_labels = ext17_rp_groups(_method_df['ID_RP'], _split['TRAIN']['ID_RP'],
                                        _split['VALIDATION']['ID_RP'])
        for _group in ['Seen_TRAIN', 'Validation_RP', 'Unseen_TRAIN_and_VALIDATION']:
            _g = _method_df.loc[_local_labels == _group].copy()
            if 'Valid' in _g:
                _g.loc[~_g['Valid'].astype(bool), 'Error'] = np.nan
            _ext17_rows.append({'Floor': _floor, 'Seed': _seed, 'Method': _method,
                                'RP_Group': _group, **ext17_metrics(_g)})
EXT17_MULTI_METRICS = pd.DataFrame(_ext17_rows)
EXT17_MULTI_SUMMARY = EXT17_MULTI_METRICS.groupby(
    ['Floor', 'Method', 'RP_Group'], as_index=False
).agg(N_Seeds=('Seed', 'nunique'), Mean_Error_Mean=('Mean_Error', 'mean'),
      Mean_Error_SD=('Mean_Error', 'std'), Coverage_Mean=('Coverage', 'mean'),
      Min_Queries_Per_Seed=('N_Query', 'min'), Max_Queries_Per_Seed=('N_Query', 'max'))
_ext17_dir = Path(OUT) / '17_NEW_COORDINATES_EXTENSION'
_ext17_dir.mkdir(parents=True, exist_ok=True)
EXT17_MULTI_METRICS.to_csv(_ext17_dir / 'existing_multiseed_group_metrics.csv', index=False)
EXT17_MULTI_SUMMARY.to_csv(_ext17_dir / 'existing_multiseed_group_summary.csv', index=False)
EXT17_GROUPED_MULTI.to_csv(_ext17_dir / 'existing_multiseed_predictions_grouped.csv', index=False)
display(EXT17_MULTI_SUMMARY)
print('Validation_RP đã được dùng chọn checkpoint/k; không gọi là TEST hoàn toàn chưa thấy.')
print('Nhóm Unseen có 0 query: cần thí nghiệm holdout độc lập mới để kết luận.')


## 17D/E. Đánh giá đầy đủ phần bổ sung và xuất báo cáo

- So sánh ORIGINAL, CGAN tọa độ mới, KNN K=3/5/7 và APMean R=3/5/7 trên cùng candidate ban đầu. Số điểm được chấp nhận sau QC được báo cáo để thấy khác biệt mật độ thực tế.
- Mean/median error, RMSE, P90, coverage, fallback theo tầng và nhóm RP.
- Mức giảm sai số ghép cặp và CI95% bootstrap cụm theo RP; dương là giảm sai số. CI chưa hiệu chỉnh cho nhiều so sánh, dùng thăm dò.
- Sinh 20 mẫu/RP VALIDATION để chẩn đoán MAE/RMSE/bias RSSI từng AP, Brier mask so với mask RP tổng hợp và khoảng cách Wasserstein biên. Đọc kèm số RP có AP so sánh được. Các chỉ số VALIDATION không phải kết quả cuối độc lập.
- Biểu đồ CDF sai số, mean error, bản đồ điểm mới; Excel, CSV, báo cáo kết luận tự động và ZIP.

Thí nghiệm này sử dụng mô hình single-seed gốc. Chưa thay thế benchmark 5 seed và thí nghiệm ngân sách khảo sát. TEST cuối theo tọa độ độc lập, nhiều tỷ lệ RP và chi phí thực địa vẫn cần nghiên cứu tiếp. Không tự khẳng định CGAN tốt hơn khi kết quả không hỗ trợ.

Lưu ý: cột Mean_Common_APs cũ trong Cell 16E thực chất lấy từ số reference neighbors. Phần mới giữ nguyên cell cũ và dùng Used_Reference_Count đúng nghĩa.


In [ ]:
# 17D — ĐÁNH GIÁ BỔ SUNG: CÙNG TỌA ĐỘ, RSSI THEO AP, SAI SỐ GHÉP CẶP
# Chạy sau 17A và 17B. Giữ nguyên các kết quả/biến gốc.
import matplotlib.pyplot as plt
from scipy.stats import wasserstein_distance
import zipfile


def ext17_interpolate(train, targets, ap_cols, k=None, radius=None):
    values = train[ap_cols].to_numpy(float)
    xy = train[['X', 'Y']].to_numpy(float)
    output = targets.copy().reset_index(drop=True)
    result = np.full((len(output), len(ap_cols)), np.nan)
    for i, point in enumerate(output[['X', 'Y']].to_numpy(float)):
        distances = np.linalg.norm(xy - point, axis=1)
        if k is not None:
            ids = np.argsort(distances)[:min(k, len(xy))]
            weights = 1 / np.maximum(distances[ids], 1e-8)
        else:
            ids = np.flatnonzero(distances <= radius)
            weights = np.ones(len(ids))
        if not len(ids):
            continue
        observed = np.isfinite(values[ids])
        denominator = (observed * weights[:, None]).sum(axis=0)
        numerator = (np.where(observed, values[ids], 0) * weights[:, None]).sum(axis=0)
        result[i] = np.divide(numerator, denominator, out=np.full(len(ap_cols), np.nan), where=denominator > 0)
    output[ap_cols] = result
    return output


def ext17_cluster_ci(paired, seed, n_bootstrap=1000):
    # Delta = ORIGINAL - phương pháp, dương là phương pháp giảm sai số.
    # Resample theo RP, giữ các query lặp cùng RP trong một cụm.
    clusters = paired.groupby('ID_RP')['Delta'].agg(['sum', 'count'])
    if len(clusters) < 2:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    ix = rng.integers(0, len(clusters), size=(n_bootstrap, len(clusters)))
    draws = clusters['sum'].to_numpy()[ix].sum(axis=1) / clusters['count'].to_numpy()[ix].sum(axis=1)
    return tuple(np.percentile(draws, [2.5, 97.5]))

EXT17_ALL_PREDICTIONS, _ext17_eval_rows, _ext17_rssi_rows = [], [], []
_ext17_plot_errors = {}
for _fi, _floor in enumerate(FLOORS):
    _tr = splits[_floor]['TRAIN'].copy()
    _va = splits[_floor]['VALIDATION'].copy()
    _te = splits[_floor]['TEST'].copy().reset_index(drop=True)
    _targets = EXT17_SYNTHETIC[_floor][['ID_RP', 'X', 'Y', 'Floor']].drop_duplicates('ID_RP')
    _refs = dict(EXT17_REFERENCES[_floor])
    _method_specs = [(f'KNN_K{k}_NEW', k, None) for k in [3,5,7]] + [(f'APMEAN_R{r}_NEW', None, r) for r in [3,5,7]]
    for _method, _k, _radius in _method_specs:
        _synth = ext17_interpolate(_tr, _targets, AP_COLS, k=_k, radius=_radius)
        _synth['Source'] = _method
        _synth = _synth[_synth[AP_COLS].notna().sum(axis=1) >= EXT17_CONFIG['min_observed_ap']]
        _synth.to_csv(EXT17_DIR / f'{_floor}_{_method}_reference_only.csv', index=False)
        _refs[_method] = pd.concat([_tr, _synth], ignore_index=True)
    _scaler, _med = fit_scaler(_tr)
    _best_k = int(EXT17_METRICS.query('Floor == @_floor').iloc[0]['Selected_k'])
    _baseline = None
    for _method, _ref in _refs.items():
        _pred, _fb, _used = wknn(_ref, _te, _scaler, _med, _best_k)
        _d = _te[['ID_RP', 'X', 'Y']].copy()
        _d['Query_ID'] = np.arange(len(_d))
        _d['Floor'], _d['Method'] = _floor, _method
        _d['Pred_X'], _d['Pred_Y'] = _pred[:,0], _pred[:,1]
        _d['Error'] = np.linalg.norm(_pred - _te[['X','Y']].to_numpy(float), axis=1)
        _d['Fallback'], _d['Used_Reference_Count'] = _fb, _used
        _d['RP_Group'] = ext17_rp_groups(_te.ID_RP, _tr.ID_RP, _va.ID_RP)
        EXT17_ALL_PREDICTIONS.append(_d)
        if _method == 'ORIGINAL':
            _baseline = _d.copy()
        for _group in ['ALL','Seen_TRAIN','Validation_RP','Unseen_TRAIN_and_VALIDATION']:
            _sub = _d if _group == 'ALL' else _d[_d.RP_Group == _group]
            _base = _baseline[['Query_ID','Error']].rename(columns={'Error':'Original_Error'})
            _paired = _sub.merge(_base, on='Query_ID', validate='one_to_one')
            _paired = _paired[np.isfinite(_paired.Error) & np.isfinite(_paired.Original_Error)].copy()
            _paired['Delta'] = _paired.Original_Error - _paired.Error
            _lo, _hi = ext17_cluster_ci(_paired, EXT17_CONFIG['seed']+_fi)
            _ext17_eval_rows.append({
                'Floor':_floor, 'Method':_method, 'RP_Group':_group,
                'Selected_k':_best_k, 'Reference_Rows':len(_ref),
                'Reference_Coordinates':len(_ref[['X','Y']].drop_duplicates()),
                'Fallback_Count':int(_sub.Fallback.sum()), **ext17_metrics(_sub),
                'N_Paired_Query':len(_paired), 'N_Paired_RP':_paired.ID_RP.nunique(),
                'Paired_Improvement':_paired.Delta.mean(),
                'Paired_Improvement_CI95_Low':_lo, 'Paired_Improvement_CI95_High':_hi,
            })
        _ext17_plot_errors[(_floor,_method)] = _d.Error.dropna().to_numpy()
    # Validation dùng chẩn đoán RSSI, không tuyên bố là đánh giá cuối độc lập.
    _val_targets = _va[['ID_RP','X','Y']].reset_index(drop=True)
    _val_fake = ext17_generate(cgan_models[_floor]['G'], cgan_data[_floor]['NORMALIZATION'],
                               _val_targets, AP_COLS, EXT17_CONFIG['seed']+500+_fi,
                               CGAN_NOISE_DIM, 20, EXT17_CONFIG['min_observed_ap'])
    _val_fake.to_csv(EXT17_DIR / f'validation_generated_{_floor}.csv', index=False)
    _val_methods = {'CGAN':_val_fake}
    for _method,_k,_radius in _method_specs:
        _val_methods[_method] = ext17_interpolate(_tr,_val_targets,AP_COLS,k=_k,radius=_radius)
    _true = _va.set_index('ID_RP')[AP_COLS]
    for _method,_fake in _val_methods.items():
        _mean = _fake.groupby('ID_RP')[AP_COLS].mean().reindex(_true.index)
        _prob = _fake.assign(**{ap:_fake[ap].notna().astype(float) for ap in AP_COLS}).groupby('ID_RP')[AP_COLS].mean().reindex(_true.index)
        for _ap in AP_COLS:
            _ok = _true[_ap].notna() & _mean[_ap].notna()
            _diff = _mean.loc[_ok,_ap] - _true.loc[_ok,_ap]
            _real_values = _true[_ap].dropna().to_numpy(float)
            _fake_values = _fake[_ap].dropna().to_numpy(float)
            _ext17_rssi_rows.append({
                'Floor':_floor,'Method':_method,'AP':_ap,'Role':'VALIDATION_DIAGNOSTIC',
                'N_Comparable_RP':int(_ok.sum()),'N_Validation_RP':len(_true),
                'RSSI_MAE_dBm':_diff.abs().mean(),
                'RSSI_RMSE_dBm':np.sqrt((_diff**2).mean()),
                'RSSI_Bias_dBm':_diff.mean(),
                'Mask_Brier_vs_RP_Aggregate':((_prob[_ap] - _true[_ap].notna().astype(float))**2).mean(),
                'Marginal_Wasserstein_dBm':wasserstein_distance(_real_values,_fake_values) if len(_real_values) and len(_fake_values) else np.nan,
            })
EXT17_EVALUATION = pd.DataFrame(_ext17_eval_rows)
EXT17_RSSI_EVALUATION = pd.DataFrame(_ext17_rssi_rows)
EXT17_EVALUATION.to_csv(EXT17_DIR / 'fair_new_coordinate_evaluation.csv',index=False)
EXT17_RSSI_EVALUATION.to_csv(EXT17_DIR / 'validation_rssi_per_ap.csv',index=False)
pd.concat(EXT17_ALL_PREDICTIONS,ignore_index=True).to_csv(EXT17_DIR / 'fair_new_coordinate_predictions.csv',index=False)
with pd.ExcelWriter(EXT17_DIR / 'DANH_GIA_BO_SUNG.xlsx') as _writer:
    EXT17_DENSITY.to_excel(_writer,sheet_name='Coordinate_Density',index=False)
    EXT17_EVALUATION.to_excel(_writer,sheet_name='Localization',index=False)
    EXT17_RSSI_EVALUATION.to_excel(_writer,sheet_name='RSSI_Validation',index=False)
_fig,_axes = plt.subplots(1,len(FLOORS),figsize=(6*len(FLOORS),5),squeeze=False)
for _ax,_floor in zip(_axes.ravel(),FLOORS):
    _sub=EXT17_EVALUATION[(EXT17_EVALUATION.Floor==_floor)&(EXT17_EVALUATION.RP_Group=='ALL')]
    _ax.bar(_sub.Method,_sub.Mean_Error)
    _ax.tick_params(axis='x',rotation=70)
    _ax.set(title=f'{_floor}: mean localization error',ylabel='Source coordinate units')
_fig.tight_layout();_fig.savefig(EXT17_DIR/'mean_error_all_methods.png',dpi=160);plt.show();plt.close(_fig)
_fig,_axes=plt.subplots(1,len(FLOORS),figsize=(6*len(FLOORS),4),squeeze=False)
for _ax,_floor in zip(_axes.ravel(),FLOORS):
    for _method in ['ORIGINAL','CGAN_NEW_COORDINATE','KNN_K7_NEW']:
        _e=np.sort(_ext17_plot_errors[(_floor,_method)])
        if len(_e):_ax.plot(_e,np.arange(1,len(_e)+1)/len(_e),label=_method)
    _ax.set(title=f'{_floor}: CDF, valid predictions',xlabel='Source coordinate units',ylabel='CDF')
    _ax.legend(fontsize=7);_ax.grid(alpha=.2)
_fig.tight_layout();_fig.savefig(EXT17_DIR/'error_cdf.png',dpi=160);plt.show();plt.close(_fig)
display(EXT17_EVALUATION)
display(EXT17_RSSI_EVALUATION)
print('CI là bootstrap cụm theo RP; delta dương = giảm sai số trên các query hợp lệ ghép cặp.')
print('RSSI đánh giá trên VALIDATION đã dùng chọn mô hình; không phải kiểm thử cuối độc lập.')
print('Brier dùng mask của RP tổng hợp, không phải xác suất quan sát đo lặp thực tế.')
print('Chưa chứng minh giảm chi phí thực địa hoặc độ chính xác vật lý tại vị trí hoàn toàn mới.')
# 17E — BIỂU ĐỒ RADIO MAP, BÁO CÁO TỰ ĐỘNG VÀ XUẤT FILE
# Chạy sau 17D. Kết luận được viết từ kết quả hiện tại, không giả định CGAN tốt hơn.
import shutil
import platform

_fig,_axes=plt.subplots(1,len(FLOORS),figsize=(6*len(FLOORS),5),squeeze=False)
for _ax,_floor in zip(_axes.ravel(),FLOORS):
    _tr=splits[_floor]['TRAIN']
    _new=EXT17_REFERENCES[_floor]['CGAN_NEW_COORDINATE'].iloc[len(_tr):]
    _ax.scatter(_tr.X,_tr.Y,s=10,label='TRAIN measured',alpha=.7)
    _ax.scatter(_new.X,_new.Y,s=14,marker='x',label='New synthetic coordinates',alpha=.7)
    _ax.set(title=_floor,xlabel='X (source units)',ylabel='Y (source units)')
    _ax.set_aspect('equal',adjustable='datalim');_ax.legend(fontsize=8);_ax.grid(alpha=.2)
_fig.tight_layout();_fig.savefig(EXT17_DIR/'new_coordinate_radio_map.png',dpi=160);plt.show();plt.close(_fig)
_ext17_lines=['# KẾT QUẢ PHẦN BỔ SUNG','',
 'Các chỉ số vị trí dùng đơn vị tọa độ nguồn, chưa xác nhận là mét.',
 'Đây là thí nghiệm single-seed bổ sung; không thay thế thí nghiệm 5 seed gốc.',
 'Candidate tạo chỉ từ TRAIN; chưa có polygon đi lại được nếu Walkable_Polygon_Provided=False.',
 'TEST chứa cả RP TRAIN và VALIDATION; đọc riêng các nhóm trước khi kết luận tổng quát hóa.','']
for _floor in FLOORS:
    _density=EXT17_DENSITY[EXT17_DENSITY.Floor==_floor].iloc[0]
    _sub=EXT17_EVALUATION[(EXT17_EVALUATION.Floor==_floor)&(EXT17_EVALUATION.RP_Group=='ALL')]
    _ext17_lines += [f'## {_floor}',
      f'- Tọa độ reference: {int(_density.Original_Coordinates)} -> {int(_density.Reference_Coordinates_After)}.',
      '| Phương pháp | Mean error | Coverage | Delta ghép cặp | CI95% delta |',
      '|---|---:|---:|---:|---:|']
    for _,_r in _sub.iterrows():
        _ext17_lines.append(f'| {_r.Method} | {_r.Mean_Error:.4f} | {_r.Coverage:.2%} | {_r.Paired_Improvement:.4f} | [{_r.Paired_Improvement_CI95_Low:.4f}, {_r.Paired_Improvement_CI95_High:.4f}] |')
    _cg=_sub[_sub.Method=='CGAN_NEW_COORDINATE'].iloc[0]
    _interpret=('CI nằm trên 0: có bằng chứng giảm sai số trong giao thức này.' if _cg.Paired_Improvement_CI95_Low>0
                 else 'CI nằm dưới 0: sai số tăng trong giao thức này.' if _cg.Paired_Improvement_CI95_High<0
                 else 'CI chứa 0: chưa đủ bằng chứng về thay đổi sai số.')
    _ext17_lines += ['',f'CGAN: {_interpret}','']
_ext17_lines += ['## Cách đọc',
 '- Delta = ORIGINAL - phương pháp, chỉ trên query hợp lệ ở cả hai; dương là giảm sai số.',
 '- CI95% dùng bootstrap theo cụm RP, 1000 lần; chưa hiệu chỉnh so sánh nhiều phương pháp.',
 '- Bảng RSSI là chẩn đoán VALIDATION đã dùng chọn mô hình; chưa phải kiểm thử cuối độc lập.',
 '- MAE/RMSE RSSI dựa trên các RP có AP quan sát ở cả hai; đọc kèm N_Comparable_RP.',
 '- Các phương pháp dùng cùng candidate ban đầu; số reference được giữ có thể khác do QC.',
 '- Không suy ra chi phí/giờ đo thực tế từ số điểm ảo.',
 '- Kiểm chứng vật lý tại điểm chưa khảo sát cần phép đo thật giữ riêng và bản đồ/vị trí AP nếu có.']
(EXT17_DIR/'BAO_CAO_KET_QUA.md').write_text('\n'.join(_ext17_lines),encoding='utf-8')
import torch, scipy, sklearn
(EXT17_DIR/'runtime_versions.json').write_text(json.dumps({
    'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,
    'torch':torch.__version__,'scipy':scipy.__version__,'sklearn':sklearn.__version__},indent=2))
_ext17_archive=shutil.make_archive(str(Path(OUT).parent/'KLTN_KET_QUA_BO_SUNG'),'zip',EXT17_DIR)
print('Bảng Excel:',EXT17_DIR/'DANH_GIA_BO_SUNG.xlsx')
print('Báo cáo:',EXT17_DIR/'BAO_CAO_KET_QUA.md')
print('Toàn bộ kết quả bổ sung:',_ext17_archive)
# Colab tự tải ZIP về máy; khi chạy local, file nằm tại đường dẫn phía trên.
try:
    from google.colab import files as _ext17_files
except ImportError:
    from IPython.display import FileLink
    display(FileLink(_ext17_archive))
else:
    _ext17_files.download(_ext17_archive)


# 18. TRƯỚC ẨN → SAU ẨN → SINH BÙ ĐÚNG RP ẨN → ĐÁNH GIÁ

Bản hiện tại: **92 cell**. Giữ nguyên 86 cell đã có (gồm 77 cell gốc), chỉ nối thêm phần 18. Không xóa ORIGINAL, CGAN, KNN hoặc các bảng/biểu đồ theo tầng. Thư mục kết quả mới: `OUT/18_HIDE_AND_RECOVER`; kết quả cũ giữ ở thư mục cũ.

**Chạy nhanh phần 18:** chạy phần dữ liệu và hàm WKNN gốc tới notebook index 12, rồi 18A → 18B → 18C. Không cần chạy lại phần 17 hoặc multi-seed cũ để dùng phần 18. Chạy Run all vẫn giữ toàn bộ thực nghiệm cũ và mới. Bản GitHub chỉ chứa code, không đăng output/dataset; chạy lại để tạo kết quả.

**Khác với phần 17:** phần 18 sinh bù tại chính tọa độ bị ẩn; phần 17 sinh trên lưới tọa độ mới. Đây là hai thí nghiệm riêng, không thay kết quả cũ.

## 18A. Thiết kế chống rò rỉ và CGAN riêng cho split mới

Chia nguyên cụm tọa độ thành OBSERVED (~65%), VALIDATION (~15%), HIDDEN (~20%), seed 2026–2030. Tỷ lệ theo cụm vị trí, số RP có thể khác khi trùng tọa độ. ID khác nhau cùng/gần tọa độ được gom cùng nhóm. Cả RSSI HIDDEN và TEST không dùng fit scaler, train GAN, chọn checkpoint hoặc chọn k.

Tọa độ mục tiêu HIDDEN được biết từ kế hoạch khảo sát; RSSI thật bị che. Phải train CGAN mới chỉ từ OBSERVED, vì checkpoint cũ có thể đã thấy các RP được chọn ẩn trong thí nghiệm mới. Mô hình mask-aware/LayerNorm ở đây chỉ phục vụ split mới, không sửa kiến trúc hoặc checkpoint cũ. Huấn luyện trên fingerprint tổng hợp theo RP; chưa tuyên bố học đầy đủ biến thiên đo lặp.

Cấu hình khóa trước TEST; không đổi seed/tỷ lệ/tham số dựa trên kết quả HIDDEN sau khi xem.


In [ ]:
# 18A — GIAO THỨC ĐỘC LẬP: ẨN NGUYÊN NHÓM TỌA ĐỘ, KHÔNG GHI ĐÈ KẾT QUẢ CŨ
from pathlib import Path
import copy, json, time, hashlib, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn
import torch.nn.functional as F

EXT18_CONFIG = dict(seeds=[2026,2027,2028,2029,2030], hidden_ratio=.20,
    validation_ratio=.15, coordinate_tolerance=1e-5, epochs=300, patience=35,
    batch_size=32, noise_dim=32, hidden_dim=128, learning_rate=.0002,
    reconstruction_weight=10., mask_weight=1., samples_per_hidden_rp=20,
    min_observed_ap=3, bootstrap_iterations=1000, k_candidates=[3,5,7])


def ext18_metrics(errors):
    errors=np.asarray(errors,float);e=errors[np.isfinite(errors)]
    return dict(N_Query=len(errors),N_Valid=len(e),Coverage=len(e)/len(errors) if len(errors) else np.nan,
        Mean_Error=e.mean() if len(e) else np.nan,Median_Error=np.median(e) if len(e) else np.nan,
        RMSE=np.sqrt(np.mean(e**2)) if len(e) else np.nan,P90=np.percentile(e,90) if len(e) else np.nan)


def ext18_coordinate_groups(xy,tolerance):
    # Gộp cả ID khác nhau có cùng/gần tọa độ vào một cụm, tránh rò rỉ vị trí.
    xy=np.asarray(xy,float)
    if not np.isfinite(xy).all():raise ValueError('Tọa độ không hợp lệ.')
    parent=np.arange(len(xy))
    def root(i):
        while parent[i]!=i:
            parent[i]=parent[parent[i]];i=parent[i]
        return i
    for a,b in cKDTree(xy).query_pairs(tolerance):
        ra,rb=root(a),root(b)
        if ra!=rb:parent[rb]=ra
    return np.array([root(i) for i in range(len(xy))])


def ext18_split(source,seed,config):
    source=source.copy().sort_values('ID_RP').reset_index(drop=True)
    if source.ID_RP.duplicated().any():raise ValueError('Bảng nguồn phải có một dòng/RP.')
    source['Coordinate_Group']=ext18_coordinate_groups(source[['X','Y']],config['coordinate_tolerance'])
    groups=np.unique(source.Coordinate_Group);rng=np.random.default_rng(seed)
    groups=rng.permutation(groups)
    nv=max(1,int(round(len(groups)*config['validation_ratio'])))
    nh=max(1,int(round(len(groups)*config['hidden_ratio'])))
    if len(groups)-nv-nh<3:raise ValueError('Không đủ vị trí để chia ba tập.')
    roles={'VALIDATION':groups[:nv],'HIDDEN':groups[nv:nv+nh],'OBSERVED':groups[nv+nh:]}
    result={name:source[source.Coordinate_Group.isin(ids)].copy().reset_index(drop=True)
            for name,ids in roles.items()}
    for a,b in [('OBSERVED','HIDDEN'),('OBSERVED','VALIDATION'),('HIDDEN','VALIDATION')]:
        if set(result[a].Coordinate_Group)&set(result[b].Coordinate_Group):raise AssertionError('Leakage')
        if cKDTree(result[a][['X','Y']]).query(result[b][['X','Y']])[0].min()<=config['coordinate_tolerance']:
            raise AssertionError('Có vị trí quá gần ở hai tập.')
    return result


def ext18_fit_scaler(observed,ap_cols):
    x=observed[ap_cols].to_numpy(float)
    med=np.array([np.median(col[np.isfinite(col)]) if np.isfinite(col).any() else 0. for col in x.T])
    return StandardScaler().fit(np.where(np.isfinite(x),x,med)),med


def ext18_interpolate(observed,targets,ap_cols,k=7):
    out=targets.copy().reset_index(drop=True);x=observed[ap_cols].to_numpy(float)
    values=np.full((len(out),len(ap_cols)),np.nan)
    for i,point in enumerate(out[['X','Y']].to_numpy(float)):
        distance=np.linalg.norm(observed[['X','Y']].to_numpy(float)-point,axis=1)
        ids=np.argsort(distance)[:min(k,len(observed))];weights=1/np.maximum(distance[ids],1e-8)
        mask=np.isfinite(x[ids]);den=(weights[:,None]*mask).sum(0)
        values[i]=np.divide((np.where(mask,x[ids],0)*weights[:,None]).sum(0),den,
                             out=np.full(len(ap_cols),np.nan),where=den>0)
    out[ap_cols]=values
    return out


class Ext18Generator(nn.Module):
    def __init__(self,n_ap,noise_dim,hidden):
        super().__init__()
        self.body=nn.Sequential(nn.Linear(2+noise_dim,hidden),nn.LayerNorm(hidden),nn.LeakyReLU(.2),
                                nn.Linear(hidden,hidden),nn.LayerNorm(hidden),nn.LeakyReLU(.2))
        self.rssi=nn.Linear(hidden,n_ap);self.mask=nn.Linear(hidden,n_ap)
    def forward(self,xy,z):
        h=self.body(torch.cat([xy,z],1))
        return torch.tanh(self.rssi(h)),torch.sigmoid(self.mask(h))


class Ext18Discriminator(nn.Module):
    def __init__(self,n_ap,hidden):
        super().__init__()
        self.body=nn.Sequential(nn.Linear(2+2*n_ap,hidden),nn.LeakyReLU(.2),
                                nn.Linear(hidden,hidden),nn.LeakyReLU(.2),nn.Linear(hidden,1))
    def forward(self,xy,rssi,mask):return self.body(torch.cat([xy,rssi*mask,mask],1))


def ext18_train(observed,validation,ap_cols,seed,config):
    # Chỉ nhận OBSERVED và VALIDATION; hàm này không có đối số HIDDEN/TEST.
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    x=observed[ap_cols].to_numpy(float);active=np.isfinite(x).any(0)
    low=np.array([col[np.isfinite(col)].min() if np.isfinite(col).any() else -100. for col in x.T])
    high=np.array([col[np.isfinite(col)].max() if np.isfinite(col).any() else 0. for col in x.T])
    span=np.maximum(high-low,1.)
    xymin=observed[['X','Y']].min().to_numpy(float)
    xyspan=np.maximum(observed[['X','Y']].max().to_numpy(float)-xymin,1.)
    norm=dict(rssi_min=low,rssi_range=span,xy_min=xymin,xy_range=xyspan,active=active)
    def tensors(frame):
        raw=frame[ap_cols].to_numpy(float);mask=np.isfinite(raw)
        scaled=np.where(mask,2*(raw-low)/span-1,0)
        coords=2*(frame[['X','Y']].to_numpy(float)-xymin)/xyspan-1
        return [torch.as_tensor(a,dtype=torch.float32,device=device) for a in [coords,scaled,mask]]
    trxy,trr,trm=tensors(observed);vxy,vr,vm=tensors(validation)
    history=[];best=np.inf;best_state=None;wait=0;start=time.perf_counter()
    devices=[device.index or 0] if device.type=='cuda' else []
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(seed)
        g=Ext18Generator(len(ap_cols),config['noise_dim'],config['hidden_dim']).to(device)
        d=Ext18Discriminator(len(ap_cols),config['hidden_dim']).to(device)
        go=torch.optim.Adam(g.parameters(),lr=config['learning_rate'],betas=(.5,.999))
        do=torch.optim.Adam(d.parameters(),lr=config['learning_rate'],betas=(.5,.999))
        vz=torch.randn(len(vxy),config['noise_dim'],device=device)
        for epoch in range(1,config['epochs']+1):
            g.train();d.train();losses=[]
            order=torch.randperm(len(trxy),device=device)
            for start_id in range(0,len(order),config['batch_size']):
                ids=order[start_id:start_id+config['batch_size']];c,r,m=trxy[ids],trr[ids],trm[ids]
                z=torch.randn(len(ids),config['noise_dim'],device=device)
                fr,mp=g(c,z);hard=(torch.rand_like(mp)<mp).float()
                do.zero_grad();real=d(c,r,m);fake=d(c,fr.detach(),hard.detach())
                dl=F.binary_cross_entropy_with_logits(real,torch.ones_like(real))+F.binary_cross_entropy_with_logits(fake,torch.zeros_like(fake))
                dl.backward();do.step()
                go.zero_grad()
                for p in d.parameters():p.requires_grad_(False)
                fr,mp=g(c,torch.randn_like(z));hard=(torch.rand_like(mp)<mp).float();straight=hard+mp-mp.detach()
                score=d(c,fr,straight)
                recon=((fr-r).abs()*m).sum()/m.sum().clamp(min=1)
                ml=F.binary_cross_entropy(mp,m)
                gl=F.binary_cross_entropy_with_logits(score,torch.ones_like(score))+config['reconstruction_weight']*recon+config['mask_weight']*ml
                gl.backward();go.step()
                for p in d.parameters():p.requires_grad_(True)
                losses.append((float(dl.detach()),float(gl.detach()),float(recon.detach()),float(ml.detach())))
            g.eval()
            with torch.no_grad():
                fr,mp=g(vxy,vz)
                val_rssi=((fr-vr).abs()*vm).sum()/vm.sum().clamp(min=1)
                val_mask=F.binary_cross_entropy(mp,vm)
                val=float((config['reconstruction_weight']*val_rssi+config['mask_weight']*val_mask).cpu())
            means=np.mean(losses,axis=0)
            history.append(dict(Epoch=epoch,D_Loss=means[0],G_Loss=means[1],Train_Reconstruction=means[2],Train_Mask_BCE=means[3],Validation_Score=val))
            if not np.isfinite(val):raise RuntimeError('Validation score không hữu hạn.')
            if val<best:
                best=val;best_state=copy.deepcopy(g.state_dict());best_epoch=epoch;wait=0
            else:wait+=1
            if wait>=config['patience']:break
        g.load_state_dict(best_state);g.eval()
    return g,norm,pd.DataFrame(history),dict(Best_Epoch=best_epoch,Epochs_Run=len(history),
        Best_Validation_Score=best,Training_Seconds=time.perf_counter()-start,Active_AP_Count=int(active.sum()))


def ext18_generate(g,norm,targets,ap_cols,seed,config):
    # targets chỉ gồm ID/tọa độ kế hoạch, không được chứa RSSI HIDDEN.
    if set(ap_cols)&set(targets.columns):raise ValueError('targets không được có RSSI thật.')
    device=next(g.parameters()).device;count=config['samples_per_hidden_rp']
    repeated=targets.loc[targets.index.repeat(count)].reset_index(drop=True)
    xy=2*(repeated[['X','Y']].to_numpy(float)-norm['xy_min'])/norm['xy_range']-1
    generator=torch.Generator(device=device).manual_seed(seed+100000)
    rssis=[];masks=[];probs=[]
    with torch.no_grad():
        for start in range(0,len(repeated),128):
            c=torch.as_tensor(xy[start:start+128],dtype=torch.float32,device=device)
            z=torch.randn(len(c),config['noise_dim'],device=device,generator=generator)
            r,p=g(c,z);m=torch.bernoulli(p,generator=generator)
            rssis.append(r.cpu().numpy());masks.append(m.cpu().numpy().astype(bool));probs.append(p.cpu().numpy())
    raw=(np.concatenate(rssis)+1)/2*norm['rssi_range']+norm['rssi_min']
    mask=np.concatenate(masks)&norm['active'];raw=np.where(mask,raw,np.nan)
    output=repeated.copy();output[ap_cols]=raw
    output['Mask_Probability_Mean']=np.concatenate(probs).mean(1)
    output['Pass_Basic_QC']=(np.isfinite(raw).sum(1)>=config['min_observed_ap'])&(~np.isfinite(raw)|((raw>-100)&(raw<=0))).all(1)
    good=output[output.Pass_Basic_QC]
    reference=good.groupby(['ID_RP','X','Y'],as_index=False)[ap_cols].mean()
    return output,reference


def ext18_paired(original,candidate,seed,n_boot):
    keys=['Floor','Seed','Query_ID','ID_RP','X','Y']
    paired=original[keys+['Error']].merge(candidate[keys+['Error']],on=keys,suffixes=('_Original','_Candidate'),validate='one_to_one')
    paired=paired[np.isfinite(paired.Error_Original)&np.isfinite(paired.Error_Candidate)].copy()
    paired['Delta']=paired.Error_Original-paired.Error_Candidate
    clusters=paired.groupby('ID_RP').Delta.agg(['sum','count'])
    low=high=np.nan
    if len(clusters)>=2:
        rng=np.random.default_rng(seed);ix=rng.integers(0,len(clusters),size=(n_boot,len(clusters)))
        draws=clusters['sum'].to_numpy()[ix].sum(1)/clusters['count'].to_numpy()[ix].sum(1)
        low,high=np.percentile(draws,[2.5,97.5])
    return dict(N_Paired_Query=len(paired),N_Paired_RP=len(clusters),
        Paired_Improvement=paired.Delta.mean(),CI95_Low=low,CI95_High=high)


## 18B. Bốn mốc trên cùng truy vấn TEST tại RP ẩn

1. **BEFORE_HIDE_MEASURED:** reference có OBSERVED + RSSI đo thật tại HIDDEN; không gồm VALIDATION. Đây là mốc ưu thế thông tin trước khi che, không phải một mô hình dự đoán công bằng.
2. **AFTER_HIDE_ORIGINAL:** chỉ còn reference OBSERVED.
3. **AFTER_FILL_CGAN:** OBSERVED + fingerprint CGAN sinh đúng tọa độ HIDDEN.
4. **AFTER_FILL_KNN_K7:** OBSERVED + fingerprint nội suy KNN tại cùng HIDDEN.

Cùng scaler fit từ OBSERVED và k chọn bằng OBSERVED → VALIDATION cho cả bốn mốc. Query TEST là phép đo độc lập tại RP ẩn; được giữ y hệt trong từng floor/seed, có SHA256 audit. HIDDEN RSSI nguồn chỉ mở sau chọn checkpoint/k để so chất lượng sinh; trường hợp trước ẩn được dùng thông tin này có chủ đích và được gắn nhãn rõ.

Báo cáo Mean/Median Error, RMSE, P90, coverage, fallback, mật độ reference; CI bootstrap theo cụm RP riêng từng seed; MAE/RMSE/bias RSSI theo AP và diversity. Không gộp query qua các seed như các mẫu độc lập. Nhóm không có query hoặc không chọn được k sẽ báo lỗi rõ, không tạo kết quả giả.


In [ ]:
# 18B — TRƯỚC ẨN / SAU ẨN / SINH BÙ, CÙNG QUERY TEST TẠI RP ẨN
_ext18_required=['data','FLOORS','AP_COLS','OUT','wknn']
if any(x not in globals() for x in _ext18_required):
    raise RuntimeError('Chạy phần dữ liệu gốc và hàm WKNN (tới index 12) rồi 18A trước.')
EXT18_DIR=Path(OUT)/'18_HIDE_AND_RECOVER';EXT18_DIR.mkdir(parents=True,exist_ok=True)
# Khóa cấu hình trước khi mở kết quả TEST. Không chọn seed/tỷ lệ/k theo kết quả đẹp.
(EXT18_DIR/'config.json').write_text(json.dumps(EXT18_CONFIG,indent=2))
_ext18_result_rows=[];_ext18_rssi_rows=[];_ext18_audit_rows=[];_ext18_pair_rows=[]
_ext18_training_rows=[];_ext18_predictions=[];_ext18_k_rows=[]
EXT18_SPLITS={};EXT18_REFERENCES={}
for _floor18 in FLOORS:
    _source18=data[_floor18]['train_rp'].copy()
    _source18['ID_RP']=_source18.ID_RP.astype(str).str.strip()
    _a18=_source18[AP_COLS].apply(pd.to_numeric,errors='coerce').replace([-100,100],np.nan)
    _source18[AP_COLS]=_a18.mask((_a18<-100)|(_a18>0))
    _floor_dir18=EXT18_DIR/_floor18;_floor_dir18.mkdir(exist_ok=True)
    for _seed18 in EXT18_CONFIG['seeds']:
        print(f'18B {_floor18}, seed {_seed18}: training observed-only CGAN',flush=True)
        _split18=ext18_split(_source18,_seed18,EXT18_CONFIG)
        EXT18_SPLITS[(_floor18,_seed18)]=_split18
        _obs18=_split18['OBSERVED'];_val18=_split18['VALIDATION'];_hidden18=_split18['HIDDEN']
        _seed_dir18=_floor_dir18/str(_seed18);_seed_dir18.mkdir(exist_ok=True)
        _audit18=pd.concat([v[['ID_RP','X','Y','Coordinate_Group']].assign(Role=role) for role,v in _split18.items()],ignore_index=True)
        _audit18.to_csv(_seed_dir18/'split_manifest.csv',index=False)
        _g18,_norm18,_history18,_training18=ext18_train(_obs18,_val18,AP_COLS,_seed18,EXT18_CONFIG)
        _history18.to_csv(_seed_dir18/'training_history.csv',index=False)
        _ext18_training_rows.append(dict(Floor=_floor18,Seed=_seed18,**_training18))
        torch.save({'generator_state_dict':_g18.state_dict(),'normalization':_norm18,
                    'config':EXT18_CONFIG,'seed':_seed18,'floor':_floor18},_seed_dir18/'observed_only_cgan.pt')
        _sc18,_med18=ext18_fit_scaler(_obs18,AP_COLS)
        # k được chọn chỉ trên OBSERVED -> VALIDATION, trước khi mở TEST/HIDDEN RSSI.
        _k_candidates18=[]
        for _k18 in EXT18_CONFIG['k_candidates']:
            _vp18,_,_=wknn(_obs18,_val18,_sc18,_med18,_k18)
            _vm18=ext18_metrics(np.linalg.norm(_vp18-_val18[['X','Y']].to_numpy(float),axis=1))
            _k_candidates18.append(dict(Floor=_floor18,Seed=_seed18,k=_k18,**_vm18))
        _kt18=pd.DataFrame(_k_candidates18);_eligible18=_kt18[_kt18.N_Valid>0]
        if _eligible18.empty:raise RuntimeError('Không chọn được k trên VALIDATION.')
        _selected_k18=int(_eligible18.sort_values(['Mean_Error','Coverage','k'],ascending=[True,False,True]).iloc[0].k)
        _kt18['Selected_k']=_selected_k18;_ext18_k_rows.append(_kt18)
        _targets18=_hidden18[['ID_RP','X','Y']].copy().reset_index(drop=True)
        _generated18,_cgan_ref18=ext18_generate(_g18,_norm18,_targets18,AP_COLS,_seed18,EXT18_CONFIG)
        _knn18=ext18_interpolate(_obs18,_targets18,AP_COLS,k=7)
        _knn18=_knn18[_knn18[AP_COLS].notna().sum(1)>=EXT18_CONFIG['min_observed_ap']].copy()
        _refs18={
            'BEFORE_HIDE_MEASURED':pd.concat([_obs18,_hidden18],ignore_index=True),
            'AFTER_HIDE_ORIGINAL':_obs18.copy(),
            'AFTER_FILL_CGAN':pd.concat([_obs18,_cgan_ref18],ignore_index=True),
            'AFTER_FILL_KNN_K7':pd.concat([_obs18,_knn18],ignore_index=True)}
        EXT18_REFERENCES[(_floor18,_seed18)]=_refs18
        # Chỉ bây giờ mở phép đo TEST độc lập tại các RP ẩn. Không dùng RSSI TRAIN ẩn làm query.
        _te18=data[_floor18]['test_queries'].copy()
        _te18['ID_RP']=_te18.ID_RP.astype(str).str.strip()
        _te18=_te18[_te18.ID_RP.isin(set(_hidden18.ID_RP))].reset_index(drop=True)
        if _te18.empty:raise ValueError('Không có TEST độc lập tại HIDDEN RP.')
        _v18=_te18[AP_COLS].apply(pd.to_numeric,errors='coerce').replace([-100,100],np.nan)
        _te18[AP_COLS]=_v18.mask((_v18<-100)|(_v18>0))
        assert not set(_te18.ID_RP)&set(_obs18.ID_RP)
        assert not set(_te18.ID_RP)&set(_val18.ID_RP)
        _query_hash18=hashlib.sha256(pd.util.hash_pandas_object(_te18[['ID_RP','X','Y',*AP_COLS]],index=True).to_numpy().tobytes()).hexdigest()
        _ext18_audit_rows.append(dict(Floor=_floor18,Seed=_seed18,Observed_RP=len(_obs18),Validation_RP=len(_val18),Hidden_RP=len(_hidden18),
            Hidden_TEST_RP=_te18.ID_RP.nunique(),Hidden_TEST_Queries=len(_te18),
            Train_Val_Group_Overlap=len(set(_obs18.Coordinate_Group)&set(_val18.Coordinate_Group)),
            Train_Hidden_Group_Overlap=len(set(_obs18.Coordinate_Group)&set(_hidden18.Coordinate_Group)),
            Val_Hidden_Group_Overlap=len(set(_val18.Coordinate_Group)&set(_hidden18.Coordinate_Group)),
            CGAN_Recovered_RP=len(_cgan_ref18),KNN_Recovered_RP=len(_knn18),Query_SHA256=_query_hash18,
            Scaler_Fit='OBSERVED_ONLY',Checkpoint_Selection='VALIDATION_ONLY',k_Selection='OBSERVED_TO_VALIDATION',
            Hidden_RSSI_In_Training=False,Test_RSSI_In_Training=False))
        _local_preds18={}
        for _method18,_reference18 in _refs18.items():
            # Không reference nào có VALIDATION, kể cả baseline trước ẩn.
            assert not set(_reference18.ID_RP)&set(_val18.ID_RP)
            _p18,_fb18,_used18=wknn(_reference18,_te18,_sc18,_med18,_selected_k18)
            _pred18=_te18[['ID_RP','X','Y']].copy()
            _pred18['Query_ID']=np.arange(len(_pred18));_pred18['Floor']=_floor18;_pred18['Seed']=_seed18
            _pred18['Method']=_method18;_pred18['Pred_X']=_p18[:,0];_pred18['Pred_Y']=_p18[:,1]
            _pred18['Error']=np.linalg.norm(_p18-_te18[['X','Y']].to_numpy(float),axis=1)
            _pred18['Fallback']=_fb18;_pred18['Used_Reference_Count']=_used18
            _local_preds18[_method18]=_pred18;_ext18_predictions.append(_pred18)
            _ext18_result_rows.append(dict(Floor=_floor18,Seed=_seed18,Method=_method18,
                Reference_Rows=len(_reference18),Reference_Coordinates=len(_reference18[['X','Y']].drop_duplicates()),
                Selected_k=_selected_k18,Fallback_Count=int(_fb18.sum()),Query_SHA256=_query_hash18,
                **ext18_metrics(_pred18.Error)))
        for _comparison18,_base_name18,_candidate_name18 in [
            ('After_hide_vs_before','BEFORE_HIDE_MEASURED','AFTER_HIDE_ORIGINAL'),
            ('CGAN_vs_after_hide','AFTER_HIDE_ORIGINAL','AFTER_FILL_CGAN'),
            ('KNN_vs_after_hide','AFTER_HIDE_ORIGINAL','AFTER_FILL_KNN_K7'),
            ('CGAN_vs_KNN','AFTER_FILL_KNN_K7','AFTER_FILL_CGAN')]:
            _ext18_pair_rows.append(dict(Floor=_floor18,Seed=_seed18,Comparison=_comparison18,
                Baseline=_base_name18,Candidate=_candidate_name18,
                **ext18_paired(_local_preds18[_base_name18],_local_preds18[_candidate_name18],_seed18,EXT18_CONFIG['bootstrap_iterations'])))
        # Đối chiếu RSSI HIDDEN chỉ sau khi mọi cấu hình/checkpoint/k đã cố định.
        _truth18=_hidden18.set_index('ID_RP')[AP_COLS]
        _std18=_generated18.groupby('ID_RP')[AP_COLS].std(ddof=0)
        for _name18,_syn18 in [('CGAN',_cgan_ref18),('KNN_K7',_knn18)]:
            _synthetic18=_syn18.set_index('ID_RP')[AP_COLS].reindex(_truth18.index)
            for _ap18 in AP_COLS:
                _ok18=_truth18[_ap18].notna()&_synthetic18[_ap18].notna()
                _diff18=_synthetic18.loc[_ok18,_ap18]-_truth18.loc[_ok18,_ap18]
                _ext18_rssi_rows.append(dict(Floor=_floor18,Seed=_seed18,Method=_name18,AP=_ap18,
                    Hidden_RP=len(_truth18),Comparable_RP=int(_ok18.sum()),
                    RSSI_MAE_dBm=_diff18.abs().mean(),RSSI_RMSE_dBm=np.sqrt((_diff18**2).mean()),RSSI_Bias_dBm=_diff18.mean(),
                    Mean_Conditional_Generated_SD_dBm=_std18[_ap18].mean() if _name18=='CGAN' else np.nan))
        _generated18.to_csv(_seed_dir18/'all_cgan_hidden_candidates.csv',index=False)
        _cgan_ref18.to_csv(_seed_dir18/'cgan_hidden_reference.csv',index=False)
        _knn18.to_csv(_seed_dir18/'knn_hidden_reference.csv',index=False)
        print(f'  done: observed={len(_obs18)}, validation={len(_val18)}, hidden={len(_hidden18)}, queries={len(_te18)}',flush=True)
EXT18_RESULTS=pd.DataFrame(_ext18_result_rows)
EXT18_PREDICTIONS=pd.concat(_ext18_predictions,ignore_index=True)
EXT18_AUDIT=pd.DataFrame(_ext18_audit_rows)
EXT18_RSSI=pd.DataFrame(_ext18_rssi_rows)
EXT18_PAIRED=pd.DataFrame(_ext18_pair_rows)
EXT18_TRAINING=pd.DataFrame(_ext18_training_rows)
EXT18_SUMMARY=EXT18_RESULTS.groupby(['Floor','Method'],as_index=False).agg(
    Mean_Error_Mean=('Mean_Error','mean'),Mean_Error_SD=('Mean_Error','std'),
    RMSE_Mean=('RMSE','mean'),P90_Mean=('P90','mean'),Coverage_Mean=('Coverage','mean'),
    Query_Count_Min=('N_Query','min'),Seeds=('Seed','nunique'))
for _name18,_df18 in {'results':EXT18_RESULTS,'predictions':EXT18_PREDICTIONS,'split_audit':EXT18_AUDIT,
    'rssi_hidden':EXT18_RSSI,'paired_comparisons':EXT18_PAIRED,'training_summary':EXT18_TRAINING,
    'mean_std_summary':EXT18_SUMMARY,'validation_k':pd.concat(_ext18_k_rows)}.items():
    _df18.to_csv(EXT18_DIR/f'{_name18}.csv',index=False)
assert len(EXT18_RESULTS)==len(FLOORS)*len(EXT18_CONFIG['seeds'])*4
assert EXT18_RESULTS.groupby(['Floor','Seed']).Query_SHA256.nunique().eq(1).all()
assert EXT18_AUDIT[['Train_Val_Group_Overlap','Train_Hidden_Group_Overlap','Val_Hidden_Group_Overlap']].eq(0).all().all()
print('18B hoàn tất. BEFORE_HIDE_MEASURED là mốc tham chiếu có RSSI thật tại HIDDEN, không phải phương pháp dự đoán công bằng.')
display(EXT18_SUMMARY)


## 18C. Giữ kết quả cũ, phân tích CGAN và kết luận có giới hạn

Xuất bảng theo tầng/seed, biểu đồ bốn mốc, CDF, CI, Excel, CSV, checkpoint và báo cáo Markdown/ZIP trong thư mục 18 riêng. Bảng cũ còn trong RAM được sao chép bổ sung, không chỉnh sửa bảng gốc.

Phân tích đối chiếu RSSI sinh với RSSI thật HIDDEN, tỷ lệ phục hồi RP, diversity và lịch sử loss. Báo cáo phân biệt **quan sát từ số liệu** với **giả thuyết nguyên nhân** (ít vị trí train, fingerprint trung bình, reconstruction ưu tiên mean, lệch mask/RSSI, loss khác mục tiêu định vị). Chưa thực hiện ablation thì không gọi các giả thuyết là nguyên nhân đã chứng minh.

Ưu tiên giao thức chia tập và đánh giá RP ẩn, chưa triển khai mọi hướng cải tiến. Kết luận không mặc định CGAN tốt hơn; nêu kết quả không rõ hoặc kém hơn. Đây là mô phỏng bỏ khảo sát trên dataset đã đo, chưa kiểm chứng ngoài dataset, vùng/hành lang độc lập, độ chính xác vật lý đầy đủ hoặc chi phí thực địa.


In [ ]:
# 18C — BẢNG / BIỂU ĐỒ / PHÂN TÍCH NGUYÊN NHÂN VÀ KẾT LUẬN TRUNG THỰC
# Tất cả file nằm riêng trong 18_HIDE_AND_RECOVER, không xóa hay ghi đè kết quả cũ.
_ext18_stage_order=['BEFORE_HIDE_MEASURED','AFTER_HIDE_ORIGINAL','AFTER_FILL_CGAN','AFTER_FILL_KNN_K7']
_ext18_stage_labels=['Trước ẩn\n(đo thật)','Sau ẩn\nORIGINAL','Sinh bù\nCGAN','Sinh bù\nKNN K7']
_ext18_colors=['#64748b','#f59e0b','#8b5cf6','#0ea5e9']
_fig18,_axes18=plt.subplots(1,len(FLOORS),figsize=(6*len(FLOORS),5),squeeze=False)
for _ax18,_fl18 in zip(_axes18.ravel(),FLOORS):
    _s18=EXT18_SUMMARY[EXT18_SUMMARY.Floor==_fl18].set_index('Method').reindex(_ext18_stage_order)
    _ax18.bar(np.arange(4),_s18.Mean_Error_Mean,yerr=_s18.Mean_Error_SD.fillna(0),capsize=4,color=_ext18_colors)
    _ax18.set_xticks(np.arange(4),_ext18_stage_labels,rotation=20)
    _ax18.set(title=f'{_fl18}: sai số trên TEST tại RP ẩn',ylabel='Đơn vị tọa độ nguồn')
    _ax18.grid(axis='y',alpha=.2)
_fig18.suptitle('Trung bình ± SD qua các seed; cùng query trong từng seed')
_fig18.tight_layout();_fig18.savefig(EXT18_DIR/'18_before_hide_after_fill.png',dpi=180);plt.show();plt.close(_fig18)
_fig18,_axes18=plt.subplots(1,len(FLOORS),figsize=(6*len(FLOORS),4),squeeze=False)
for _ax18,_fl18 in zip(_axes18.ravel(),FLOORS):
    for _method18,_color18 in zip(['CGAN_vs_after_hide','KNN_vs_after_hide'],['#8b5cf6','#0ea5e9']):
        _p18=EXT18_PAIRED[(EXT18_PAIRED.Floor==_fl18)&(EXT18_PAIRED.Comparison==_method18)]
        _x18=np.arange(len(_p18))+(0 if _method18=='CGAN_vs_after_hide' else .15)
        _ax18.scatter(_x18,_p18.Paired_Improvement,label=_method18,color=_color18)
        _ax18.vlines(_x18,_p18.CI95_Low,_p18.CI95_High,color=_color18,alpha=.7)
    _ax18.axhline(0,color='black',linewidth=.8)
    _ax18.set_xticks(np.arange(len(EXT18_CONFIG['seeds'])),EXT18_CONFIG['seeds'],rotation=30)
    _ax18.set(title=_fl18,ylabel='Giảm sai số so với sau ẩn (dương = tốt hơn)')
    _ax18.legend(fontsize=7)
_fig18.tight_layout();_fig18.savefig(EXT18_DIR/'18_paired_improvement_ci.png',dpi=180);plt.show();plt.close(_fig18)
_fig18,_axes18=plt.subplots(1,len(FLOORS),figsize=(6*len(FLOORS),4),squeeze=False)
for _ax18,_fl18 in zip(_axes18.ravel(),FLOORS):
    # Một seed đã định trước để tránh gộp các query lặp giữa các seed như mẫu độc lập.
    _first18=EXT18_CONFIG['seeds'][0]
    _p18=EXT18_PREDICTIONS[(EXT18_PREDICTIONS.Floor==_fl18)&(EXT18_PREDICTIONS.Seed==_first18)]
    for _method18,_label18,_color18 in zip(_ext18_stage_order,_ext18_stage_labels,_ext18_colors):
        _e18=np.sort(_p18.loc[_p18.Method==_method18,'Error'].dropna())
        if len(_e18):_ax18.plot(_e18,np.arange(1,len(_e18)+1)/len(_e18),label=_label18.replace('\n',' '),color=_color18)
    _ax18.set(title=f'{_fl18}: seed {_first18}',xlabel='Sai số (đơn vị nguồn)',ylabel='CDF query hợp lệ')
    _ax18.legend(fontsize=7);_ax18.grid(alpha=.2)
_fig18.tight_layout();_fig18.savefig(EXT18_DIR/'18_error_cdf_first_seed.png',dpi=180);plt.show();plt.close(_fig18)
# Chẩn đoán theo seed: fidelity tại điểm ẩn, số điểm được sinh bù, diversity, khoảng cách không gian.
_ext18_diagnostic=[]
for (_fl18,_se18),_split18 in EXT18_SPLITS.items():
    _rs18=EXT18_RSSI[(EXT18_RSSI.Floor==_fl18)&(EXT18_RSSI.Seed==_se18)]
    _au18=EXT18_AUDIT[(EXT18_AUDIT.Floor==_fl18)&(EXT18_AUDIT.Seed==_se18)].iloc[0]
    _spatial18=cKDTree(_split18['OBSERVED'][['X','Y']]).query(_split18['HIDDEN'][['X','Y']])[0]
    _cg18=_rs18[_rs18.Method=='CGAN'];_kn18=_rs18[_rs18.Method=='KNN_K7']
    _ext18_diagnostic.append(dict(Floor=_fl18,Seed=_se18,Observed_RP=len(_split18['OBSERVED']),
        Hidden_RP=len(_split18['HIDDEN']),CGAN_Recovered_RP=_au18.CGAN_Recovered_RP,
        CGAN_Recovery_Coverage=_au18.CGAN_Recovered_RP/len(_split18['HIDDEN']),
        Mean_Nearest_Observed_Distance=_spatial18.mean(),
        CGAN_Macro_AP_MAE_dBm=_cg18.RSSI_MAE_dBm.mean(),KNN_Macro_AP_MAE_dBm=_kn18.RSSI_MAE_dBm.mean(),
        CGAN_Mean_Conditional_SD_dBm=_cg18.Mean_Conditional_Generated_SD_dBm.mean(),
        CGAN_Mean_Comparable_RP=_cg18.Comparable_RP.mean(),KNN_Mean_Comparable_RP=_kn18.Comparable_RP.mean()))
EXT18_DIAGNOSTICS=pd.DataFrame(_ext18_diagnostic)
EXT18_DIAGNOSTICS.to_csv(EXT18_DIR/'diagnostics.csv',index=False)
with pd.ExcelWriter(EXT18_DIR/'18_DANH_GIA_TRUOC_AN_SAU_SINH.xlsx') as _writer18:
    for _name18,_frame18 in {'Mean_SD':EXT18_SUMMARY,'Per_Seed':EXT18_RESULTS,'Split_Audit':EXT18_AUDIT,
        'Paired_CI':EXT18_PAIRED,'RSSI_Hidden_AP':EXT18_RSSI,'Diagnostics':EXT18_DIAGNOSTICS,
        'Training':EXT18_TRAINING}.items():_frame18.to_excel(_writer18,sheet_name=_name18,index=False)
# Lưu bản sao bảng cũ nếu có trong RAM, không sửa bảng/file gốc.
_ext18_legacy_names=['results_16e_df','summary_16e_df','EXT17_EVALUATION','EXT17_RSSI_EVALUATION']
_ext18_preserved=[]
for _name18 in _ext18_legacy_names:
    _obj18=globals().get(_name18)
    if isinstance(_obj18,pd.DataFrame):
        _ld18=EXT18_DIR/'legacy_table_copies';_ld18.mkdir(exist_ok=True)
        _obj18.copy(deep=True).to_csv(_ld18/f'{_name18}.csv',index=False);_ext18_preserved.append(_name18)
_ext18_lines=['# THỰC NGHIỆM ẨN RP VÀ SINH BÙ','',
 '## Giao thức',
 '- Tách cụm tọa độ thành OBSERVED, VALIDATION, HIDDEN trước huấn luyện. RSSI HIDDEN không dùng train, scaler, chọn checkpoint hoặc k.',
 '- Tọa độ HIDDEN được xem là vị trí mục tiêu đã biết từ kế hoạch khảo sát; không phải thông tin RSSI.',
 '- TEST là các phép đo độc lập trong workbook TEST tại HIDDEN RP. Trong từng floor/seed, cả bốn mốc dùng cùng query, scaler OBSERVED và k đã chọn trên VALIDATION.',
 '- BEFORE_HIDE_MEASURED có fingerprint đo thật tại HIDDEN, là mốc ưu thế thông tin để so sánh mức suy giảm, không phải phương pháp tổng quát hóa công bằng.',
 '- Các kết quả ORIGINAL/CGAN/KNN cũ vẫn được giữ; phần 18 lưu trong thư mục riêng. Bảng cũ có trong RAM được sao chép bổ sung: '+', '.join(_ext18_preserved),
 '- Mean ± SD qua seed chỉ là thống kê mô tả; không xem các seed với query/RP chồng lặp là mẫu độc lập. CI theo cụm RP được báo cáo riêng từng seed.','']
for _fl18 in FLOORS:
    _s18=EXT18_SUMMARY[EXT18_SUMMARY.Floor==_fl18]
    _ext18_lines += [f'## {_fl18}','| Mốc | Mean error ± SD | RMSE | P90 | Coverage |','|---|---:|---:|---:|---:|']
    for _,_row18 in _s18.iterrows():
        _ext18_lines.append(f'| {_row18.Method} | {_row18.Mean_Error_Mean:.4f} ± {_row18.Mean_Error_SD:.4f} | {_row18.RMSE_Mean:.4f} | {_row18.P90_Mean:.4f} | {_row18.Coverage_Mean:.2%} |')
    _pa18=EXT18_PAIRED[(EXT18_PAIRED.Floor==_fl18)&(EXT18_PAIRED.Comparison=='CGAN_vs_after_hide')]
    _positive18=int((_pa18.CI95_Low>0).sum());_negative18=int((_pa18.CI95_High<0).sum())
    _uncertain18=len(_pa18)-_positive18-_negative18
    _ext18_lines += ['',f'CGAN so với sau ẩn: {_positive18} seed có CI95% hoàn toàn dương, {_negative18} seed hoàn toàn âm, {_uncertain18} seed chưa rõ/không đủ CI. Các CI là thăm dò, chưa hiệu chỉnh nhiều so sánh.',
        'Không suy ra kết luận chung CGAN tốt hơn chỉ từ một seed hoặc một tầng.','']
_ext18_lines += ['## Phân tích vì sao CGAN có thể chưa cải thiện',
 '**Bằng chứng cần đọc từ kết quả:**',
 '- diagnostics.csv: MAE RSSI theo AP của CGAN/KNN, số RP so sánh được, tỷ lệ RP sinh bù đạt QC, độ đa dạng khi thay noise và khoảng cách từ RP ẩn đến RP quan sát.',
 '- training_history.csv: loss train/validation, checkpoint và số epoch. Loss GAN thấp không tự chứng minh sai số định vị thấp.',
 '- paired_comparisons.csv và coverage: kết luận sai số chỉ trên query ghép cặp hợp lệ, luôn công bố số thất bại và số RP thực tế.',
 '**Giả thuyết giải thích, chưa được xác minh nhân quả bằng ablation:**',
 '- CGAN học từ các fingerprint trung bình theo RP, số vị trí huấn luyện hạn chế; chưa trực tiếp học đầy đủ biến thiên của các phép đo lặp.',
 '- Thành phần reconstruction có thể ưu tiên giá trị trung bình và làm giảm ảnh hưởng noise. Độ đa dạng thấp là dấu hiệu cần kiểm tra, không tự chứng minh mode collapse.',
 '- Sai lệch RSSI/mask tại điểm ẩn có thể làm WKNN chọn sai láng giềng dù dữ liệu sinh nằm trong miền RSSI hợp lệ.',
 '- KNN nội suy khai thác trực tiếp tính cục bộ của fingerprint; mô hình sinh phức tạp không mặc định tốt hơn khi dữ liệu vị trí ít.',
 '- Mục tiêu loss tái tạo/đối kháng không trùng trực tiếp với mục tiêu sai số định vị. Chưa kết luận tăng epoch hoặc đổi kiến trúc sẽ khắc phục.',
 'Các giả thuyết chỉ được củng cố khi đối chiếu số liệu chẩn đoán; không viết thành nguyên nhân đã được chứng minh.','',
 '## Hạn chế và kết luận có thể dùng trong luận văn',
 'Nghiên cứu bổ sung giao thức ẩn vị trí và sinh bù với dữ liệu kiểm thử không dùng để chọn mô hình. Các kết quả hiện có được giữ lại; hiệu quả CGAN được đánh giá theo tầng, seed, sai số và coverage. Chỉ kết luận cải thiện ở điều kiện có bằng chứng tương ứng; nếu CI chưa rõ hoặc sai số tăng, cần nêu trực tiếp và không khẳng định CGAN ưu việt.',
 '- Đây là mô phỏng bỏ khảo sát bằng cách che dữ liệu tại RP đã đo; chưa chứng minh hiệu quả tại các vị trí ngoài dataset.',
 '- Cụm tọa độ tránh rò rỉ cùng vị trí, nhưng random spatial holdout chưa thay thế đánh giá vùng/hành lang độc lập hoặc môi trường mới.',
 '- VALIDATION dùng chọn checkpoint/k, HIDDEN mới là tập đối chiếu cuối. Không điều chỉnh cấu hình theo kết quả HIDDEN sau khi xem.',
 '- Mốc trước ẩn dùng thêm RSSI thật; độ chênh với sau ẩn không phải bằng chứng CGAN thất bại khi không phục hồi được toàn bộ thông tin đo thật.',
 '- Chưa có floorplan/vị trí AP để xác thực đầy đủ các ràng buộc truyền sóng; chưa định lượng chi phí/giờ khảo sát thực địa.',
 '- Huấn luyện phần 18 là CGAN riêng cho split mới, giữ ý tưởng mask-aware và LayerNorm; không dùng lại checkpoint cũ đã có thể thấy RP ẩn. Kết quả 18 không so sánh trực tiếp như cùng một giao thức với kết quả cũ.',
 '- Chưa triển khai mọi hướng cải tiến, ưu tiên sửa chia tập và đánh giá đúng vị trí bị ẩn theo yêu cầu.']
(EXT18_DIR/'18_PHAN_TICH_VA_KET_LUAN.md').write_text('\n'.join(_ext18_lines),encoding='utf-8')
_ext18_zip=shutil.make_archive(str(Path(OUT).parent/'KLTN_18_TRUOC_AN_SAU_SINH'),'zip',EXT18_DIR)
print('Kết quả riêng:',EXT18_DIR)
print('ZIP:',_ext18_zip)
display(EXT18_SUMMARY);display(EXT18_DIAGNOSTICS)
try:
    from google.colab import files as _files18
except ImportError:
    pass
else:_files18.download(_ext18_zip)
